# Daily Coding Problem


Daily Coding Problem questions with notes and Python solutions, one section per problem.

The **Progress** summary is the last cell: run it once per session and it then refreshes itself whenever you run a cell (it reads the saved notebook). When a problem has more than one solution, caption each one: `#### Solution 1: <title>`, `#### Solution 2: <title>`.

### Problem #1 <a id="p1"></a>

**Easy** · Google · 2019-12-18

#### Problem

Given a list of numbers and a number k, return whether any two numbers from the list add up to k.

For example, given `[10, 15, 3, 7]` and k of 17, return true since 10 + 7 is 17.

Bonus: Can you do this in one pass?

#### Solution 1: Hash set, one pass

One pass with a set of the values seen so far. For each `x`, if `k - x` is already in the set, some earlier number pairs with `x`; otherwise add `x` and move on. Checking before adding means a number never pairs with itself, while equal values at different positions (`[5, 5]`, `k = 10`) still count.

*O(N) time, O(N) space.*

In [1]:
def has_pair_with_sum(nums, k):
    """Return True if two numbers at different positions in nums add up to k."""
    seen = set()
    for x in nums:
        if k - x in seen:
            return True
        seen.add(x)
    return False


assert has_pair_with_sum([10, 15, 3, 7], 17)      # example: 10 + 7
assert not has_pair_with_sum([10, 15, 3, 7], 50)
assert not has_pair_with_sum([], 5)
assert not has_pair_with_sum([5], 10)              # a number can't pair with itself
assert has_pair_with_sum([5, 5], 10)               # equal values at different positions
assert has_pair_with_sum([-2, 4, 1], 2)            # negatives
print("all tests passed")

all tests passed


#### Solution 2: Sort, then two pointers

Sort a copy and put one pointer at each end. If the pair sums to less than `k`, only moving the left pointer right can raise the sum; if it is more, only moving the right pointer left can lower it. Each step drops a value that cannot be part of any pair, so the pointers meet within N steps.

*O(N log N) time, O(N) space for the sorted copy* (O(1) extra if you sort the input in place and don't mind changing it).

In [ ]:
def has_pair_with_sum_sorted(nums, k):
    """Return True if two numbers at different positions in nums add up to k."""
    nums = sorted(nums)
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        total = nums[lo] + nums[hi]
        if total == k:
            return True
        if total < k:
            lo += 1     # need a bigger sum
        else:
            hi -= 1     # need a smaller sum
    return False


assert has_pair_with_sum_sorted([10, 15, 3, 7], 17)      # example: 10 + 7
assert not has_pair_with_sum_sorted([10, 15, 3, 7], 50)
assert not has_pair_with_sum_sorted([], 5)
assert not has_pair_with_sum_sorted([5], 10)              # a number can't pair with itself
assert has_pair_with_sum_sorted([5, 5], 10)               # equal values at different positions
assert has_pair_with_sum_sorted([-2, 4, 1], 2)            # negatives
print("all tests passed")

all tests passed


### Problem #2 <a id="p2"></a>

**Hard** · Uber · 2019-12-19

#### Problem

Given an array of integers, return a new array such that each element at index i of the new array is the product of all the numbers in the original array except the one at i.

For example, if our input was `[1, 2, 3, 4, 5]`, the expected output would be `[120, 60, 40, 30, 24]`. If our input was `[3, 2, 1]`, the expected output would be `[2, 3, 6]`.

Follow-up: what if you can't use division?

#### Solution 1: Divide the total product

Multiply everything once; the answer at `i` is then `total / nums[i]`. Zeros break the division, so they get their own cases: with two or more zeros every product includes one and all answers are 0; with exactly one, only the zero's own position gets the product of the other numbers. Integer division is exact because `total` is a multiple of every nonzero element.

*O(N) time, O(1) extra space.* It uses division, so it doesn't answer the follow-up.

In [ ]:
from math import prod


def products_except_self_div(nums):
    """Product of all other elements at each index, using division."""
    zeros = nums.count(0)
    if zeros > 1:
        return [0] * len(nums)            # every product includes a zero
    total = prod(x for x in nums if x != 0)
    if zeros == 1:
        return [total if x == 0 else 0 for x in nums]
    return [total // x for x in nums]     # exact: total is a multiple of x


f = products_except_self_div
assert f([1, 2, 3, 4, 5]) == [120, 60, 40, 30, 24]   # examples
assert f([3, 2, 1]) == [2, 3, 6]
assert f([1, 0, 3]) == [0, 3, 0]                     # one zero
assert f([0, 4, 0]) == [0, 0, 0]                     # two zeros
assert f([-1, 2, -3]) == [-6, 3, -2]                 # negatives
assert f([5]) == [1]                                 # empty product is 1
assert f([]) == []
print("all tests passed")

all tests passed


#### Solution 2: Prefix and suffix products

The answer at `i` is (product of everything left of `i`) × (product of everything right of `i`). One pass builds `left[i] = left[i - 1] * nums[i - 1]`, a second builds `right[i] = right[i + 1] * nums[i + 1]`, and the answers are their pairwise products. No division, so zeros need no special case.

*O(N) time, O(N) extra space* for the two helper arrays.

In [ ]:
def products_except_self_prefix(nums):
    """Product of all other elements at each index, without division."""
    n = len(nums)
    left = [1] * n     # left[i]  = product of nums[:i]
    right = [1] * n    # right[i] = product of nums[i + 1:]
    for i in range(1, n):
        left[i] = left[i - 1] * nums[i - 1]
    for i in range(n - 2, -1, -1):
        right[i] = right[i + 1] * nums[i + 1]
    return [l * r for l, r in zip(left, right)]


f = products_except_self_prefix
assert f([1, 2, 3, 4, 5]) == [120, 60, 40, 30, 24]   # examples
assert f([3, 2, 1]) == [2, 3, 6]
assert f([1, 0, 3]) == [0, 3, 0]                     # one zero
assert f([0, 4, 0]) == [0, 0, 0]                     # two zeros
assert f([-1, 2, -3]) == [-6, 3, -2]                 # negatives
assert f([5]) == [1]                                 # empty product is 1
assert f([]) == []
print("all tests passed")

all tests passed


#### Solution 3: Prefix and suffix products in constant space

Solution 2 without the helper arrays: the first pass writes each left product straight into the result, and the second pass walks right to left, multiplying in a running right product.

*O(N) time, O(1) extra space* besides the output. This is the usual answer to the follow-up.

In [ ]:
def products_except_self(nums):
    """Product of all other elements at each index, without division, O(1) extra space."""
    result = [1] * len(nums)
    left = 1                                # product of everything before i
    for i, x in enumerate(nums):
        result[i] = left
        left *= x
    right = 1                               # product of everything after i
    for i in range(len(nums) - 1, -1, -1):
        result[i] *= right
        right *= nums[i]
    return result


f = products_except_self
assert f([1, 2, 3, 4, 5]) == [120, 60, 40, 30, 24]   # examples
assert f([3, 2, 1]) == [2, 3, 6]
assert f([1, 0, 3]) == [0, 3, 0]                     # one zero
assert f([0, 4, 0]) == [0, 0, 0]                     # two zeros
assert f([-1, 2, -3]) == [-6, 3, -2]                 # negatives
assert f([5]) == [1]                                 # empty product is 1
assert f([]) == []
print("all tests passed")

all tests passed


### Problem #3 <a id="p3"></a>

**Medium** · Google · 2019-12-20

#### Problem

Given the root to a binary tree, implement `serialize(root)`, which serializes the tree into a string, and deserialize(s), which deserializes the string back into the tree.

For example, given the following Node class

```
class Node:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
```

The following test should pass:

```
node = Node('root', Node('left', Node('left.left')), Node('right'))
assert deserialize(serialize(node)).left.left.val == 'left.left'
```

### Problem #4 <a id="p4"></a>

**Hard** · Stripe · 2019-12-21

#### Problem

Given an array of integers, find the first missing positive integer in linear time and constant space. In other words, find the lowest positive integer that does not exist in the array. The array can contain duplicates and negative numbers as well.

For example, the input `[3, 4, -1, 1]` should give 2. The input `[1, 2, 0]` should give 3.

You can modify the input array in-place.

### Problem #5 <a id="p5"></a>

**Medium** · Jane Street · 2019-12-22

#### Problem

`cons(a, b)` constructs a pair, and `car(pair)` and `cdr(pair)` returns the first and last element of that pair. For example, `car(cons(3, 4))` returns 3, and `cdr(cons(3, 4))` returns 4.

Given this implementation of cons:

```
def cons(a, b):
    def pair(f):
        return f(a, b)
    return pair
```

Implement car and cdr.

### Problem #6 <a id="p6"></a>

**Hard** · Google · 2019-12-23

#### Problem

An XOR linked list is a more memory efficient doubly linked list. Instead of each node holding next and prev fields, it holds a field named both, which is an XOR of the next node and the previous node. Implement an XOR linked list; it has an `add(element)` which adds the element to the end, and a `get(index)` which returns the node at index.

If using a language that has no pointers (such as Python), you can assume you have access to `get_pointer` and `dereference_pointer` functions that converts between nodes and memory addresses.

### Problem #7 <a id="p7"></a>

**Medium** · Facebook · 2019-12-24

#### Problem

Given the mapping a = 1, b = 2, ... z = 26, and an encoded message, count the number of ways it can be decoded.

For example, the message `'111'` would give 3, since it could be decoded as `'aaa'`, `'ka'`, and `'ak'`.

You can assume that the messages are decodable. For example, `'001'` is not allowed.

### Problem #8 <a id="p8"></a>

**Easy** · Google · 2019-12-25

#### Problem

A unival tree (which stands for `"universal value"`) is a tree where all nodes under it have the same value.

Given the root to a binary tree, count the number of unival subtrees.

For example, the following tree has 5 unival subtrees:

```
  0
 / \
1   0
   / \
  1   0
 / \
1   1
```

### Problem #9 <a id="p9"></a>

**Hard** · Airbnb · 2019-12-26

#### Problem

Given a list of integers, write a function that returns the largest sum of non-adjacent numbers. Numbers can be 0 or negative.

For example, `[2, 4, 6, 2, 5]` should return 13, since we pick 2, 6, and 5. `[5, 1, 1, 5]` should return 10, since we pick 5 and 5.

Follow-up: Can you do this in `O(N)` time and constant space?

### Problem #10 <a id="p10"></a>

**Medium** · Apple · 2019-12-27

#### Problem

Implement a job scheduler which takes in a function f and an integer n, and calls f after n milliseconds.

### Problem #11 <a id="p11"></a>

**Medium** · Twitter · 2019-12-28

#### Problem

Implement an autocomplete system. That is, given a query string s and a set of all possible query strings, return all strings in the set that have s as a prefix.

For example, given the query string de and the set of strings `[dog, deer, deal]`, return `[deer, deal]`.

Hint: Try preprocessing the dictionary into a more efficient data structure to speed up queries.

### Problem #12 <a id="p12"></a>

**Hard** · Amazon · 2019-12-29

#### Problem

There exists a staircase with N steps, and you can climb up either 1 or 2 steps at a time. Given N, write a function that returns the number of unique ways you can climb the staircase. The order of the steps matters.

For example, if N is 4, then there are 5 unique ways:

- 1, 1, 1, 1
- 2, 1, 1
- 1, 2, 1
- 1, 1, 2
- 2, 2

What if, instead of being able to climb 1 or 2 steps at a time, you could climb any number from a set of positive integers X? For example, if X = `{1, 3, 5}`, you could climb 1, 3, or 5 steps at a time.

### Problem #13 <a id="p13"></a>

**Hard** · Amazon · 2019-12-30

#### Problem

Given an integer k and a string s, find the length of the longest substring that contains at most k distinct characters.

For example, given s = `"abcba"` and k = 2, the longest substring with k distinct characters is `"bcb"`.

### Problem #14 <a id="p14"></a>

**Medium** · Google · 2019-12-31

#### Problem

The area of a circle is defined as πr^2. Estimate π to 3 decimal places using a Monte Carlo method.

Hint: The basic equation of a circle is x2 + y2 = r2.

### Problem #15 <a id="p15"></a>

**Medium** · Facebook · 2020-01-01

#### Problem

Given a stream of elements too large to store in memory, pick a random element from the stream with uniform probability.

### Problem #16 <a id="p16"></a>

**Easy** · Twitter · 2020-01-02

#### Problem

You run an e-commerce website and want to record the last N order ids in a log. Implement a data structure to accomplish this, with the following API:

- `record(order_id)`: adds the `order_id` to the log
- `get_last(i)`: gets the ith last element from the log. i is guaranteed to be smaller than or equal to N.

You should be as efficient with time and space as possible.

### Problem #17 <a id="p17"></a>

**Hard** · Google · 2020-01-03

#### Problem

Suppose we represent our file system by a string in the following manner:

The string `"dir\n\tsubdir1\n\tsubdir2\n\t\tfile.ext"` represents:

```
dir
    subdir1
    subdir2
        file.ext
```

The directory dir contains an empty sub-directory subdir1 and a sub-directory subdir2 containing a file file.ext.

The string `"dir\n\tsubdir1\n\t\tfile1.ext\n\t\tsubsubdir1\n\tsubdir2\n\t\tsubsubdir2\n\t\t\tfile2.ext"` represents:

```
dir
    subdir1
        file1.ext
        subsubdir1
    subdir2
        subsubdir2
            file2.ext
```

The directory dir contains two sub-directories subdir1 and subdir2. subdir1 contains a file file1.ext and an empty second-level sub-directory subsubdir1. subdir2 contains a second-level sub-directory subsubdir2 containing a file file2.ext.

We are interested in finding the longest (number of characters) absolute path to a file within our file system. For example, in the second example above, the longest absolute path is `"dir/subdir2/subsubdir2/file2.ext"`, and its length is 32 (not including the double quotes).

Given a string representing the file system in the above format, return the length of the longest absolute path to a file in the abstracted file system. If there is no file in the system, return 0.

Note:

The name of a file contains at least a period and an extension.

The name of a directory or sub-directory will not contain a period.

### Problem #18 <a id="p18"></a>

**Hard** · Google · 2020-01-04

#### Problem

Given an array of integers and a number k, where 1 \<= k \<= length of the array, compute the maximum values of each subarray of length k.

For example, given array = `[10, 5, 2, 7, 8, 7]` and k = 3, we should get: `[10, 7, 8, 8]`, since:

- 10 = `max(10, 5, 2)`
- 7 = `max(5, 2, 7)`
- 8 = `max(2, 7, 8)`
- 8 = `max(7, 8, 7)`

Do this in `O(n)` time and `O(k)` space. You can modify the input array in-place and you do not need to store the results. You can simply print them out as you compute them.

### Problem #19 <a id="p19"></a>

**Medium** · Facebook · 2020-01-05

#### Problem

A builder is looking to build a row of N houses that can be of K different colors. He has a goal of minimizing cost while ensuring that no two neighboring houses are of the same color.

Given an N by K matrix where the nth row and kth column represents the cost to build the nth house with kth color, return the minimum cost which achieves this goal.

### Problem #20 <a id="p20"></a>

**Easy** · Google · 2020-01-06

#### Problem

Given two singly linked lists that intersect at some point, find the intersecting node. The lists are non-cyclical.

For example, given A = 3 -> 7 -> 8 -> 10 and B = 99 -> 1 -> 8 -> 10, return the node with value 8.

In this example, assume nodes with the same value are the exact same node objects.

Do this in `O(M + N)` time (where M and N are the lengths of the lists) and constant space.

### Problem #21 <a id="p21"></a>

**Easy** · Snapchat · 2020-01-07

#### Problem

Given an array of time intervals (start, end) for classroom lectures (possibly overlapping), find the minimum number of rooms required.

For example, given `[(30, 75), (0, 50), (60, 150)]`, you should return 2.

### Problem #22 <a id="p22"></a>

**Medium** · Microsoft · 2020-01-08

#### Problem

Given a dictionary of words and a string made up of those words (no spaces), return the original sentence in a list. If there is more than one possible reconstruction, return any of them. If there is no possible reconstruction, then return null.

For example, given the set of words `'quick'`, `'brown'`, `'the'`, `'fox'`, and the string `"thequickbrownfox"`, you should return `['the', 'quick', 'brown', 'fox']`.

Given the set of words `'bed'`, `'bath'`, `'bedbath'`, `'and'`, `'beyond'`, and the string `"bedbathandbeyond"`, return either `['bed', 'bath', 'and', 'beyond]` or `['bedbath', 'and', 'beyond']`.

### Problem #23 <a id="p23"></a>

**Easy** · Google · 2020-01-09

#### Problem

You are given an M by N matrix consisting of booleans that represents a board. Each True boolean represents a wall. Each False boolean represents a tile you can walk on.

Given this matrix, a start coordinate, and an end coordinate, return the minimum number of steps required to reach the end coordinate from the start. If there is no possible path, then return null. You can move up, left, down, and right. You cannot move through walls. You cannot wrap around the edges of the board.

For example, given the following board:

```
[[f, f, f, f],
[t, t, f, t],
[f, f, f, f],
[f, f, f, f]]
```

and start = (3, 0) (bottom left) and end = (0, 0) (top left), the minimum number of steps required to reach the end is 7, since we would need to go through (1, 2) because there is a wall everywhere else on the second row.

### Problem #24 <a id="p24"></a>

**Medium** · Google · 2020-01-10

#### Problem

Implement locking in a binary tree. A binary tree node can be locked or unlocked only if all of its descendants or ancestors are not locked.

Design a binary tree node class with the following methods:

- `is_locked`, which returns whether the node is locked
- lock, which attempts to lock the node. If it cannot be locked, then it should return false. Otherwise, it should lock it and return true.
- unlock, which unlocks the node. If it cannot be unlocked, then it should return false. Otherwise, it should unlock it and return true.

You may augment the node to add parent pointers or any other property you would like. You may assume the class is used in a single-threaded program, so there is no need for actual locks or mutexes. Each method should run in `O(h)`, where h is the height of the tree.

### Problem #25 <a id="p25"></a>

**Hard** · Facebook · 2020-01-11

#### Problem

Implement regular expression matching with the following special characters:

- . (period) which matches any single character
- \* (asterisk) which matches zero or more of the preceding element

That is, implement a function that takes in a string and a valid regular expression and returns whether or not the string matches the regular expression.

For example, given the regular expression `"ra."` and the string `"ray"`, your function should return true. The same regular expression on the string `"raymond"` should return false.

Given the regular expression `".*at"` and the string `"chat"`, your function should return true. The same regular expression on the string `"chats"` should return false.

### Problem #26 <a id="p26"></a>

**Medium** · Google · 2020-01-12

#### Problem

Given a singly linked list and an integer k, remove the kth last element from the list. k is guaranteed to be smaller than the length of the list.

The list is very long, so making more than one pass is prohibitively expensive.

Do this in constant space and in one pass.

### Problem #27 <a id="p27"></a>

**Easy** · Facebook · 2020-01-13

#### Problem

Given a string of round, curly, and square open and closing brackets, return whether the brackets are balanced (well-formed).

For example, given the string `"([])[]({})"`, you should return true.

Given the string `"([)]"` or `"((()"`, you should return false.

### Problem #28 <a id="p28"></a>

**Medium** · Palantir · 2020-01-14

#### Problem

Write an algorithm to justify text. Given a sequence of words and an integer line length k, return a list of strings which represents each line, fully justified.

More specifically, you should have as many words as possible in each line. There should be at least one space between each word. Pad extra spaces when necessary so that each line has exactly length k. Spaces should be distributed as equally as possible, with the extra spaces, if any, distributed starting from the left.

If you can only fit one word on a line, then you should pad the right-hand side with spaces.

Each word is guaranteed not to be longer than k.

For example, given the list of words `["the", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]` and k = 16, you should return the following:

```
["the  quick brown", # 1 extra space on the left
"fox  jumps  over", # 2 extra spaces distributed evenly
"the   lazy   dog"] # 4 extra spaces distributed evenly
```

### Problem #29 <a id="p29"></a>

**Easy** · Amazon · 2020-01-15

#### Problem

Run-length encoding is a fast and simple method of encoding strings. The basic idea is to represent repeated successive characters as a single count and character. For example, the string `"AAAABBBCCDAA"` would be encoded as `"4A3B2C1D2A"`.

Implement run-length encoding and decoding. You can assume the string to be encoded have no digits and consists solely of alphabetic characters. You can assume the string to be decoded is valid.

### Problem #30 <a id="p30"></a>

**Medium** · Facebook · 2020-01-16

#### Problem

You are given an array of non-negative integers that represents a two-dimensional elevation map where each element is unit-width wall and the integer is the height. Suppose it will rain and all spots between two walls get filled up.

Compute how many units of water remain trapped on the map in `O(N)` time and `O(1)` space.

For example, given the input `[2, 1, 2]`, we can hold 1 unit of water in the middle.

Given the input `[3, 0, 1, 3, 0, 5]`, we can hold 3 units in the first index, 2 in the second, and 3 in the fourth index (we cannot hold 5 since it would run off to the left), so we can trap 8 units of water.

### Problem #31 <a id="p31"></a>

**Easy** · Google · 2020-01-17

#### Problem

The edit distance between two strings refers to the minimum number of character insertions, deletions, and substitutions required to change one string to the other. For example, the edit distance between “kitten” and “sitting” is three: substitute the “k” for “s”, substitute the “e” for “i”, and append a “g”.

Given two strings, compute the edit distance between them.

### Problem #32 <a id="p32"></a>

**Hard** · Jane Street · 2020-01-18

#### Problem

Suppose you are given a table of currency exchange rates, represented as a 2D array. Determine whether there is a possible arbitrage: that is, whether there is some sequence of trades you can make, starting with some amount A of any currency, so that you can end up with some amount greater than A of that currency.

There are no transaction costs and you can trade fractional quantities.

### Problem #33 <a id="p33"></a>

**Easy** · Microsoft · 2020-01-19

#### Problem

Compute the running median of a sequence of numbers. That is, given a stream of numbers, print out the median of the list so far on each new element.

Recall that the median of an even-numbered list is the average of the two middle numbers.

For example, given the sequence `[2, 1, 5, 7, 2, 0, 5]`, your algorithm should print out:

```
2
1.5
2
3.5
2
2
2
```

### Problem #34 <a id="p34"></a>

**Medium** · Quora · 2020-01-20

#### Problem

Given a string, find the palindrome that can be made by inserting the fewest number of characters as possible anywhere in the word. If there is more than one palindrome of minimum length that can be made, return the lexicographically earliest one (the first one alphabetically).

For example, given the string `"race"`, you should return `"ecarace"`, since we can add three letters to it (which is the smallest amount to make a palindrome). There are seven other palindromes that can be made from `"race"` by adding three letters, but `"ecarace"` comes first alphabetically.

As another example, given the string `"google"`, you should return `"elgoogle"`.

### Problem #35 <a id="p35"></a>

**Hard** · Google · 2020-01-21

#### Problem

Given an array of strictly the characters `'R'`, `'G'`, and `'B'`, segregate the values of the array so that all the Rs come first, the Gs come second, and the Bs come last. You can only swap elements of the array.

Do this in linear time and in-place.

For example, given the array `['G', 'B', 'R', 'R', 'B', 'R', 'G']`, it should become `['R', 'R', 'R', 'G', 'G', 'B', 'B']`.

### Problem #36 <a id="p36"></a>

**Medium** · Dropbox · 2020-01-22

#### Problem

Given the root to a binary search tree, find the second largest node in the tree.

### Problem #37 <a id="p37"></a>

**Easy** · Google · 2020-01-23

#### Problem

The power set of a set is the set of all its subsets. Write a function that, given a set, generates its power set.

For example, given the set `{1, 2, 3}`, it should return `{{}, {1}, {2}, {3}, {1, 2}, {1, 3}, {2, 3}, {1, 2, 3}}`.

You may also use a list or array to represent a set.

### Problem #38 <a id="p38"></a>

**Hard** · Microsoft · 2020-01-24

#### Problem

You have an N by N board. Write a function that, given N, returns the number of possible arrangements of the board where N queens can be placed on the board without threatening each other, i.e. no two queens share the same row, column, or diagonal.

### Problem #39 <a id="p39"></a>

**Medium** · Dropbox · 2020-01-25

#### Problem

Conway's Game of Life takes place on an infinite two-dimensional board of square cells. Each cell is either dead or alive, and at each tick, the following rules apply:

- Any live cell with less than two live neighbours dies.
- Any live cell with two or three live neighbours remains living.
- Any live cell with more than three live neighbours dies.
- Any dead cell with exactly three live neighbours becomes a live cell.

A cell neighbours another cell if it is horizontally, vertically, or diagonally adjacent.

Implement Conway's Game of Life. It should be able to be initialized with a starting list of live cell coordinates and the number of steps it should run for. Once initialized, it should print out the board state at each step. Since it's an infinite board, print out only the relevant coordinates, i.e. from the top-leftmost live cell to bottom-rightmost live cell.

You can represent a live cell with an asterisk (\*) and a dead cell with a dot (.).

### Problem #40 <a id="p40"></a>

**Hard** · Google · 2020-01-26

#### Problem

Given an array of integers where every integer occurs three times except for one integer, which only occurs once, find and return the non-duplicated integer.

For example, given `[6, 1, 3, 3, 3, 6, 6]`, return 1. Given `[13, 19, 13, 13]`, return 19.

Do this in `O(N)` time and `O(1)` space.

### Problem #41 <a id="p41"></a>

**Medium** · Facebook · 2020-01-27

#### Problem

Given an unordered list of flights taken by someone, each represented as (origin, destination) pairs, and a starting airport, compute the person's itinerary. If no such itinerary exists, return null. If there are multiple possible itineraries, return the lexicographically smallest one. All flights must be used in the itinerary.

For example, given the list of flights `[('SFO', 'HKO'), ('YYZ', 'SFO'), ('YUL', 'YYZ'), ('HKO', 'ORD')]` and starting airport `'YUL'`, you should return the list `['YUL', 'YYZ', 'SFO', 'HKO', 'ORD']`.

Given the list of flights `[('SFO', 'COM'), ('COM', 'YYZ')]` and starting airport `'COM'`, you should return null.

Given the list of flights `[('A', 'B'), ('A', 'C'), ('B', 'C'), ('C', 'A')]` and starting airport `'A'`, you should return the list `['A', 'B', 'C', 'A', 'C']` even though `['A', 'C', 'A', 'B', 'C']` is also a valid itinerary. However, the first one is lexicographically smaller.

### Problem #42 <a id="p42"></a>

**Hard** · Google · 2020-01-28

#### Problem

Given a list of integers S and a target number k, write a function that returns a subset of S that adds up to k. If such a subset cannot be made, then return null.

Integers can appear more than once in the list. You may assume all numbers in the list are positive.

For example, given S = `[12, 1, 61, 5, 9, 2]` and k = 24, return `[12, 9, 2, 1]` since it sums up to 24.

### Problem #43 <a id="p43"></a>

**Easy** · Amazon · 2020-01-29

#### Problem

Implement a stack that has the following methods:

- `push(val)`, which pushes an element onto the stack
- `pop()`, which pops off and returns the topmost element of the stack. If there are no elements in the stack, then it should throw an error or return null.
- `max()`, which returns the maximum value in the stack currently. If there are no elements in the stack, then it should throw an error or return null.

Each method should run in constant time.

### Problem #44 <a id="p44"></a>

**Medium** · Google · 2020-01-30

#### Problem

We can determine how `"out of order"` an array A is by counting the number of inversions it has. Two elements `A[i]` and `A[j]` form an inversion if `A[i]` > `A[j]` but i \< j. That is, a smaller element appears after a larger element.

Given an array, count the number of inversions it has. Do this faster than `O(N^2)` time.

You may assume each element in the array is distinct.

For example, a sorted list has zero inversions. The array `[2, 4, 1, 3, 5]` has three inversions: (2, 1), (4, 1), and (4, 3). The array `[5, 4, 3, 2, 1]` has ten inversions: every distinct pair forms an inversion.

### Problem #45 <a id="p45"></a>

**Easy** · Two Sigma · 2020-01-31

#### Problem

Using a function `rand5()` that returns an integer from 1 to 5 (inclusive) with uniform probability, implement a function `rand7()` that returns an integer from 1 to 7 (inclusive).

### Problem #46 <a id="p46"></a>

**Hard** · Amazon · 2020-02-01

#### Problem

Given a string, find the longest palindromic contiguous substring. If there are more than one with the maximum length, return any one.

For example, the longest palindromic substring of `"aabcdcb"` is `"bcdcb"`. The longest palindromic substring of `"bananas"` is `"anana"`.

### Problem #47 <a id="p47"></a>

**Easy** · Facebook · 2020-02-02

#### Problem

Given a array of numbers representing the stock prices of a company in chronological order, write a function that calculates the maximum profit you could have made from buying and selling that stock once. You must buy before you can sell it.

For example, given `[9, 11, 8, 5, 7, 10]`, you should return 5, since you could buy the stock at 5 dollars and sell it at 10 dollars.

### Problem #48 <a id="p48"></a>

**Medium** · Google · 2020-02-03

#### Problem

Given pre-order and in-order traversals of a binary tree, write a function to reconstruct the tree.

For example, given the following preorder traversal:

`[a, b, d, e, c, f, g]`

And the following inorder traversal:

`[d, b, e, a, f, c, g]`

You should return the following tree:

```
    a
   / \
  b   c
 / \ / \
d  e f  g
```

### Problem #49 <a id="p49"></a>

**Medium** · Amazon · 2020-02-04

#### Problem

Given an array of numbers, find the maximum sum of any contiguous subarray of the array.

For example, given the array `[34, -50, 42, 14, -5, 86]`, the maximum sum would be 137, since we would take elements 42, 14, -5, and 86.

Given the array `[-5, -1, -8, -9]`, the maximum sum would be 0, since we would not take any elements.

Do this in `O(N)` time.

### Problem #50 <a id="p50"></a>

**Easy** · Microsoft · 2020-02-05

#### Problem

Suppose an arithmetic expression is given as a binary tree. Each leaf is an integer and each internal node is one of `'+'`, `'−'`, `'∗'`, or `'/'`.

Given the root to such a tree, write a function to evaluate it.

For example, given the following tree:

```
    *
   / \
  +    +
 / \  / \
3  2  4  5
```

You should return 45, as it is (3 + 2) \* (4 + 5).

### Problem #51 <a id="p51"></a>

**Medium** · Facebook · 2020-02-06

#### Problem

Given a function that generates perfectly random numbers between 1 and k (inclusive), where k is an input, write a function that shuffles a deck of cards represented as an array using only swaps.

It should run in `O(N)` time.

Hint: Make sure each one of the 52! permutations of the deck is equally likely.

### Problem #52 <a id="p52"></a>

**Hard** · Google · 2020-02-07

#### Problem

Implement an LRU (Least Recently Used) cache. It should be able to be initialized with a cache size n, and contain the following methods:

- `set(key, value)`: sets key to value. If there are already n items in the cache and we are adding a new item, then it should also remove the least recently used item.
- `get(key)`: gets the value at key. If no such key exists, return null.

Each operation should run in `O(1)` time.

### Problem #53 <a id="p53"></a>

**Medium** · Apple · 2020-02-08

#### Problem

Implement a queue using two stacks. Recall that a queue is a FIFO (first-in, first-out) data structure with the following methods: enqueue, which inserts an element into the queue, and dequeue, which removes it.

### Problem #54 <a id="p54"></a>

**Hard** · Dropbox · 2020-02-09

#### Problem

Sudoku is a puzzle where you're given a partially-filled 9 by 9 grid with digits. The objective is to fill the grid with the constraint that every row, column, and box (3 by 3 subgrid) must contain all of the digits from 1 to 9.

Implement an efficient sudoku solver.

### Problem #55 <a id="p55"></a>

**Easy** · Microsoft · 2020-02-10

#### Problem

Implement a URL shortener with the following methods:

- `shorten(url)`, which shortens the url into a six-character alphanumeric string, such as zLg6wl.
- `restore(short)`, which expands the shortened string into the original url. If no such shortened string exists, return null.

Hint: What if we enter the same URL twice?

### Problem #56 <a id="p56"></a>

**Medium** · Google · 2020-02-11

#### Problem

Given an undirected graph represented as an adjacency matrix and an integer k, write a function to determine whether each vertex in the graph can be colored such that no two adjacent vertices share the same color using at most k colors.

### Problem #57 <a id="p57"></a>

**Medium** · Amazon · 2020-02-12

#### Problem

Given a string s and an integer k, break up the string into multiple lines such that each line has a length of k or less. You must break it up so that words don't break across lines. Each line has to have the maximum possible amount of words. If there's no way to break the text up, then return null.

You can assume that there are no spaces at the ends of the string and that there is exactly one space between each word.

For example, given the string `"the quick brown fox jumps over the lazy dog"` and k = 10, you should return: `["the quick", "brown fox", "jumps over", "the lazy", "dog"]`. No string in the list has a length of more than 10.

### Problem #58 <a id="p58"></a>

**Medium** · Amazon · 2020-02-13

#### Problem

An sorted array of integers was rotated an unknown number of times.

Given such an array, find the index of the element in the array in faster than linear time. If the element doesn't exist in the array, return null.

For example, given the array `[13, 18, 25, 2, 8, 10]` and the element 8, return 4 (the index of 8 in the array).

You can assume all the integers in the array are unique.

### Problem #59 <a id="p59"></a>

**Hard** · Google · 2020-02-14

#### Problem

Implement a file syncing algorithm for two computers over a low-bandwidth network. What if we know the files in the two computers are mostly the same?

### Problem #60 <a id="p60"></a>

**Medium** · Facebook · 2020-02-15

#### Problem

Given a multiset of integers, return whether it can be partitioned into two subsets whose sums are the same.

For example, given the multiset `{15, 5, 20, 10, 35, 15, 10}`, it would return true, since we can split it up into `{15, 5, 10, 15, 10}` and `{20, 35}`, which both add up to 55.

Given the multiset `{15, 5, 20, 10, 35}`, it would return false, since we can't split it up into two subsets that add up to the same sum.

### Problem #61 <a id="p61"></a>

**Medium** · Google · 2020-02-16

#### Problem

Implement integer exponentiation. That is, implement the `pow(x, y)` function, where x and y are integers and returns x^y.

Do this faster than the naive method of repeated multiplication.

For example, `pow(2, 10)` should return 1024.

### Problem #62 <a id="p62"></a>

**Medium** · Facebook · 2020-02-17

#### Problem

There is an N by M matrix of zeroes. Given N and M, write a function to count the number of ways of starting at the top-left corner and getting to the bottom-right corner. You can only move right or down.

For example, given a 2 by 2 matrix, you should return 2, since there are two ways to get to the bottom-right:

- Right, then down
- Down, then right

Given a 5 by 5 matrix, there are 70 ways to get to the bottom-right.

### Problem #63 <a id="p63"></a>

**Easy** · Microsoft · 2020-02-18

#### Problem

Given a 2D matrix of characters and a target word, write a function that returns whether the word can be found in the matrix by going left-to-right, or up-to-down.

For example, given the following matrix:

```
[['F', 'A', 'C', 'I'],
 ['O', 'B', 'Q', 'P'],
 ['A', 'N', 'O', 'B'],
 ['M', 'A', 'S', 'S']]
```

and the target word `'FOAM'`, you should return true, since it's the leftmost column. Similarly, given the target word `'MASS'`, you should return true, since it's the last row.

### Problem #64 <a id="p64"></a>

**Hard** · Google · 2020-02-19

#### Problem

A knight's tour is a sequence of moves by a knight on a chessboard such that all squares are visited once.

Given N, write a function to return the number of knight's tours on an N by N chessboard.

### Problem #65 <a id="p65"></a>

**Easy** · Amazon · 2020-02-20

#### Problem

Given a N by M matrix of numbers, print out the matrix in a clockwise spiral.

For example, given the following matrix:

```
[[1,  2,  3,  4,  5],
 [6,  7,  8,  9,  10],
 [11, 12, 13, 14, 15],
 [16, 17, 18, 19, 20]]
```

You should print out the following:

```
1
2
3
4
5
10
15
20
19
18
17
16
11
6
7
8
9
14
13
12
```

### Problem #66 <a id="p66"></a>

**Medium** · Square · 2020-02-21

#### Problem

Assume you have access to a function `toss_biased()` which returns 0 or 1 with a probability that's not 50-50 (but also not 0-100 or 100-0). You do not know the bias of the coin.

Write a function to simulate an unbiased coin toss.

### Problem #67 <a id="p67"></a>

**Hard** · Google · 2020-02-22

#### Problem

Implement an LFU (Least Frequently Used) cache. It should be able to be initialized with a cache size n, and contain the following methods:

- `set(key, value)`: sets key to value. If there are already n items in the cache and we are adding a new item, then it should also remove the least frequently used item. If there is a tie, then the least recently used key should be removed.
- `get(key)`: gets the value at key. If no such key exists, return null.

Each operation should run in `O(1)` time.

### Problem #68 <a id="p68"></a>

**Medium** · Google · 2020-02-23

#### Problem

On our special chessboard, two bishops attack each other if they share the same diagonal. This includes bishops that have another bishop located between them, i.e. bishops can attack through pieces.

You are given N bishops, represented as (row, column) tuples on a M by M chessboard. Write a function to count the number of pairs of bishops that attack each other. The ordering of the pair doesn't matter: (1, 2) is considered the same as (2, 1).

For example, given M = 5 and the list of bishops:

- (0, 0)
- (1, 2)
- (2, 2)
- (4, 0)

The board would look like this:

```
[b 0 0 0 0]
[0 0 b 0 0]
[0 0 b 0 0]
[0 0 0 0 0]
[b 0 0 0 0]
```

You should return 2, since bishops 1 and 3 attack each other, as well as bishops 3 and 4.

### Problem #69 <a id="p69"></a>

**Easy** · Facebook · 2020-02-24

#### Problem

Given a list of integers, return the largest product that can be made by multiplying any three integers.

For example, if the list is `[-10, -10, 5, 2]`, we should return 500, since that's -10 \* -10 \* 5.

You can assume the list has at least three integers.

### Problem #70 <a id="p70"></a>

**Easy** · Microsoft · 2020-02-25

#### Problem

A number is considered perfect if its digits sum up to exactly 10.

Given a positive integer n, return the n-th perfect number.

For example, given 1, you should return 19. Given 2, you should return 28.

### Problem #71 <a id="p71"></a>

**Easy** · Two Sigma · 2020-02-26

#### Problem

Using a function `rand7()` that returns an integer from 1 to 7 (inclusive) with uniform probability, implement a function `rand5()` that returns an integer from 1 to 5 (inclusive).

### Problem #72 <a id="p72"></a>

**Hard** · Google · 2020-02-27

#### Problem

In a directed graph, each node is assigned an uppercase letter. We define a path's value as the number of most frequently-occurring letter along that path. For example, if a path in the graph goes through `"ABACA"`, the value of the path is 3, since there are 3 occurrences of `'A'` on the path.

Given a graph with n nodes and m directed edges, return the largest value path of the graph. If the largest value is infinite, then return null.

The graph is represented with a string and an edge list. The i-th character represents the uppercase letter of the i-th node. Each tuple in the edge list (i, j) means there is a directed edge from the i-th node to the j-th node. Self-edges are possible, as well as multi-edges.

For example, the following input graph:

ABACA

```
[(0, 1),
 (0, 2),
 (2, 3),
 (3, 4)]
```

Would have maximum value 3 using the path of vertices `[0, 2, 3, 4]`, (A, A, C, A).

The following input graph:

A

`[(0, 0)]`

Should return null, since we have an infinite loop.

### Problem #73 <a id="p73"></a>

**Easy** · Google · 2020-02-28

#### Problem

Given the head of a singly linked list, reverse it in-place.

### Problem #74 <a id="p74"></a>

**Medium** · Apple · 2020-02-29

#### Problem

Suppose you have a multiplication table that is N by N. That is, a 2D array where the value at the i-th row and j-th column is (i + 1) \* (j + 1) (if 0-indexed) or i \* j (if 1-indexed).

Given integers N and X, write a function that returns the number of times X appears as a value in an N by N multiplication table.

For example, given N = 6 and X = 12, you should return 4, since the multiplication table looks like this:

| 1 | 2 | 3 | 4 | 5 | 6 |

| 2 | 4 | 6 | 8 | 10 | 12 |

| 3 | 6 | 9 | 12 | 15 | 18 |

| 4 | 8 | 12 | 16 | 20 | 24 |

| 5 | 10 | 15 | 20 | 25 | 30 |

| 6 | 12 | 18 | 24 | 30 | 36 |

And there are 4 12's in the table.

### Problem #75 <a id="p75"></a>

**Hard** · Microsoft · 2020-03-01

#### Problem

Given an array of numbers, find the length of the longest increasing subsequence in the array. The subsequence does not necessarily have to be contiguous.

For example, given the array `[0, 8, 4, 12, 2, 10, 6, 14, 1, 9, 5, 13, 3, 11, 7, 15]`, the longest increasing subsequence has length 6: it is 0, 2, 6, 9, 11, 15.

### Problem #76 <a id="p76"></a>

**Medium** · Google · 2020-03-02

#### Problem

You are given an N by M 2D matrix of lowercase letters. Determine the minimum number of columns that can be removed to ensure that each row is ordered from top to bottom lexicographically. That is, the letter at each column is lexicographically later as you go down each row. It does not matter whether each row itself is ordered lexicographically.

For example, given the following table:

```
cba
daf
ghi
```

This is not ordered because of the a in the center. We can remove the second column to make it ordered:

```
ca
df
gi
```

So your function should return 1, since we only needed to remove 1 column.

As another example, given the following table:

abcdef

Your function should return 0, since the rows are already ordered (there's only one row).

As another example, given the following table:

```
zyx
wvu
tsr
```

Your function should return 3, since we would need to remove all the columns to order it.

### Problem #77 <a id="p77"></a>

**Easy** · Snapchat · 2020-03-03

#### Problem

Given a list of possibly overlapping intervals, return a new list of intervals where all overlapping intervals have been merged.

The input list is not necessarily ordered in any way.

For example, given `[(1, 3), (5, 8), (4, 10), (20, 25)]`, you should return `[(1, 3), (4, 10), (20, 25)]`.

### Problem #78 <a id="p78"></a>

**Medium** · Google · 2020-03-04

#### Problem

Given k sorted singly linked lists, write a function to merge all the lists into one sorted singly linked list.

### Problem #79 <a id="p79"></a>

**Medium** · Facebook · 2020-03-05

#### Problem

Given an array of integers, write a function to determine whether the array could become non-decreasing by modifying at most 1 element.

For example, given the array `[10, 5, 7]`, you should return true, since we can modify the 10 into a 1 to make the array non-decreasing.

Given the array `[10, 5, 1]`, you should return false, since we can't modify any one element to get a non-decreasing array.

### Problem #80 <a id="p80"></a>

**Easy** · Google · 2020-03-06

#### Problem

Given the root of a binary tree, return a deepest node. For example, in the following tree, return d.

```
    a
   / \
  b   c
 /
d
```

### Problem #81 <a id="p81"></a>

**Easy** · Yelp · 2020-03-07

#### Problem

Given a mapping of digits to letters (as in a phone number), and a digit string, return all possible letters the number could represent. You can assume each valid number in the mapping is a single digit.

For example if `{“2”: [“a”, “b”, “c”], 3: [“d”, “e”, “f”], …}` then “23” should return \[“ad”, “ae”, “af”, “bd”, “be”, “bf”, “cd”, “ce”, “cf"\].

### Problem #82 <a id="p82"></a>

**Easy** · Microsoft · 2020-03-08

#### Problem

Using a `read7()` method that returns 7 characters from a file, implement `readN(n)` which reads n characters.

For example, given a file with the content “Hello world”, three `read7()` returns “Hello w”, “orld” and then “”.

### Problem #83 <a id="p83"></a>

**Medium** · Google · 2020-03-09

#### Problem

Invert a binary tree.

For example, given the following tree:

```
    a
   / \
  b   c
 / \  /
d   e f
```

should become:

```
 a
/ \
c  b
\  / \
 f e  d
```

### Problem #84 <a id="p84"></a>

**Medium** · Amazon · 2020-03-10

#### Problem

Given a matrix of 1s and 0s, return the number of `"islands"` in the matrix. A 1 represents land and 0 represents water, so an island is a group of 1s that are neighboring whose perimeter is surrounded by water.

For example, this matrix has 4 islands.

```
1 0 0 0 0
0 0 1 1 0
0 1 1 0 0
0 0 0 0 0
1 1 0 0 1
1 1 0 0 1
```

### Problem #85 <a id="p85"></a>

**Medium** · Facebook · 2020-03-11

#### Problem

Given three 32-bit integers x, y, and b, return x if b is 1 and y if b is 0, using only mathematical or bit operations. You can assume b can only be 1 or 0.

### Problem #86 <a id="p86"></a>

**Medium** · Google · 2020-03-12

#### Problem

Given a string of parentheses, write a function to compute the minimum number of parentheses to be removed to make the string valid (i.e. each open parenthesis is eventually closed).

For example, given the string `"()())()"`, you should return 1. Given the string `")("`, you should return 2, since we must remove all of them.

### Problem #87 <a id="p87"></a>

**Hard** · Uber · 2020-03-13

#### Problem

A rule looks like this:

A NE B

This means this means point A is located northeast of point B.

A SW C

means that point A is southwest of C.

Given a list of rules, check if the sum of the rules validate. For example:

```
A N B
B NE C
C N A
```

does not validate, since A cannot be both north and south of C.

```
A NW B
A N B
```

is considered valid.

### Problem #88 <a id="p88"></a>

**Medium** · ContextLogic · 2020-03-14

#### Problem

Implement division of two positive integers without using the division, multiplication, or modulus operators. Return the quotient as an integer, ignoring the remainder.

### Problem #89 <a id="p89"></a>

**Medium** · LinkedIn · 2020-03-15

#### Problem

Determine whether a tree is a valid binary search tree.

A binary search tree is a tree with two children, left and right, and satisfies the constraint that the key in the left child must be less than or equal to the root and the key in the right child must be greater than or equal to the root.

### Problem #90 <a id="p90"></a>

**Medium** · Google · 2020-03-16

#### Problem

Given an integer n and a list of integers l, write a function that randomly generates a number from 0 to n-1 that isn't in l (uniform).

### Problem #91 <a id="p91"></a>

**Easy** · Dropbox · 2020-03-17

#### Problem

What does the below code snippet print out? How can we fix the anonymous functions to behave as we'd expect?

```
functions = []
for i in range(10):
    functions.append(lambda : i)
```

```
for f in functions:
    print(f())
```

### Problem #92 <a id="p92"></a>

**Hard** · Airbnb · 2020-03-18

#### Problem

We're given a hashmap associating each courseId key with a list of courseIds values, which represents that the prerequisites of courseId are courseIds. Return a sorted ordering of courses such that we can finish all courses.

Return null if there is no such ordering.

For example, given `{'CSC300': ['CSC100', 'CSC200'], 'CSC200': ['CSC100'], 'CSC100': []}`, should return `['CSC100', 'CSC200', 'CSCS300']`.

### Problem #93 <a id="p93"></a>

**Hard** · Apple · 2020-03-19

#### Problem

Given a tree, find the largest tree/subtree that is a BST.

Given a tree, return the size of the largest tree/subtree that is a BST.

### Problem #94 <a id="p94"></a>

**Easy** · Google · 2020-03-20

#### Problem

Given a binary tree of integers, find the maximum path sum between two nodes. The path must go through at least one node, and does not need to go through the root.

### Problem #95 <a id="p95"></a>

**Hard** · Palantir · 2020-03-21

#### Problem

Given a number represented by a list of digits, find the next greater permutation of a number, in terms of lexicographic ordering. If there is not greater permutation possible, return the permutation with the lowest value/ordering.

For example, the list `[1,2,3]` should return `[1,3,2]`. The list `[1,3,2]` should return `[2,1,3]`. The list `[3,2,1]` should return `[1,2,3]`.

Can you perform the operation without allocating extra memory (disregarding the input memory)?

### Problem #96 <a id="p96"></a>

**Easy** · Microsoft · 2020-03-22

#### Problem

Given a number in the form of a list of digits, return all possible permutations.

For example, given `[1,2,3]`, return `[[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]`.

### Problem #97 <a id="p97"></a>

**Medium** · Stripe · 2020-03-23

#### Problem

Write a map implementation with a get function that lets you retrieve the value of a key at a particular time.

It should contain the following methods:

- `set(key, value, time)`: sets key to value for t = time.
- `get(key, time)`: gets the key at t = time.

The map should work like this. If we set a key at a particular time, it will maintain that value forever or until it gets set at a later time. In other words, when we get a key at a time, it should return the value that was set for that key set at the most recent time.

Consider the following examples:

```
d.set(1, 1, 0) # set key 1 to value 1 at time 0
d.set(1, 2, 2) # set key 1 to value 2 at time 2
d.get(1, 1) # get key 1 at time 1 should be 1
d.get(1, 3) # get key 1 at time 3 should be 2
```

```
d.set(1, 1, 5) # set key 1 to value 1 at time 5
d.get(1, 0) # get key 1 at time 0 should be null
d.get(1, 10) # get key 1 at time 10 should be 1
```

```
d.set(1, 1, 0) # set key 1 to value 1 at time 0
d.set(1, 2, 0) # set key 1 to value 2 at time 0
d.get(1, 0) # get key 1 at time 0 should be 2
```

### Problem #98 <a id="p98"></a>

**Easy** · Coursera · 2020-03-24

#### Problem

Given a 2D board of characters and a word, find if the word exists in the grid.

The word can be constructed from letters of sequentially adjacent cell, where `"adjacent"` cells are those horizontally or vertically neighboring. The same letter cell may not be used more than once.

For example, given the following board:

```
[
  ['A','B','C','E'],
  ['S','F','C','S'],
  ['A','D','E','E']
]
```

`exists(board, "ABCCED")` returns true, `exists(board, "SEE")` returns true, `exists(board, "ABCB")` returns false.

### Problem #99 <a id="p99"></a>

**Medium** · Microsoft · 2020-03-25

#### Problem

Given an unsorted array of integers, find the length of the longest consecutive elements sequence.

For example, given `[100, 4, 200, 1, 3, 2]`, the longest consecutive element sequence is `[1, 2, 3, 4]`. Return its length: 4.

Your algorithm should run in `O(n)` complexity.

### Problem #100 <a id="p100"></a>

**Easy** · Google · 2020-03-26

#### Problem

You are in an infinite 2D grid where you can move in any of the 8 directions:

```
(x,y) to
   (x+1, y),
   (x - 1, y),
   (x, y+1),
   (x, y-1),
   (x-1, y-1),
   (x+1,y+1),
   (x-1,y+1),
   (x+1,y-1)
```

You are given a sequence of points and the order in which you need to cover the points. Give the minimum number of steps in which you can achieve it. You start from the first point.

Example:

```
Input: [(0, 0), (1, 1), (1, 2)]
Output: 2
```

It takes 1 step to move from (0, 0) to (1, 1). It takes one more step to move from (1, 1) to (1, 2).

### Problem #101 <a id="p101"></a>

**Easy** · Alibaba · 2020-03-27

#### Problem

Given an even number (greater than 2), return two prime numbers whose sum will be equal to the given number.

A solution will always exist. See Goldbach’s conjecture.

Example:

```
Input: 4
Output: 2 + 2 = 4
```

If there are more than one solution possible, return the lexicographically smaller solution.

If `[a, b]` is one solution with a \<= b, and `[c, d]` is another solution with c \<= d, then

`[a, b]` \< `[c, d]`

If a \< c OR a==c AND b \< d.

### Problem #102 <a id="p102"></a>

**Medium** · Lyft · 2020-03-28

#### Problem

Given a list of integers and a number K, return which contiguous elements of the list sum to K.

For example, if the list is `[1, 2, 3, 4, 5]` and K is 9, then it should return `[2, 3, 4]`, since 2 + 3 + 4 = 9.

### Problem #103 <a id="p103"></a>

**Medium** · Square · 2020-03-29

#### Problem

Given a string and a set of characters, return the shortest substring containing all the characters in the set.

For example, given the string `"figehaeci"` and the set of characters `{a, e, i}`, you should return `"aeci"`.

If there is no substring containing all the characters in the set, return null.

### Problem #104 <a id="p104"></a>

**Easy** · Google · 2020-03-30

#### Problem

Determine whether a doubly linked list is a palindrome. What if it’s singly linked?

For example, 1 -> 4 -> 3 -> 4 -> 1 returns True while 1 -> 4 returns False.

### Problem #105 <a id="p105"></a>

**Easy** · Facebook · 2020-03-31

#### Problem

Given a function f, and N return a debounced f of N milliseconds.

That is, as long as the debounced f continues to be invoked, f itself will not be called for N milliseconds.

### Problem #106 <a id="p106"></a>

**Medium** · Pinterest · 2020-04-01

#### Problem

Given an integer list where each number represents the number of hops you can make, determine whether you can reach to the last index starting at index 0.

For example, `[2, 0, 1, 0]` returns True while `[1, 1, 0, 1]` returns False.

### Problem #107 <a id="p107"></a>

**Easy** · Microsoft · 2020-04-02

#### Problem

Print the nodes in a binary tree level-wise. For example, the following should print 1, 2, 3, 4, 5.

```
  1
 / \
2   3
   / \
  4   5
```

### Problem #108 <a id="p108"></a>

**Easy** · Google · 2020-04-03

#### Problem

Given two strings A and B, return whether or not A can be shifted some number of times to get B.

For example, if A is abcde and B is cdeab, return true. If A is abc and B is acb, return false.

### Problem #109 <a id="p109"></a>

**Medium** · Cisco · 2020-04-04

#### Problem

Given an unsigned 8-bit integer, swap its even and odd bits. The 1st and 2nd bit should be swapped, the 3rd and 4th bit should be swapped, and so on.

For example, 10101010 should be 01010101. 11100010 should be 11010001.

Bonus: Can you do this in one line?

### Problem #110 <a id="p110"></a>

**Medium** · Facebook · 2020-04-05

#### Problem

Given a binary tree, return all paths from the root to leaves.

For example, given the tree:

```
  1
 / \
2   3
   / \
  4   5
```

Return `[[1, 2], [1, 3, 4], [1, 3, 5]]`.

### Problem #111 <a id="p111"></a>

**Hard** · Google · 2020-04-06

#### Problem

Given a word W and a string S, find all starting indices in S which are anagrams of W.

For example, given that W is `"ab"`, and S is `"abxaba"`, return 0, 3, and 4.

### Problem #112 <a id="p112"></a>

**Hard** · Twitter · 2020-04-07

#### Problem

Given a binary tree, find the lowest common ancestor (LCA) of two given nodes in the tree. Assume that each node in the tree also has a pointer to its parent.

According to the definition of LCA on Wikipedia: “The lowest common ancestor is defined between two nodes v and w as the lowest node in T that has both v and w as descendants (where we allow a node to be a descendant of itself).”

### Problem #113 <a id="p113"></a>

**Medium** · Google · 2020-04-08

#### Problem

Given a string of words delimited by spaces, reverse the words in string. For example, given `"hello world here"`, return `"here world hello"`

Follow-up: given a mutable string representation, can you perform this operation in-place?

### Problem #114 <a id="p114"></a>

**Hard** · Facebook · 2020-04-09

#### Problem

Given a string and a set of delimiters, reverse the words in the string while maintaining the relative order of the delimiters. For example, given `"hello/world:here"`, return `"here/world:hello"`

Follow-up: Does your solution work for the following cases: `"hello/world:here/"`, `"hello//world:here"`

### Problem #115 <a id="p115"></a>

**Hard** · Google · 2020-04-10

#### Problem

Given two non-empty binary trees s and t, check whether tree t has exactly the same structure and node values with a subtree of s. A subtree of s is a tree consists of a node in s and all of this node's descendants. The tree s could also be considered as a subtree of itself.

### Problem #116 <a id="p116"></a>

**Medium** · Jane Street · 2020-04-11

#### Problem

Generate a finite, but an arbitrarily large binary tree quickly in `O(1)`.

That is, `generate()` should return a tree whose size is unbounded but finite.

### Problem #117 <a id="p117"></a>

**Easy** · Facebook · 2020-04-12

#### Problem

Given a binary tree, return the level of the tree with minimum sum.

### Problem #118 <a id="p118"></a>

**Easy** · Google · 2020-04-13

#### Problem

Given a sorted list of integers, square the elements and give the output in sorted order.

For example, given `[-9, -2, 0, 2, 3]`, return `[0, 4, 4, 9, 81]`.

### Problem #119 <a id="p119"></a>

**Medium** · Google · 2020-04-14

#### Problem

Given a set of closed intervals, find the smallest set of numbers that covers all the intervals. If there are multiple smallest sets, return any of them.

For example, given the intervals `[0, 3]`, `[2, 6]`, `[3, 4]`, `[6, 9]`, one set of numbers that covers all these intervals is `{3, 6}`.

### Problem #120 <a id="p120"></a>

**Medium** · Microsoft · 2020-04-15

#### Problem

Implement the singleton pattern with a twist. First, instead of storing one instance, store two instances. And in every even call of `getInstance()`, return the first instance and in every odd call of `getInstance()`, return the second instance.

### Problem #121 <a id="p121"></a>

**Hard** · Google · 2020-04-16

#### Problem

Given a string which we can delete at most k, return whether you can make a palindrome.

For example, given `'waterrfetawx'` and a k of 2, you could delete f and x to get `'waterretaw'`.

### Problem #122 <a id="p122"></a>

**Medium** · Zillow · 2020-04-17

#### Problem

You are given a 2-d matrix where each cell represents number of coins in that cell. Assuming we start at `matrix[0][0]`, and can only move right or down, find the maximum number of coins you can collect by the bottom right corner.

For example, in this matrix

```
0 3 1 1
2 0 0 4
1 5 3 1
```

The most we can collect is 0 + 2 + 1 + 5 + 3 + 1 = 12 coins.

### Problem #123 <a id="p123"></a>

**Hard** · LinkedIn · 2020-04-18

#### Problem

Given a string, return whether it represents a number. Here are the different kinds of numbers:

- `"10"`, a positive integer
- `"-10"`, a negative integer
- `"10.1"`, a positive real number
- `"-10.1"`, a negative real number
- `"1e5"`, a number in scientific notation

And here are examples of non-numbers:

- `"a"`
- `"x 1"`
- `"a -2"`
- `"-"`

### Problem #124 <a id="p124"></a>

**Easy** · Microsoft · 2020-04-19

#### Problem

You have n fair coins and you flip them all at the same time. Any that come up tails you set aside. The ones that come up heads you flip again. How many rounds do you expect to play before only one coin remains?

Write a function that, given n, returns the number of rounds you'd expect to play until one coin remains.

### Problem #125 <a id="p125"></a>

**Easy** · Google · 2020-04-20

#### Problem

Given the root of a binary search tree, and a target K, return two nodes in the tree whose sum equals K.

For example, given the following tree and K of 20

```
   10
  /   \
5      15
      /  \
    11    15
```

Return the nodes 5 and 15.

### Problem #126 <a id="p126"></a>

**Medium** · Facebook · 2020-04-21

#### Problem

Write a function that rotates a list by k elements. For example, `[1, 2, 3, 4, 5, 6]` rotated by two becomes `[3, 4, 5, 6, 1, 2]`. Try solving this without creating a copy of the list. How many swap or move operations do you need?

### Problem #127 <a id="p127"></a>

**Easy** · Microsoft · 2020-04-22

#### Problem

Let's represent an integer in a linked list format by having each node represent a digit in the number. The nodes make up the number in reversed order.

For example, the following linked list:

1 -> 2 -> 3 -> 4 -> 5

is the number 54321.

Given two linked lists in this format, return their sum in the same linked list format.

For example, given

9 -> 9

5 -> 2

return 124 (99 + 25) as:

4 -> 2 -> 1

### Problem #128 <a id="p128"></a>

**Medium** · 2020-04-23

#### Problem

The Tower of Hanoi is a puzzle game with three rods and n disks, each a different size.

All the disks start off on the first rod in a stack. They are ordered by size, with the largest disk on the bottom and the smallest one at the top.

The goal of this puzzle is to move all the disks from the first rod to the last rod while following these rules:

- You can only move one disk at a time.
- A move consists of taking the uppermost disk from one of the stacks and placing it on top of another stack.
- You cannot place a larger disk on top of a smaller disk.

Write a function that prints out all the steps necessary to complete the Tower of Hanoi. You should assume that the rods are numbered, with the first rod being 1, the second (auxiliary) rod being 2, and the last (goal) rod being 3.

For example, with n = 3, we can do this in 7 moves:

```
Move 1 to 3
Move 1 to 2
Move 3 to 2
Move 1 to 3
Move 2 to 1
Move 2 to 3
Move 1 to 3
```

### Problem #129 <a id="p129"></a>

**Medium** · 2020-04-24

#### Problem

Given a real number n, find the square root of n. For example, given n = 9, return 3.

### Problem #130 <a id="p130"></a>

**Medium** · Facebook · 2020-04-25

#### Problem

Given an array of numbers representing the stock prices of a company in chronological order and an integer k, return the maximum profit you can make from k buys and sells. You must buy the stock before you can sell it, and you must sell the stock before you can buy it again.

For example, given k = 2 and the array `[5, 2, 4, 0, 1]`, you should return 3.

### Problem #131 <a id="p131"></a>

**Medium** · Snapchat · 2020-04-26

#### Problem

Given the head to a singly linked list, where each node also has a “random” pointer that points to anywhere in the linked list, deep clone the list.

### Problem #132 <a id="p132"></a>

**Easy** · Riot Games · 2020-04-27

#### Problem

Design and implement a HitCounter class that keeps track of requests (or hits). It should support the following operations:

- `record(timestamp)`: records a hit that happened at timestamp
- `total()`: returns the total number of hits recorded
- `range(lower, upper)`: returns the number of hits that occurred between timestamps lower and upper (inclusive)

Follow-up: What if our system has limited memory?

### Problem #133 <a id="p133"></a>

**Medium** · Amazon · 2020-04-28

#### Problem

Given a node in a binary search tree, return the next bigger element, also known as the inorder successor.

For example, the inorder successor of 22 is 30.

```
  10
 /  \
5    30
    /  \
  22    35
```

You can assume each node has a parent pointer.

### Problem #134 <a id="p134"></a>

**Easy** · Facebook · 2020-04-29

#### Problem

You have a large array with most of the elements as zero.

Use a more space-efficient data structure, SparseArray, that implements the same interface:

- `init(arr, size)`: initialize with the original large array and size.
- `set(i, val)`: updates index at i with val.
- `get(i)`: gets the value at index i.

### Problem #135 <a id="p135"></a>

**Easy** · Apple · 2020-04-30

#### Problem

Given a binary tree, find a minimum path sum from root to a leaf.

For example, the minimum path in this tree is `[10, 5, 1, -1]`, which has sum 15.

```
  10
 /  \
5    5
 \     \
   2    1
       /
     -1
```

### Problem #136 <a id="p136"></a>

**Medium** · Google · 2020-05-01

#### Problem

Given an N by M matrix consisting only of 1's and 0's, find the largest rectangle containing only 1's and return its area.

For example, given the following matrix:

```
[[1, 0, 0, 0],
 [1, 0, 1, 1],
 [1, 0, 1, 1],
 [0, 1, 0, 0]]
```

Return 4.

### Problem #137 <a id="p137"></a>

**Medium** · Amazon · 2020-05-02

#### Problem

Implement a bit array.

A bit array is a space efficient array that holds a value of 1 or 0 at each index.

- `init(size)`: initialize the array with size
- `set(i, val)`: updates index at i with val where val is either 1 or 0.
- `get(i)`: gets the value at index i.

### Problem #138 <a id="p138"></a>

**Hard** · Google · 2020-05-03

#### Problem

Find the minimum number of coins required to make n cents.

You can use standard American denominations, that is, 1¢, 5¢, 10¢, and 25¢.

For example, given n = 16, return 3 since we can make it with a 10¢, a 5¢, and a 1¢.

### Problem #139 <a id="p139"></a>

**Medium** · Google · 2020-05-04

#### Problem

Given an iterator with methods `next()` and `hasNext()`, create a wrapper iterator, PeekableInterface, which also implements `peek()`. peek shows the next element that would be returned on `next()`.

Here is the interface:

```
class PeekableInterface(object):
    def __init__(self, iterator):
        pass
```

```
def peek(self):
    pass
```

```
def next(self):
    pass
```

```
def hasNext(self):
    pass
```

### Problem #140 <a id="p140"></a>

**Medium** · Facebook · 2020-05-05

#### Problem

Given an array of integers in which two elements appear exactly once and all other elements appear exactly twice, find the two elements that appear only once.

For example, given the array `[2, 4, 6, 8, 10, 2, 6, 10]`, return 4 and 8. The order does not matter.

Follow-up: Can you do this in linear time and constant space?

### Problem #141 <a id="p141"></a>

**Hard** · Microsoft · 2020-05-06

#### Problem

Implement 3 stacks using a single list:

```
class Stack:
    def __init__(self):
        self.list = []
```

```
def pop(self, stack_number):
    pass
```

```
def push(self, item, stack_number):
    pass
```

### Problem #142 <a id="p142"></a>

**Hard** · Google · 2020-05-07

#### Problem

You're given a string consisting solely of (, ), and \*. \* can represent either a (, ), or an empty string. Determine whether the parentheses are balanced.

For example, (()\* and (\*) are balanced. )\*( is not balanced.

### Problem #143 <a id="p143"></a>

**Medium** · Amazon · 2020-05-08

#### Problem

Given a pivot x, and a list lst, partition the list into three parts.

- The first part contains all elements in lst that are less than x
- The second part contains all elements in lst that are equal to x
- The third part contains all elements in lst that are larger than x

Ordering within a part can be arbitrary.

For example, given x = 10 and lst = `[9, 12, 3, 5, 14, 10, 10]`, one partition may be `[9, 3, 5, 10, 10, 12, 14]`.

### Problem #144 <a id="p144"></a>

**Medium** · Google · 2020-05-09

#### Problem

Given an array of numbers and an index i, return the index of the nearest larger number of the number at index i, where distance is measured in array indices.

For example, given `[4, 1, 3, 5, 6]` and index 0, you should return 3.

If two distances to larger numbers are the equal, then return any one of them. If the array at i doesn't have a nearest larger integer, then return null.

Follow-up: If you can preprocess the array, can you do this in constant time?

### Problem #145 <a id="p145"></a>

**Easy** · Google · 2020-05-10

#### Problem

Given the head of a singly linked list, swap every two nodes and return its head.

For example, given 1 -> 2 -> 3 -> 4, return 2 -> 1 -> 4 -> 3.

### Problem #146 <a id="p146"></a>

**Medium** · BufferBox · 2020-05-11

#### Problem

Given a binary tree where all nodes are either 0 or 1, prune the tree so that subtrees containing all 0s are removed.

For example, given the following tree:

```
  0
 / \
1   0
   / \
  1   0
 / \
0   0
```

should be pruned to:

```
  0
 / \
1   0
   /
  1
```

We do not remove the tree at the root or its left child because it still has a 1 as a descendant.

### Problem #147 <a id="p147"></a>

**Hard** · 2020-05-12

#### Problem

Given a list, sort it using this method: `reverse(lst, i, j)`, which reverses lst from i to j.

### Problem #148 <a id="p148"></a>

**Medium** · Apple · 2020-05-13

#### Problem

Gray code is a binary code where each successive value differ in only one bit, as well as when wrapping around. Gray code is common in hardware so that we don't see temporary spurious values during transitions.

Given a number of bits n, generate a possible gray code for it.

For example, for n = 2, one gray code would be `[00, 01, 11, 10]`.

### Problem #149 <a id="p149"></a>

**Hard** · Goldman Sachs · 2020-05-14

#### Problem

Given a list of numbers L, implement a method `sum(i, j)` which returns the sum from the sublist `L[i:j]` (including i, excluding j).

For example, given L = `[1, 2, 3, 4, 5]`, `sum(1, 3)` should return `sum([2, 3])`, which is 5.

You can assume that you can do some pre-processing. `sum()` should be optimized over the pre-processing step.

### Problem #150 <a id="p150"></a>

**Hard** · LinkedIn · 2020-05-15

#### Problem

Given a list of points, a central point, and an integer k, find the nearest k points from the central point.

For example, given the list of points `[(0, 0), (5, 4), (3, 1)]`, the central point (1, 2), and k = 2, return `[(0, 0), (3, 1)]`.

### Problem #151 <a id="p151"></a>

**Medium** · 2020-05-16

#### Problem

Given a 2-D matrix representing an image, a location of a pixel in the screen and a color C, replace the color of the given pixel and all adjacent same colored pixels with C.

For example, given the following matrix, and location pixel of (2, 2), and `'G'` for green:

```
B B W
W W W
W W W
B B B
```

Becomes

```
B B G
G G G
G G G
B B B
```

### Problem #152 <a id="p152"></a>

**Medium** · Triplebyte · 2020-05-17

#### Problem

You are given n numbers as well as n probabilities that sum up to 1. Write a function to generate one of the numbers with its corresponding probability.

For example, given the numbers `[1, 2, 3, 4]` and probabilities `[0.1, 0.5, 0.2, 0.2]`, your function should return 1 10% of the time, 2 50% of the time, and 3 and 4 20% of the time.

You can generate random numbers between 0 and 1 uniformly.

### Problem #153 <a id="p153"></a>

**Hard** · 2020-05-18

#### Problem

Find an efficient algorithm to find the smallest distance (measured in number of words) between any two given words in a string.

For example, given words `"hello"`, and `"world"` and a text content of `"dog cat hello cat dog dog hello cat world"`, return 1 because there's only one word `"cat"` in between the two words.

### Problem #154 <a id="p154"></a>

**Easy** · Amazon · 2020-05-19

#### Problem

Implement a stack API using only a heap. A stack implements the following methods:

- `push(item)`, which adds an element to the stack
- `pop()`, which removes and returns the most recently added element (or throws an error if there is nothing on the stack)

Recall that a heap has the following operations:

- `push(item)`, which adds a new key to the heap
- `pop()`, which removes and returns the max value of the heap

### Problem #155 <a id="p155"></a>

**Medium** · MongoDB · 2020-05-20

#### Problem

Given a list of elements, find the majority element, which appears more than half the time (> `floor(len(lst) / 2.0)`).

You can assume that such element exists.

For example, given `[1, 2, 1, 1, 3, 4, 0]`, return 1.

### Problem #156 <a id="p156"></a>

**Medium** · Facebook · 2020-05-21

#### Problem

Given a positive integer n, find the smallest number of squared integers which sum to n.

For example, given n = 13, return 2 since 13 = 32 + 22 = 9 + 4.

Given n = 27, return 3 since 27 = 32 + 32 + 32 = 9 + 9 + 9.

### Problem #157 <a id="p157"></a>

**Easy** · Amazon · 2020-05-22

#### Problem

Given a string, determine whether any permutation of it is a palindrome.

For example, carrace should return true, since it can be rearranged to form racecar, which is a palindrome. daily should return false, since there's no rearrangement that can form a palindrome.

### Problem #158 <a id="p158"></a>

**Medium** · Slack · 2020-05-23

#### Problem

You are given an N by M matrix of 0s and 1s. Starting from the top left corner, how many ways are there to reach the bottom right corner?

You can only move right and down. 0 represents an empty space while 1 represents a wall you cannot walk through.

For example, given the following matrix:

```
[[0, 0, 1],
 [0, 0, 1],
 [1, 0, 0]]
```

Return two, as there are only two ways to get to the bottom right:

- Right, down, down, right
- Down, right, down, right

The top left corner and bottom right corner will always be 0.

### Problem #159 <a id="p159"></a>

**Easy** · Google · 2020-05-24

#### Problem

Given a string, return the first recurring character in it, or null if there is no recurring character.

For example, given the string `"acbbac"`, return `"b"`. Given the string `"abcdef"`, return null.

### Problem #160 <a id="p160"></a>

**Hard** · Uber · 2020-05-25

#### Problem

Given a tree where each edge has a weight, compute the length of the longest path in the tree.

For example, given the following tree:

```
  a
 /|\
b c d
   / \
  e   f
 / \
g   h
```

and the weights: a-b: 3, a-c: 5, a-d: 8, d-e: 2, d-f: 4, e-g: 1, e-h: 1, the longest path would be c -> a -> d -> f, with a length of 17.

The path does not have to pass through the root, and each node can have any amount of children.

### Problem #161 <a id="p161"></a>

**Easy** · Facebook · 2020-05-26

#### Problem

Given a 32-bit integer, return the number with its bits reversed.

For example, given the binary number 1111 0000 1111 0000 1111 0000 1111 0000, return 0000 1111 0000 1111 0000 1111 0000 1111.

### Problem #162 <a id="p162"></a>

**Medium** · Square · 2020-05-27

#### Problem

Given a list of words, return the shortest unique prefix of each word. For example, given the list:

- dog
- cat
- apple
- apricot
- fish

Return the list:

- d
- c
- app
- apr
- f

### Problem #163 <a id="p163"></a>

**Hard** · Jane Street · 2020-05-28

#### Problem

Given an arithmetic expression in Reverse Polish Notation, write a program to evaluate it.

The expression is given as a list of numbers and operands. For example: `[5, 3, '+']` should return 5 + 3 = 8.

For example, `[15, 7, 1, 1, '+', '-', '/', 3, '*', 2, 1, 1, '+', '+', '-']` should return 5, since it is equivalent to ((15 / (7 - (1 + 1))) \* 3) - (2 + (1 + 1)) = 5.

You can assume the given expression is always valid.

### Problem #164 <a id="p164"></a>

**Medium** · Google · 2020-05-29

#### Problem

You are given an array of length n + 1 whose elements belong to the set `{1, 2, ..., n}`. By the pigeonhole principle, there must be a duplicate. Find it in linear time and space.

### Problem #165 <a id="p165"></a>

**Medium** · Google · 2020-05-30

#### Problem

Given an array of integers, return a new array where each element in the new array is the number of smaller elements to the right of that element in the original input array.

For example, given the array `[3, 4, 9, 6, 1]`, return `[1, 1, 2, 1, 0]`, since:

- There is 1 smaller element to the right of 3
- There is 1 smaller element to the right of 4
- There are 2 smaller elements to the right of 9
- There is 1 smaller element to the right of 6
- There are no smaller elements to the right of 1

### Problem #166 <a id="p166"></a>

**Medium** · Uber · 2020-05-31

#### Problem

Implement a 2D iterator class. It will be initialized with an array of arrays, and should implement the following methods:

- `next()`: returns the next element in the array of arrays. If there are no more elements, raise an exception.
- `has_next()`: returns whether or not the iterator still has elements left.

For example, given the input `[[1, 2], [3], [], [4, 5, 6]]`, calling `next()` repeatedly should output 1, 2, 3, 4, 5, 6.

Do not use flatten or otherwise clone the arrays. Some of the arrays can be empty.

### Problem #167 <a id="p167"></a>

**Hard** · Airbnb · 2020-06-01

#### Problem

Given a list of words, find all pairs of unique indices such that the concatenation of the two words is a palindrome.

For example, given the list `["code", "edoc", "da", "d"]`, return `[(0, 1), (1, 0), (2, 3)]`.

### Problem #168 <a id="p168"></a>

**Medium** · Facebook · 2020-06-02

#### Problem

Given an N by N matrix, rotate it by 90 degrees clockwise.

For example, given the following matrix:

```
[[1, 2, 3],
 [4, 5, 6],
 [7, 8, 9]]
```

you should return:

```
[[7, 4, 1],
 [8, 5, 2],
 [9, 6, 3]]
```

Follow-up: What if you couldn't use any extra space?

### Problem #169 <a id="p169"></a>

**Medium** · Google · 2020-06-03

#### Problem

Given a linked list, sort it in `O(n log n)` time and constant space.

For example, the linked list 4 -> 1 -> -3 -> 99 should become -3 -> 1 -> 4 -> 99.

### Problem #170 <a id="p170"></a>

**Medium** · Facebook · 2020-06-04

#### Problem

Given a start word, an end word, and a dictionary of valid words, find the shortest transformation sequence from start to end such that only one letter is changed at each step of the sequence, and each transformed word exists in the dictionary. If there is no possible transformation, return null. Each word in the dictionary have the same length as start and end and is lowercase.

For example, given start = `"dog"`, end = `"cat"`, and dictionary = `{"dot", "dop", "dat", "cat"}`, return `["dog", "dot", "dat", "cat"]`.

Given start = `"dog"`, end = `"cat"`, and dictionary = `{"dot", "tod", "dat", "dar"}`, return null as there is no possible transformation from dog to cat.

### Problem #171 <a id="p171"></a>

**Easy** · Amazon · 2020-06-05

#### Problem

You are given a list of data entries that represent entries and exits of groups of people into a building. An entry looks like this:

`{"timestamp": 1526579928, count: 3, "type": "enter"}`

This means 3 people entered the building. An exit looks like this:

`{"timestamp": 1526580382, count: 2, "type": "exit"}`

This means that 2 people exited the building. timestamp is in Unix time.

Find the busiest period in the building, that is, the time with the most people in the building. Return it as a pair of (start, end) timestamps. You can assume the building always starts off and ends up empty, i.e. with 0 people inside.

### Problem #172 <a id="p172"></a>

**Medium** · Dropbox · 2020-06-06

#### Problem

Given a string s and a list of words words, where each word is the same length, find all starting indices of substrings in s that is a concatenation of every word in words exactly once.

For example, given s = `"dogcatcatcodecatdog"` and words = `["cat", "dog"]`, return `[0, 13]`, since `"dogcat"` starts at index 0 and `"catdog"` starts at index 13.

Given s = `"barfoobazbitbyte"` and words = `["dog", "cat"]`, return \[\] since there are no substrings composed of `"dog"` and `"cat"` in s.

The order of the indices does not matter.

### Problem #173 <a id="p173"></a>

**Easy** · Stripe · 2020-06-07

#### Problem

Write a function to flatten a nested dictionary. Namespace the keys with a period.

For example, given the following dictionary:

```
{
    "key": 3,
    "foo": {
        "a": 5,
        "bar": {
            "baz": 8
        }
    }
}
```

it should become:

```
{
    "key": 3,
    "foo.a": 5,
    "foo.bar.baz": 8
}
```

You can assume keys do not contain dots in them, i.e. no clobbering will occur.

### Problem #174 <a id="p174"></a>

**Medium** · Microsoft · 2020-06-08

#### Problem

Describe and give an example of each of the following types of polymorphism:

- Ad-hoc polymorphism
- Parametric polymorphism
- Subtype polymorphism

### Problem #175 <a id="p175"></a>

**Easy** · Google · 2020-06-09

#### Problem

You are given a starting state start, a list of transition probabilities for a Markov chain, and a number of steps `num_steps`. Run the Markov chain starting from start for `num_steps` and compute the number of times we visited each state.

For example, given the starting state a, number of steps 5000, and the following transition probabilities:

```
[
  ('a', 'a', 0.9),
  ('a', 'b', 0.075),
  ('a', 'c', 0.025),
  ('b', 'a', 0.15),
  ('b', 'b', 0.8),
  ('b', 'c', 0.05),
  ('c', 'a', 0.25),
  ('c', 'b', 0.25),
  ('c', 'c', 0.5)
]
```

One instance of running this Markov chain might produce `{ 'a': 3012, 'b': 1656, 'c': 332 }`.

### Problem #176 <a id="p176"></a>

**Easy** · Bloomberg · 2020-06-10

#### Problem

Determine whether there exists a one-to-one character mapping from one string s1 to another s2.

For example, given s1 = abc and s2 = bcd, return true since we can map a to b, b to c, and c to d.

Given s1 = foo and s2 = bar, return false since the o cannot map to two characters.

### Problem #177 <a id="p177"></a>

**Easy** · Airbnb · 2020-06-11

#### Problem

Given a linked list and a positive integer k, rotate the list to the right by k places.

For example, given the linked list 7 -> 7 -> 3 -> 5 and k = 2, it should become 3 -> 5 -> 7 -> 7.

Given the linked list 1 -> 2 -> 3 -> 4 -> 5 and k = 3, it should become 3 -> 4 -> 5 -> 1 -> 2.

### Problem #178 <a id="p178"></a>

**Hard** · Two Sigma · 2020-06-12

#### Problem

Alice wants to join her school's Probability Student Club. Membership dues are computed via one of two simple probabilistic games.

The first game: roll a die repeatedly. Stop rolling once you get a five followed by a six. Your number of rolls is the amount you pay, in dollars.

The second game: same, except that the stopping condition is a five followed by a five.

Which of the two games should Alice elect to play? Does it even matter? Write a program to simulate the two games and calculate their expected value.

### Problem #179 <a id="p179"></a>

**Medium** · Google · 2020-06-13

#### Problem

Given the sequence of keys visited by a postorder traversal of a binary search tree, reconstruct the tree.

For example, given the sequence 2, 4, 3, 8, 7, 5, you should construct the following tree:

```
    5
   / \
  3   7
 / \   \
2   4   8
```

### Problem #180 <a id="p180"></a>

**Medium** · Google · 2020-06-14

#### Problem

Given a stack of N elements, interleave the first half of the stack with the second half reversed using only one other queue. This should be done in-place.

Recall that you can only push or pop from a stack, and enqueue or dequeue from a queue.

For example, if the stack is `[1, 2, 3, 4, 5]`, it should become `[1, 5, 2, 4, 3]`. If the stack is `[1, 2, 3, 4]`, it should become `[1, 4, 2, 3]`.

Hint: Try working backwards from the end state.

### Problem #181 <a id="p181"></a>

**Hard** · Google · 2020-06-15

#### Problem

Given a string, split it into as few strings as possible such that each string is a palindrome.

For example, given the input string racecarannakayak, return `["racecar", "anna", "kayak"]`.

Given the input string abc, return `["a", "b", "c"]`.

### Problem #182 <a id="p182"></a>

**Medium** · Facebook · 2020-06-16

#### Problem

A graph is minimally-connected if it is connected and there is no edge that can be removed while still leaving the graph connected. For example, any binary tree is minimally-connected.

Given an undirected graph, check if the graph is minimally-connected. You can choose to represent the graph as either an adjacency matrix or adjacency list.

### Problem #183 <a id="p183"></a>

**Hard** · Twitch · 2020-06-17

#### Problem

Describe what happens when you type a URL into your browser and press Enter.

### Problem #184 <a id="p184"></a>

**Easy** · Amazon · 2020-06-18

#### Problem

Given n numbers, find the greatest common denominator between them.

For example, given the numbers `[42, 56, 14]`, return 14.

### Problem #185 <a id="p185"></a>

**Easy** · Google · 2020-06-19

#### Problem

Given two rectangles on a 2D graph, return the area of their intersection. If the rectangles don't intersect, return 0.

For example, given the following rectangles:

```
{
    "top_left": (1, 4),
    "dimensions": (3, 3) # width, height
}
```

and

```
{
    "top_left": (0, 5),
    "dimensions": (4, 3) # width, height
}
```

return 6.

### Problem #186 <a id="p186"></a>

**Hard** · Microsoft · 2020-06-20

#### Problem

Given an array of positive integers, divide the array into two subsets such that the difference between the sum of the subsets is as small as possible.

For example, given `[5, 10, 15, 20, 25]`, return the sets `{10, 25}` and `{5, 15, 20}`, which has a difference of 5, which is the smallest possible difference.

### Problem #187 <a id="p187"></a>

**Easy** · Google · 2020-06-21

#### Problem

You are given given a list of rectangles represented by min and max x- and y-coordinates. Compute whether or not a pair of rectangles overlap each other. If one rectangle completely covers another, it is considered overlapping.

For example, given the following rectangles:

```
{
    "top_left": (1, 4),
    "dimensions": (3, 3) # width, height
},
{
    "top_left": (-1, 3),
    "dimensions": (2, 1)
},
{
    "top_left": (0, 5),
    "dimensions": (4, 3)
}
```

return true as the first and third rectangle overlap each other.

### Problem #188 <a id="p188"></a>

**Medium** · Google · 2020-06-22

#### Problem

What will this code print out?

```
def make_functions():
    flist = []
```

```
for i in [1, 2, 3]:
    def print_i():
        print(i)
    flist.append(print_i)
```

```
return flist
```

```
functions = make_functions()
for f in functions:
    f()
```

How can we make it print out what we apparently want?

### Problem #189 <a id="p189"></a>

**Easy** · Google · 2020-06-23

#### Problem

Given an array of elements, return the length of the longest subarray where all its elements are distinct.

For example, given the array `[5, 1, 3, 5, 2, 3, 4, 1]`, return 5 as the longest subarray of distinct elements is `[5, 2, 3, 4, 1]`.

### Problem #190 <a id="p190"></a>

**Medium** · Facebook · 2020-06-24

#### Problem

Given a circular array, compute its maximum subarray sum in `O(n)` time. A subarray can be empty, and in this case the sum is 0.

For example, given `[8, -1, 3, 4]`, return 15 as we choose the numbers 3, 4, and 8 where the 8 is obtained from wrapping around.

Given `[-4, 5, 1, 0]`, return 6 as we choose the numbers 5 and 1.

### Problem #191 <a id="p191"></a>

**Easy** · Stripe · 2020-06-25

#### Problem

Given a collection of intervals, find the minimum number of intervals you need to remove to make the rest of the intervals non-overlapping.

Intervals can `"touch"`, such as `[0, 1]` and `[1, 2]`, but they won't be considered overlapping.

For example, given the intervals (7, 9), (2, 4), (5, 8), return 1 as the last interval can be removed and the first two won't overlap.

The intervals are not necessarily sorted in any order.

### Problem #192 <a id="p192"></a>

**Medium** · Google · 2020-06-26

#### Problem

You are given an array of nonnegative integers. Let's say you start at the beginning of the array and are trying to advance to the end. You can advance at most, the number of steps that you're currently on. Determine whether you can get to the end of the array.

For example, given the array `[1, 3, 1, 2, 0, 1]`, we can go from indices 0 -> 1 -> 3 -> 5, so return true.

Given the array `[1, 2, 1, 0, 0]`, we can't reach the end, so return false.

### Problem #193 <a id="p193"></a>

**Hard** · Affirm · 2020-06-27

#### Problem

Given a array of numbers representing the stock prices of a company in chronological order, write a function that calculates the maximum profit you could have made from buying and selling that stock. You're also given a number fee that represents a transaction fee for each buy and sell transaction.

You must buy before you can sell the stock, but you can make as many transactions as you like.

For example, given `[1, 3, 2, 8, 4, 10]` and fee = 2, you should return 9, since you could buy the stock at 1 dollar, and sell at 8 dollars, and then buy it at 4 dollars and sell it at 10 dollars. Since we did two transactions, there is a 4 dollar fee, so we have 7 + 6 = 13 profit minus 4 dollars of fees.

### Problem #194 <a id="p194"></a>

**Easy** · Facebook · 2020-06-28

#### Problem

Suppose you are given two lists of n points, one list p1, p2, ..., pn on the line y = 0 and the other list q1, q2, ..., qn on the line y = 1. Imagine a set of n line segments connecting each point pi to qi. Write an algorithm to determine how many pairs of the line segments intersect.

### Problem #195 <a id="p195"></a>

**Hard** · Google · 2020-06-29

#### Problem

Let A be an N by M matrix in which every row and every column is sorted.

Given i1, j1, i2, and j2, compute the number of elements of M smaller than `M[i1, j1]` and larger than `M[i2, j2]`.

For example, given the following matrix:

```
[[1, 3, 7, 10, 15, 20],
 [2, 6, 9, 14, 22, 25],
 [3, 8, 10, 15, 25, 30],
 [10, 11, 12, 23, 30, 35],
 [20, 25, 30, 35, 40, 45]]
```

And i1 = 1, j1 = 1, i2 = 3, j2 = 3, return 15 as there are 15 numbers in the matrix smaller than 6 or greater than 23.

### Problem #196 <a id="p196"></a>

**Easy** · Apple · 2020-06-30

#### Problem

Given the root of a binary tree, find the most frequent subtree sum. The subtree sum of a node is the sum of all values under a node, including the node itself.

For example, given the following tree:

```
  5
 / \
2  -5
```

Return 2 as it occurs twice: once as the left leaf, and once as the sum of 2 + 5 - 5.

### Problem #197 <a id="p197"></a>

**Easy** · Amazon · 2020-07-01

#### Problem

Given an array and a number k that's smaller than the length of the array, rotate the array to the right k elements in-place.

### Problem #198 <a id="p198"></a>

**Medium** · Google · 2020-07-02

#### Problem

Given a set of distinct positive integers, find the largest subset such that every pair of elements in the subset (i, j) satisfies either i % j = 0 or j % i = 0.

For example, given the set `[3, 5, 10, 20, 21]`, you should return `[5, 10, 20]`. Given `[1, 3, 6, 24]`, return `[1, 3, 6, 24]`.

### Problem #199 <a id="p199"></a>

**Hard** · Facebook · 2020-07-03

#### Problem

Given a string of parentheses, find the balanced string that can be produced from it using the minimum number of insertions and deletions. If there are multiple solutions, return any of them.

For example, given `"(()"`, you could return `"(())"`. Given `"))()("`, you could return `"()()()()"`.

### Problem #200 <a id="p200"></a>

**Hard** · Microsoft · 2020-07-04

#### Problem

Let X be a set of n intervals on the real line. We say that a set of points P `"stabs"` X if every interval in X contains at least one point in P. Compute the smallest set of points that stabs X.

For example, given the intervals `[(1, 4), (4, 5), (7, 9), (9, 12)]`, you should return `[4, 9]`.

### Problem #201 <a id="p201"></a>

**Easy** · Google · 2020-07-05

#### Problem

You are given an array of arrays of integers, where each array corresponds to a row in a triangle of numbers. For example, `[[1], [2, 3], [1, 5, 1]]` represents the triangle:

```
  1
 2 3
1 5 1
```

We define a path in the triangle to start at the top and go down one row at a time to an adjacent value, eventually ending with an entry on the bottom row. For example, 1 -> 3 -> 5. The weight of the path is the sum of the entries.

Write a program that returns the weight of the maximum weight path.

### Problem #202 <a id="p202"></a>

**Easy** · Palantir · 2020-07-06

#### Problem

Write a program that checks whether an integer is a palindrome. For example, 121 is a palindrome, as well as 888. 678 is not a palindrome. Do not convert the integer into a string.

### Problem #203 <a id="p203"></a>

**Medium** · Uber · 2020-07-07

#### Problem

Suppose an array sorted in ascending order is rotated at some pivot unknown to you beforehand. Find the minimum element in `O(log N)` time. You may assume the array does not contain duplicates.

For example, given `[5, 7, 10, 3, 4]`, return 3.

### Problem #204 <a id="p204"></a>

**Easy** · Amazon · 2020-07-08

#### Problem

Given a complete binary tree, count the number of nodes in faster than `O(n)` time. Recall that a complete binary tree has every level filled except the last, and the nodes in the last level are filled starting from the left.

### Problem #205 <a id="p205"></a>

**Easy** · IBM · 2020-07-09

#### Problem

Given an integer, find the next permutation of it in absolute order. For example, given 48975, the next permutation would be 49578.

### Problem #206 <a id="p206"></a>

**Easy** · Twitter · 2020-07-10

#### Problem

A permutation can be specified by an array P, where `P[i]` represents the location of the element at i in the permutation. For example, `[2, 1, 0]` represents the permutation where elements at the index 0 and 2 are swapped.

Given an array and a permutation, apply the permutation to the array. For example, given the array `["a", "b", "c"]` and the permutation `[2, 1, 0]`, return `["c", "b", "a"]`.

### Problem #207 <a id="p207"></a>

**Medium** · Dropbox · 2020-07-11

#### Problem

Given an undirected graph G, check whether it is bipartite. Recall that a graph is bipartite if its vertices can be divided into two independent sets, U and V, such that no edge connects vertices of the same set.

### Problem #208 <a id="p208"></a>

**Medium** · LinkedIn · 2020-07-12

#### Problem

Given a linked list of numbers and a pivot k, partition the linked list so that all nodes less than k come before nodes greater than or equal to k.

For example, given the linked list 5 -> 1 -> 8 -> 0 -> 3 and k = 3, the solution could be 1 -> 0 -> 5 -> 8 -> 3.

### Problem #209 <a id="p209"></a>

**Hard** · YouTube · 2020-07-13

#### Problem

Write a program that computes the length of the longest common subsequence of three given strings. For example, given `"epidemiologist"`, `"refrigeration"`, and `"supercalifragilisticexpialodocious"`, it should return 5, since the longest common subsequence is `"eieio"`.

### Problem #210 <a id="p210"></a>

**Easy** · Apple · 2020-07-14

#### Problem

A Collatz sequence in mathematics can be defined as follows. Starting with any positive integer:

- if n is even, the next number in the sequence is n / 2
- if n is odd, the next number in the sequence is 3n + 1

It is conjectured that every such sequence eventually reaches the number 1. Test this conjecture.

Bonus: What input n \<= 1000000 gives the longest sequence?

### Problem #211 <a id="p211"></a>

**Medium** · Microsoft · 2020-07-15

#### Problem

Given a string and a pattern, find the starting indices of all occurrences of the pattern in the string. For example, given the string `"abracadabra"` and the pattern `"abr"`, you should return `[0, 7]`.

### Problem #212 <a id="p212"></a>

**Easy** · Dropbox · 2020-07-16

#### Problem

Spreadsheets often use this alphabetical encoding for its columns: `"A"`, `"B"`, `"C"`, ..., `"AA"`, `"AB"`, ..., `"ZZ"`, `"AAA"`, `"AAB"`, ....

Given a column number, return its alphabetical column id. For example, given 1, return `"A"`. Given 27, return `"AA"`.

### Problem #213 <a id="p213"></a>

**Medium** · Snapchat · 2020-07-17

#### Problem

Given a string of digits, generate all possible valid IP address combinations.

IP addresses must follow the format A.B.C.D, where A, B, C, and D are numbers between 0 and 255. Zero-prefixed numbers, such as 01 and 065, are not allowed, except for 0 itself.

For example, given `"2542540123"`, you should return `['254.25.40.123', '254.254.0.123']`.

### Problem #214 <a id="p214"></a>

**Easy** · Stripe · 2020-07-18

#### Problem

Given an integer n, return the length of the longest consecutive run of 1s in its binary representation.

For example, given 156, you should return 3.

### Problem #215 <a id="p215"></a>

**Medium** · Yelp · 2020-07-19

#### Problem

The horizontal distance of a binary tree node describes how far left or right the node will be when the tree is printed out.

More rigorously, we can define it as follows:

- The horizontal distance of the root is 0.
- The horizontal distance of a left child is `hd(parent)` - 1.
- The horizontal distance of a right child is `hd(parent)` + 1.

For example, for the following tree, `hd(1)` = -2, and `hd(6)` = 0.

```
           5
        /     \
      3         7
    /  \      /   \
  1     4    6     9
 /                /
0                8
```

The bottom view of a tree, then, consists of the lowest node at each horizontal distance. If there are two nodes at the same depth and horizontal distance, either is acceptable.

For this tree, for example, the bottom view could be `[0, 1, 3, 6, 8, 9]`.

Given the root to a binary tree, return its bottom view.

### Problem #216 <a id="p216"></a>

**Medium** · Facebook · 2020-07-20

#### Problem

Given a number in Roman numeral format, convert it to decimal.

The values of Roman numerals are as follows:

```
{
    'M': 1000,
    'D': 500,
    'C': 100,
    'L': 50,
    'X': 10,
    'V': 5,
    'I': 1
}
```

In addition, note that the Roman numeral system uses subtractive notation for numbers such as IV and XL.

For the input XIV, for instance, you should return 14.

### Problem #217 <a id="p217"></a>

**Hard** · Oracle · 2020-07-21

#### Problem

We say a number is sparse if there are no adjacent ones in its binary representation. For example, 21 (10101) is sparse, but 22 (10110) is not. For a given input N, find the smallest sparse number greater than or equal to N.

Do this in faster than `O(N log N)` time.

### Problem #218 <a id="p218"></a>

**Medium** · Yahoo · 2020-07-22

#### Problem

Write an algorithm that computes the reversal of a directed graph. For example, if a graph consists of A -> B -> C, it should become A \<- B \<- C.

### Problem #219 <a id="p219"></a>

**Hard** · Salesforce · 2020-07-23

#### Problem

Connect 4 is a game where opponents take turns dropping red or black discs into a 7 x 6 vertically suspended grid. The game ends either when one player creates a line of four consecutive discs of their color (horizontally, vertically, or diagonally), or when there are no more spots left in the grid.

Design and implement Connect 4.

### Problem #220 <a id="p220"></a>

**Medium** · Square · 2020-07-24

#### Problem

In front of you is a row of N coins, with values v1, v1, ..., vn.

You are asked to play the following game. You and an opponent take turns choosing either the first or last coin from the row, removing it from the row, and receiving the value of the coin.

Write a program that returns the maximum amount of money you can win with certainty, if you move first, assuming your opponent plays optimally.

### Problem #221 <a id="p221"></a>

**Easy** · Zillow · 2020-07-25

#### Problem

Let's define a `"sevenish"` number to be one which is either a power of 7, or the sum of unique powers of 7. The first few sevenish numbers are 1, 7, 8, 49, and so on. Create an algorithm to find the nth sevenish number.

### Problem #222 <a id="p222"></a>

**Medium** · Quora · 2020-07-26

#### Problem

Given an absolute pathname that may have . or .. as part of it, return the shortest standardized path.

For example, given `"/usr/bin/../bin/./scripts/../"`, return `"/usr/bin/"`.

### Problem #223 <a id="p223"></a>

**Hard** · Palantir · 2020-07-27

#### Problem

Typically, an implementation of in-order traversal of a binary tree has `O(h)` space complexity, where h is the height of the tree. Write a program to compute the in-order traversal of a binary tree using `O(1)` space.

### Problem #224 <a id="p224"></a>

**Easy** · Amazon · 2020-07-28

#### Problem

Given a sorted array, find the smallest positive integer that is not the sum of a subset of the array.

For example, for the input `[1, 2, 3, 10]`, you should return 7.

Do this in `O(N)` time.

### Problem #225 <a id="p225"></a>

**Easy** · Bloomberg · 2020-07-29

#### Problem

There are N prisoners standing in a circle, waiting to be executed. The executions are carried out starting with the kth person, and removing every successive kth person going clockwise until there is no one left.

Given N and k, write an algorithm to determine where a prisoner should stand in order to be the last survivor.

For example, if N = 5 and k = 2, the order of executions would be `[2, 4, 1, 5, 3]`, so you should return 3.

Bonus: Find an `O(log N)` solution if k = 2.

### Problem #226 <a id="p226"></a>

**Hard** · Airbnb · 2020-07-30

#### Problem

You come across a dictionary of sorted words in a language you've never seen before. Write a program that returns the correct order of letters in this language.

For example, given `['xww', 'wxyz', 'wxyw', 'ywx', 'ywz']`, you should return `['x', 'z', 'w', 'y']`.

### Problem #227 <a id="p227"></a>

**Easy** · Facebook · 2020-07-31

#### Problem

Boggle is a game played on a 4 x 4 grid of letters. The goal is to find as many words as possible that can be formed by a sequence of adjacent letters in the grid, using each cell at most once. Given a game board and a dictionary of valid words, implement a Boggle solver.

### Problem #228 <a id="p228"></a>

**Medium** · Twitter · 2020-08-01

#### Problem

Given a list of numbers, create an algorithm that arranges them in order to form the largest possible integer. For example, given `[10, 7, 76, 415]`, you should return 77641510.

### Problem #229 <a id="p229"></a>

**Medium** · Flipkart · 2020-08-02

#### Problem

Snakes and Ladders is a game played on a 10 x 10 board, the goal of which is get from square 1 to square 100. On each turn players will roll a six-sided die and move forward a number of spaces equal to the result. If they land on a square that represents a snake or ladder, they will be transported ahead or behind, respectively, to a new square.

Find the smallest number of turns it takes to play snakes and ladders.

For convenience, here are the squares representing snakes and ladders, and their outcomes:

```
snakes = {16: 6, 48: 26, 49: 11, 56: 53, 62: 19, 64: 60, 87: 24, 93: 73, 95: 75, 98: 78}
ladders = {1: 38, 4: 14, 9: 31, 21: 42, 28: 84, 36: 44, 51: 67, 71: 91, 80: 100}
```

### Problem #230 <a id="p230"></a>

**Medium** · Goldman Sachs · 2020-08-03

#### Problem

You are given N identical eggs and access to a building with k floors. Your task is to find the lowest floor that will cause an egg to break, if dropped from that floor. Once an egg breaks, it cannot be dropped again. If an egg breaks when dropped from the xth floor, you can assume it will also break when dropped from any floor greater than x.

Write an algorithm that finds the minimum number of trial drops it will take, in the worst case, to identify this floor.

For example, if N = 1 and k = 5, we will need to try dropping the egg at every floor, beginning with the first, until we reach the fifth floor, so our solution will be 5.

### Problem #231 <a id="p231"></a>

**Easy** · IBM · 2020-08-04

#### Problem

Given a string with repeated characters, rearrange the string so that no two adjacent characters are the same. If this is not possible, return None.

For example, given `"aaabbc"`, you could return `"ababac"`. Given `"aaab"`, return None.

### Problem #232 <a id="p232"></a>

**Easy** · Google · 2020-08-05

#### Problem

Implement a PrefixMapSum class with the following methods:

- `insert(key: str, value: int)`: Set a given key's value in the map. If the key already exists, overwrite the value.
- `sum(prefix: str)`: Return the sum of all values of keys that begin with a given prefix.

For example, you should be able to run the following code:

```
mapsum.insert("columnar", 3)
assert mapsum.sum("col") == 3
```

```
mapsum.insert("column", 2)
assert mapsum.sum("col") == 5
```

### Problem #233 <a id="p233"></a>

**Easy** · Apple · 2020-08-06

#### Problem

Implement the function `fib(n)`, which returns the nth number in the Fibonacci sequence, using only `O(1)` space.

### Problem #234 <a id="p234"></a>

**Hard** · Microsoft · 2020-08-07

#### Problem

Recall that the minimum spanning tree is the subset of edges of a tree that connect all its vertices with the smallest possible total edge weight. Given an undirected graph with weighted edges, compute the maximum weight spanning tree.

### Problem #235 <a id="p235"></a>

**Hard** · Facebook · 2020-08-08

#### Problem

Given an array of numbers of length N, find both the minimum and maximum using less than 2 \* (N - 2) comparisons.

### Problem #236 <a id="p236"></a>

**Medium** · Nvidia · 2020-08-09

#### Problem

You are given a list of N points (x1, y1), (x2, y2), ..., (xN, yN) representing a polygon. You can assume these points are given in order; that is, you can construct the polygon by connecting point 1 to point 2, point 2 to point 3, and so on, finally looping around to connect point N to point 1.

Determine if a new point p lies inside this polygon. (If p is on the boundary of the polygon, you should return False).

### Problem #237 <a id="p237"></a>

**Easy** · Amazon · 2020-08-10

#### Problem

A tree is symmetric if its data and shape remain unchanged when it is reflected about the root node. The following tree is an example:

```
        4
      / | \
    3   5   3
  /           \
9              9
```

Given a k-ary tree, determine whether it is symmetric.

### Problem #238 <a id="p238"></a>

**Hard** · MIT · 2020-08-11

#### Problem

Blackjack is a two player card game whose rules are as follows:

- The player and then the dealer are each given two cards.
- The player can then `"hit"`, or ask for arbitrarily many additional cards, so long as their total does not exceed 21.
- The dealer must then hit if their total is 16 or lower, otherwise pass.
- Finally, the two compare totals, and the one with the greatest sum not exceeding 21 is the winner.

For this problem, cards values are counted as follows: each card between 2 and 10 counts as their face value, face cards count as 10, and aces count as 1.

Given perfect knowledge of the sequence of cards in the deck, implement a blackjack solver that maximizes the player's score (that is, wins minus losses).

### Problem #239 <a id="p239"></a>

**Medium** · Uber · 2020-08-12

#### Problem

One way to unlock an Android phone is through a pattern of swipes across a 1-9 keypad.

For a pattern to be valid, it must satisfy the following:

- All of its keys must be distinct.
- It must not connect two keys by jumping over a third key, unless that key has already been used.

For example, 4 - 2 - 1 - 7 is a valid pattern, whereas 2 - 1 - 7 is not.

Find the total number of valid unlock patterns of length N, where 1 \<= N \<= 9.

### Problem #240 <a id="p240"></a>

**Hard** · Spotify · 2020-08-13

#### Problem

There are N couples sitting in a row of length 2 \* N. They are currently ordered randomly, but would like to rearrange themselves so that each couple's partners can sit side by side.

What is the minimum number of swaps necessary for this to happen?

### Problem #241 <a id="p241"></a>

**Easy** · Palantir · 2020-08-14

#### Problem

In academia, the h-index is a metric used to calculate the impact of a researcher's papers. It is calculated as follows:

A researcher has index h if at least h of her N papers have h citations each. If there are multiple h satisfying this formula, the maximum is chosen.

For example, suppose N = 5, and the respective citations of each paper are `[4, 3, 0, 1, 5]`. Then the h-index would be 3, since the researcher has 3 papers with at least 3 citations.

Given a list of paper citations of a researcher, calculate their h-index.

### Problem #242 <a id="p242"></a>

**Hard** · Twitter · 2020-08-15

#### Problem

You are given an array of length 24, where each element represents the number of new subscribers during the corresponding hour. Implement a data structure that efficiently supports the following:

- `update(hour: int, value: int)`: Increment the element at index hour by value.
- `query(start: int, end: int)`: Retrieve the number of subscribers that have signed up between start and end (inclusive).

You can assume that all values get cleared at the end of the day, and that you will not be asked for start and end values that wrap around midnight.

### Problem #243 <a id="p243"></a>

**Medium** · Etsy · 2020-08-16

#### Problem

Given an array of numbers N and an integer k, your task is to split N into k partitions such that the maximum sum of any partition is minimized. Return this sum.

For example, given N = `[5, 1, 2, 7, 3, 4]` and k = 3, you should return 8, since the optimal partition is `[5, 1, 2]`, `[7]`, `[3, 4]`.

### Problem #244 <a id="p244"></a>

**Easy** · Square · 2020-08-17

#### Problem

The Sieve of Eratosthenes is an algorithm used to generate all prime numbers smaller than N. The method is to take increasingly larger prime numbers, and mark their multiples as composite.

For example, to find all primes less than 100, we would first mark `[4, 6, 8, ...]` (multiples of two), then `[6, 9, 12, ...]` (multiples of three), and so on. Once we have done this for all primes less than N, the unmarked numbers that remain will be prime.

Implement this algorithm.

Bonus: Create a generator that produces primes indefinitely (that is, without taking N as an input).

### Problem #245 <a id="p245"></a>

**Medium** · Yelp · 2020-08-18

#### Problem

You are given an array of integers, where each element represents the maximum number of steps that can be jumped going forward from that element. Write a function to return the minimum number of jumps you must take in order to get from the start to the end of the array.

For example, given `[6, 2, 4, 0, 5, 1, 1, 4, 2, 9]`, you should return 2, as the optimal solution involves jumping from 6 to 5, and then from 5 to 9.

### Problem #246 <a id="p246"></a>

**Medium** · Dropbox · 2020-08-19

#### Problem

Given a list of words, determine whether the words can be chained to form a circle. A word X can be placed in front of another word Y in a circle if the last character of X is same as the first character of Y.

For example, the words `['chair', 'height', 'racket', touch', 'tunic']` can form the following circle: chair --> racket --> touch --> height --> tunic --> chair.

### Problem #247 <a id="p247"></a>

**Easy** · PayPal · 2020-08-20

#### Problem

Given a binary tree, determine whether or not it is height-balanced. A height-balanced binary tree can be defined as one in which the heights of the two subtrees of any node never differ by more than one.

### Problem #248 <a id="p248"></a>

**Hard** · Nvidia · 2020-08-21

#### Problem

Find the maximum of two numbers without using any if-else statements, branching, or direct comparisons.

### Problem #249 <a id="p249"></a>

**Hard** · Salesforce · 2020-08-22

#### Problem

Given an array of integers, find the maximum XOR of any two elements.

### Problem #250 <a id="p250"></a>

**Medium** · Google · 2020-08-23

#### Problem

A cryptarithmetic puzzle is a mathematical game where the digits of some numbers are represented by letters. Each letter represents a unique digit.

For example, a puzzle of the form:

```
  SEND
+ MORE
--------
 MONEY
```

may have the solution:

`{'S': 9, 'E': 5, 'N': 6, 'D': 7, 'M': 1, 'O', 0, 'R': 8, 'Y': 2}`

Given a three-word puzzle like the one above, create an algorithm that finds a solution.

### Problem #251 <a id="p251"></a>

**Medium** · Amazon · 2020-08-24

#### Problem

Given an array of a million integers between zero and a billion, out of order, how can you efficiently sort it? Assume that you cannot store an array of a billion elements in memory.

### Problem #252 <a id="p252"></a>

**Easy** · Palantir · 2020-08-25

#### Problem

The ancient Egyptians used to express fractions as a sum of several terms where each numerator is one. For example, 4 / 13 can be represented as 1 / 4 + 1 / 18 + 1 / 468.

Create an algorithm to turn an ordinary fraction a / b, where a \< b, into an Egyptian fraction.

### Problem #253 <a id="p253"></a>

**Medium** · PayPal · 2020-08-26

#### Problem

Given a string and a number of lines k, print the string in zigzag form. In zigzag, characters are printed out diagonally from top left to bottom right until reaching the kth line, then back up to top right, and so on.

For example, given the sentence `"thisisazigzag"` and k = 4, you should print:

```
t     a     g
 h   s z   a
  i i   i z
   s     g
```

### Problem #254 <a id="p254"></a>

**Medium** · Yahoo · 2020-08-27

#### Problem

Recall that a full binary tree is one in which each node is either a leaf node, or has two children. Given a binary tree, convert it to a full one by removing nodes with only one child.

For example, given the following tree:

```
         0
      /     \
    1         2
  /            \
3                 4
  \             /   \
    5          6     7
```

You should convert it to:

```
     0
  /     \
5         4
        /   \
       6     7
```

### Problem #255 <a id="p255"></a>

**Easy** · Microsoft · 2020-08-28

#### Problem

The transitive closure of a graph is a measure of which vertices are reachable from other vertices. It can be represented as a matrix M, where `M[i][j]` == 1 if there is a path between vertices i and j, and otherwise 0.

For example, suppose we are given the following graph in adjacency list form:

```
graph = [
    [0, 1, 3],
    [1, 2],
    [2],
    [3]
]
```

The transitive closure of this graph would be:

```
[1, 1, 1, 1]
[0, 1, 1, 0]
[0, 0, 1, 0]
[0, 0, 0, 1]
```

Given a graph, find its transitive closure.

### Problem #256 <a id="p256"></a>

**Medium** · Fitbit · 2020-08-29

#### Problem

Given a linked list, rearrange the node values such that they appear in alternating low -> high -> low -> high ... form. For example, given 1 -> 2 -> 3 -> 4 -> 5, you should return 1 -> 3 -> 2 -> 5 -> 4.

### Problem #257 <a id="p257"></a>

**Easy** · WhatsApp · 2020-08-30

#### Problem

Given an array of integers out of order, determine the bounds of the smallest window that must be sorted in order for the entire array to be sorted. For example, given `[3, 7, 5, 6, 9]`, you should return (1, 3).

### Problem #258 <a id="p258"></a>

**Easy** · Morgan Stanley · 2020-08-31

#### Problem

In Ancient Greece, it was common to write text with the first line going left to right, the second line going right to left, and continuing to go back and forth. This style was called `"boustrophedon"`.

Given a binary tree, write an algorithm to print the nodes in boustrophedon order.

For example, given the following tree:

```
       1
    /     \
  2         3
 / \       / \
4   5     6   7
```

You should return `[1, 3, 2, 4, 5, 6, 7]`.

### Problem #259 <a id="p259"></a>

**Hard** · Two Sigma · 2020-09-01

#### Problem

Ghost is a two-person word game where players alternate appending letters to a word. The first person who spells out a word, or creates a prefix for which there is no possible continuation, loses. Here is a sample game:

- Player 1: g
- Player 2: h
- Player 1: o
- Player 2: s
- Player 1: t \[loses\]

Given a dictionary of words, determine the letters the first player should start with, such that with optimal play they cannot lose.

For example, if the dictionary is `["cat", "calf", "dog", "bear"]`, the only winning start letter would be b.

### Problem #260 <a id="p260"></a>

**Medium** · Pinterest · 2020-09-02

#### Problem

The sequence `[0, 1, ..., N]` has been jumbled, and the only clue you have for its order is an array representing whether each number is larger or smaller than the last. Given this information, reconstruct an array that is consistent with it. For example, given `[None, +, +, -, +]`, you could return `[1, 2, 3, 0, 4]`.

### Problem #261 <a id="p261"></a>

**Easy** · Amazon · 2020-09-03

#### Problem

Huffman coding is a method of encoding characters based on their frequency. Each letter is assigned a variable-length binary string, such as 0101 or 111110, where shorter lengths correspond to more common letters. To accomplish this, a binary tree is built such that the path from the root to any leaf uniquely maps to a character. When traversing the path, descending to a left child corresponds to a 0 in the prefix, while descending right corresponds to 1.

Here is an example tree (note that only the leaf nodes have letters):

```
        *
      /   \
    *       *
   / \     / \
  *   a   t   *
 /             \
c               s
```

With this encoding, cats would be represented as 0000110111.

Given a dictionary of character frequencies, build a Huffman tree, and use it to determine a mapping between characters and their encoded binary strings.

### Problem #262 <a id="p262"></a>

**Medium** · Mozilla · 2020-09-04

#### Problem

A bridge in a connected (undirected) graph is an edge that, if removed, causes the graph to become disconnected. Find all the bridges in a graph.

### Problem #263 <a id="p263"></a>

**Medium** · Nest · 2020-09-05

#### Problem

Create a basic sentence checker that takes in a stream of characters and determines whether they form valid sentences. If a sentence is valid, the program should print it out.

We can consider a sentence valid if it conforms to the following rules:

- The sentence must start with a capital letter, followed by a lowercase letter or a space.
- All other characters must be lowercase letters, separators (,,;,:) or terminal marks (.,?,!,‽).
- There must be a single space between each word.
- The sentence must end with a terminal mark immediately following a word.

### Problem #264 <a id="p264"></a>

**Hard** · LinkedIn · 2020-09-06

#### Problem

Given a set of characters C and an integer k, a De Bruijn sequence is a cyclic sequence in which every possible k-length string of characters in C occurs exactly once.

For example, suppose C = `{0, 1}` and k = 3. Then our sequence should contain the substrings `{'000', '001', '010', '011', '100', '101', '110', '111'}`, and one possible solution would be 00010111.

Create an algorithm that finds a De Bruijn sequence.

### Problem #265 <a id="p265"></a>

**Easy** · Atlassian · 2020-09-07

#### Problem

MegaCorp wants to give bonuses to its employees based on how many lines of codes they have written. They would like to give the smallest positive amount to each worker consistent with the constraint that if a developer has written more lines of code than their neighbor, they should receive more money.

Given an array representing a line of seats of employees at MegaCorp, determine how much each one should get paid.

For example, given `[10, 40, 200, 1000, 60, 30]`, you should return `[1, 2, 3, 4, 2, 1]`.

### Problem #266 <a id="p266"></a>

**Easy** · Pivotal · 2020-09-08

#### Problem

A step word is formed by taking a given word, adding a letter, and anagramming the result. For example, starting with the word `"APPLE"`, you can add an `"A"` and anagram to get `"APPEAL"`.

Given a dictionary of words and an input word, create a function that returns all valid step words.

### Problem #267 <a id="p267"></a>

**Hard** · Oracle · 2020-09-09

#### Problem

You are presented with an 8 by 8 matrix representing the positions of pieces on a chess board. The only pieces on the board are the black king and various white pieces. Given this matrix, determine whether the king is in check.

For details on how each piece moves, see here.

For example, given the following matrix:

```
...K....
........
.B......
......P.
.......R
..N.....
........
.....Q..
```

You should return True, since the bishop is attacking the king diagonally.

### Problem #268 <a id="p268"></a>

**Medium** · Indeed · 2020-09-10

#### Problem

Given a 32-bit positive integer N, determine whether it is a power of four in faster than `O(log N)` time.

### Problem #269 <a id="p269"></a>

**Easy** · Microsoft · 2020-09-11

#### Problem

You are given an string representing the initial conditions of some dominoes. Each element can take one of three values:

- L, meaning the domino has just been pushed to the left,
- R, meaning the domino has just been pushed to the right, or
- ., meaning the domino is standing still.

Determine the orientation of each tile when the dominoes stop falling. Note that if a domino receives a force from the left and right side simultaneously, it will remain upright.

For example, given the string .L.R....L, you should return LL.RRRLLL.

Given the string ..R...L.L, you should return ..RR.LLLL.

### Problem #270 <a id="p270"></a>

**Medium** · Twitter · 2020-09-12

#### Problem

A network consists of nodes labeled 0 to N. You are given a list of edges (a, b, t), describing the time t it takes for a message to be sent from node a to node b. Whenever a node receives a message, it immediately passes the message on to a neighboring node, if possible.

Assuming all nodes are connected, determine how long it will take for every node to receive a message that begins at node 0.

For example, given N = 5, and the following edges:

```
edges = [
    (0, 1, 5),
    (0, 2, 3),
    (0, 5, 4),
    (1, 3, 8),
    (2, 3, 1),
    (3, 5, 10),
    (3, 4, 5)
]
```

You should return 9, because propagating the message from 0 -> 2 -> 3 -> 4 will take that much time.

### Problem #271 <a id="p271"></a>

**Hard** · Netflix · 2020-09-13

#### Problem

Given a sorted list of integers of length N, determine if an element x is in the list without performing any multiplication, division, or bit-shift operations.

Do this in `O(log N)` time.

### Problem #272 <a id="p272"></a>

**Medium** · Spotify · 2020-09-14

#### Problem

Write a function, `throw_dice(N, faces, total)`, that determines how many ways it is possible to throw N dice with some number of faces each to get a specific total.

For example, `throw_dice(3, 6, 7)` should equal 15.

### Problem #273 <a id="p273"></a>

**Easy** · Apple · 2020-09-15

#### Problem

A fixed point in an array is an element whose value is equal to its index. Given a sorted array of distinct elements, return a fixed point, if one exists. Otherwise, return False.

For example, given `[-6, 0, 2, 40]`, you should return 2. Given `[1, 5, 7, 8]`, you should return False.

### Problem #274 <a id="p274"></a>

**Hard** · Facebook · 2020-09-16

#### Problem

Given a string consisting of parentheses, single digits, and positive and negative signs, convert the string into a mathematical expression to obtain the answer.

Don't use eval or a similar built-in parser.

For example, given `'-1 + (2 + 3)'`, you should return 4.

### Problem #275 <a id="p275"></a>

**Medium** · Epic · 2020-09-17

#### Problem

The `"look and say"` sequence is defined as follows: beginning with the term 1, each subsequent term visually describes the digits appearing in the previous term. The first few terms are as follows:

```
1
11
21
1211
111221
```

As an example, the fourth term is 1211, since the third term consists of one 2 and one 1.

Given an integer N, print the Nth term of this sequence.

### Problem #276 <a id="p276"></a>

**Hard** · Dropbox · 2020-09-18

#### Problem

Implement an efficient string matching algorithm.

That is, given a string of length N and a pattern of length k, write a program that searches for the pattern in the string with less than `O(N * k)` worst-case time complexity.

If the pattern is found, return the start index of its location. If not, return False.

### Problem #277 <a id="p277"></a>

**Easy** · Google · 2020-09-19

#### Problem

UTF-8 is a character encoding that maps each symbol to one, two, three, or four bytes.

For example, the Euro sign, €, corresponds to the three bytes 11100010 10000010 10101100. The rules for mapping characters are as follows:

- For a single-byte character, the first bit must be zero.
- For an n-byte character, the first byte starts with n ones and a zero. The other n - 1 bytes all start with 10.

Visually, this can be represented as follows.

```
 Bytes   |           Byte format
-----------------------------------------------
   1     | 0xxxxxxx
   2     | 110xxxxx 10xxxxxx
   3     | 1110xxxx 10xxxxxx 10xxxxxx
   4     | 11110xxx 10xxxxxx 10xxxxxx 10xxxxxx
```

Write a program that takes in an array of integers representing byte values, and returns whether it is a valid UTF-8 encoding.

### Problem #278 <a id="p278"></a>

**Easy** · Amazon · 2020-09-20

#### Problem

Given an integer N, construct all possible binary search trees with N nodes.

### Problem #279 <a id="p279"></a>

**Easy** · Twitter · 2020-09-21

#### Problem

A classroom consists of N students, whose friendships can be represented in an adjacency list. For example, the following descibes a situation where 0 is friends with 1 and 2, 3 is friends with 6, and so on.

```
{0: [1, 2],
 1: [0, 5],
 2: [0],
 3: [6],
 4: [],
 5: [1],
 6: [3]}
```

Each student can be placed in a friend group, which can be defined as the transitive closure of that student's friendship relations. In other words, this is the smallest set such that no student in the group has any friends outside this group. For the example above, the friend groups would be `{0, 1, 2, 5}`, `{3, 6}`, `{4}`.

Given a friendship list such as the one above, determine the number of friend groups in the class.

### Problem #280 <a id="p280"></a>

**Easy** · Pandora · 2020-09-22

#### Problem

Given an undirected graph, determine if it contains a cycle.

### Problem #281 <a id="p281"></a>

**Medium** · LinkedIn · 2020-09-23

#### Problem

A wall consists of several rows of bricks of various integer lengths and uniform height. Your goal is to find a vertical line going from the top to the bottom of the wall that cuts through the fewest number of bricks. If the line goes through the edge between two bricks, this does not count as a cut.

For example, suppose the input is as follows, where values in each row represent the lengths of bricks in that row:

```
[[3, 5, 1, 1],
 [2, 3, 3, 2],
 [5, 5],
 [4, 4, 2],
 [1, 3, 3, 3],
 [1, 1, 6, 1, 1]]
```

The best we can we do here is to draw a line after the eighth brick, which will only require cutting through the bricks in the third and fifth row.

Given an input consisting of brick lengths for each row such as the one above, return the fewest number of bricks that must be cut to create a vertical line.

### Problem #282 <a id="p282"></a>

**Easy** · Netflix · 2020-09-24

#### Problem

Given an array of integers, determine whether it contains a Pythagorean triplet. Recall that a Pythogorean triplet (a, b, c) is defined by the equation a2+ b2= c2.

### Problem #283 <a id="p283"></a>

**Easy** · Google · 2020-09-25

#### Problem

A regular number in mathematics is defined as one which evenly divides some power of 60. Equivalently, we can say that a regular number is one whose only prime divisors are 2, 3, and 5.

These numbers have had many applications, from helping ancient Babylonians keep time to tuning instruments according to the diatonic scale.

Given an integer N, write a program that returns, in order, the first N regular numbers.

### Problem #284 <a id="p284"></a>

**Medium** · Yext · 2020-09-26

#### Problem

Two nodes in a binary tree can be called cousins if they are on the same level of the tree but have different parents. For example, in the following diagram 4 and 6 are cousins.

```
    1
   / \
  2   3
 / \   \
4   5   6
```

Given a binary tree and a particular node, find all cousins of that node.

### Problem #285 <a id="p285"></a>

**Medium** · Mailchimp · 2020-09-27

#### Problem

You are given an array representing the heights of neighboring buildings on a city street, from east to west. The city assessor would like you to write an algorithm that returns how many of these buildings have a view of the setting sun, in order to properly value the street.

For example, given the array `[3, 7, 8, 3, 6, 1]`, you should return 3, since the top floors of the buildings with heights 8, 6, and 1 all have an unobstructed view to the west.

Can you do this using just one forward pass through the array?

### Problem #286 <a id="p286"></a>

**Hard** · VMware · 2020-09-28

#### Problem

The skyline of a city is composed of several buildings of various widths and heights, possibly overlapping one another when viewed from a distance. We can represent the buildings using an array of (left, right, height) tuples, which tell us where on an imaginary x-axis a building begins and ends, and how tall it is. The skyline itself can be described by a list of (x, height) tuples, giving the locations at which the height visible to a distant observer changes, and each new height.

Given an array of buildings as described above, create a function that returns the skyline.

For example, suppose the input consists of the buildings `[(0, 15, 3), (4, 11, 5), (19, 23, 4)]`. In aggregate, these buildings would create a skyline that looks like the one below.

### Problem #287 <a id="p287"></a>

**Medium** · Quora · 2020-09-29

#### Problem

You are given a list of (website, user) pairs that represent users visiting websites. Come up with a program that identifies the top k pairs of websites with the greatest similarity.

For example, suppose k = 1, and the list of tuples is:

```
[('a', 1), ('a', 3), ('a', 5),
 ('b', 2), ('b', 6),
 ('c', 1), ('c', 2), ('c', 3), ('c', 4), ('c', 5)
 ('d', 4), ('d', 5), ('d', 6), ('d', 7),
 ('e', 1), ('e', 3), ('e': 5), ('e', 6)]
```

Then a reasonable similarity metric would most likely conclude that a and e are the most similar, so your program should return `[('a', 'e')]`.

### Problem #288 <a id="p288"></a>

**Medium** · Salesforce · 2020-09-30

#### Problem

The number 6174 is known as Kaprekar's contant, after the mathematician who discovered an associated property: for all four-digit numbers with at least two distinct digits, repeatedly applying a simple procedure eventually results in this value. The procedure is as follows:

- For a given input x, create two new numbers that consist of the digits in x in ascending and descending order.
- Subtract the smaller number from the larger number.

For example, this algorithm terminates in three steps when starting from 1234:

- 4321 - 1234 = 3087
- 8730 - 0378 = 8352
- 8532 - 2358 = 6174

Write a function that returns how many steps this will take for a given input N.

### Problem #289 <a id="p289"></a>

**Hard** · Google · 2020-10-01

#### Problem

The game of Nim is played as follows. Starting with three heaps, each containing a variable number of items, two players take turns removing one or more items from a single pile. The player who eventually is forced to take the last stone loses. For example, if the initial heap sizes are 3, 4, and 5, a game could be played as shown below:

```
  A  |  B  |  C
-----------------
  3  |  4  |  5
  3  |  1  |  3
  3  |  1  |  3
  0  |  1  |  3
  0  |  1  |  0
  0  |  0  |  0
```

In other words, to start, the first player takes three items from pile B. The second player responds by removing two stones from pile C. The game continues in this way until player one takes last stone and loses.

Given a list of non-zero starting values `[a, b, c]`, and assuming optimal play, determine whether the first player has a forced win.

### Problem #290 <a id="p290"></a>

**Easy** · Facebook · 2020-10-02

#### Problem

On a mysterious island there are creatures known as Quxes which come in three colors: red, green, and blue. One power of the Qux is that if two of them are standing next to each other, they can transform into a single creature of the third color.

Given N Quxes standing in a line, determine the smallest number of them remaining after any possible sequence of such transformations.

For example, given the input `['R', 'G', 'B', 'G', 'B']`, it is possible to end up with a single Qux through the following steps:

```
        Arrangement       |   Change
----------------------------------------
['R', 'G', 'B', 'G', 'B'] | (R, G) -> B
['B', 'B', 'G', 'B']      | (B, G) -> R
['B', 'R', 'B']           | (R, B) -> G
['B', 'G']                | (B, G) -> R
['R']                     |
```

### Problem #291 <a id="p291"></a>

**Medium** · Glassdoor · 2020-10-03

#### Problem

An imminent hurricane threatens the coastal town of Codeville. If at most two people can fit in a rescue boat, and the maximum weight limit for a given boat is k, determine how many boats will be needed to save everyone.

For example, given a population with weights `[100, 200, 150, 80]` and a boat limit of 200, the smallest number of boats required will be three.

### Problem #292 <a id="p292"></a>

**Hard** · Twitter · 2020-10-04

#### Problem

A teacher must divide a class of students into two teams to play dodgeball. Unfortunately, not all the kids get along, and several refuse to be put on the same team as that of their enemies.

Given an adjacency list of students and their enemies, write an algorithm that finds a satisfactory pair of teams, or returns False if none exists.

For example, given the following enemy graph you should return the teams `{0, 1, 4, 5}` and `{2, 3}`.

```
students = {
    0: [3],
    1: [2],
    2: [1, 4],
    3: [0, 4, 5],
    4: [2, 3],
    5: [3]
}
```

On the other hand, given the input below, you should return False.

```
students = {
    0: [3],
    1: [2],
    2: [1, 3, 4],
    3: [0, 2, 4, 5],
    4: [2, 3],
    5: [3]
}
```

### Problem #293 <a id="p293"></a>

**Hard** · Uber · 2020-10-05

#### Problem

You have N stones in a row, and would like to create from them a pyramid. This pyramid should be constructed such that the height of each stone increases by one until reaching the tallest stone, after which the heights decrease by one. In addition, the start and end stones of the pyramid should each be one stone high.

You can change the height of any stone by paying a cost of 1 unit to lower its height by 1, as many times as necessary. Given this information, determine the lowest cost method to produce this pyramid.

For example, given the stones `[1, 1, 3, 3, 2, 1]`, the optimal solution is to pay 2 to create `[0, 1, 2, 3, 2, 1]`.

### Problem #294 <a id="p294"></a>

**Medium** · Square · 2020-10-06

#### Problem

A competitive runner would like to create a route that starts and ends at his house, with the condition that the route goes entirely uphill at first, and then entirely downhill.

Given a dictionary of places of the form `{location: elevation}`, and a dictionary mapping paths between some of these locations to their corresponding distances, find the length of the shortest route satisfying the condition above. Assume the runner's home is location 0.

For example, suppose you are given the following input:

```
elevations = {0: 5, 1: 25, 2: 15, 3: 20, 4: 10}
paths = {
    (0, 1): 10,
    (0, 2): 8,
    (0, 3): 15,
    (1, 3): 12,
    (2, 4): 10,
    (3, 4): 5,
    (3, 0): 17,
    (4, 0): 10
}
```

In this case, the shortest valid path would be 0 -> 2 -> 4 -> 0, with a distance of 28.

### Problem #295 <a id="p295"></a>

**Medium** · Stitch Fix · 2020-10-07

#### Problem

Pascal's triangle is a triangular array of integers constructed with the following formula:

- The first row consists of the number 1.
- For each subsequent row, each element is the sum of the numbers directly above it, on either side.

For example, here are the first few rows:

```
    1
   1 1
  1 2 1
 1 3 3 1
1 4 6 4 1
```

Given an input k, return the kth row of Pascal's triangle.

Bonus: Can you do this using only `O(k)` space?

### Problem #296 <a id="p296"></a>

**Hard** · Etsy · 2020-10-08

#### Problem

Given a sorted array, convert it into a height-balanced binary search tree.

### Problem #297 <a id="p297"></a>

**Medium** · Amazon · 2020-10-09

#### Problem

At a popular bar, each customer has a set of favorite drinks, and will happily accept any drink among this set. For example, in the following situation, customer 0 will be satisfied with drinks 0, 1, 3, or 6.

```
preferences = {
    0: [0, 1, 3, 6],
    1: [1, 4, 7],
    2: [2, 4, 7, 5],
    3: [3, 2, 5],
    4: [5, 8]
}
```

A lazy bartender working at this bar is trying to reduce his effort by limiting the drink recipes he must memorize. Given a dictionary input such as the one above, return the fewest number of drinks he must learn in order to satisfy all customers.

For the input above, the answer would be 2, as drinks 1 and 5 will satisfy everyone.

### Problem #298 <a id="p298"></a>

**Easy** · Google · 2020-10-10

#### Problem

A girl is walking along an apple orchard with a bag in each hand. She likes to pick apples from each tree as she goes along, but is meticulous about not putting different kinds of apples in the same bag.

Given an input describing the types of apples she will pass on her path, in order, determine the length of the longest portion of her path that consists of just two types of apple trees.

For example, given the input `[2, 1, 2, 3, 3, 1, 3, 5]`, the longest portion will involve types 1 and 3, with a length of four.

### Problem #299 <a id="p299"></a>

**Medium** · Samsung · 2020-10-11

#### Problem

A group of houses is connected to the main water plant by means of a set of pipes. A house can either be connected by a set of pipes extending directly to the plant, or indirectly by a pipe to a nearby house which is otherwise connected.

For example, here is a possible configuration, where A, B, and C are houses, and arrows represent pipes:

A \<--> B \<--> C \<--> plant

Each pipe has an associated cost, which the utility company would like to minimize. Given an undirected graph of pipe connections, return the lowest cost configuration of pipes such that each house has access to water.

In the following setup, for example, we can remove all but the pipes from plant to A, plant to B, and B to C, for a total cost of 16.

```
pipes = {
    'plant': {'A': 1, 'B': 5, 'C': 20},
    'A': {'C': 15},
    'B': {'C': 10},
    'C': {}
}
```

### Problem #300 <a id="p300"></a>

**Easy** · Uber · 2020-10-12

#### Problem

On election day, a voting machine writes data in the form (`voter_id`, `candidate_id`) to a text file. Write a program that reads this file as a stream and returns the top 3 candidates at any given time. If you find a voter voting more than once, report this as fraud.

### Problem #301 <a id="p301"></a>

**Medium** · Triplebyte · 2020-10-13

#### Problem

Implement a data structure which carries out the following operations without resizing the underlying array:

- `add(value)`: Add a value to the set of values.
- `check(value)`: Check whether a value is in the set.

The check method may return occasional false positives (in other words, incorrectly identifying an element as part of the set), but should always correctly identify a true element.

### Problem #302 <a id="p302"></a>

**Medium** · Uber · 2020-10-14

#### Problem

You are given a 2-d matrix where each cell consists of either /, \\, or an empty space. Write an algorithm that determines into how many regions the slashes divide the space.

For example, suppose the input for a three-by-six grid is the following:

```
\    /
 \  /
  \/
```

Considering the edges of the matrix as boundaries, this divides the grid into three triangles, so you should return 3.

### Problem #303 <a id="p303"></a>

**Easy** · Microsoft · 2020-10-15

#### Problem

Given a clock time in hh:mm format, determine, to the nearest degree, the angle between the hour and the minute hands.

Bonus: When, during the course of a day, will the angle be zero?

### Problem #304 <a id="p304"></a>

**Hard** · Two Sigma · 2020-10-16

#### Problem

A knight is placed on a given square on an 8 x 8 chessboard. It is then moved randomly several times, where each move is a standard knight move. If the knight jumps off the board at any point, however, it is not allowed to jump back on.

After k moves, what is the probability that the knight remains on the board?

### Problem #305 <a id="p305"></a>

**Easy** · Amazon · 2020-10-17

#### Problem

Given a linked list, remove all consecutive nodes that sum to zero. Print out the remaining nodes.

For example, suppose you are given the input 3 -> 4 -> -7 -> 5 -> -6 -> 6. In this case, you should first remove 3 -> 4 -> -7, then -6 -> 6, leaving only 5.

### Problem #306 <a id="p306"></a>

**Medium** · Palantir · 2020-10-18

#### Problem

You are given a list of N numbers, in which each number is located at most k places away from its sorted position. For example, if k = 1, a given element at index 4 might end up at indices 3, 4, or 5.

Come up with an algorithm that sorts this list in `O(N log k)` time.

### Problem #307 <a id="p307"></a>

**Easy** · Oracle · 2020-10-19

#### Problem

Given a binary search tree, find the floor and ceiling of a given integer. The floor is the highest element in the tree less than or equal to an integer, while the ceiling is the lowest element in the tree greater than or equal to an integer.

If either value does not exist, return None.

### Problem #308 <a id="p308"></a>

**Hard** · Quantcast · 2020-10-20

#### Problem

You are presented with an array representing a Boolean expression. The elements are of two kinds:

- T and F, representing the values True and False.
- &, |, and ^, representing the bitwise operators for AND, OR, and XOR.

Determine the number of ways to group the array elements using parentheses so that the entire expression evaluates to True.

For example, suppose the input is `['F', '|', 'T', '&', 'T']`. In this case, there are two acceptable groupings: (F | T) & T and F | (T & T).

### Problem #309 <a id="p309"></a>

**Medium** · Walmart Labs · 2020-10-21

#### Problem

There are M people sitting in a row of N seats, where M \< N. Your task is to redistribute people such that there are no gaps between any of them, while keeping overall movement to a minimum.

For example, suppose you are faced with an input of `[0, 1, 1, 0, 1, 0, 0, 0, 1]`, where 0 represents an empty seat and 1 represents a person. In this case, one solution would be to place the person on the right in the fourth seat. We can consider the cost of a solution to be the sum of the absolute distance each person must move, so that the cost here would be five.

Given an input such as the one above, return the lowest possible cost of moving people to remove all gaps.

### Problem #310 <a id="p310"></a>

**Easy** · Pivotal · 2020-10-22

#### Problem

Write an algorithm that finds the total number of set bits in all integers between 1 and N.

### Problem #311 <a id="p311"></a>

**Easy** · Sumo Logic · 2020-10-24

#### Problem

Given a array that's sorted but rotated at some unknown pivot, in which all elements are distinct, find a `"peak"` element in `O(log N)` time.

An element is considered a peak if it is greater than both its left and right neighbors. It is guaranteed that the first and last elements are lower than all others.

### Problem #312 <a id="p312"></a>

**Easy** · Wayfair · 2020-10-25

#### Problem

You are given a 2 x N board, and instructed to completely cover the board with the following shapes:

- Dominoes, or 2 x 1 rectangles.
- Trominoes, or L-shapes.

For example, if N = 4, here is one possible configuration, where A is a domino, and B and C are trominoes.

```
A B B C
A B C C
```

Given an integer N, determine in how many ways this task is possible.

### Problem #313 <a id="p313"></a>

**Hard** · Citrix · 2020-10-26

#### Problem

You are given a circular lock with three wheels, each of which display the numbers 0 through 9 in order. Each of these wheels rotate clockwise and counterclockwise.

In addition, the lock has a certain number of `"dead ends"`, meaning that if you turn the wheels to one of these combinations, the lock becomes stuck in that state and cannot be opened.

Let us consider a `"move"` to be a rotation of a single wheel by one digit, in either direction. Given a lock initially set to 000, a target combination, and a list of dead ends, write a function that returns the minimum number of moves required to reach the target state, or None if this is impossible.

### Problem #314 <a id="p314"></a>

**Medium** · Spotify · 2020-10-27

#### Problem

You are the technical director of WSPT radio, serving listeners nationwide. For simplicity's sake we can consider each listener to live along a horizontal line stretching from 0 (west) to 1000 (east).

Given a list of N listeners, and a list of M radio towers, each placed at various locations along this line, determine what the minimum broadcast range would have to be in order for each listener's home to be covered.

For example, suppose listeners = `[1, 5, 11, 20]`, and towers = `[4, 8, 15]`. In this case the minimum range would be 5, since that would be required for the tower at position 15 to reach the listener at position 20.

### Problem #315 <a id="p315"></a>

**Easy** · Google · 2020-10-28

#### Problem

In linear algebra, a Toeplitz matrix is one in which the elements on any given diagonal from top left to bottom right are identical.

Here is an example:

```
1 2 3 4 8
5 1 2 3 4
4 5 1 2 3
7 4 5 1 2
```

Write a program to determine whether a given input is a Toeplitz matrix.

### Problem #316 <a id="p316"></a>

**Medium** · Snapchat · 2020-10-29

#### Problem

You are given an array of length N, where each element i represents the number of ways we can produce i units of change. For example, `[1, 0, 1, 1, 2]` would indicate that there is only one way to make 0, 2, or 3 units, and two ways of making 4 units.

Given such an array, determine the denominations that must be in use. In the case above, for example, there must be coins with value 2, 3, and 4.

### Problem #317 <a id="p317"></a>

**Medium** · Yahoo · 2020-10-30

#### Problem

Write a function that returns the bitwise AND of all integers between M and N, inclusive.

### Problem #318 <a id="p318"></a>

**Hard** · Apple · 2020-10-31

#### Problem

You are going on a road trip, and would like to create a suitable music playlist. The trip will require N songs, though you only have M songs downloaded, where M \< N. A valid playlist should select each song at least once, and guarantee a buffer of B songs between repeats.

Given N, M, and B, determine the number of valid playlists.

### Problem #319 <a id="p319"></a>

**Hard** · Airbnb · 2020-11-01

#### Problem

An 8-puzzle is a game played on a 3 x 3 board of tiles, with the ninth tile missing. The remaining tiles are labeled 1 through 8 but shuffled randomly. Tiles may slide horizontally or vertically into an empty space, but may not be removed from the board.

Design a class to represent the board, and find a series of steps to bring the board to the state `[[1, 2, 3], [4, 5, 6], [7, 8, None]]`.

### Problem #320 <a id="p320"></a>

**Medium** · Amazon · 2020-11-02

#### Problem

Given a string, find the length of the smallest window that contains every distinct character. Characters may appear more than once in the window.

For example, given `"jiujitsu"`, you should return 5, corresponding to the final five letters.

### Problem #321 <a id="p321"></a>

**Easy** · PagerDuty · 2020-11-03

#### Problem

Given a positive integer N, find the smallest number of steps it will take to reach 1.

There are two kinds of permitted steps:

- You may decrement N to N - 1.
- If a \* b = N, you may decrement N to the larger of a and b.

For example, given 100, you can reach 1 in five steps with the following route: 100 -> 10 -> 9 -> 3 -> 2 -> 1.

### Problem #322 <a id="p322"></a>

**Medium** · Flipkart · 2020-11-04

#### Problem

Starting from 0 on a number line, you would like to make a series of jumps that lead to the integer N.

On the ith jump, you may move exactly i places to the left or right.

Find a path with the fewest number of jumps required to get from 0 to N.

### Problem #323 <a id="p323"></a>

**Medium** · Dropbox · 2020-11-05

#### Problem

Create an algorithm to efficiently compute the approximate median of a list of numbers.

More precisely, given an unordered list of N numbers, find an element whose rank is between N / 4 and 3 \* N / 4, with a high level of certainty, in less than `O(N)` time.

### Problem #324 <a id="p324"></a>

**Easy** · Amazon · 2020-11-06

#### Problem

Consider the following scenario: there are N mice and N holes placed at integer points along a line. Given this, find a method that maps mice to holes such that the largest number of steps any mouse takes is minimized.

Each move consists of moving one mouse one unit to the left or right, and only one mouse can fit inside each hole.

For example, suppose the mice are positioned at `[1, 4, 9, 15]`, and the holes are located at `[10, -5, 0, 16]`. In this case, the best pairing would require us to send the mouse at 1 to the hole at -5, so our function should return 6.

### Problem #325 <a id="p325"></a>

**Easy** · Jane Street · 2020-11-07

#### Problem

The United States uses the imperial system of weights and measures, which means that there are many different, seemingly arbitrary units to measure distance. There are 12 inches in a foot, 3 feet in a yard, 22 yards in a chain, and so on.

Create a data structure that can efficiently convert a certain quantity of one unit to the correct amount of any other unit. You should also allow for additional units to be added to the system.

### Problem #326 <a id="p326"></a>

**Hard** · Netflix · 2020-11-08

#### Problem

A Cartesian tree with sequence S is a binary tree defined by the following two properties:

- It is heap-ordered, so that each parent value is strictly less than that of its children.
- An in-order traversal of the tree produces nodes with values that correspond exactly to S.

For example, given the sequence `[3, 2, 6, 1, 9]`, the resulting Cartesian tree would be:

```
      1
    /   \   
  2       9
 / \
3   6
```

Given a sequence S, construct the corresponding Cartesian tree.

### Problem #327 <a id="p327"></a>

**Easy** · Salesforce · 2020-11-09

#### Problem

Write a program to merge two binary trees. Each node in the new tree should hold a value equal to the sum of the values of the corresponding nodes of the input trees.

If only one input tree has a node in a given position, the corresponding node in the new tree should match that input node.

### Problem #328 <a id="p328"></a>

**Medium** · Facebook · 2020-11-10

#### Problem

In chess, the Elo rating system is used to calculate player strengths based on game results.

A simplified description of the Elo system is as follows. Every player begins at the same score. For each subsequent game, the loser transfers some points to the winner, where the amount of points transferred depends on how unlikely the win is. For example, a 1200-ranked player should gain much more points for beating a 2000-ranked player than for beating a 1300-ranked player.

Implement this system.

### Problem #329 <a id="p329"></a>

**Hard** · Amazon · 2020-11-11

#### Problem

The stable marriage problem is defined as follows:

Suppose you have N men and N women, and each person has ranked their prospective opposite-sex partners in order of preference.

For example, if N = 3, the input could be something like this:

```
guy_preferences = {
    'andrew': ['caroline', 'abigail', 'betty'],
    'bill': ['caroline', 'betty', 'abigail'],
    'chester': ['betty', 'caroline', 'abigail'],
}
```

```
gal_preferences = {
    'abigail': ['andrew', 'bill', 'chester'],
    'betty': ['bill', 'andrew', 'chester'],
    'caroline': ['bill', 'chester', 'andrew']
}
```

Write an algorithm that pairs the men and women together in such a way that no two people of opposite sex would both rather be with each other than with their current partners.

### Problem #330 <a id="p330"></a>

**Hard** · Dropbox · 2020-11-12

#### Problem

A Boolean formula can be said to be satisfiable if there is a way to assign truth values to each variable such that the entire formula evaluates to true.

For example, suppose we have the following formula, where the symbol ¬ is used to denote negation:

(¬c OR b) AND (b OR c) AND (¬b OR c) AND (¬c OR ¬a)

One way to satisfy this formula would be to let a = False, b = True, and c = True.

This type of formula, with AND statements joining tuples containing exactly one OR, is known as 2-CNF.

Given a 2-CNF formula, find a way to assign truth values to satisfy it, or return False if this is impossible.

### Problem #331 <a id="p331"></a>

**Medium** · LinkedIn · 2020-11-13

#### Problem

You are given a string consisting of the letters x and y, such as xyxxxyxyy. In addition, you have an operation called flip, which changes a single x to y or vice versa.

Determine how many times you would need to apply this operation to ensure that all x's come before all y's. In the preceding example, it suffices to flip the second and sixth characters, so you should return 2.

### Problem #332 <a id="p332"></a>

**Easy** · Jane Street · 2020-11-14

#### Problem

Given integers M and N, write a program that counts how many positive integer pairs (a, b) satisfy the following conditions:

- a + b = M
- a XOR b = N

### Problem #333 <a id="p333"></a>

**Medium** · Pinterest · 2020-11-15

#### Problem

At a party, there is a single person who everyone knows, but who does not know anyone in return (the `"celebrity"`). To help figure out who this is, you have access to an `O(1)` method called `knows(a, b)`, which returns True if person a knows person b, else False.

Given a list of N people and the above operation, find a way to identify the celebrity in `O(N)` time.

### Problem #334 <a id="p334"></a>

**Easy** · Twitter · 2020-11-16

#### Problem

The 24 game is played as follows. You are given a list of four integers, each between 1 and 9, in a fixed order. By placing the operators +, -, \*, and / between the numbers, and grouping them with parentheses, determine whether it is possible to reach the value 24.

For example, given the input `[5, 2, 7, 8]`, you should return True, since (5 \* 2 - 7) \* 8 = 24.

Write a function that plays the 24 game.

### Problem #335 <a id="p335"></a>

**Hard** · Google · 2020-11-17

#### Problem

PageRank is an algorithm used by Google to rank the importance of different websites. While there have been changes over the years, the central idea is to assign each site a score based on the importance of other pages that link to that page.

More mathematically, suppose there are N sites, and each site i has a certain count Ci of outgoing links. Then the score for a particular site Sj is defined as :

`score(Sj)` = (1 - d) / N + d \* (`score(Sx)` / Cx+ `score(Sy)` / Cy+ ... + `score(Sz)` / Cz))

Here, Sx, Sy, ..., Sz denote the scores of all the other sites that have outgoing links to Sj, and d is a damping factor, usually set to around 0.85, used to model the probability that a user will stop searching.

Given a directed graph of links between various websites, write a program that calculates each site's page rank.

### Problem #336 <a id="p336"></a>

**Medium** · Microsoft · 2020-11-18

#### Problem

Write a program to determine how many distinct ways there are to create a max heap from a list of N given integers.

For example, if N = 3, and our integers are `[1, 2, 3]`, there are two ways, shown below.

```
  3      3
 / \    / \
1   2  2   1
```

### Problem #337 <a id="p337"></a>

**Hard** · Apple · 2020-11-19

#### Problem

Given a linked list, uniformly shuffle the nodes. What if we want to prioritize space over time?

### Problem #338 <a id="p338"></a>

**Medium** · Facebook · 2020-11-20

#### Problem

Given an integer n, find the next biggest integer with the same number of 1-bits on. For example, given the number 6 (0110 in binary), return 9 (1001).

### Problem #339 <a id="p339"></a>

**Easy** · Microsoft · 2020-11-21

#### Problem

Given an array of numbers and a number k, determine if there are three entries in the array which add up to the specified number k. For example, given `[20, 303, 3, 4, 25]` and k = 49, return true as 20 + 4 + 25 = 49.

### Problem #340 <a id="p340"></a>

**Easy** · Google · 2020-11-22

#### Problem

Given a set of points (x, y) on a 2D cartesian plane, find the two closest points. For example, given the points `[(1, 1), (-1, -1), (3, 4), (6, 1), (-1, -6), (-4, -3)]`, return `[(-1, -1), (1, 1)]`.

### Problem #341 <a id="p341"></a>

**Easy** · Google · 2020-11-23

#### Problem

You are given an N by N matrix of random letters and a dictionary of words. Find the maximum number of words that can be packed on the board from the given dictionary.

A word is considered to be able to be packed on the board if:

- It can be found in the dictionary
- It can be constructed from untaken letters by other words found so far on the board
- The letters are adjacent to each other (vertically and horizontally, not diagonally).

Each tile can be visited only once by any word.

For example, given the following dictionary:

`{ 'eat', 'rain', 'in', 'rat' }`

and matrix:

```
[['e', 'a', 'n'],
 ['t', 't', 'i'],
 ['a', 'r', 'a']]
```

Your function should return 3, since we can make the words `'eat'`, `'in'`, and `'rat'` without them touching each other. We could have alternatively made `'eat'` and `'rain'`, but that would be incorrect since that's only 2 words.

### Problem #342 <a id="p342"></a>

**Medium** · Stripe · 2020-11-24

#### Problem

reduce (also known as fold) is a function that takes in an array, a combining function, and an initial value and builds up a result by calling the combining function on each element of the array, left to right. For example, we can write `sum()` in terms of reduce:

```
def add(a, b):
    return a + b
```

```
def sum(lst):
    return reduce(lst, add, 0)
```

This should call add on the initial value with the first element of the array, and then the result of that with the second element of the array, and so on until we reach the end, when we return the sum of the array.

Implement your own version of reduce.

### Problem #343 <a id="p343"></a>

**Medium** · Google · 2020-11-25

#### Problem

Given a binary search tree and a range `[a, b]` (inclusive), return the sum of the elements of the binary search tree within the range.

For example, given the following tree:

```
    5
   / \
  3   8
 / \ / \
2  4 6  10
```

and the range `[4, 9]`, return 23 (5 + 4 + 6 + 8).

### Problem #344 <a id="p344"></a>

**Hard** · Adobe · 2020-11-26

#### Problem

You are given a tree with an even number of nodes. Consider each connection between a parent and child node to be an `"edge"`. You would like to remove some of these edges, such that the disconnected subtrees that remain each have an even number of nodes.

For example, suppose your input was the following tree:

```
   1
  / \ 
 2   3
    / \ 
   4   5
 / | \
6  7  8
```

In this case, removing the edge (3, 4) satisfies our requirement.

Write a function that returns the maximum number of edges you can remove while still satisfying this requirement.

### Problem #345 <a id="p345"></a>

**Medium** · Google · 2020-11-27

#### Problem

You are given a set of synonyms, such as (big, large) and (eat, consume). Using this set, determine if two sentences with the same number of words are equivalent.

For example, the following two sentences are equivalent:

- `"He wants to eat food."`
- `"He wants to consume food."`

Note that the synonyms (a, b) and (a, c) do not necessarily imply (b, c): consider the case of (coach, bus) and (coach, teacher).

Follow-up: what if we can assume that (a, b) and (a, c) do in fact imply (b, c)?

### Problem #346 <a id="p346"></a>

**Medium** · Airbnb · 2020-11-28

#### Problem

You are given a huge list of airline ticket prices between different cities around the world on a given day. These are all direct flights. Each element in the list has the format (`source_city`, destination, price).

Consider a user who is willing to take up to k connections from their origin city A to their destination B. Find the cheapest fare possible for this journey and print the itinerary for that journey.

For example, our traveler wants to go from JFK to LAX with up to 3 connections, and our input flights are as follows:

```
[
    ('JFK', 'ATL', 150),
    ('ATL', 'SFO', 400),
    ('ORD', 'LAX', 200),
    ('LAX', 'DFW', 80),
    ('JFK', 'HKG', 800),
    ('ATL', 'ORD', 90),
    ('JFK', 'LAX', 500),
]
```

Due to some improbably low flight prices, the cheapest itinerary would be JFK -> ATL -> ORD -> LAX, costing \$440.

### Problem #347 <a id="p347"></a>

**Easy** · Yahoo · 2020-11-29

#### Problem

You are given a string of length N and a parameter k. The string can be manipulated by taking one of the first k letters and moving it to the end.

Write a program to determine the lexicographically smallest string that can be created after an unlimited number of moves.

For example, suppose we are given the string daily and k = 1. The best we can create in this case is ailyd.

### Problem #348 <a id="p348"></a>

**Easy** · Zillow · 2020-11-30

#### Problem

A ternary search tree is a trie-like data structure where each node may have up to three children. Here is an example which represents the words code, cob, be, ax, war, and we.

```
       c
    /  |  \
   b   o   w
 / |   |   |
a  e   d   a
|    / |   | \ 
x   b  e   r  e
```

The tree is structured according to the following rules:

- left child nodes link to words lexicographically earlier than the parent prefix
- right child nodes link to words lexicographically later than the parent prefix
- middle child nodes continue the current word

For instance, since code is the first word inserted in the tree, and cob lexicographically precedes cod, cob is represented as a left child extending from cod.

Implement insertion and search functions for a ternary search tree.

### Problem #349 <a id="p349"></a>

**Hard** · Grammarly · 2020-12-01

#### Problem

Soundex is an algorithm used to categorize phonetically, such that two names that sound alike but are spelled differently have the same representation.

Soundex maps every name to a string consisting of one letter and three numbers, like M460.

One version of the algorithm is as follows:

- Remove consecutive consonants with the same sound (for example, change ck -> c).
- Keep the first letter. The remaining steps only apply to the rest of the string.
- Remove all vowels, including y, w, and h.
- Replace all consonants with the following digits:

- b, f, p, v → 1
- c, g, j, k, q, s, x, z → 2
- d, t → 3
- l → 4
- m, n → 5
- r → 6

- If you don't have three numbers yet, append zeros until you do. Keep the first three numbers.

Using this scheme, Jackson and Jaxen both map to J250.

Implement Soundex.

### Problem #350 <a id="p350"></a>

**Medium** · Uber · 2020-12-02

#### Problem

Write a program that determines the smallest number of perfect squares that sum up to N.

Here are a few examples:

- Given N = 4, return 1 (4)
- Given N = 17, return 2 (16 + 1)
- Given N = 18, return 2 (9 + 9)

### Problem #351 <a id="p351"></a>

**Hard** · Quora · 2020-12-03

#### Problem

Word sense disambiguation is the problem of determining which sense a word takes on in a particular setting, if that word has multiple meanings. For example, in the sentence `"I went to get money from the bank"`, bank probably means the place where people deposit money, not the land beside a river or lake.

Suppose you are given a list of meanings for several words, formatted like so:

```
{
    "word_1": ["meaning one", "meaning two", ...],
    ...
    "word_n": ["meaning one", "meaning two", ...]
}
```

Given a sentence, most of whose words are contained in the meaning list above, create an algorithm that determines the likely sense of each possibly ambiguous word.

### Problem #352 <a id="p352"></a>

**Easy** · Palantir · 2020-12-04

#### Problem

A typical American-style crossword puzzle grid is an N x N matrix with black and white squares, which obeys the following rules:

- Every white square must be part of an `"across"` word and a `"down"` word.
- No word can be fewer than three letters long.
- Every white square must be reachable from every other white square.
- The grid is rotationally symmetric (for example, the colors of the top left and bottom right squares must match).

Write a program to determine whether a given matrix qualifies as a crossword grid.

### Problem #353 <a id="p353"></a>

**Medium** · Square · 2020-12-05

#### Problem

You are given a histogram consisting of rectangles of different heights. These heights are represented in an input list, such that `[1, 3, 2, 5]` corresponds to the following diagram:

```
      x
      x  
  x   x
  x x x
x x x x
```

Determine the area of the largest rectangle that can be formed only from the bars of the histogram. For the diagram above, for example, this would be six, representing the 2 x 3 area at the bottom right.

### Problem #354 <a id="p354"></a>

**Hard** · Google · 2020-12-06

#### Problem

Design a system to crawl and copy all of Wikipedia using a distributed network of machines.

More specifically, suppose your server has access to a set of client machines. Your client machines can execute code you have written to access Wikipedia pages, download and parse their data, and write the results to a database.

Some questions you may want to consider as part of your solution are:

- How will you reach as many pages as possible?
- How can you keep track of pages that have already been visited?
- How will you deal with your client machines being blacklisted?
- How can you update your database when Wikipedia pages are added or updated?

### Problem #355 <a id="p355"></a>

**Hard** · Airbnb · 2020-12-07

#### Problem

You are given an array X of floating-point numbers x1, x2, ... xn. These can be rounded up or down to create a corresponding array Y of integers y1, y2, ... yn.

Write an algorithm that finds an appropriate Y array with the following properties:

- The rounded sums of both arrays should be equal.
- The absolute pairwise difference between elements is minimized. In other words, |x1- y1| + |x2- y2| + ... + |xn- yn| should be as small as possible.

For example, suppose your input is `[1.3, 2.3, 4.4]`. In this case you cannot do better than `[1, 2, 5]`, which has an absolute difference of |1.3 - 1| + |2.3 - 2| + |4.4 - 5| = 1.

### Problem #356 <a id="p356"></a>

**Hard** · Netflix · 2020-12-08

#### Problem

Implement a queue using a set of fixed-length arrays.

The queue should support enqueue, dequeue, and `get_size` operations.

### Problem #357 <a id="p357"></a>

**Hard** · LinkedIn · 2020-12-09

#### Problem

You are given a binary tree in a peculiar string representation. Each node is written in the form (lr), where l corresponds to the left child and r corresponds to the right child.

If either l or r is null, it will be represented as a zero. Otherwise, it will be represented by a new (lr) pair.

Here are a few examples:

- A root node with no children: (00)
- A root node with two children: ((00)(00))
- An unbalanced tree with three consecutive left children: ((((00)0)0)0)

Given this representation, determine the depth of the tree.

### Problem #358 <a id="p358"></a>

**Hard** · Dropbox · 2020-12-10

#### Problem

Create a data structure that performs all the following operations in `O(1)` time:

- plus: Add a key with value 1. If the key already exists, increment its value by one.
- minus: Decrement the value of a key. If the key's value is currently 1, remove it.
- `get_max`: Return a key with the highest value.
- `get_min`: Return a key with the lowest value.

### Problem #359 <a id="p359"></a>

**Easy** · Slack · 2020-12-11

#### Problem

You are given a string formed by concatenating several words corresponding to the integers zero through nine and then anagramming.

For example, the input could be `'niesevehrtfeev'`, which is an anagram of `'threefiveseven'`. Note that there can be multiple instances of each integer.

Given this string, return the original integers in sorted order. In the example above, this would be 357.

### Problem #360 <a id="p360"></a>

**Medium** · Spotify · 2020-12-12

#### Problem

You have access to ranked lists of songs for various users. Each song is represented as an integer, and more preferred songs appear earlier in each list. For example, the list `[4, 1, 7]` indicates that a user likes song 4 the best, followed by songs 1 and 7.

Given a set of these ranked lists, interleave them to create a playlist that satisfies everyone's priorities.

For example, suppose your input is `{[1, 7, 3], [2, 1, 6, 7, 9], [3, 9, 5]}`. In this case a satisfactory playlist could be `[2, 1, 6, 7, 3, 9, 5]`.

### Problem #361 <a id="p361"></a>

**Medium** · Facebook · 2020-12-13

#### Problem

Mastermind is a two-player game in which the first player attempts to guess the secret code of the second. In this version, the code may be any six-digit number with all distinct digits.

Each turn the first player guesses some number, and the second player responds by saying how many digits in this number correctly matched their location in the secret code. For example, if the secret code were 123456, then a guess of 175286 would score two, since 1 and 6 were correctly placed.

Write an algorithm which, given a sequence of guesses and their scores, determines whether there exists some secret code that could have produced them.

For example, for the following scores you should return True, since they correspond to the secret code 123456:

`{175286: 2, 293416: 3, 654321: 0}`

However, it is impossible for any key to result in the following scores, so in this case you should return False:

`{123456: 4, 345678: 4, 567890: 4}`

### Problem #362 <a id="p362"></a>

**Easy** · Twitter · 2020-12-14

#### Problem

A strobogrammatic number is a positive number that appears the same after being rotated 180 degrees. For example, 16891 is strobogrammatic.

Create a program that finds all strobogrammatic numbers with N digits.

### Problem #363 <a id="p363"></a>

**Medium** · Squarespace · 2020-12-15

#### Problem

Write a function, `add_subtract`, which alternately adds and subtracts curried arguments. Here are some sample operations:

`add_subtract(7)` -> 7

`add_subtract(1)(2)(3)` -> 1 + 2 - 3 -> 0

`add_subtract(-5)(10)(3)(9)` -> -5 + 10 - 3 + 9 -> 11

### Problem #364 <a id="p364"></a>

**Medium** · Facebook · 2020-12-16

#### Problem

Describe an algorithm to compute the longest increasing subsequence of an array of numbers in `O(n log n)` time.

### Problem #365 <a id="p365"></a>

**Hard** · Google · 2020-12-17

#### Problem

A quack is a data structure combining properties of both stacks and queues. It can be viewed as a list of elements written left to right such that three operations are possible:

- `push(x)`: add a new item x to the left end of the list
- `pop()`: remove and return the item on the left end of the list
- `pull()`: remove the item on the right end of the list.

Implement a quack using three stacks and `O(1)` additional memory, so that the amortized time for any push, pop, or pull operation is `O(1)`.

### Problem #366 <a id="p366"></a>

**Medium** · Flexport · 2020-12-18

#### Problem

Given a string s, rearrange the characters so that any two adjacent characters are not the same. If this is not possible, return null.

For example, if s = yyz then return yzy. If s = yyy then return null.

### Problem #367 <a id="p367"></a>

**Medium** · Two Sigma · 2020-12-19

#### Problem

Given two sorted iterators, merge it into one iterator.

For example, given these two iterators:

```
foo = iter([5, 10, 15])
bar = iter([3, 8, 9])
```

You should be able to do:

```
for num in merge_iterators(foo, bar):
    print(num)
```

```
# 3
# 5
# 8
# 9
# 10
# 15
```

Bonus: Make it work without pulling in the contents of the iterators in memory.

### Problem #368 <a id="p368"></a>

**Hard** · Google · 2020-12-20

#### Problem

Implement a key value store, where keys and values are integers, with the following methods:

- `update(key, vl)`: updates the value at key to val, or sets it if doesn't exist
- `get(key)`: returns the value with key, or None if no such value exists
- `max_key(val)`: returns the largest key with value val, or None if no key with that value exists

For example, if we ran the following calls:

```
kv.update(1, 1)
kv.update(2, 1)
```

And then called `kv.max_key(1)`, it should return 2, since it's the largest key with value 1.

### Problem #369 <a id="p369"></a>

**Medium** · Two Sigma · 2020-12-21

#### Problem

You’re tracking stock price at a given instance of time. Implement an API with the following functions: `add()`, `update()`, `remove()`, which adds/updates/removes a datapoint for the stock price you are tracking. The data is given as (timestamp, price), where timestamp is specified in unix epoch time.

Also, provide `max()`, `min()`, and `average()` functions that give the max/min/average of all values seen thus far.

### Problem #370 <a id="p370"></a>

**Easy** · Postmates · 2020-12-22

#### Problem

The “active time” of a courier is the time between the pickup and dropoff of a delivery. Given a set of data formatted like the following:

(delivery id, timestamp, pickup/dropoff)

Calculate the total active time in seconds. A courier can pick up multiple orders before dropping them off. The timestamp is in unix epoch seconds.

For example, if the input is the following:

```
(1, 1573280047, 'pickup')
(1, 1570320725, 'dropoff')
(2, 1570321092, 'pickup')
(3, 1570321212, 'pickup')
(3, 1570322352, 'dropoff')
(2, 1570323012, 'dropoff')
```

The total active time would be 1260 seconds.

### Problem #371 <a id="p371"></a>

**Hard** · Google · 2020-12-23

#### Problem

You are given a series of arithmetic equations as a string, such as:

```
y = x + 1
5 = x + 3
10 = z + y + 2
```

The equations use addition only and are separated by newlines. Return a mapping of all variables to their values. If it's not possible, then return null. In this example, you should return:

```
{
  x: 2,
  y: 3,
  z: 5
}
```

### Problem #372 <a id="p372"></a>

**Easy** · Amazon · 2020-12-24

#### Problem

Write a function that takes a natural number as input and returns the number of digits the input has.

Constraint: don't use any loops.

### Problem #373 <a id="p373"></a>

**Hard** · Facebook · 2020-12-25

#### Problem

Given a list of integers L, find the maximum length of a sequence of consecutive numbers that can be formed using elements from L.

For example, given L = `[5, 2, 99, 3, 4, 1, 100]`, return 5 as we can build a sequence `[1, 2, 3, 4, 5]` which has length 5.

### Problem #374 <a id="p374"></a>

**Hard** · Amazon · 2020-12-26

#### Problem

Given a sorted array arr of distinct integers, return the lowest index i for which `arr[i]` == i. Return null if there is no such index.

For example, given the array `[-5, -3, 2, 3]`, return 2 since `arr[2]` == 2. Even though `arr[3]` == 3, we return 2 since it's the lowest index.

### Problem #375 <a id="p375"></a>

**Medium** · Google · 2020-12-27

#### Problem

The h-index is a metric used to measure the impact and productivity of a scientist or researcher.

A scientist has index h if h of their N papers have at least h citations each, and the other N - h papers have no more than h citations each. If there are multiple possible values for h, the maximum value is used.

Given an array of natural numbers, with each value representing the number of citations of a researcher's paper, return the h-index of that researcher.

For example, if the array was:

`[4, 0, 0, 2, 3]`

This means the researcher has 5 papers with 4, 1, 0, 2, and 3 citations respectively. The h-index for this researcher is 2, since they have 2 papers with at least 2 citations and the remaining 3 papers have no more than 2 citations.

### Problem #376 <a id="p376"></a>

**Easy** · Google · 2020-12-28

#### Problem

You are writing an AI for a 2D map game. You are somewhere in a 2D grid, and there are coins strewn about over the map.

Given the position of all the coins and your current position, find the closest coin to you in terms of Manhattan distance. That is, you can move around up, down, left, and right, but not diagonally. If there are multiple possible closest coins, return any of them.

For example, given the following map, where you are x, coins are o, and empty spaces are . (top left is 0, 0):

```
---------------------
| . | . | x | . | o |
---------------------
| o | . | . | . | . |
---------------------
| o | . | . | . | o |
---------------------
| . | . | o | . | . |
---------------------
```

return (0, 4), since that coin is closest. This map would be represented in our question as:

```
Our position: (0, 2)
Coins: [(0, 4), (1, 0), (2, 0), (3, 2)]
```

### Problem #377 <a id="p377"></a>

**Hard** · Microsoft · 2020-12-29

#### Problem

Given an array of numbers arr and a window of size k, print out the median of each window of size k starting from the left and moving right by one position each time.

For example, given the following array and k = 3:

`[-1, 5, 13, 8, 2, 3, 3, 1]`

Your function should print out the following:

```
5 <- median of [-1, 5, 13]
8 <- median of [5, 13, 8]
8 <- median of [13, 8, 2]
3 <- median of [8, 2, 3]
3 <- median of [2, 3, 3]
3 <- median of [3, 3, 1]
```

Recall that the median of an even-sized list is the average of the two middle numbers.

### Problem #378 <a id="p378"></a>

**Medium** · Coinbase · 2020-12-30

#### Problem

Write a function that takes in a number, string, list, or dictionary and returns its JSON encoding. It should also handle nulls.

For example, given the following input:

`[None, 123, ["a", "b"], {"c":"d"}]`

You should return the following, as a string:

`'[null, 123, ["a", "b"], {"c": "d"}]'`

### Problem #379 <a id="p379"></a>

**Easy** · Microsoft · 2020-12-31

#### Problem

Given a string, generate all possible subsequences of the string.

For example, given the string xyz, return an array or set with the following strings:

```
x
y
z
xy
xz
yz
xyz
```

Note that zx is not a valid subsequence since it is not in the order of the given string.

### Problem #380 <a id="p380"></a>

**Medium** · Nextdoor · 2021-01-01

#### Problem

Implement integer division without using the division operator. Your function should return a tuple of (dividend, remainder) and it should take two numbers, the product and divisor.

For example, calling `divide(10, 3)` should return (3, 1) since the divisor is 3 and the remainder is 1.

Bonus: Can you do it in `O(log n)` time?

### Problem #381 <a id="p381"></a>

**Easy** · Paypal · 2021-01-02

#### Problem

Read this Wikipedia article on Base64 encoding.

Implement a function that converts a hex string to base64.

For example, the string:

deadbeef

should produce:

3q2+7w==

### Problem #382 <a id="p382"></a>

**Easy** · Google · 2021-01-03

#### Problem

Yesterday you implemented a function that encodes a hexadecimal string into Base64.

Write a function to decode a Base64 string back to a hexadecimal string.

For example, the following string:

3q2+7w==

should produce:

deadbeef

### Problem #383 <a id="p383"></a>

**Medium** · Gusto · 2021-01-04

#### Problem

Implement the function `embolden(s, lst)` which takes in a string s and list of substrings lst, and wraps all substrings in s with an HTML bold tag \<b> and \</b>.

If two bold tags overlap or are contiguous, they should be merged.

For example, given s = abcdefg and lst = `["bc", "ef"]`, return the string a\<b>bc\</b>d\<b>ef\</b>g.

Given s = abcdefg and lst = `["bcd", "def"]`, return the string a\<b>bcdef\</b>g, since they overlap.

### Problem #384 <a id="p384"></a>

**Hard** · WeWork · 2021-01-05

#### Problem

You are given an array of integers representing coin denominations and a total amount of money. Write a function to compute the fewest number of coins needed to make up that amount. If it is not possible to make that amount, return null.

For example, given an array of `[1, 5, 10]` and an amount 56, return 7 since we can use 5 dimes, 1 nickel, and 1 penny.

Given an array of `[5, 8]` and an amount 15, return 3 since we can use 5 5-cent coins.

### Problem #385 <a id="p385"></a>

**Medium** · Apple · 2021-01-06

#### Problem

You are given a hexadecimal-encoded string that has been XOR'd against a single char.

Decrypt the message. For example, given the string:

7a575e5e5d12455d405e561254405d5f1276535b5e4b12715d565b5c551262405d505e575f

You should be able to decrypt it and get:

Hello world from Daily Coding Problem

### Problem #386 <a id="p386"></a>

**Easy** · Twitter · 2021-01-07

#### Problem

Given a string, sort it in decreasing order based on the frequency of characters. If there are multiple possible solutions, return any of them.

For example, given the string tweet, return tteew. eettw would also be acceptable.

### Problem #387 <a id="p387"></a>

**Medium** · Amazon · 2021-01-08

#### Problem

How would you explain the difference between an API and SDK to a non-technical person?

### Problem #388 <a id="p388"></a>

**Medium** · Airtable · 2021-01-09

#### Problem

How would you explain web cookies to someone non-technical?

### Problem #389 <a id="p389"></a>

**Hard** · Google · 2021-01-10

#### Problem

Explain the difference between composition and inheritance. In which cases would you use each?

### Problem #390 <a id="p390"></a>

**Medium** · Two Sigma · 2021-01-11

#### Problem

You are given an unsorted list of 999,000 unique integers, each from 1 and 1,000,000. Find the missing 1000 numbers. What is the computational and space complexity of your solution?

### Problem #391 <a id="p391"></a>

**Hard** · Facebook · 2021-01-12

#### Problem

We have some historical clickstream data gathered from our site anonymously using cookies. The histories contain URLs that users have visited in chronological order.

Write a function that takes two users' browsing histories as input and returns the longest contiguous sequence of URLs that appear in both.

For example, given the following two users' histories:

```
user1 = ['/home', '/register', '/login', '/user', '/one', '/two']
user2 = ['/home', '/red', '/login', '/user', '/one', '/pink']
```

You should return the following:

`['/login', '/user', '/one']`

### Problem #392 <a id="p392"></a>

**Hard** · Google · 2021-01-13

#### Problem

You are given a 2D matrix of 1s and 0s where 1 represents land and 0 represents water.

Grid cells are connected horizontally orvertically (not diagonally). The grid is completely surrounded by water, and there is exactly one island (i.e., one or more connected land cells).

An island is a group is cells connected horizontally or vertically, but not diagonally. There is guaranteed to be exactly one island in this grid, and the island doesn't have water inside that isn't connected to the water around the island. Each cell has a side length of 1.

Determine the perimeter of this island.

For example, given the following matrix:

```
[[0, 1, 1, 0],
[1, 1, 1, 0],
[0, 1, 1, 0],
[0, 0, 1, 0]]
```

Return 14.

### Problem #393 <a id="p393"></a>

**Medium** · Airbnb · 2021-01-14

#### Problem

Given an array of integers, return the largest range, inclusive, of integers that are all included in the array.

For example, given the array `[9, 6, 1, 3, 8, 10, 12, 11]`, return (8, 12) since 8, 9, 10, 11, and 12 are all in the array.

### Problem #394 <a id="p394"></a>

**Easy** · Uber · 2021-01-15

#### Problem

Given a binary tree and an integer k, return whether there exists a root-to-leaf path that sums up to k.

For example, given k = 18 and the following binary tree:

```
    8
   / \
  4   13
 / \   \
2   6   19
```

Return True since the path 8 -> 4 -> 6 sums to 18.

### Problem #395 <a id="p395"></a>

**Medium** · Robinhood · 2021-01-16

#### Problem

Given an array of strings, group anagrams together.

For example, given the following array:

`['eat', 'ate', 'apt', 'pat', 'tea', 'now']`

Return:

```
[['eat', 'ate', 'tea'],
 ['apt', 'pat'],
 ['now']]
```

### Problem #396 <a id="p396"></a>

**Hard** · Google · 2021-01-17

#### Problem

Given a string, return the length of the longest palindromic subsequence in the string.

For example, given the following string:

MAPTPTMTPA

Return 7, since the longest palindromic subsequence in the string is APTMTPA. Recall that a subsequence of a string does not have to be contiguous!

Your algorithm should run in `O(n^2)` time and space.

### Problem #397 <a id="p397"></a>

**Medium** · Microsoft · 2021-01-18

#### Problem

You are given a list of jobs to be done, where each job is represented by a start time and end time. Two jobs are compatible if they don't overlap. Find the largest subset of compatible jobs.

For example, given the following jobs (there is no guarantee that jobs will be sorted):

```
[(0, 6),
(1, 4),
(3, 5),
(3, 8),
(4, 7),
(5, 9),
(6, 10),
(8, 11)]
```

Return:

```
[(1, 4),
(4, 7),
(8, 11)]
```

### Problem #398 <a id="p398"></a>

**Medium** · Amazon · 2021-01-19

#### Problem

Given a linked list and an integer k, remove the k-th node from the end of the list and return the head of the list.

k is guaranteed to be smaller than the length of the list.

Do this in one pass.

### Problem #399 <a id="p399"></a>

**Hard** · Facebook · 2021-01-20

#### Problem

Given a list of strictly positive integers, partition the list into 3 contiguous partitions which each sum up to the same value. If not possible, return null.

For example, given the following list:

`[3, 5, 8, 0, 8]`

Return the following 3 partitions:

```
[[3, 5],
 [8, 0],
 [8]]
```

Which each add up to 8.

### Problem #400 <a id="p400"></a>

**Hard** · Goldman Sachs · 2021-01-21

#### Problem

Given a list of numbers L, implement a method `sum(i, j)` which returns the sum from the sublist `L[i:j]` (including i, excluding j).

For example, given L = `[1, 2, 3, 4, 5]`, `sum(1, 3)` should return `sum([2, 3])`, which is 5.

You can assume that you can do some pre-processing. `sum()` should be optimized over the pre-processing step.

### Problem #401 <a id="p401"></a>

**Easy** · Twitter · 2021-01-22

#### Problem

A permutation can be specified by an array P, where `P[i]` represents the location of the element at i in the permutation. For example, `[2, 1, 0]` represents the permutation where elements at the index 0 and 2 are swapped.

Given an array and a permutation, apply the permutation to the array. For example, given the array `["a", "b", "c"]` and the permutation `[2, 1, 0]`, return `["c", "b", "a"]`.

### Problem #402 <a id="p402"></a>

**Easy** · Twitter · 2021-01-23

#### Problem

A strobogrammatic number is a positive number that appears the same after being rotated 180 degrees. For example, 16891 is strobogrammatic.

Create a program that finds all strobogrammatic numbers with N digits.

### Problem #403 <a id="p403"></a>

**Easy** · Two Sigma · 2021-01-24

#### Problem

Using a function `rand5()` that returns an integer from 1 to 5 (inclusive) with uniform probability, implement a function `rand7()` that returns an integer from 1 to 7 (inclusive).

### Problem #404 <a id="p404"></a>

**Easy** · Snapchat · 2021-01-25

#### Problem

Given an array of time intervals (start, end) for classroom lectures (possibly overlapping), find the minimum number of rooms required.

For example, given `[(30, 75), (0, 50), (60, 150)]`, you should return 2.

### Problem #405 <a id="p405"></a>

**Hard** · Apple · 2021-01-26

#### Problem

Given a tree, find the largest tree/subtree that is a BST.

Given a tree, return the size of the largest tree/subtree that is a BST.

### Problem #406 <a id="p406"></a>

**Hard** · Quantcast · 2021-01-27

#### Problem

You are presented with an array representing a Boolean expression. The elements are of two kinds:

- T and F, representing the values True and False.
- &, |, and ^, representing the bitwise operators for AND, OR, and XOR.

Determine the number of ways to group the array elements using parentheses so that the entire expression evaluates to True.

For example, suppose the input is `['F', '|', 'T', '&', 'T']`. In this case, there are two acceptable groupings: (F | T) & T and F | (T & T).

### Problem #407 <a id="p407"></a>

**Medium** · Samsung · 2021-01-28

#### Problem

A group of houses is connected to the main water plant by means of a set of pipes. A house can either be connected by a set of pipes extending directly to the plant, or indirectly by a pipe to a nearby house which is otherwise connected.

For example, here is a possible configuration, where A, B, and C are houses, and arrows represent pipes:

A \<--> B \<--> C \<--> plant

Each pipe has an associated cost, which the utility company would like to minimize. Given an undirected graph of pipe connections, return the lowest cost configuration of pipes such that each house has access to water.

In the following setup, for example, we can remove all but the pipes from plant to A, plant to B, and B to C, for a total cost of 16.

```
pipes = {
    'plant': {'A': 1, 'B': 5, 'C': 20},
    'A': {'C': 15},
    'B': {'C': 10},
    'C': {}
}
```

### Problem #408 <a id="p408"></a>

**Medium** · Facebook · 2021-01-29

#### Problem

Given an array of numbers representing the stock prices of a company in chronological order and an integer k, return the maximum profit you can make from k buys and sells. You must buy the stock before you can sell it, and you must sell the stock before you can buy it again.

For example, given k = 2 and the array `[5, 2, 4, 0, 1]`, you should return 3.

### Problem #409 <a id="p409"></a>

**Hard** · Google · 2021-01-30

#### Problem

PageRank is an algorithm used by Google to rank the importance of different websites. While there have been changes over the years, the central idea is to assign each site a score based on the importance of other pages that link to that page.

More mathematically, suppose there are N sites, and each site i has a certain count Ci of outgoing links. Then the score for a particular site Sj is defined as :

`score(Sj)` = (1 - d) / N + d \* (`score(Sx)` / Cx+ `score(Sy)` / Cy+ ... + `score(Sz)` / Cz))

Here, Sx, Sy, ..., Sz denote the scores of all the other sites that have outgoing links to Sj, and d is a damping factor, usually set to around 0.85, used to model the probability that a user will stop searching.

Given a directed graph of links between various websites, write a program that calculates each site's page rank.

### Problem #410 <a id="p410"></a>

**Hard** · Airbnb · 2021-01-31

#### Problem

You are given an array X of floating-point numbers x1, x2, ... xn. These can be rounded up or down to create a corresponding array Y of integers y1, y2, ... yn.

Write an algorithm that finds an appropriate Y array with the following properties:

- The rounded sums of both arrays should be equal.
- The absolute pairwise difference between elements is minimized. In other words, |x1- y1| + |x2- y2| + ... + |xn- yn| should be as small as possible.

For example, suppose your input is `[1.3, 2.3, 4.4]`. In this case you cannot do better than `[1, 2, 5]`, which has an absolute difference of |1.3 - 1| + |2.3 - 2| + |4.4 - 5| = 1.

### Problem #411 <a id="p411"></a>

**Hard** · Google · 2021-02-01

#### Problem

Given a string which we can delete at most k, return whether you can make a palindrome.

For example, given `'waterrfetawx'` and a k of 2, you could delete f and x to get `'waterretaw'`.

### Problem #412 <a id="p412"></a>

**Medium** · Epic · 2021-02-02

#### Problem

The `"look and say"` sequence is defined as follows: beginning with the term 1, each subsequent term visually describes the digits appearing in the previous term. The first few terms are as follows:

```
1
11
21
1211
111221
```

As an example, the fourth term is 1211, since the third term consists of one 2 and one 1.

Given an integer N, print the Nth term of this sequence.

### Problem #413 <a id="p413"></a>

**Hard** · Amazon · 2021-02-03

#### Problem

There exists a staircase with N steps, and you can climb up either 1 or 2 steps at a time. Given N, write a function that returns the number of unique ways you can climb the staircase. The order of the steps matters.

For example, if N is 4, then there are 5 unique ways:

- 1, 1, 1, 1
- 2, 1, 1
- 1, 2, 1
- 1, 1, 2
- 2, 2

What if, instead of being able to climb 1 or 2 steps at a time, you could climb any number from a set of positive integers X? For example, if X = `{1, 3, 5}`, you could climb 1, 3, or 5 steps at a time.

### Problem #414 <a id="p414"></a>

**Hard** · Microsoft · 2021-02-04

#### Problem

You have an N by N board. Write a function that, given N, returns the number of possible arrangements of the board where N queens can be placed on the board without threatening each other, i.e. no two queens share the same row, column, or diagonal.

### Problem #415 <a id="p415"></a>

**Hard** · Affirm · 2021-02-05

#### Problem

Given a array of numbers representing the stock prices of a company in chronological order, write a function that calculates the maximum profit you could have made from buying and selling that stock. You're also given a number fee that represents a transaction fee for each buy and sell transaction.

You must buy before you can sell the stock, but you can make as many transactions as you like.

For example, given `[1, 3, 2, 8, 4, 10]` and fee = 2, you should return 9, since you could buy the stock at 1 dollar, and sell at 8 dollars, and then buy it at 4 dollars and sell it at 10 dollars. Since we did two transactions, there is a 4 dollar fee, so we have 7 + 6 = 13 profit minus 4 dollars of fees.

### Problem #416 <a id="p416"></a>

**Easy** · Google · 2021-02-06

#### Problem

You are in an infinite 2D grid where you can move in any of the 8 directions:

```
(x,y) to
   (x+1, y),
   (x - 1, y),
   (x, y+1),
   (x, y-1),
   (x-1, y-1),
   (x+1,y+1),
   (x-1,y+1),
   (x+1,y-1)
```

You are given a sequence of points and the order in which you need to cover the points. Give the minimum number of steps in which you can achieve it. You start from the first point.

Example:

```
Input: [(0, 0), (1, 1), (1, 2)]
Output: 2
```

It takes 1 step to move from (0, 0) to (1, 1). It takes one more step to move from (1, 1) to (1, 2).

### Problem #417 <a id="p417"></a>

**Easy** · Amazon · 2021-02-07

#### Problem

Given a linked list, remove all consecutive nodes that sum to zero. Print out the remaining nodes.

For example, suppose you are given the input 3 -> 4 -> -7 -> 5 -> -6 -> 6. In this case, you should first remove 3 -> 4 -> -7, then -6 -> 6, leaving only 5.

### Problem #418 <a id="p418"></a>

**Easy** · Atlassian · 2021-02-08

#### Problem

MegaCorp wants to give bonuses to its employees based on how many lines of codes they have written. They would like to give the smallest positive amount to each worker consistent with the constraint that if a developer has written more lines of code than their neighbor, they should receive more money.

Given an array representing a line of seats of employees at MegaCorp, determine how much each one should get paid.

For example, given `[10, 40, 200, 1000, 60, 30]`, you should return `[1, 2, 3, 4, 2, 1]`.

### Problem #419 <a id="p419"></a>

**Easy** · PagerDuty · 2021-02-09

#### Problem

Given a positive integer N, find the smallest number of steps it will take to reach 1.

There are two kinds of permitted steps:

- You may decrement N to N - 1.
- If a \* b = N, you may decrement N to the larger of a and b.

For example, given 100, you can reach 1 in five steps with the following route: 100 -> 10 -> 9 -> 3 -> 2 -> 1.

### Problem #420 <a id="p420"></a>

**Easy** · Microsoft · 2021-02-10

#### Problem

A number is considered perfect if its digits sum up to exactly 10.

Given a positive integer n, return the n-th perfect number.

For example, given 1, you should return 19. Given 2, you should return 28.

### Problem #421 <a id="p421"></a>

**Medium** · Amazon · 2021-02-11

#### Problem

Given an array of a million integers between zero and a billion, out of order, how can you efficiently sort it? Assume that you cannot store an array of a billion elements in memory.

### Problem #422 <a id="p422"></a>

**Easy** · Salesforce · 2021-02-12

#### Problem

Write a program to merge two binary trees. Each node in the new tree should hold a value equal to the sum of the values of the corresponding nodes of the input trees.

If only one input tree has a node in a given position, the corresponding node in the new tree should match that input node.

### Problem #423 <a id="p423"></a>

**Easy** · Microsoft · 2021-02-13

#### Problem

The transitive closure of a graph is a measure of which vertices are reachable from other vertices. It can be represented as a matrix M, where `M[i][j]` == 1 if there is a path between vertices i and j, and otherwise 0.

For example, suppose we are given the following graph in adjacency list form:

```
graph = [
    [0, 1, 3],
    [1, 2],
    [2],
    [3]
]
```

The transitive closure of this graph would be:

```
[1, 1, 1, 1]
[0, 1, 1, 0]
[0, 0, 1, 0]
[0, 0, 0, 1]
```

Given a graph, find its transitive closure.

### Problem #424 <a id="p424"></a>

**Medium** · Facebook · 2021-02-14

#### Problem

Given an array of integers in which two elements appear exactly once and all other elements appear exactly twice, find the two elements that appear only once.

For example, given the array `[2, 4, 6, 8, 10, 2, 6, 10]`, return 4 and 8. The order does not matter.

Follow-up: Can you do this in linear time and constant space?

### Problem #425 <a id="p425"></a>

**Hard** · Oracle · 2021-02-15

#### Problem

You are presented with an 8 by 8 matrix representing the positions of pieces on a chess board. The only pieces on the board are the black king and various white pieces. Given this matrix, determine whether the king is in check.

For details on how each piece moves, see here.

For example, given the following matrix:

```
...K....
........
.B......
......P.
.......R
..N.....
........
.....Q..
```

You should return True, since the bishop is attacking the king diagonally.

### Problem #426 <a id="p426"></a>

**Easy** · Facebook · 2021-02-16

#### Problem

Given a binary tree, return the level of the tree with minimum sum.

### Problem #427 <a id="p427"></a>

**Medium** · Square · 2021-02-17

#### Problem

A competitive runner would like to create a route that starts and ends at his house, with the condition that the route goes entirely uphill at first, and then entirely downhill.

Given a dictionary of places of the form `{location: elevation}`, and a dictionary mapping paths between some of these locations to their corresponding distances, find the length of the shortest route satisfying the condition above. Assume the runner's home is location 0.

For example, suppose you are given the following input:

```
elevations = {0: 5, 1: 25, 2: 15, 3: 20, 4: 10}
paths = {
    (0, 1): 10,
    (0, 2): 8,
    (0, 3): 15,
    (1, 3): 12,
    (2, 4): 10,
    (3, 4): 5,
    (3, 0): 17,
    (4, 0): 10
}
```

In this case, the shortest valid path would be 0 -> 2 -> 4 -> 0, with a distance of 28.

### Problem #428 <a id="p428"></a>

**Hard** · Uber · 2021-02-18

#### Problem

You have N stones in a row, and would like to create from them a pyramid. This pyramid should be constructed such that the height of each stone increases by one until reaching the tallest stone, after which the heights decrease by one. In addition, the start and end stones of the pyramid should each be one stone high.

You can change the height of any stone by paying a cost of 1 unit to lower its height by 1, as many times as necessary. Given this information, determine the lowest cost method to produce this pyramid.

For example, given the stones `[1, 1, 3, 3, 2, 1]`, the optimal solution is to pay 2 to create `[0, 1, 2, 3, 2, 1]`.

### Problem #429 <a id="p429"></a>

**Medium** · Stitch Fix · 2021-02-19

#### Problem

Pascal's triangle is a triangular array of integers constructed with the following formula:

- The first row consists of the number 1.
- For each subsequent row, each element is the sum of the numbers directly above it, on either side.

For example, here are the first few rows:

```
    1
   1 1
  1 2 1
 1 3 3 1
1 4 6 4 1
```

Given an input k, return the kth row of Pascal's triangle.

Bonus: Can you do this using only `O(k)` space?

### Problem #430 <a id="p430"></a>

**Hard** · Facebook · 2021-02-20

#### Problem

Given a string of parentheses, find the balanced string that can be produced from it using the minimum number of insertions and deletions. If there are multiple solutions, return any of them.

For example, given `"(()"`, you could return `"(())"`. Given `"))()("`, you could return `"()()()()"`.

### Problem #431 <a id="p431"></a>

**Medium** · Nest · 2021-02-21

#### Problem

Create a basic sentence checker that takes in a stream of characters and determines whether they form valid sentences. If a sentence is valid, the program should print it out.

We can consider a sentence valid if it conforms to the following rules:

- The sentence must start with a capital letter, followed by a lowercase letter or a space.
- All other characters must be lowercase letters, separators (,,;,:) or terminal marks (.,?,!,‽).
- There must be a single space between each word.
- The sentence must end with a terminal mark immediately following a word.

### Problem #432 <a id="p432"></a>

**Hard** · Google · 2021-02-22

#### Problem

Design a system to crawl and copy all of Wikipedia using a distributed network of machines.

More specifically, suppose your server has access to a set of client machines. Your client machines can execute code you have written to access Wikipedia pages, download and parse their data, and write the results to a database.

Some questions you may want to consider as part of your solution are:

- How will you reach as many pages as possible?
- How can you keep track of pages that have already been visited?
- How will you deal with your client machines being blacklisted?
- How can you update your database when Wikipedia pages are added or updated?

### Problem #433 <a id="p433"></a>

**Medium** · Facebook · 2021-02-23

#### Problem

Given an integer n, find the next biggest integer with the same number of 1-bits on. For example, given the number 6 (0110 in binary), return 9 (1001).

### Problem #434 <a id="p434"></a>

**Easy** · Oracle · 2021-02-24

#### Problem

Given a binary search tree, find the floor and ceiling of a given integer. The floor is the highest element in the tree less than or equal to an integer, while the ceiling is the lowest element in the tree greater than or equal to an integer.

If either value does not exist, return None.

### Problem #435 <a id="p435"></a>

**Medium** · Google · 2021-02-25

#### Problem

Given pre-order and in-order traversals of a binary tree, write a function to reconstruct the tree.

For example, given the following preorder traversal:

`[a, b, d, e, c, f, g]`

And the following inorder traversal:

`[d, b, e, a, f, c, g]`

You should return the following tree:

```
    a
   / \
  b   c
 / \ / \
d  e f  g
```

### Problem #436 <a id="p436"></a>

**Hard** · Microsoft · 2021-02-26

#### Problem

Implement 3 stacks using a single list:

```
class Stack:
    def __init__(self):
        self.list = []
```

```
def pop(self, stack_number):
    pass
```

```
def push(self, item, stack_number):
    pass
```

### Problem #437 <a id="p437"></a>

**Medium** · Square · 2021-02-27

#### Problem

Given a string and a set of characters, return the shortest substring containing all the characters in the set.

For example, given the string `"figehaeci"` and the set of characters `{a, e, i}`, you should return `"aeci"`.

If there is no substring containing all the characters in the set, return null.

### Problem #438 <a id="p438"></a>

**Easy** · Amazon · 2021-02-28

#### Problem

Implement a stack API using only a heap. A stack implements the following methods:

- `push(item)`, which adds an element to the stack
- `pop()`, which removes and returns the most recently added element (or throws an error if there is nothing on the stack)

Recall that a heap has the following operations:

- `push(item)`, which adds a new key to the heap
- `pop()`, which removes and returns the max value of the heap

### Problem #439 <a id="p439"></a>

**Medium** · Facebook · 2021-03-01

#### Problem

Given an unordered list of flights taken by someone, each represented as (origin, destination) pairs, and a starting airport, compute the person's itinerary. If no such itinerary exists, return null. If there are multiple possible itineraries, return the lexicographically smallest one. All flights must be used in the itinerary.

For example, given the list of flights `[('SFO', 'HKO'), ('YYZ', 'SFO'), ('YUL', 'YYZ'), ('HKO', 'ORD')]` and starting airport `'YUL'`, you should return the list `['YUL', 'YYZ', 'SFO', 'HKO', 'ORD']`.

Given the list of flights `[('SFO', 'COM'), ('COM', 'YYZ')]` and starting airport `'COM'`, you should return null.

Given the list of flights `[('A', 'B'), ('A', 'C'), ('B', 'C'), ('C', 'A')]` and starting airport `'A'`, you should return the list `['A', 'B', 'C', 'A', 'C']` even though `['A', 'C', 'A', 'B', 'C']` is also a valid itinerary. However, the first one is lexicographically smaller.

### Problem #440 <a id="p440"></a>

**Medium** · Microsoft · 2021-03-02

#### Problem

Describe and give an example of each of the following types of polymorphism:

- Ad-hoc polymorphism
- Parametric polymorphism
- Subtype polymorphism

### Problem #441 <a id="p441"></a>

**Medium** · Amazon · 2021-03-03

#### Problem

Given a pivot x, and a list lst, partition the list into three parts.

- The first part contains all elements in lst that are less than x
- The second part contains all elements in lst that are equal to x
- The third part contains all elements in lst that are larger than x

Ordering within a part can be arbitrary.

For example, given x = 10 and lst = `[9, 12, 3, 5, 14, 10, 10]`, one partition may be `[9, 3, 5, 10, 10, 12, 14]`.

### Problem #442 <a id="p442"></a>

**Hard** · Netflix · 2021-03-04

#### Problem

A Cartesian tree with sequence S is a binary tree defined by the following two properties:

- It is heap-ordered, so that each parent value is strictly less than that of its children.
- An in-order traversal of the tree produces nodes with values that correspond exactly to S.

For example, given the sequence `[3, 2, 6, 1, 9]`, the resulting Cartesian tree would be:

```
      1
    /   \   
  2       9
 / \
3   6
```

Given a sequence S, construct the corresponding Cartesian tree.

### Problem #443 <a id="p443"></a>

**Medium** · Apple · 2021-03-05

#### Problem

Implement a queue using two stacks. Recall that a queue is a FIFO (first-in, first-out) data structure with the following methods: enqueue, which inserts an element into the queue, and dequeue, which removes it.

### Problem #444 <a id="p444"></a>

**Hard** · Dropbox · 2021-03-06

#### Problem

Implement an efficient string matching algorithm.

That is, given a string of length N and a pattern of length k, write a program that searches for the pattern in the string with less than `O(N * k)` worst-case time complexity.

If the pattern is found, return the start index of its location. If not, return False.

### Problem #445 <a id="p445"></a>

**Medium** · BufferBox · 2021-03-07

#### Problem

Given a binary tree where all nodes are either 0 or 1, prune the tree so that subtrees containing all 0s are removed.

For example, given the following tree:

```
  0
 / \
1   0
   / \
  1   0
 / \
0   0
```

should be pruned to:

```
  0
 / \
1   0
   /
  1
```

We do not remove the tree at the root or its left child because it still has a 1 as a descendant.

### Problem #446 <a id="p446"></a>

**Medium** · Indeed · 2021-03-08

#### Problem

Given a 32-bit positive integer N, determine whether it is a power of four in faster than `O(log N)` time.

### Problem #447 <a id="p447"></a>

**Medium** · Google · 2021-03-09

#### Problem

Implement integer exponentiation. That is, implement the `pow(x, y)` function, where x and y are integers and returns x^y.

Do this faster than the naive method of repeated multiplication.

For example, `pow(2, 10)` should return 1024.

### Problem #448 <a id="p448"></a>

**Hard** · Google · 2021-03-10

#### Problem

Given an array of strictly the characters `'R'`, `'G'`, and `'B'`, segregate the values of the array so that all the Rs come first, the Gs come second, and the Bs come last. You can only swap elements of the array.

Do this in linear time and in-place.

For example, given the array `['G', 'B', 'R', 'R', 'B', 'R', 'G']`, it should become `['R', 'R', 'R', 'G', 'G', 'B', 'B']`.

### Problem #449 <a id="p449"></a>

**Easy** · Alibaba · 2021-03-11

#### Problem

Given an even number (greater than 2), return two prime numbers whose sum will be equal to the given number.

A solution will always exist. See Goldbach’s conjecture.

Example:

```
Input: 4
Output: 2 + 2 = 4
```

If there are more than one solution possible, return the lexicographically smaller solution.

If `[a, b]` is one solution with a \<= b, and `[c, d]` is another solution with c \<= d, then

`[a, b]` \< `[c, d]`

If a \< c OR a==c AND b \< d.

### Problem #450 <a id="p450"></a>

**Hard** · Google · 2021-03-12

#### Problem

You're given a string consisting solely of (, ), and \*. \* can represent either a (, ), or an empty string. Determine whether the parentheses are balanced.

For example, (()\* and (\*) are balanced. )\*( is not balanced.

### Problem #451 <a id="p451"></a>

**Easy** · Apple · 2021-03-13

#### Problem

Implement the function `fib(n)`, which returns the nth number in the Fibonacci sequence, using only `O(1)` space.

### Problem #452 <a id="p452"></a>

**Easy** · Microsoft · 2021-03-14

#### Problem

Let's represent an integer in a linked list format by having each node represent a digit in the number. The nodes make up the number in reversed order.

For example, the following linked list:

1 -> 2 -> 3 -> 4 -> 5

is the number 54321.

Given two linked lists in this format, return their sum in the same linked list format.

For example, given

9 -> 9

5 -> 2

return 124 (99 + 25) as:

4 -> 2 -> 1

### Problem #453 <a id="p453"></a>

**Easy** · Google · 2021-03-15

#### Problem

Given the root of a binary search tree, and a target K, return two nodes in the tree whose sum equals K.

For example, given the following tree and K of 20

```
   10
  /   \
5      15
      /  \
    11    15
```

Return the nodes 5 and 15.

### Problem #454 <a id="p454"></a>

**Medium** · Facebook · 2021-03-16

#### Problem

Describe an algorithm to compute the longest increasing subsequence of an array of numbers in `O(n log n)` time.

### Problem #455 <a id="p455"></a>

**Medium** · Dropbox · 2021-03-17

#### Problem

Conway's Game of Life takes place on an infinite two-dimensional board of square cells. Each cell is either dead or alive, and at each tick, the following rules apply:

- Any live cell with less than two live neighbours dies.
- Any live cell with two or three live neighbours remains living.
- Any live cell with more than three live neighbours dies.
- Any dead cell with exactly three live neighbours becomes a live cell.

A cell neighbours another cell if it is horizontally, vertically, or diagonally adjacent.

Implement Conway's Game of Life. It should be able to be initialized with a starting list of live cell coordinates and the number of steps it should run for. Once initialized, it should print out the board state at each step. Since it's an infinite board, print out only the relevant coordinates, i.e. from the top-leftmost live cell to bottom-rightmost live cell.

You can represent a live cell with an asterisk (\*) and a dead cell with a dot (.).

### Problem #456 <a id="p456"></a>

**Easy** · Amazon · 2021-03-18

#### Problem

You are given a list of data entries that represent entries and exits of groups of people into a building. An entry looks like this:

`{"timestamp": 1526579928, count: 3, "type": "enter"}`

This means 3 people entered the building. An exit looks like this:

`{"timestamp": 1526580382, count: 2, "type": "exit"}`

This means that 2 people exited the building. timestamp is in Unix time.

Find the busiest period in the building, that is, the time with the most people in the building. Return it as a pair of (start, end) timestamps. You can assume the building always starts off and ends up empty, i.e. with 0 people inside.

### Problem #457 <a id="p457"></a>

**Hard** · Google · 2021-03-19

#### Problem

Given a word W and a string S, find all starting indices in S which are anagrams of W.

For example, given that W is `"ab"`, and S is `"abxaba"`, return 0, 3, and 4.

### Problem #458 <a id="p458"></a>

**Hard** · Uber · 2021-03-20

#### Problem

A rule looks like this:

A NE B

This means this means point A is located northeast of point B.

A SW C

means that point A is southwest of C.

Given a list of rules, check if the sum of the rules validate. For example:

```
A N B
B NE C
C N A
```

does not validate, since A cannot be both north and south of C.

```
A NW B
A N B
```

is considered valid.

### Problem #459 <a id="p459"></a>

**Medium** · Uber · 2021-03-21

#### Problem

Write a program that determines the smallest number of perfect squares that sum up to N.

Here are a few examples:

- Given N = 4, return 1 (4)
- Given N = 17, return 2 (16 + 1)
- Given N = 18, return 2 (9 + 9)

### Problem #460 <a id="p460"></a>

**Medium** · LinkedIn · 2021-03-22

#### Problem

You are given a string consisting of the letters x and y, such as xyxxxyxyy. In addition, you have an operation called flip, which changes a single x to y or vice versa.

Determine how many times you would need to apply this operation to ensure that all x's come before all y's. In the preceding example, it suffices to flip the second and sixth characters, so you should return 2.

### Problem #461 <a id="p461"></a>

**Medium** · Facebook · 2021-03-23

#### Problem

There is an N by M matrix of zeroes. Given N and M, write a function to count the number of ways of starting at the top-left corner and getting to the bottom-right corner. You can only move right or down.

For example, given a 2 by 2 matrix, you should return 2, since there are two ways to get to the bottom-right:

- Right, then down
- Down, then right

Given a 5 by 5 matrix, there are 70 ways to get to the bottom-right.

### Problem #462 <a id="p462"></a>

**Hard** · Oracle · 2021-03-24

#### Problem

We say a number is sparse if there are no adjacent ones in its binary representation. For example, 21 (10101) is sparse, but 22 (10110) is not. For a given input N, find the smallest sparse number greater than or equal to N.

Do this in faster than `O(N log N)` time.

### Problem #463 <a id="p463"></a>

**Easy** · Facebook · 2021-03-25

#### Problem

On a mysterious island there are creatures known as Quxes which come in three colors: red, green, and blue. One power of the Qux is that if two of them are standing next to each other, they can transform into a single creature of the third color.

Given N Quxes standing in a line, determine the smallest number of them remaining after any possible sequence of such transformations.

For example, given the input `['R', 'G', 'B', 'G', 'B']`, it is possible to end up with a single Qux through the following steps:

```
        Arrangement       |   Change
----------------------------------------
['R', 'G', 'B', 'G', 'B'] | (R, G) -> B
['B', 'B', 'G', 'B']      | (B, G) -> R
['B', 'R', 'B']           | (R, B) -> G
['B', 'G']                | (B, G) -> R
['R']                     |
```

### Problem #464 <a id="p464"></a>

**Medium** · Google · 2021-03-26

#### Problem

Given a set of distinct positive integers, find the largest subset such that every pair of elements in the subset (i, j) satisfies either i % j = 0 or j % i = 0.

For example, given the set `[3, 5, 10, 20, 21]`, you should return `[5, 10, 20]`. Given `[1, 3, 6, 24]`, return `[1, 3, 6, 24]`.

### Problem #465 <a id="p465"></a>

**Easy** · Google · 2021-03-27

#### Problem

Given the head of a singly linked list, reverse it in-place.

### Problem #466 <a id="p466"></a>

**Easy** · Amazon · 2021-03-28

#### Problem

A tree is symmetric if its data and shape remain unchanged when it is reflected about the root node. The following tree is an example:

```
        4
      / | \
    3   5   3
  /           \
9              9
```

Given a k-ary tree, determine whether it is symmetric.

### Problem #467 <a id="p467"></a>

**Medium** · 2021-03-29

#### Problem

Given a real number n, find the square root of n. For example, given n = 9, return 3.

### Problem #468 <a id="p468"></a>

**Medium** · Facebook · 2021-03-30

#### Problem

Given an N by N matrix, rotate it by 90 degrees clockwise.

For example, given the following matrix:

```
[[1, 2, 3],
 [4, 5, 6],
 [7, 8, 9]]
```

you should return:

```
[[7, 4, 1],
 [8, 5, 2],
 [9, 6, 3]]
```

Follow-up: What if you couldn't use any extra space?

### Problem #469 <a id="p469"></a>

**Medium** · Facebook · 2021-03-31

#### Problem

Mastermind is a two-player game in which the first player attempts to guess the secret code of the second. In this version, the code may be any six-digit number with all distinct digits.

Each turn the first player guesses some number, and the second player responds by saying how many digits in this number correctly matched their location in the secret code. For example, if the secret code were 123456, then a guess of 175286 would score two, since 1 and 6 were correctly placed.

Write an algorithm which, given a sequence of guesses and their scores, determines whether there exists some secret code that could have produced them.

For example, for the following scores you should return True, since they correspond to the secret code 123456:

`{175286: 2, 293416: 3, 654321: 0}`

However, it is impossible for any key to result in the following scores, so in this case you should return False:

`{123456: 4, 345678: 4, 567890: 4}`

### Problem #470 <a id="p470"></a>

**Medium** · Google · 2021-04-01

#### Problem

Given an array of numbers and an index i, return the index of the nearest larger number of the number at index i, where distance is measured in array indices.

For example, given `[4, 1, 3, 5, 6]` and index 0, you should return 3.

If two distances to larger numbers are the equal, then return any one of them. If the array at i doesn't have a nearest larger integer, then return null.

Follow-up: If you can preprocess the array, can you do this in constant time?

### Problem #471 <a id="p471"></a>

**Easy** · Amazon · 2021-04-02

#### Problem

Given an integer N, construct all possible binary search trees with N nodes.

### Problem #472 <a id="p472"></a>

**Medium** · Facebook · 2021-04-03

#### Problem

Given the mapping a = 1, b = 2, ... z = 26, and an encoded message, count the number of ways it can be decoded.

For example, the message `'111'` would give 3, since it could be decoded as `'aaa'`, `'ka'`, and `'ak'`.

You can assume that the messages are decodable. For example, `'001'` is not allowed.

### Problem #473 <a id="p473"></a>

**Medium** · Yahoo · 2021-04-04

#### Problem

Write an algorithm that computes the reversal of a directed graph. For example, if a graph consists of A -> B -> C, it should become A \<- B \<- C.

### Problem #474 <a id="p474"></a>

**Hard** · Google · 2021-04-05

#### Problem

Find the minimum number of coins required to make n cents.

You can use standard American denominations, that is, 1¢, 5¢, 10¢, and 25¢.

For example, given n = 16, return 3 since we can make it with a 10¢, a 5¢, and a 1¢.

### Problem #475 <a id="p475"></a>

**Medium** · Google · 2021-04-06

#### Problem

Implement locking in a binary tree. A binary tree node can be locked or unlocked only if all of its descendants or ancestors are not locked.

Design a binary tree node class with the following methods:

- `is_locked`, which returns whether the node is locked
- lock, which attempts to lock the node. If it cannot be locked, then it should return false. Otherwise, it should lock it and return true.
- unlock, which unlocks the node. If it cannot be unlocked, then it should return false. Otherwise, it should unlock it and return true.

You may augment the node to add parent pointers or any other property you would like. You may assume the class is used in a single-threaded program, so there is no need for actual locks or mutexes. Each method should run in `O(h)`, where h is the height of the tree.

### Problem #476 <a id="p476"></a>

**Medium** · Google · 2021-04-07

#### Problem

You are given an array of length n + 1 whose elements belong to the set `{1, 2, ..., n}`. By the pigeonhole principle, there must be a duplicate. Find it in linear time and space.

### Problem #477 <a id="p477"></a>

**Easy** · Dropbox · 2021-04-08

#### Problem

What does the below code snippet print out? How can we fix the anonymous functions to behave as we'd expect?

```
functions = []
for i in range(10):
    functions.append(lambda : i)
```

```
for f in functions:
    print(f())
```

### Problem #478 <a id="p478"></a>

**Hard** · Google · 2021-04-09

#### Problem

Implement a file syncing algorithm for two computers over a low-bandwidth network. What if we know the files in the two computers are mostly the same?

### Problem #479 <a id="p479"></a>

**Easy** · Microsoft · 2021-04-10

#### Problem

Given a number in the form of a list of digits, return all possible permutations.

For example, given `[1,2,3]`, return `[[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]`.

### Problem #480 <a id="p480"></a>

**Medium** · Microsoft · 2021-04-11

#### Problem

Given a dictionary of words and a string made up of those words (no spaces), return the original sentence in a list. If there is more than one possible reconstruction, return any of them. If there is no possible reconstruction, then return null.

For example, given the set of words `'quick'`, `'brown'`, `'the'`, `'fox'`, and the string `"thequickbrownfox"`, you should return `['the', 'quick', 'brown', 'fox']`.

Given the set of words `'bed'`, `'bath'`, `'bedbath'`, `'and'`, `'beyond'`, and the string `"bedbathandbeyond"`, return either `['bed', 'bath', 'and', 'beyond]` or `['bedbath', 'and', 'beyond']`.

### Problem #481 <a id="p481"></a>

**Hard** · Jane Street · 2021-04-14

#### Problem

Given an arithmetic expression in Reverse Polish Notation, write a program to evaluate it.

The expression is given as a list of numbers and operands. For example: `[5, 3, '+']` should return 5 + 3 = 8.

For example, `[15, 7, 1, 1, '+', '-', '/', 3, '*', 2, 1, 1, '+', '+', '-']` should return 5, since it is equivalent to ((15 / (7 - (1 + 1))) \* 3) - (2 + (1 + 1)) = 5.

You can assume the given expression is always valid.

### Problem #482 <a id="p482"></a>

**Medium** · Google · 2021-04-15

#### Problem

Given a binary search tree and a range `[a, b]` (inclusive), return the sum of the elements of the binary search tree within the range.

For example, given the following tree:

```
    5
   / \
  3   8
 / \ / \
2  4 6  10
```

and the range `[4, 9]`, return 23 (5 + 4 + 6 + 8).

### Problem #483 <a id="p483"></a>

**Easy** · Bloomberg · 2021-04-16

#### Problem

There are N prisoners standing in a circle, waiting to be executed. The executions are carried out starting with the kth person, and removing every successive kth person going clockwise until there is no one left.

Given N and k, write an algorithm to determine where a prisoner should stand in order to be the last survivor.

For example, if N = 5 and k = 2, the order of executions would be `[2, 4, 1, 5, 3]`, so you should return 3.

Bonus: Find an `O(log N)` solution if k = 2.

### Problem #484 <a id="p484"></a>

**Medium** · Dropbox · 2021-04-17

#### Problem

Given the root to a binary search tree, find the second largest node in the tree.

### Problem #485 <a id="p485"></a>

**Hard** · Amazon · 2021-04-18

#### Problem

Given an integer k and a string s, find the length of the longest substring that contains at most k distinct characters.

For example, given s = `"abcba"` and k = 2, the longest substring with k distinct characters is `"bcb"`.

### Problem #486 <a id="p486"></a>

**Medium** · Pinterest · 2021-04-19

#### Problem

At a party, there is a single person who everyone knows, but who does not know anyone in return (the `"celebrity"`). To help figure out who this is, you have access to an `O(1)` method called `knows(a, b)`, which returns True if person a knows person b, else False.

Given a list of N people and the above operation, find a way to identify the celebrity in `O(N)` time.

### Problem #487 <a id="p487"></a>

**Medium** · Yext · 2021-04-20

#### Problem

Two nodes in a binary tree can be called cousins if they are on the same level of the tree but have different parents. For example, in the following diagram 4 and 6 are cousins.

```
    1
   / \
  2   3
 / \   \
4   5   6
```

Given a binary tree and a particular node, find all cousins of that node.

### Problem #488 <a id="p488"></a>

**Hard** · Netflix · 2021-04-21

#### Problem

Implement a queue using a set of fixed-length arrays.

The queue should support enqueue, dequeue, and `get_size` operations.

### Problem #489 <a id="p489"></a>

**Easy** · Google · 2021-04-22

#### Problem

Given an array of elements, return the length of the longest subarray where all its elements are distinct.

For example, given the array `[5, 1, 3, 5, 2, 3, 4, 1]`, return 5 as the longest subarray of distinct elements is `[5, 2, 3, 4, 1]`.

### Problem #490 <a id="p490"></a>

**Medium** · Yelp · 2021-04-23

#### Problem

The horizontal distance of a binary tree node describes how far left or right the node will be when the tree is printed out.

More rigorously, we can define it as follows:

- The horizontal distance of the root is 0.
- The horizontal distance of a left child is `hd(parent)` - 1.
- The horizontal distance of a right child is `hd(parent)` + 1.

For example, for the following tree, `hd(1)` = -2, and `hd(6)` = 0.

```
           5
        /     \
      3         7
    /  \      /   \
  1     4    6     9
 /                /
0                8
```

The bottom view of a tree, then, consists of the lowest node at each horizontal distance. If there are two nodes at the same depth and horizontal distance, either is acceptable.

For this tree, for example, the bottom view could be `[0, 1, 3, 6, 8, 9]`.

Given the root to a binary tree, return its bottom view.

### Problem #491 <a id="p491"></a>

**Easy** · Palantir · 2021-04-24

#### Problem

Write a program that checks whether an integer is a palindrome. For example, 121 is a palindrome, as well as 888. 678 is not a palindrome. Do not convert the integer into a string.

### Problem #492 <a id="p492"></a>

**Medium** · Google · 2021-04-25

#### Problem

Given an undirected graph represented as an adjacency matrix and an integer k, write a function to determine whether each vertex in the graph can be colored such that no two adjacent vertices share the same color using at most k colors.

### Problem #493 <a id="p493"></a>

**Medium** · Triplebyte · 2021-04-26

#### Problem

You are given n numbers as well as n probabilities that sum up to 1. Write a function to generate one of the numbers with its corresponding probability.

For example, given the numbers `[1, 2, 3, 4]` and probabilities `[0.1, 0.5, 0.2, 0.2]`, your function should return 1 10% of the time, 2 50% of the time, and 3 and 4 20% of the time.

You can generate random numbers between 0 and 1 uniformly.

### Problem #494 <a id="p494"></a>

**Medium** · Facebook · 2021-04-27

#### Problem

Given a circular array, compute its maximum subarray sum in `O(n)` time. A subarray can be empty, and in this case the sum is 0.

For example, given `[8, -1, 3, 4]`, return 15 as we choose the numbers 3, 4, and 8 where the 8 is obtained from wrapping around.

Given `[-4, 5, 1, 0]`, return 6 as we choose the numbers 5 and 1.

### Problem #495 <a id="p495"></a>

**Medium** · Facebook · 2021-04-28

#### Problem

Given a stream of elements too large to store in memory, pick a random element from the stream with uniform probability.

### Problem #496 <a id="p496"></a>

**Easy** · Pivotal · 2021-04-29

#### Problem

Write an algorithm that finds the total number of set bits in all integers between 1 and N.

### Problem #497 <a id="p497"></a>

**Medium** · Yahoo · 2021-04-30

#### Problem

Recall that a full binary tree is one in which each node is either a leaf node, or has two children. Given a binary tree, convert it to a full one by removing nodes with only one child.

For example, given the following tree:

```
         0
      /     \
    1         2
  /            \
3                 4
  \             /   \
    5          6     7
```

You should convert it to:

```
     0
  /     \
5         4
        /   \
       6     7
```

### Problem #498 <a id="p498"></a>

**Easy** · WhatsApp · 2021-05-01

#### Problem

Given an array of integers out of order, determine the bounds of the smallest window that must be sorted in order for the entire array to be sorted. For example, given `[3, 7, 5, 6, 9]`, you should return (1, 3).

### Problem #499 <a id="p499"></a>

**Easy** · Palantir · 2021-05-02

#### Problem

A typical American-style crossword puzzle grid is an N x N matrix with black and white squares, which obeys the following rules:

- Every white square must be part of an `"across"` word and a `"down"` word.
- No word can be fewer than three letters long.
- Every white square must be reachable from every other white square.
- The grid is rotationally symmetric (for example, the colors of the top left and bottom right squares must match).

Write a program to determine whether a given matrix qualifies as a crossword grid.

### Problem #500 <a id="p500"></a>

**Easy** · Google · 2021-05-03

#### Problem

You are given an M by N matrix consisting of booleans that represents a board. Each True boolean represents a wall. Each False boolean represents a tile you can walk on.

Given this matrix, a start coordinate, and an end coordinate, return the minimum number of steps required to reach the end coordinate from the start. If there is no possible path, then return null. You can move up, left, down, and right. You cannot move through walls. You cannot wrap around the edges of the board.

For example, given the following board:

```
[[f, f, f, f],
[t, t, f, t],
[f, f, f, f],
[f, f, f, f]]
```

and start = (3, 0) (bottom left) and end = (0, 0) (top left), the minimum number of steps required to reach the end is 7, since we would need to go through (1, 2) because there is a wall everywhere else on the second row.

### Problem #501 <a id="p501"></a>

**Medium** · Facebook · 2021-05-04

#### Problem

Given a function that generates perfectly random numbers between 1 and k (inclusive), where k is an input, write a function that shuffles a deck of cards represented as an array using only swaps.

It should run in `O(N)` time.

Hint: Make sure each one of the 52! permutations of the deck is equally likely.

### Problem #502 <a id="p502"></a>

**Easy** · PayPal · 2021-05-05

#### Problem

Given a binary tree, determine whether or not it is height-balanced. A height-balanced binary tree can be defined as one in which the heights of the two subtrees of any node never differ by more than one.

### Problem #503 <a id="p503"></a>

**Medium** · Google · 2021-05-06

#### Problem

Given a linked list, sort it in `O(n log n)` time and constant space.

For example, the linked list 4 -> 1 -> -3 -> 99 should become -3 -> 1 -> 4 -> 99.

### Problem #504 <a id="p504"></a>

**Easy** · Twitter · 2021-05-07

#### Problem

You run an e-commerce website and want to record the last N order ids in a log. Implement a data structure to accomplish this, with the following API:

- `record(order_id)`: adds the `order_id` to the log
- `get_last(i)`: gets the ith last element from the log. i is guaranteed to be smaller than or equal to N.

You should be as efficient with time and space as possible.

### Problem #505 <a id="p505"></a>

**Easy** · Amazon · 2021-05-08

#### Problem

Given an array and a number k that's smaller than the length of the array, rotate the array to the right k elements in-place.

### Problem #506 <a id="p506"></a>

**Medium** · Fitbit · 2021-05-09

#### Problem

Given a linked list, rearrange the node values such that they appear in alternating low -> high -> low -> high ... form. For example, given 1 -> 2 -> 3 -> 4 -> 5, you should return 1 -> 3 -> 2 -> 5 -> 4.

### Problem #507 <a id="p507"></a>

**Easy** · Uber · 2021-05-10

#### Problem

On election day, a voting machine writes data in the form (`voter_id`, `candidate_id`) to a text file. Write a program that reads this file as a stream and returns the top 3 candidates at any given time. If you find a voter voting more than once, report this as fraud.

### Problem #508 <a id="p508"></a>

**Medium** · Dropbox · 2021-05-11

#### Problem

Create an algorithm to efficiently compute the approximate median of a list of numbers.

More precisely, given an unordered list of N numbers, find an element whose rank is between N / 4 and 3 \* N / 4, with a high level of certainty, in less than `O(N)` time.

### Problem #509 <a id="p509"></a>

**Medium** · Quora · 2021-05-12

#### Problem

Given a string, find the palindrome that can be made by inserting the fewest number of characters as possible anywhere in the word. If there is more than one palindrome of minimum length that can be made, return the lexicographically earliest one (the first one alphabetically).

For example, given the string `"race"`, you should return `"ecarace"`, since we can add three letters to it (which is the smallest amount to make a palindrome). There are seven other palindromes that can be made from `"race"` by adding three letters, but `"ecarace"` comes first alphabetically.

As another example, given the string `"google"`, you should return `"elgoogle"`.

### Problem #510 <a id="p510"></a>

**Hard** · Airbnb · 2021-05-13

#### Problem

Given a list of words, find all pairs of unique indices such that the concatenation of the two words is a palindrome.

For example, given the list `["code", "edoc", "da", "d"]`, return `[(0, 1), (1, 0), (2, 3)]`.

### Problem #511 <a id="p511"></a>

**Medium** · Yelp · 2021-05-14

#### Problem

You are given an array of integers, where each element represents the maximum number of steps that can be jumped going forward from that element. Write a function to return the minimum number of jumps you must take in order to get from the start to the end of the array.

For example, given `[6, 2, 4, 0, 5, 1, 1, 4, 2, 9]`, you should return 2, as the optimal solution involves jumping from 6 to 5, and then from 5 to 9.

### Problem #512 <a id="p512"></a>

**Medium** · Google · 2021-05-15

#### Problem

You are given an array of nonnegative integers. Let's say you start at the beginning of the array and are trying to advance to the end. You can advance at most, the number of steps that you're currently on. Determine whether you can get to the end of the array.

For example, given the array `[1, 3, 1, 2, 0, 1]`, we can go from indices 0 -> 1 -> 3 -> 5, so return true.

Given the array `[1, 2, 1, 0, 0]`, we can't reach the end, so return false.

### Problem #513 <a id="p513"></a>

**Medium** · Lyft · 2021-05-16

#### Problem

Given a list of integers and a number K, return which contiguous elements of the list sum to K.

For example, if the list is `[1, 2, 3, 4, 5]` and K is 9, then it should return `[2, 3, 4]`, since 2 + 3 + 4 = 9.

### Problem #514 <a id="p514"></a>

**Medium** · Microsoft · 2021-05-17

#### Problem

Given an unsorted array of integers, find the length of the longest consecutive elements sequence.

For example, given `[100, 4, 200, 1, 3, 2]`, the longest consecutive element sequence is `[1, 2, 3, 4]`. Return its length: 4.

Your algorithm should run in `O(n)` complexity.

### Problem #515 <a id="p515"></a>

**Medium** · LinkedIn · 2021-05-18

#### Problem

Given a linked list of numbers and a pivot k, partition the linked list so that all nodes less than k come before nodes greater than or equal to k.

For example, given the linked list 5 -> 1 -> 8 -> 0 -> 3 and k = 3, the solution could be 1 -> 0 -> 5 -> 8 -> 3.

### Problem #516 <a id="p516"></a>

**Easy** · Zillow · 2021-05-19

#### Problem

Let's define a `"sevenish"` number to be one which is either a power of 7, or the sum of unique powers of 7. The first few sevenish numbers are 1, 7, 8, 49, and so on. Create an algorithm to find the nth sevenish number.

### Problem #517 <a id="p517"></a>

**Easy** · Google · 2021-05-20

#### Problem

Given two singly linked lists that intersect at some point, find the intersecting node. The lists are non-cyclical.

For example, given A = 3 -> 7 -> 8 -> 10 and B = 99 -> 1 -> 8 -> 10, return the node with value 8.

In this example, assume nodes with the same value are the exact same node objects.

Do this in `O(M + N)` time (where M and N are the lengths of the lists) and constant space.

### Problem #518 <a id="p518"></a>

**Easy** · Microsoft · 2021-05-21

#### Problem

Given an array of numbers and a number k, determine if there are three entries in the array which add up to the specified number k. For example, given `[20, 303, 3, 4, 25]` and k = 49, return true as 20 + 4 + 25 = 49.

### Problem #519 <a id="p519"></a>

**Medium** · Facebook · 2021-05-22

#### Problem

Given three 32-bit integers x, y, and b, return x if b is 1 and y if b is 0, using only mathematical or bit operations. You can assume b can only be 1 or 0.

### Problem #520 <a id="p520"></a>

**Hard** · LinkedIn · 2021-05-23

#### Problem

You are given a binary tree in a peculiar string representation. Each node is written in the form (lr), where l corresponds to the left child and r corresponds to the right child.

If either l or r is null, it will be represented as a zero. Otherwise, it will be represented by a new (lr) pair.

Here are a few examples:

- A root node with no children: (00)
- A root node with two children: ((00)(00))
- An unbalanced tree with three consecutive left children: ((((00)0)0)0)

Given this representation, determine the depth of the tree.

### Problem #521 <a id="p521"></a>

**Medium** · PayPal · 2021-05-24

#### Problem

Given a string and a number of lines k, print the string in zigzag form. In zigzag, characters are printed out diagonally from top left to bottom right until reaching the kth line, then back up to top right, and so on.

For example, given the sentence `"thisisazigzag"` and k = 4, you should print:

```
t     a     g
 h   s z   a
  i i   i z
   s     g
```

### Problem #522 <a id="p522"></a>

**Medium** · Microsoft · 2021-05-25

#### Problem

Given a string and a pattern, find the starting indices of all occurrences of the pattern in the string. For example, given the string `"abracadabra"` and the pattern `"abr"`, you should return `[0, 7]`.

### Problem #523 <a id="p523"></a>

**Easy** · Jane Street · 2021-05-26

#### Problem

Given integers M and N, write a program that counts how many positive integer pairs (a, b) satisfy the following conditions:

- a + b = M
- a XOR b = N

### Problem #524 <a id="p524"></a>

**Medium** · Amazon · 2021-05-27

#### Problem

Given an array of numbers, find the maximum sum of any contiguous subarray of the array.

For example, given the array `[34, -50, 42, 14, -5, 86]`, the maximum sum would be 137, since we would take elements 42, 14, -5, and 86.

Given the array `[-5, -1, -8, -9]`, the maximum sum would be 0, since we would not take any elements.

Do this in `O(N)` time.

### Problem #525 <a id="p525"></a>

**Easy** · Amazon · 2021-05-28

#### Problem

Given a N by M matrix of numbers, print out the matrix in a clockwise spiral.

For example, given the following matrix:

```
[[1,  2,  3,  4,  5],
 [6,  7,  8,  9,  10],
 [11, 12, 13, 14, 15],
 [16, 17, 18, 19, 20]]
```

You should print out the following:

```
1
2
3
4
5
10
15
20
19
18
17
16
11
6
7
8
9
14
13
12
```

### Problem #526 <a id="p526"></a>

**Easy** · Yahoo · 2021-05-29

#### Problem

You are given a string of length N and a parameter k. The string can be manipulated by taking one of the first k letters and moving it to the end.

Write a program to determine the lexicographically smallest string that can be created after an unlimited number of moves.

For example, suppose we are given the string daily and k = 1. The best we can create in this case is ailyd.

### Problem #527 <a id="p527"></a>

**Medium** · Microsoft · 2021-05-30

#### Problem

Write a program to determine how many distinct ways there are to create a max heap from a list of N given integers.

For example, if N = 3, and our integers are `[1, 2, 3]`, there are two ways, shown below.

```
  3      3
 / \    / \
1   2  2   1
```

### Problem #528 <a id="p528"></a>

**Easy** · Amazon · 2021-05-31

#### Problem

Huffman coding is a method of encoding characters based on their frequency. Each letter is assigned a variable-length binary string, such as 0101 or 111110, where shorter lengths correspond to more common letters. To accomplish this, a binary tree is built such that the path from the root to any leaf uniquely maps to a character. When traversing the path, descending to a left child corresponds to a 0 in the prefix, while descending right corresponds to 1.

Here is an example tree (note that only the leaf nodes have letters):

```
        *
      /   \
    *       *
   / \     / \
  *   a   t   *
 /             \
c               s
```

With this encoding, cats would be represented as 0000110111.

Given a dictionary of character frequencies, build a Huffman tree, and use it to determine a mapping between characters and their encoded binary strings.

### Problem #529 <a id="p529"></a>

**Hard** · Google · 2021-06-01

#### Problem

Given a string, split it into as few strings as possible such that each string is a palindrome.

For example, given the input string racecarannakayak, return `["racecar", "anna", "kayak"]`.

Given the input string abc, return `["a", "b", "c"]`.

### Problem #531 <a id="p531"></a>

**Easy** · Microsoft · 2021-06-03

#### Problem

Using a `read7()` method that returns 7 characters from a file, implement `readN(n)` which reads n characters.

For example, given a file with the content “Hello world”, three `read7()` returns “Hello w”, “orld” and then “”.

### Problem #532 <a id="p532"></a>

**Medium** · Google · 2021-06-04

#### Problem

On our special chessboard, two bishops attack each other if they share the same diagonal. This includes bishops that have another bishop located between them, i.e. bishops can attack through pieces.

You are given N bishops, represented as (row, column) tuples on a M by M chessboard. Write a function to count the number of pairs of bishops that attack each other. The ordering of the pair doesn't matter: (1, 2) is considered the same as (2, 1).

For example, given M = 5 and the list of bishops:

- (0, 0)
- (1, 2)
- (2, 2)
- (4, 0)

The board would look like this:

```
[b 0 0 0 0]
[0 0 b 0 0]
[0 0 b 0 0]
[0 0 0 0 0]
[b 0 0 0 0]
```

You should return 2, since bishops 1 and 3 attack each other, as well as bishops 3 and 4.

### Problem #533 <a id="p533"></a>

**Easy** · Facebook · 2021-06-05

#### Problem

Boggle is a game played on a 4 x 4 grid of letters. The goal is to find as many words as possible that can be formed by a sequence of adjacent letters in the grid, using each cell at most once. Given a game board and a dictionary of valid words, implement a Boggle solver.

### Problem #534 <a id="p534"></a>

**Easy** · Amazon · 2021-06-06

#### Problem

Given a string, determine whether any permutation of it is a palindrome.

For example, carrace should return true, since it can be rearranged to form racecar, which is a palindrome. daily should return false, since there's no rearrangement that can form a palindrome.

### Problem #535 <a id="p535"></a>

**Medium** · Goldman Sachs · 2021-06-07

#### Problem

You are given N identical eggs and access to a building with k floors. Your task is to find the lowest floor that will cause an egg to break, if dropped from that floor. Once an egg breaks, it cannot be dropped again. If an egg breaks when dropped from the xth floor, you can assume it will also break when dropped from any floor greater than x.

Write an algorithm that finds the minimum number of trial drops it will take, in the worst case, to identify this floor.

For example, if N = 1 and k = 5, we will need to try dropping the egg at every floor, beginning with the first, until we reach the fifth floor, so our solution will be 5.

### Problem #536 <a id="p536"></a>

**Medium** · Google · 2021-06-09

#### Problem

Given the sequence of keys visited by a postorder traversal of a binary search tree, reconstruct the tree.

For example, given the sequence 2, 4, 3, 8, 7, 5, you should construct the following tree:

```
    5
   / \
  3   7
 / \   \
2   4   8
```

### Problem #537 <a id="p537"></a>

**Easy** · Apple · 2021-06-10

#### Problem

A Collatz sequence in mathematics can be defined as follows. Starting with any positive integer:

- if n is even, the next number in the sequence is n / 2
- if n is odd, the next number in the sequence is 3n + 1

It is conjectured that every such sequence eventually reaches the number 1. Test this conjecture.

Bonus: What input n \<= 1000000 gives the longest sequence?

### Problem #538 <a id="p538"></a>

**Hard** · LinkedIn · 2021-06-11

#### Problem

Given a set of characters C and an integer k, a De Bruijn sequence is a cyclic sequence in which every possible k-length string of characters in C occurs exactly once.

For example, suppose C = `{0, 1}` and k = 3. Then our sequence should contain the substrings `{'000', '001', '010', '011', '100', '101', '110', '111'}`, and one possible solution would be 00010111.

Create an algorithm that finds a De Bruijn sequence.

### Problem #539 <a id="p539"></a>

**Easy** · Pandora · 2021-06-12

#### Problem

Given an undirected graph, determine if it contains a cycle.

### Problem #540 <a id="p540"></a>

**Easy** · Morgan Stanley · 2021-06-13

#### Problem

In Ancient Greece, it was common to write text with the first line going left to right, the second line going right to left, and continuing to go back and forth. This style was called `"boustrophedon"`.

Given a binary tree, write an algorithm to print the nodes in boustrophedon order.

For example, given the following tree:

```
       1
    /     \
  2         3
 / \       / \
4   5     6   7
```

You should return `[1, 3, 2, 4, 5, 6, 7]`.

### Problem #541 <a id="p541"></a>

**Easy** · Amazon · 2021-06-14

#### Problem

Run-length encoding is a fast and simple method of encoding strings. The basic idea is to represent repeated successive characters as a single count and character. For example, the string `"AAAABBBCCDAA"` would be encoded as `"4A3B2C1D2A"`.

Implement run-length encoding and decoding. You can assume the string to be encoded have no digits and consists solely of alphabetic characters. You can assume the string to be decoded is valid.

### Problem #542 <a id="p542"></a>

**Medium** · Dropbox · 2021-06-15

#### Problem

Given an undirected graph G, check whether it is bipartite. Recall that a graph is bipartite if its vertices can be divided into two independent sets, U and V, such that no edge connects vertices of the same set.

### Problem #543 <a id="p543"></a>

**Medium** · Google · 2021-06-16

#### Problem

Given a singly linked list and an integer k, remove the kth last element from the list. k is guaranteed to be smaller than the length of the list.

The list is very long, so making more than one pass is prohibitively expensive.

Do this in constant space and in one pass.

### Problem #544 <a id="p544"></a>

**Hard** · Google · 2021-06-17

#### Problem

Given a list of integers S and a target number k, write a function that returns a subset of S that adds up to k. If such a subset cannot be made, then return null.

Integers can appear more than once in the list. You may assume all numbers in the list are positive.

For example, given S = `[12, 1, 61, 5, 9, 2]` and k = 24, return `[12, 9, 2, 1]` since it sums up to 24.

### Problem #545 <a id="p545"></a>

**Hard** · Twitter · 2021-06-18

#### Problem

Given a binary tree, find the lowest common ancestor (LCA) of two given nodes in the tree. Assume that each node in the tree also has a pointer to its parent.

According to the definition of LCA on Wikipedia: “The lowest common ancestor is defined between two nodes v and w as the lowest node in T that has both v and w as descendants (where we allow a node to be a descendant of itself).”

### Problem #546 <a id="p546"></a>

**Medium** · Google · 2021-06-19

#### Problem

Given an array of integers, return a new array where each element in the new array is the number of smaller elements to the right of that element in the original input array.

For example, given the array `[3, 4, 9, 6, 1]`, return `[1, 1, 2, 1, 0]`, since:

- There is 1 smaller element to the right of 3
- There is 1 smaller element to the right of 4
- There are 2 smaller elements to the right of 9
- There is 1 smaller element to the right of 6
- There are no smaller elements to the right of 1

### Problem #547 <a id="p547"></a>

**Hard** · Salesforce · 2021-06-20

#### Problem

Given an array of integers, find the maximum XOR of any two elements.

### Problem #548 <a id="p548"></a>

**Easy** · Microsoft · 2021-06-21

#### Problem

Given a clock time in hh:mm format, determine, to the nearest degree, the angle between the hour and the minute hands.

Bonus: When, during the course of a day, will the angle be zero?

### Problem #549 <a id="p549"></a>

**Hard** · Google · 2021-06-22

#### Problem

Given an array of integers where every integer occurs three times except for one integer, which only occurs once, find and return the non-duplicated integer.

For example, given `[6, 1, 3, 3, 3, 6, 6]`, return 1. Given `[13, 19, 13, 13]`, return 19.

Do this in `O(N)` time and `O(1)` space.

### Problem #550 <a id="p550"></a>

**Hard** · Jane Street · 2021-06-23

#### Problem

Suppose you are given a table of currency exchange rates, represented as a 2D array. Determine whether there is a possible arbitrage: that is, whether there is some sequence of trades you can make, starting with some amount A of any currency, so that you can end up with some amount greater than A of that currency.

There are no transaction costs and you can trade fractional quantities.

### Problem #551 <a id="p551"></a>

**Medium** · Square · 2021-06-24

#### Problem

Assume you have access to a function `toss_biased()` which returns 0 or 1 with a probability that's not 50-50 (but also not 0-100 or 100-0). You do not know the bias of the coin.

Write a function to simulate an unbiased coin toss.

### Problem #552 <a id="p552"></a>

**Easy** · Wayfair · 2021-06-25

#### Problem

You are given a 2 x N board, and instructed to completely cover the board with the following shapes:

- Dominoes, or 2 x 1 rectangles.
- Trominoes, or L-shapes.

For example, if N = 4, here is one possible configuration, where A is a domino, and B and C are trominoes.

```
A B B C
A B C C
```

Given an integer N, determine in how many ways this task is possible.

### Problem #553 <a id="p553"></a>

**Medium** · Google · 2021-06-26

#### Problem

You are given an N by M 2D matrix of lowercase letters. Determine the minimum number of columns that can be removed to ensure that each row is ordered from top to bottom lexicographically. That is, the letter at each column is lexicographically later as you go down each row. It does not matter whether each row itself is ordered lexicographically.

For example, given the following table:

```
cba
daf
ghi
```

This is not ordered because of the a in the center. We can remove the second column to make it ordered:

```
ca
df
gi
```

So your function should return 1, since we only needed to remove 1 column.

As another example, given the following table:

abcdef

Your function should return 0, since the rows are already ordered (there's only one row).

As another example, given the following table:

```
zyx
wvu
tsr
```

Your function should return 3, since we would need to remove all the columns to order it.

### Problem #554 <a id="p554"></a>

**Easy** · Palantir · 2021-06-27

#### Problem

The ancient Egyptians used to express fractions as a sum of several terms where each numerator is one. For example, 4 / 13 can be represented as 1 / 4 + 1 / 18 + 1 / 468.

Create an algorithm to turn an ordinary fraction a / b, where a \< b, into an Egyptian fraction.

### Problem #555 <a id="p555"></a>

**Medium** · Amazon · 2021-06-28

#### Problem

An sorted array of integers was rotated an unknown number of times.

Given such an array, find the index of the element in the array in faster than linear time. If the element doesn't exist in the array, return null.

For example, given the array `[13, 18, 25, 2, 8, 10]` and the element 8, return 4 (the index of 8 in the array).

You can assume all the integers in the array are unique.

### Problem #556 <a id="p556"></a>

**Medium** · Facebook · 2021-06-30

#### Problem

Given an array of integers, write a function to determine whether the array could become non-decreasing by modifying at most 1 element.

For example, given the array `[10, 5, 7]`, you should return true, since we can modify the 10 into a 1 to make the array non-decreasing.

Given the array `[10, 5, 1]`, you should return false, since we can't modify any one element to get a non-decreasing array.

### Problem #557 <a id="p557"></a>

**Medium** · Apple · 2021-07-01

#### Problem

Suppose you have a multiplication table that is N by N. That is, a 2D array where the value at the i-th row and j-th column is (i + 1) \* (j + 1) (if 0-indexed) or i \* j (if 1-indexed).

Given integers N and X, write a function that returns the number of times X appears as a value in an N by N multiplication table.

For example, given N = 6 and X = 12, you should return 4, since the multiplication table looks like this:

| 1 | 2 | 3 | 4 | 5 | 6 |

| 2 | 4 | 6 | 8 | 10 | 12 |

| 3 | 6 | 9 | 12 | 15 | 18 |

| 4 | 8 | 12 | 16 | 20 | 24 |

| 5 | 10 | 15 | 20 | 25 | 30 |

| 6 | 12 | 18 | 24 | 30 | 36 |

And there are 4 12's in the table.

### Problem #558 <a id="p558"></a>

**Medium** · Google · 2021-07-02

#### Problem

The area of a circle is defined as πr^2. Estimate π to 3 decimal places using a Monte Carlo method.

Hint: The basic equation of a circle is x2 + y2 = r2.

### Problem #559 <a id="p559"></a>

**Medium** · Google · 2021-07-03

#### Problem

Given k sorted singly linked lists, write a function to merge all the lists into one sorted singly linked list.

### Problem #560 <a id="p560"></a>

**Easy** · Google · 2021-07-04

#### Problem

Given a list of numbers and a number k, return whether any two numbers from the list add up to k.

For example, given `[10, 15, 3, 7]` and k of 17, return true since 10 + 7 is 17.

Bonus: Can you do this in one pass?

### Problem #561 <a id="p561"></a>

**Hard** · Etsy · 2021-07-05

#### Problem

Given a sorted array, convert it into a height-balanced binary search tree.

### Problem #562 <a id="p562"></a>

**Hard** · Uber · 2021-07-06

#### Problem

Given an array of integers, return a new array such that each element at index i of the new array is the product of all the numbers in the original array except the one at i.

For example, if our input was `[1, 2, 3, 4, 5]`, the expected output would be `[120, 60, 40, 30, 24]`. If our input was `[3, 2, 1]`, the expected output would be `[2, 3, 6]`.

Follow-up: what if you can't use division?

### Problem #563 <a id="p563"></a>

**Hard** · Nvidia · 2021-07-07

#### Problem

Find the maximum of two numbers without using any if-else statements, branching, or direct comparisons.

### Problem #564 <a id="p564"></a>

**Hard** · Airbnb · 2021-07-08

#### Problem

Given a list of integers, write a function that returns the largest sum of non-adjacent numbers. Numbers can be 0 or negative.

For example, `[2, 4, 6, 2, 5]` should return 13, since we pick 2, 6, and 5. `[5, 1, 1, 5]` should return 10, since we pick 5 and 5.

Follow-up: Can you do this in `O(N)` time and constant space?

### Problem #565 <a id="p565"></a>

**Medium** · Pinterest · 2021-07-09

#### Problem

Given an integer list where each number represents the number of hops you can make, determine whether you can reach to the last index starting at index 0.

For example, `[2, 0, 1, 0]` returns True while `[1, 1, 0, 1]` returns False.

### Problem #566 <a id="p566"></a>

**Medium** · Facebook · 2021-07-10

#### Problem

A graph is minimally-connected if it is connected and there is no edge that can be removed while still leaving the graph connected. For example, any binary tree is minimally-connected.

Given an undirected graph, check if the graph is minimally-connected. You can choose to represent the graph as either an adjacency matrix or adjacency list.

### Problem #567 <a id="p567"></a>

**Medium** · Jane Street · 2021-07-11

#### Problem

`cons(a, b)` constructs a pair, and `car(pair)` and `cdr(pair)` returns the first and last element of that pair. For example, `car(cons(3, 4))` returns 3, and `cdr(cons(3, 4))` returns 4.

Given this implementation of cons:

```
def cons(a, b):
    def pair(f):
        return f(a, b)
    return pair
```

Implement car and cdr.

### Problem #568 <a id="p568"></a>

**Easy** · Google · 2021-07-12

#### Problem

Given a sorted list of integers, square the elements and give the output in sorted order.

For example, given `[-9, -2, 0, 2, 3]`, return `[0, 4, 4, 9, 81]`.

### Problem #569 <a id="p569"></a>

**Hard** · Facebook · 2021-07-13

#### Problem

Given an array of numbers of length N, find both the minimum and maximum using less than 2 \* (N - 2) comparisons.

### Problem #570 <a id="p570"></a>

**Hard** · Google · 2021-07-14

#### Problem

Given two non-empty binary trees s and t, check whether tree t has exactly the same structure and node values with a subtree of s. A subtree of s is a tree consists of a node in s and all of this node's descendants. The tree s could also be considered as a subtree of itself.

### Problem #571 <a id="p571"></a>

**Medium** · 2021-07-15

#### Problem

The Tower of Hanoi is a puzzle game with three rods and n disks, each a different size.

All the disks start off on the first rod in a stack. They are ordered by size, with the largest disk on the bottom and the smallest one at the top.

The goal of this puzzle is to move all the disks from the first rod to the last rod while following these rules:

- You can only move one disk at a time.
- A move consists of taking the uppermost disk from one of the stacks and placing it on top of another stack.
- You cannot place a larger disk on top of a smaller disk.

Write a function that prints out all the steps necessary to complete the Tower of Hanoi. You should assume that the rods are numbered, with the first rod being 1, the second (auxiliary) rod being 2, and the last (goal) rod being 3.

For example, with n = 3, we can do this in 7 moves:

```
Move 1 to 3
Move 1 to 2
Move 3 to 2
Move 1 to 3
Move 2 to 1
Move 2 to 3
Move 1 to 3
```

### Problem #572 <a id="p572"></a>

**Hard** · Palantir · 2021-07-16

#### Problem

Given a number represented by a list of digits, find the next greater permutation of a number, in terms of lexicographic ordering. If there is not greater permutation possible, return the permutation with the lowest value/ordering.

For example, the list `[1,2,3]` should return `[1,3,2]`. The list `[1,3,2]` should return `[2,1,3]`. The list `[3,2,1]` should return `[1,2,3]`.

Can you perform the operation without allocating extra memory (disregarding the input memory)?

### Problem #573 <a id="p573"></a>

**Medium** · Google · 2021-07-17

#### Problem

Given a stack of N elements, interleave the first half of the stack with the second half reversed using only one other queue. This should be done in-place.

Recall that you can only push or pop from a stack, and enqueue or dequeue from a queue.

For example, if the stack is `[1, 2, 3, 4, 5]`, it should become `[1, 5, 2, 4, 3]`. If the stack is `[1, 2, 3, 4]`, it should become `[1, 4, 2, 3]`.

Hint: Try working backwards from the end state.

### Problem #574 <a id="p574"></a>

**Medium** · Amazon · 2021-07-18

#### Problem

Implement a bit array.

A bit array is a space efficient array that holds a value of 1 or 0 at each index.

- `init(size)`: initialize the array with size
- `set(i, val)`: updates index at i with val where val is either 1 or 0.
- `get(i)`: gets the value at index i.

### Problem #575 <a id="p575"></a>

**Medium** · Uber · 2021-07-19

#### Problem

Implement a 2D iterator class. It will be initialized with an array of arrays, and should implement the following methods:

- `next()`: returns the next element in the array of arrays. If there are no more elements, raise an exception.
- `has_next()`: returns whether or not the iterator still has elements left.

For example, given the input `[[1, 2], [3], [], [4, 5, 6]]`, calling `next()` repeatedly should output 1, 2, 3, 4, 5, 6.

Do not use flatten or otherwise clone the arrays. Some of the arrays can be empty.

### Problem #576 <a id="p576"></a>

**Medium** · Spotify · 2021-07-20

#### Problem

Write a function, `throw_dice(N, faces, total)`, that determines how many ways it is possible to throw N dice with some number of faces each to get a specific total.

For example, `throw_dice(3, 6, 7)` should equal 15.

### Problem #577 <a id="p577"></a>

**Medium** · Dropbox · 2021-07-21

#### Problem

Given a list of words, determine whether the words can be chained to form a circle. A word X can be placed in front of another word Y in a circle if the last character of X is same as the first character of Y.

For example, the words `['chair', 'height', 'racket', touch', 'tunic']` can form the following circle: chair --> racket --> touch --> height --> tunic --> chair.

### Problem #578 <a id="p578"></a>

**Easy** · Bloomberg · 2021-07-22

#### Problem

Determine whether there exists a one-to-one character mapping from one string s1 to another s2.

For example, given s1 = abc and s2 = bcd, return true since we can map a to b, b to c, and c to d.

Given s1 = foo and s2 = bar, return false since the o cannot map to two characters.

### Problem #579 <a id="p579"></a>

**Medium** · Flipkart · 2021-07-23

#### Problem

Starting from 0 on a number line, you would like to make a series of jumps that lead to the integer N.

On the ith jump, you may move exactly i places to the left or right.

Find a path with the fewest number of jumps required to get from 0 to N.

### Problem #580 <a id="p580"></a>

**Easy** · Apple · 2021-07-24

#### Problem

Given a binary tree, find a minimum path sum from root to a leaf.

For example, the minimum path in this tree is `[10, 5, 1, -1]`, which has sum 15.

```
  10
 /  \
5    5
 \     \
   2    1
       /
     -1
```

### Problem #581 <a id="p581"></a>

**Easy** · Google · 2021-07-25

#### Problem

Given two rectangles on a 2D graph, return the area of their intersection. If the rectangles don't intersect, return 0.

For example, given the following rectangles:

```
{
    "top_left": (1, 4),
    "dimensions": (3, 3) # width, height
}
```

and

```
{
    "top_left": (0, 5),
    "dimensions": (4, 3) # width, height
}
```

return 6.

### Problem #582 <a id="p582"></a>

**Hard** · Microsoft · 2021-07-26

#### Problem

Let X be a set of n intervals on the real line. We say that a set of points P `"stabs"` X if every interval in X contains at least one point in P. Compute the smallest set of points that stabs X.

For example, given the intervals `[(1, 4), (4, 5), (7, 9), (9, 12)]`, you should return `[4, 9]`.

### Problem #583 <a id="p583"></a>

**Medium** · Zillow · 2021-07-28

#### Problem

You are given a 2-d matrix where each cell represents number of coins in that cell. Assuming we start at `matrix[0][0]`, and can only move right or down, find the maximum number of coins you can collect by the bottom right corner.

For example, in this matrix

```
0 3 1 1
2 0 0 4
1 5 3 1
```

The most we can collect is 0 + 2 + 1 + 5 + 3 + 1 = 12 coins.

### Problem #584 <a id="p584"></a>

**Easy** · IBM · 2021-07-29

#### Problem

Given a string with repeated characters, rearrange the string so that no two adjacent characters are the same. If this is not possible, return None.

For example, given `"aaabbc"`, you could return `"ababac"`. Given `"aaab"`, return None.

### Problem #585 <a id="p585"></a>

**Medium** · Google · 2021-07-30

#### Problem

Given an N by M matrix consisting only of 1's and 0's, find the largest rectangle containing only 1's and return its area.

For example, given the following matrix:

```
[[1, 0, 0, 0],
 [1, 0, 1, 1],
 [1, 0, 1, 1],
 [0, 1, 0, 0]]
```

Return 4.

### Problem #586 <a id="p586"></a>

**Medium** · Quora · 2021-07-31

#### Problem

You are given a list of (website, user) pairs that represent users visiting websites. Come up with a program that identifies the top k pairs of websites with the greatest similarity.

For example, suppose k = 1, and the list of tuples is:

```
[('a', 1), ('a', 3), ('a', 5),
 ('b', 2), ('b', 6),
 ('c', 1), ('c', 2), ('c', 3), ('c', 4), ('c', 5)
 ('d', 4), ('d', 5), ('d', 6), ('d', 7),
 ('e', 1), ('e', 3), ('e': 5), ('e', 6)]
```

Then a reasonable similarity metric would most likely conclude that a and e are the most similar, so your program should return `[('a', 'e')]`.

### Problem #587 <a id="p587"></a>

**Medium** · Facebook · 2021-08-01

#### Problem

Given a binary tree, return all paths from the root to leaves.

For example, given the tree:

```
  1
 / \
2   3
   / \
  4   5
```

Return `[[1, 2], [1, 3, 4], [1, 3, 5]]`.

### Problem #588 <a id="p588"></a>

**Easy** · Facebook · 2021-08-02

#### Problem

You have a large array with most of the elements as zero.

Use a more space-efficient data structure, SparseArray, that implements the same interface:

- `init(arr, size)`: initialize with the original large array and size.
- `set(i, val)`: updates index at i with val.
- `get(i)`: gets the value at index i.

### Problem #589 <a id="p589"></a>

**Hard** · Two Sigma · 2021-08-03

#### Problem

Alice wants to join her school's Probability Student Club. Membership dues are computed via one of two simple probabilistic games.

The first game: roll a die repeatedly. Stop rolling once you get a five followed by a six. Your number of rolls is the amount you pay, in dollars.

The second game: same, except that the stopping condition is a five followed by a five.

Which of the two games should Alice elect to play? Does it even matter? Write a program to simulate the two games and calculate their expected value.

### Problem #590 <a id="p590"></a>

**Hard** · Google · 2021-08-04

#### Problem

An XOR linked list is a more memory efficient doubly linked list. Instead of each node holding next and prev fields, it holds a field named both, which is an XOR of the next node and the previous node. Implement an XOR linked list; it has an `add(element)` which adds the element to the end, and a `get(index)` which returns the node at index.

If using a language that has no pointers (such as Python), you can assume you have access to `get_pointer` and `dereference_pointer` functions that converts between nodes and memory addresses.

### Problem #591 <a id="p591"></a>

**Hard** · Quora · 2021-08-05

#### Problem

Word sense disambiguation is the problem of determining which sense a word takes on in a particular setting, if that word has multiple meanings. For example, in the sentence `"I went to get money from the bank"`, bank probably means the place where people deposit money, not the land beside a river or lake.

Suppose you are given a list of meanings for several words, formatted like so:

```
{
    "word_1": ["meaning one", "meaning two", ...],
    ...
    "word_n": ["meaning one", "meaning two", ...]
}
```

Given a sentence, most of whose words are contained in the meaning list above, create an algorithm that determines the likely sense of each possibly ambiguous word.

### Problem #592 <a id="p592"></a>

**Medium** · Amazon · 2021-08-06

#### Problem

Given a matrix of 1s and 0s, return the number of `"islands"` in the matrix. A 1 represents land and 0 represents water, so an island is a group of 1s that are neighboring whose perimeter is surrounded by water.

For example, this matrix has 4 islands.

```
1 0 0 0 0
0 0 1 1 0
0 1 1 0 0
0 0 0 0 0
1 1 0 0 1
1 1 0 0 1
```

### Problem #593 <a id="p593"></a>

**Medium** · Mailchimp · 2021-08-07

#### Problem

You are given an array representing the heights of neighboring buildings on a city street, from east to west. The city assessor would like you to write an algorithm that returns how many of these buildings have a view of the setting sun, in order to properly value the street.

For example, given the array `[3, 7, 8, 3, 6, 1]`, you should return 3, since the top floors of the buildings with heights 8, 6, and 1 all have an unobstructed view to the west.

Can you do this using just one forward pass through the array?

### Problem #594 <a id="p594"></a>

**Easy** · Google · 2021-08-08

#### Problem

You are given an N by N matrix of random letters and a dictionary of words. Find the maximum number of words that can be packed on the board from the given dictionary.

A word is considered to be able to be packed on the board if:

- It can be found in the dictionary
- It can be constructed from untaken letters by other words found so far on the board
- The letters are adjacent to each other (vertically and horizontally, not diagonally).

Each tile can be visited only once by any word.

For example, given the following dictionary:

`{ 'eat', 'rain', 'in', 'rat' }`

and matrix:

```
[['e', 'a', 'n'],
 ['t', 't', 'i'],
 ['a', 'r', 'a']]
```

Your function should return 3, since we can make the words `'eat'`, `'in'`, and `'rat'` without them touching each other. We could have alternatively made `'eat'` and `'rain'`, but that would be incorrect since that's only 2 words.

### Problem #595 <a id="p595"></a>

**Medium** · Salesforce · 2021-08-09

#### Problem

The number 6174 is known as Kaprekar's contant, after the mathematician who discovered an associated property: for all four-digit numbers with at least two distinct digits, repeatedly applying a simple procedure eventually results in this value. The procedure is as follows:

- For a given input x, create two new numbers that consist of the digits in x in ascending and descending order.
- Subtract the smaller number from the larger number.

For example, this algorithm terminates in three steps when starting from 1234:

- 4321 - 1234 = 3087
- 8730 - 0378 = 8352
- 8532 - 2358 = 6174

Write a function that returns how many steps this will take for a given input N.

### Problem #596 <a id="p596"></a>

**Medium** · Google · 2021-08-10

#### Problem

Invert a binary tree.

For example, given the following tree:

```
    a
   / \
  b   c
 / \  /
d   e f
```

should become:

```
 a
/ \
c  b
\  / \
 f e  d
```

### Problem #597 <a id="p597"></a>

**Easy** · Netflix · 2021-08-11

#### Problem

Given an array of integers, determine whether it contains a Pythagorean triplet. Recall that a Pythogorean triplet (a, b, c) is defined by the equation a2+ b2= c2.

### Problem #598 <a id="p598"></a>

**Easy** · Microsoft · 2021-08-12

#### Problem

You have n fair coins and you flip them all at the same time. Any that come up tails you set aside. The ones that come up heads you flip again. How many rounds do you expect to play before only one coin remains?

Write a function that, given n, returns the number of rounds you'd expect to play until one coin remains.

### Problem #599 <a id="p599"></a>

**Hard** · Two Sigma · 2021-08-13

#### Problem

Ghost is a two-person word game where players alternate appending letters to a word. The first person who spells out a word, or creates a prefix for which there is no possible continuation, loses. Here is a sample game:

- Player 1: g
- Player 2: h
- Player 1: o
- Player 2: s
- Player 1: t \[loses\]

Given a dictionary of words, determine the letters the first player should start with, such that with optimal play they cannot lose.

For example, if the dictionary is `["cat", "calf", "dog", "bear"]`, the only winning start letter would be b.

### Problem #600 <a id="p600"></a>

**Easy** · Google · 2021-08-14

#### Problem

Given a set of points (x, y) on a 2D cartesian plane, find the two closest points. For example, given the points `[(1, 1), (-1, -1), (3, 4), (6, 1), (-1, -6), (-4, -3)]`, return `[(-1, -1), (1, 1)]`.

### Problem #601 <a id="p601"></a>

**Medium** · Pinterest · 2021-08-15

#### Problem

The sequence `[0, 1, ..., N]` has been jumbled, and the only clue you have for its order is an array representing whether each number is larger or smaller than the last. Given this information, reconstruct an array that is consistent with it. For example, given `[None, +, +, -, +]`, you could return `[1, 2, 3, 0, 4]`.

### Problem #602 <a id="p602"></a>

**Easy** · Facebook · 2021-08-16

#### Problem

Suppose you are given two lists of n points, one list p1, p2, ..., pn on the line y = 0 and the other list q1, q2, ..., qn on the line y = 1. Imagine a set of n line segments connecting each point pi to qi. Write an algorithm to determine how many pairs of the line segments intersect.

### Problem #603 <a id="p603"></a>

**Easy** · Microsoft · 2021-08-17

#### Problem

You are given an string representing the initial conditions of some dominoes. Each element can take one of three values:

- L, meaning the domino has just been pushed to the left,
- R, meaning the domino has just been pushed to the right, or
- ., meaning the domino is standing still.

Determine the orientation of each tile when the dominoes stop falling. Note that if a domino receives a force from the left and right side simultaneously, it will remain upright.

For example, given the string .L.R....L, you should return LL.RRRLLL.

Given the string ..R...L.L, you should return ..RR.LLLL.

### Problem #604 <a id="p604"></a>

**Hard** · Grammarly · 2021-08-18

#### Problem

Soundex is an algorithm used to categorize phonetically, such that two names that sound alike but are spelled differently have the same representation.

Soundex maps every name to a string consisting of one letter and three numbers, like M460.

One version of the algorithm is as follows:

- Remove consecutive consonants with the same sound (for example, change ck -> c).
- Keep the first letter. The remaining steps only apply to the rest of the string.
- Remove all vowels, including y, w, and h.
- Replace all consonants with the following digits:

- b, f, p, v → 1
- c, g, j, k, q, s, x, z → 2
- d, t → 3
- l → 4
- m, n → 5
- r → 6

- If you don't have three numbers yet, append zeros until you do. Keep the first three numbers.

Using this scheme, Jackson and Jaxen both map to J250.

Implement Soundex.

### Problem #605 <a id="p605"></a>

**Easy** · Palantir · 2021-08-19

#### Problem

In academia, the h-index is a metric used to calculate the impact of a researcher's papers. It is calculated as follows:

A researcher has index h if at least h of her N papers have h citations each. If there are multiple h satisfying this formula, the maximum is chosen.

For example, suppose N = 5, and the respective citations of each paper are `[4, 3, 0, 1, 5]`. Then the h-index would be 3, since the researcher has 3 papers with at least 3 citations.

Given a list of paper citations of a researcher, calculate their h-index.

### Problem #606 <a id="p606"></a>

**Hard** · Google · 2021-08-20

#### Problem

A knight's tour is a sequence of moves by a knight on a chessboard such that all squares are visited once.

Given N, write a function to return the number of knight's tours on an N by N chessboard.

### Problem #607 <a id="p607"></a>

**Medium** · Walmart Labs · 2021-08-21

#### Problem

There are M people sitting in a row of N seats, where M \< N. Your task is to redistribute people such that there are no gaps between any of them, while keeping overall movement to a minimum.

For example, suppose you are faced with an input of `[0, 1, 1, 0, 1, 0, 0, 0, 1]`, where 0 represents an empty seat and 1 represents a person. In this case, one solution would be to place the person on the right in the fourth seat. We can consider the cost of a solution to be the sum of the absolute distance each person must move, so that the cost here would be five.

Given an input such as the one above, return the lowest possible cost of moving people to remove all gaps.

### Problem #608 <a id="p608"></a>

**Medium** · Facebook · 2021-08-22

#### Problem

Given a start word, an end word, and a dictionary of valid words, find the shortest transformation sequence from start to end such that only one letter is changed at each step of the sequence, and each transformed word exists in the dictionary. If there is no possible transformation, return null. Each word in the dictionary have the same length as start and end and is lowercase.

For example, given start = `"dog"`, end = `"cat"`, and dictionary = `{"dot", "dop", "dat", "cat"}`, return `["dog", "dot", "dat", "cat"]`.

Given start = `"dog"`, end = `"cat"`, and dictionary = `{"dot", "tod", "dat", "dar"}`, return null as there is no possible transformation from dog to cat.

### Problem #609 <a id="p609"></a>

**Medium** · Amazon · 2021-08-23

#### Problem

Given a node in a binary search tree, return the next bigger element, also known as the inorder successor.

For example, the inorder successor of 22 is 30.

```
  10
 /  \
5    30
    /  \
  22    35
```

You can assume each node has a parent pointer.

### Problem #610 <a id="p610"></a>

**Medium** · ContextLogic · 2021-08-24

#### Problem

Implement division of two positive integers without using the division, multiplication, or modulus operators. Return the quotient as an integer, ignoring the remainder.

### Problem #611 <a id="p611"></a>

**Hard** · Dropbox · 2021-08-25

#### Problem

Create a data structure that performs all the following operations in `O(1)` time:

- plus: Add a key with value 1. If the key already exists, increment its value by one.
- minus: Decrement the value of a key. If the key's value is currently 1, remove it.
- `get_max`: Return a key with the highest value.
- `get_min`: Return a key with the lowest value.

### Problem #612 <a id="p612"></a>

**Easy** · Stripe · 2021-08-26

#### Problem

Given a collection of intervals, find the minimum number of intervals you need to remove to make the rest of the intervals non-overlapping.

Intervals can `"touch"`, such as `[0, 1]` and `[1, 2]`, but they won't be considered overlapping.

For example, given the intervals (7, 9), (2, 4), (5, 8), return 1 as the last interval can be removed and the first two won't overlap.

The intervals are not necessarily sorted in any order.

### Problem #613 <a id="p613"></a>

**Easy** · Google · 2021-08-27

#### Problem

Implement a PrefixMapSum class with the following methods:

- `insert(key: str, value: int)`: Set a given key's value in the map. If the key already exists, overwrite the value.
- `sum(prefix: str)`: Return the sum of all values of keys that begin with a given prefix.

For example, you should be able to run the following code:

```
mapsum.insert("columnar", 3)
assert mapsum.sum("col") == 3
```

```
mapsum.insert("column", 2)
assert mapsum.sum("col") == 5
```

### Problem #614 <a id="p614"></a>

**Medium** · Twitter · 2021-08-28

#### Problem

A network consists of nodes labeled 0 to N. You are given a list of edges (a, b, t), describing the time t it takes for a message to be sent from node a to node b. Whenever a node receives a message, it immediately passes the message on to a neighboring node, if possible.

Assuming all nodes are connected, determine how long it will take for every node to receive a message that begins at node 0.

For example, given N = 5, and the following edges:

```
edges = [
    (0, 1, 5),
    (0, 2, 3),
    (0, 5, 4),
    (1, 3, 8),
    (2, 3, 1),
    (3, 5, 10),
    (3, 4, 5)
]
```

You should return 9, because propagating the message from 0 -> 2 -> 3 -> 4 will take that much time.

### Problem #615 <a id="p615"></a>

**Hard** · Amazon · 2021-08-29

#### Problem

The stable marriage problem is defined as follows:

Suppose you have N men and N women, and each person has ranked their prospective opposite-sex partners in order of preference.

For example, if N = 3, the input could be something like this:

```
guy_preferences = {
    'andrew': ['caroline', 'abigail', 'betty'],
    'bill': ['caroline', 'betty', 'abigail'],
    'chester': ['betty', 'caroline', 'abigail'],
}
```

```
gal_preferences = {
    'abigail': ['andrew', 'bill', 'chester'],
    'betty': ['bill', 'andrew', 'chester'],
    'caroline': ['bill', 'chester', 'andrew']
}
```

Write an algorithm that pairs the men and women together in such a way that no two people of opposite sex would both rather be with each other than with their current partners.

### Problem #616 <a id="p616"></a>

**Medium** · Google · 2021-08-30

#### Problem

A cryptarithmetic puzzle is a mathematical game where the digits of some numbers are represented by letters. Each letter represents a unique digit.

For example, a puzzle of the form:

```
  SEND
+ MORE
--------
 MONEY
```

may have the solution:

`{'S': 9, 'E': 5, 'N': 6, 'D': 7, 'M': 1, 'O', 0, 'R': 8, 'Y': 2}`

Given a three-word puzzle like the one above, create an algorithm that finds a solution.

### Problem #617 <a id="p617"></a>

**Medium** · Facebook · 2021-08-31

#### Problem

Given a number in Roman numeral format, convert it to decimal.

The values of Roman numerals are as follows:

```
{
    'M': 1000,
    'D': 500,
    'C': 100,
    'L': 50,
    'X': 10,
    'V': 5,
    'I': 1
}
```

In addition, note that the Roman numeral system uses subtractive notation for numbers such as IV and XL.

For the input XIV, for instance, you should return 14.

### Problem #618 <a id="p618"></a>

**Hard** · 2021-09-01

#### Problem

Given a list, sort it using this method: `reverse(lst, i, j)`, which reverses lst from i to j.

### Problem #619 <a id="p619"></a>

**Easy** · Coursera · 2021-09-02

#### Problem

Given a 2D board of characters and a word, find if the word exists in the grid.

The word can be constructed from letters of sequentially adjacent cell, where `"adjacent"` cells are those horizontally or vertically neighboring. The same letter cell may not be used more than once.

For example, given the following board:

```
[
  ['A','B','C','E'],
  ['S','F','C','S'],
  ['A','D','E','E']
]
```

`exists(board, "ABCCED")` returns true, `exists(board, "SEE")` returns true, `exists(board, "ABCB")` returns false.

### Problem #620 <a id="p620"></a>

**Medium** · LinkedIn · 2021-09-03

#### Problem

A wall consists of several rows of bricks of various integer lengths and uniform height. Your goal is to find a vertical line going from the top to the bottom of the wall that cuts through the fewest number of bricks. If the line goes through the edge between two bricks, this does not count as a cut.

For example, suppose the input is as follows, where values in each row represent the lengths of bricks in that row:

```
[[3, 5, 1, 1],
 [2, 3, 3, 2],
 [5, 5],
 [4, 4, 2],
 [1, 3, 3, 3],
 [1, 1, 6, 1, 1]]
```

The best we can we do here is to draw a line after the eighth brick, which will only require cutting through the bricks in the third and fifth row.

Given an input consisting of brick lengths for each row such as the one above, return the fewest number of bricks that must be cut to create a vertical line.

### Problem #621 <a id="p621"></a>

**Hard** · Uber · 2021-09-04

#### Problem

Given a tree where each edge has a weight, compute the length of the longest path in the tree.

For example, given the following tree:

```
  a
 /|\
b c d
   / \
  e   f
 / \
g   h
```

and the weights: a-b: 3, a-c: 5, a-d: 8, d-e: 2, d-f: 4, e-g: 1, e-h: 1, the longest path would be c -> a -> d -> f, with a length of 17.

The path does not have to pass through the root, and each node can have any amount of children.

### Problem #622 <a id="p622"></a>

**Easy** · Google · 2021-09-05

#### Problem

Given the root of a binary tree, return a deepest node. For example, in the following tree, return d.

```
    a
   / \
  b   c
 /
d
```

### Problem #623 <a id="p623"></a>

**Medium** · Palantir · 2021-09-06

#### Problem

Write an algorithm to justify text. Given a sequence of words and an integer line length k, return a list of strings which represents each line, fully justified.

More specifically, you should have as many words as possible in each line. There should be at least one space between each word. Pad extra spaces when necessary so that each line has exactly length k. Spaces should be distributed as equally as possible, with the extra spaces, if any, distributed starting from the left.

If you can only fit one word on a line, then you should pad the right-hand side with spaces.

Each word is guaranteed not to be longer than k.

For example, given the list of words `["the", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]` and k = 16, you should return the following:

```
["the  quick brown", # 1 extra space on the left
"fox  jumps  over", # 2 extra spaces distributed evenly
"the   lazy   dog"] # 4 extra spaces distributed evenly
```

### Problem #624 <a id="p624"></a>

**Medium** · Google · 2021-09-07

#### Problem

Given a string of parentheses, write a function to compute the minimum number of parentheses to be removed to make the string valid (i.e. each open parenthesis is eventually closed).

For example, given the string `"()())()"`, you should return 1. Given the string `")("`, you should return 2, since we must remove all of them.

### Problem #625 <a id="p625"></a>

**Easy** · Stripe · 2021-09-08

#### Problem

Given an integer n, return the length of the longest consecutive run of 1s in its binary representation.

For example, given 156, you should return 3.

### Problem #626 <a id="p626"></a>

**Easy** · Facebook · 2021-09-09

#### Problem

Given a list of integers, return the largest product that can be made by multiplying any three integers.

For example, if the list is `[-10, -10, 5, 2]`, we should return 500, since that's -10 \* -10 \* 5.

You can assume the list has at least three integers.

### Problem #627 <a id="p627"></a>

**Medium** · Google · 2021-09-10

#### Problem

Given an iterator with methods `next()` and `hasNext()`, create a wrapper iterator, PeekableInterface, which also implements `peek()`. peek shows the next element that would be returned on `next()`.

Here is the interface:

```
class PeekableInterface(object):
    def __init__(self, iterator):
        pass
```

```
def peek(self):
    pass
```

```
def next(self):
    pass
```

```
def hasNext(self):
    pass
```

### Problem #628 <a id="p628"></a>

**Hard** · Citrix · 2021-09-11

#### Problem

You are given a circular lock with three wheels, each of which display the numbers 0 through 9 in order. Each of these wheels rotate clockwise and counterclockwise.

In addition, the lock has a certain number of `"dead ends"`, meaning that if you turn the wheels to one of these combinations, the lock becomes stuck in that state and cannot be opened.

Let us consider a `"move"` to be a rotation of a single wheel by one digit, in either direction. Given a lock initially set to 000, a target combination, and a list of dead ends, write a function that returns the minimum number of moves required to reach the target state, or None if this is impossible.

### Problem #629 <a id="p629"></a>

**Medium** · Etsy · 2021-09-12

#### Problem

Given an array of numbers N and an integer k, your task is to split N into k partitions such that the maximum sum of any partition is minimized. Return this sum.

For example, given N = `[5, 1, 2, 7, 3, 4]` and k = 3, you should return 8, since the optimal partition is `[5, 1, 2]`, `[7]`, `[3, 4]`.

### Problem #630 <a id="p630"></a>

**Medium** · Apple · 2021-09-13

#### Problem

Implement a job scheduler which takes in a function f and an integer n, and calls f after n milliseconds.

### Problem #631 <a id="p631"></a>

**Hard** · VMware · 2021-09-14

#### Problem

The skyline of a city is composed of several buildings of various widths and heights, possibly overlapping one another when viewed from a distance. We can represent the buildings using an array of (left, right, height) tuples, which tell us where on an imaginary x-axis a building begins and ends, and how tall it is. The skyline itself can be described by a list of (x, height) tuples, giving the locations at which the height visible to a distant observer changes, and each new height.

Given an array of buildings as described above, create a function that returns the skyline.

For example, suppose the input consists of the buildings `[(0, 15, 3), (4, 11, 5), (19, 23, 4)]`. In aggregate, these buildings would create a skyline that looks like the one below.

### Problem #632 <a id="p632"></a>

**Medium** · Snapchat · 2021-09-15

#### Problem

You are given an array of length N, where each element i represents the number of ways we can produce i units of change. For example, `[1, 0, 1, 1, 2]` would indicate that there is only one way to make 0, 2, or 3 units, and two ways of making 4 units.

Given such an array, determine the denominations that must be in use. In the case above, for example, there must be coins with value 2, 3, and 4.

### Problem #633 <a id="p633"></a>

**Medium** · Palantir · 2021-09-16

#### Problem

You are given a list of N numbers, in which each number is located at most k places away from its sorted position. For example, if k = 1, a given element at index 4 might end up at indices 3, 4, or 5.

Come up with an algorithm that sorts this list in `O(N log k)` time.

### Problem #634 <a id="p634"></a>

**Medium** · Square · 2021-09-17

#### Problem

You are given a histogram consisting of rectangles of different heights. These heights are represented in an input list, such that `[1, 3, 2, 5]` corresponds to the following diagram:

```
      x
      x  
  x   x
  x x x
x x x x
```

Determine the area of the largest rectangle that can be formed only from the bars of the histogram. For the diagram above, for example, this would be six, representing the 2 x 3 area at the bottom right.

### Problem #635 <a id="p635"></a>

**Hard** · Facebook · 2021-09-18

#### Problem

Implement regular expression matching with the following special characters:

- . (period) which matches any single character
- \* (asterisk) which matches zero or more of the preceding element

That is, implement a function that takes in a string and a valid regular expression and returns whether or not the string matches the regular expression.

For example, given the regular expression `"ra."` and the string `"ray"`, your function should return true. The same regular expression on the string `"raymond"` should return false.

Given the regular expression `".*at"` and the string `"chat"`, your function should return true. The same regular expression on the string `"chats"` should return false.

### Problem #636 <a id="p636"></a>

**Medium** · Uber · 2021-09-19

#### Problem

Suppose an array sorted in ascending order is rotated at some pivot unknown to you beforehand. Find the minimum element in `O(log N)` time. You may assume the array does not contain duplicates.

For example, given `[5, 7, 10, 3, 4]`, return 3.

### Problem #637 <a id="p637"></a>

**Easy** · Snapchat · 2021-09-20

#### Problem

Given a list of possibly overlapping intervals, return a new list of intervals where all overlapping intervals have been merged.

The input list is not necessarily ordered in any way.

For example, given `[(1, 3), (5, 8), (4, 10), (20, 25)]`, you should return `[(1, 3), (4, 10), (20, 25)]`.

### Problem #638 <a id="p638"></a>

**Medium** · Google · 2021-09-21

#### Problem

Given a string of words delimited by spaces, reverse the words in string. For example, given `"hello world here"`, return `"here world hello"`

Follow-up: given a mutable string representation, can you perform this operation in-place?

### Problem #639 <a id="p639"></a>

**Easy** · Yelp · 2021-09-22

#### Problem

Given a mapping of digits to letters (as in a phone number), and a digit string, return all possible letters the number could represent. You can assume each valid number in the mapping is a single digit.

For example if `{“2”: [“a”, “b”, “c”], 3: [“d”, “e”, “f”], …}` then “23” should return \[“ad”, “ae”, “af”, “bd”, “be”, “bf”, “cd”, “ce”, “cf"\].

### Problem #640 <a id="p640"></a>

**Easy** · Google · 2021-09-23

#### Problem

You are given a starting state start, a list of transition probabilities for a Markov chain, and a number of steps `num_steps`. Run the Markov chain starting from start for `num_steps` and compute the number of times we visited each state.

For example, given the starting state a, number of steps 5000, and the following transition probabilities:

```
[
  ('a', 'a', 0.9),
  ('a', 'b', 0.075),
  ('a', 'c', 0.025),
  ('b', 'a', 0.15),
  ('b', 'b', 0.8),
  ('b', 'c', 0.05),
  ('c', 'a', 0.25),
  ('c', 'b', 0.25),
  ('c', 'c', 0.5)
]
```

One instance of running this Markov chain might produce `{ 'a': 3012, 'b': 1656, 'c': 332 }`.

### Problem #641 <a id="p641"></a>

**Easy** · Amazon · 2021-09-24

#### Problem

Given a sorted array, find the smallest positive integer that is not the sum of a subset of the array.

For example, for the input `[1, 2, 3, 10]`, you should return 7.

Do this in `O(N)` time.

### Problem #642 <a id="p642"></a>

**Easy** · Pivotal · 2021-09-25

#### Problem

A step word is formed by taking a given word, adding a letter, and anagramming the result. For example, starting with the word `"APPLE"`, you can add an `"A"` and anagram to get `"APPEAL"`.

Given a dictionary of words and an input word, create a function that returns all valid step words.

### Problem #643 <a id="p643"></a>

**Hard** · YouTube · 2021-09-26

#### Problem

Write a program that computes the length of the longest common subsequence of three given strings. For example, given `"epidemiologist"`, `"refrigeration"`, and `"supercalifragilisticexpialodocious"`, it should return 5, since the longest common subsequence is `"eieio"`.

### Problem #644 <a id="p644"></a>

**Easy** · Google · 2021-09-27

#### Problem

A unival tree (which stands for `"universal value"`) is a tree where all nodes under it have the same value.

Given the root to a binary tree, count the number of unival subtrees.

For example, the following tree has 5 unival subtrees:

```
  0
 / \
1   0
   / \
  1   0
 / \
1   1
```

### Problem #645 <a id="p645"></a>

**Easy** · Microsoft · 2021-09-28

#### Problem

Given a 2D matrix of characters and a target word, write a function that returns whether the word can be found in the matrix by going left-to-right, or up-to-down.

For example, given the following matrix:

```
[['F', 'A', 'C', 'I'],
 ['O', 'B', 'Q', 'P'],
 ['A', 'N', 'O', 'B'],
 ['M', 'A', 'S', 'S']]
```

and the target word `'FOAM'`, you should return true, since it's the leftmost column. Similarly, given the target word `'MASS'`, you should return true, since it's the last row.

### Problem #646 <a id="p646"></a>

**Easy** · Twitter · 2021-09-29

#### Problem

A classroom consists of N students, whose friendships can be represented in an adjacency list. For example, the following descibes a situation where 0 is friends with 1 and 2, 3 is friends with 6, and so on.

```
{0: [1, 2],
 1: [0, 5],
 2: [0],
 3: [6],
 4: [],
 5: [1],
 6: [3]}
```

Each student can be placed in a friend group, which can be defined as the transitive closure of that student's friendship relations. In other words, this is the smallest set such that no student in the group has any friends outside this group. For the example above, the friend groups would be `{0, 1, 2, 5}`, `{3, 6}`, `{4}`.

Given a friendship list such as the one above, determine the number of friend groups in the class.

### Problem #647 <a id="p647"></a>

**Medium** · Facebook · 2021-09-30

#### Problem

Given a multiset of integers, return whether it can be partitioned into two subsets whose sums are the same.

For example, given the multiset `{15, 5, 20, 10, 35, 15, 10}`, it would return true, since we can split it up into `{15, 5, 10, 15, 10}` and `{20, 35}`, which both add up to 55.

Given the multiset `{15, 5, 20, 10, 35}`, it would return false, since we can't split it up into two subsets that add up to the same sum.

### Problem #648 <a id="p648"></a>

**Medium** · Snapchat · 2021-10-01

#### Problem

Given the head to a singly linked list, where each node also has a “random” pointer that points to anywhere in the linked list, deep clone the list.

### Problem #649 <a id="p649"></a>

**Easy** · Google · 2021-10-02

#### Problem

Given a string, return the first recurring character in it, or null if there is no recurring character.

For example, given the string `"acbbac"`, return `"b"`. Given the string `"abcdef"`, return null.

### Problem #650 <a id="p650"></a>

**Hard** · Google · 2021-10-03

#### Problem

Let A be an N by M matrix in which every row and every column is sorted.

Given i1, j1, i2, and j2, compute the number of elements of M smaller than `M[i1, j1]` and larger than `M[i2, j2]`.

For example, given the following matrix:

```
[[1, 3, 7, 10, 15, 20],
 [2, 6, 9, 14, 22, 25],
 [3, 8, 10, 15, 25, 30],
 [10, 11, 12, 23, 30, 35],
 [20, 25, 30, 35, 40, 45]]
```

And i1 = 1, j1 = 1, i2 = 3, j2 = 3, return 15 as there are 15 numbers in the matrix smaller than 6 or greater than 23.

### Problem #651 <a id="p651"></a>

**Medium** · LinkedIn · 2021-10-04

#### Problem

Determine whether a tree is a valid binary search tree.

A binary search tree is a tree with two children, left and right, and satisfies the constraint that the key in the left child must be less than or equal to the root and the key in the right child must be greater than or equal to the root.

### Problem #652 <a id="p652"></a>

**Hard** · Apple · 2021-10-05

#### Problem

You are going on a road trip, and would like to create a suitable music playlist. The trip will require N songs, though you only have M songs downloaded, where M \< N. A valid playlist should select each song at least once, and guarantee a buffer of B songs between repeats.

Given N, M, and B, determine the number of valid playlists.

### Problem #653 <a id="p653"></a>

**Easy** · Google · 2021-10-06

#### Problem

You are given given a list of rectangles represented by min and max x- and y-coordinates. Compute whether or not a pair of rectangles overlap each other. If one rectangle completely covers another, it is considered overlapping.

For example, given the following rectangles:

```
{
    "top_left": (1, 4),
    "dimensions": (3, 3) # width, height
},
{
    "top_left": (-1, 3),
    "dimensions": (2, 1)
},
{
    "top_left": (0, 5),
    "dimensions": (4, 3)
}
```

return true as the first and third rectangle overlap each other.

### Problem #654 <a id="p654"></a>

**Medium** · Amazon · 2021-10-07

#### Problem

Given a string, find the length of the smallest window that contains every distinct character. Characters may appear more than once in the window.

For example, given `"jiujitsu"`, you should return 5, corresponding to the final five letters.

### Problem #655 <a id="p655"></a>

**Easy** · Facebook · 2021-10-08

#### Problem

Given a 32-bit integer, return the number with its bits reversed.

For example, given the binary number 1111 0000 1111 0000 1111 0000 1111 0000, return 0000 1111 0000 1111 0000 1111 0000 1111.

### Problem #656 <a id="p656"></a>

**Medium** · 2021-10-09

#### Problem

Given a 2-D matrix representing an image, a location of a pixel in the screen and a color C, replace the color of the given pixel and all adjacent same colored pixels with C.

For example, given the following matrix, and location pixel of (2, 2), and `'G'` for green:

```
B B W
W W W
W W W
B B B
```

Becomes

```
B B G
G G G
G G G
B B B
```

### Problem #657 <a id="p657"></a>

**Easy** · Google · 2021-10-10

#### Problem

The power set of a set is the set of all its subsets. Write a function that, given a set, generates its power set.

For example, given the set `{1, 2, 3}`, it should return `{{}, {1}, {2}, {3}, {1, 2}, {1, 3}, {2, 3}, {1, 2, 3}}`.

You may also use a list or array to represent a set.

### Problem #658 <a id="p658"></a>

**Hard** · Google · 2021-10-11

#### Problem

Suppose we represent our file system by a string in the following manner:

The string `"dir\n\tsubdir1\n\tsubdir2\n\t\tfile.ext"` represents:

```
dir
    subdir1
    subdir2
        file.ext
```

The directory dir contains an empty sub-directory subdir1 and a sub-directory subdir2 containing a file file.ext.

The string `"dir\n\tsubdir1\n\t\tfile1.ext\n\t\tsubsubdir1\n\tsubdir2\n\t\tsubsubdir2\n\t\t\tfile2.ext"` represents:

```
dir
    subdir1
        file1.ext
        subsubdir1
    subdir2
        subsubdir2
            file2.ext
```

The directory dir contains two sub-directories subdir1 and subdir2. subdir1 contains a file file1.ext and an empty second-level sub-directory subsubdir1. subdir2 contains a second-level sub-directory subsubdir2 containing a file file2.ext.

We are interested in finding the longest (number of characters) absolute path to a file within our file system. For example, in the second example above, the longest absolute path is `"dir/subdir2/subsubdir2/file2.ext"`, and its length is 32 (not including the double quotes).

Given a string representing the file system in the above format, return the length of the longest absolute path to a file in the abstracted file system. If there is no file in the system, return 0.

Note:

The name of a file contains at least a period and an extension.

The name of a directory or sub-directory will not contain a period.

### Problem #659 <a id="p659"></a>

**Hard** · Twitch · 2021-10-12

#### Problem

Describe what happens when you type a URL into your browser and press Enter.

### Problem #660 <a id="p660"></a>

**Hard** · Airbnb · 2021-10-13

#### Problem

You come across a dictionary of sorted words in a language you've never seen before. Write a program that returns the correct order of letters in this language.

For example, given `['xww', 'wxyz', 'wxyw', 'ywx', 'ywz']`, you should return `['x', 'z', 'w', 'y']`.

### Problem #661 <a id="p661"></a>

**Hard** · Netflix · 2021-10-14

#### Problem

Given a sorted list of integers of length N, determine if an element x is in the list without performing any multiplication, division, or bit-shift operations.

Do this in `O(log N)` time.

### Problem #662 <a id="p662"></a>

**Easy** · Amazon · 2021-10-15

#### Problem

Given n numbers, find the greatest common denominator between them.

For example, given the numbers `[42, 56, 14]`, return 14.

### Problem #663 <a id="p663"></a>

**Easy** · Riot Games · 2021-10-16

#### Problem

Design and implement a HitCounter class that keeps track of requests (or hits). It should support the following operations:

- `record(timestamp)`: records a hit that happened at timestamp
- `total()`: returns the total number of hits recorded
- `range(lower, upper)`: returns the number of hits that occurred between timestamps lower and upper (inclusive)

Follow-up: What if our system has limited memory?

### Problem #664 <a id="p664"></a>

**Easy** · Google · 2021-10-17

#### Problem

Given a binary tree of integers, find the maximum path sum between two nodes. The path must go through at least one node, and does not need to go through the root.

### Problem #665 <a id="p665"></a>

**Easy** · Microsoft · 2021-10-18

#### Problem

Implement a URL shortener with the following methods:

- `shorten(url)`, which shortens the url into a six-character alphanumeric string, such as zLg6wl.
- `restore(short)`, which expands the shortened string into the original url. If no such shortened string exists, return null.

Hint: What if we enter the same URL twice?

### Problem #666 <a id="p666"></a>

**Hard** · Microsoft · 2021-10-19

#### Problem

Given an array of positive integers, divide the array into two subsets such that the difference between the sum of the subsets is as small as possible.

For example, given `[5, 10, 15, 20, 25]`, return the sets `{10, 25}` and `{5, 15, 20}`, which has a difference of 5, which is the smallest possible difference.

### Problem #667 <a id="p667"></a>

**Medium** · Facebook · 2021-10-20

#### Problem

In chess, the Elo rating system is used to calculate player strengths based on game results.

A simplified description of the Elo system is as follows. Every player begins at the same score. For each subsequent game, the loser transfers some points to the winner, where the amount of points transferred depends on how unlikely the win is. For example, a 1200-ranked player should gain much more points for beating a 2000-ranked player than for beating a 1300-ranked player.

Implement this system.

### Problem #668 <a id="p668"></a>

**Easy** · Google · 2021-10-21

#### Problem

In linear algebra, a Toeplitz matrix is one in which the elements on any given diagonal from top left to bottom right are identical.

Here is an example:

```
1 2 3 4 8
5 1 2 3 4
4 5 1 2 3
7 4 5 1 2
```

Write a program to determine whether a given input is a Toeplitz matrix.

### Problem #669 <a id="p669"></a>

**Hard** · Google · 2021-10-22

#### Problem

The game of Nim is played as follows. Starting with three heaps, each containing a variable number of items, two players take turns removing one or more items from a single pile. The player who eventually is forced to take the last stone loses. For example, if the initial heap sizes are 3, 4, and 5, a game could be played as shown below:

```
  A  |  B  |  C
-----------------
  3  |  4  |  5
  3  |  1  |  3
  3  |  1  |  3
  0  |  1  |  3
  0  |  1  |  0
  0  |  0  |  0
```

In other words, to start, the first player takes three items from pile B. The second player responds by removing two stones from pile C. The game continues in this way until player one takes last stone and loses.

Given a list of non-zero starting values `[a, b, c]`, and assuming optimal play, determine whether the first player has a forced win.

### Problem #670 <a id="p670"></a>

**Medium** · Facebook · 2021-10-23

#### Problem

Given a positive integer n, find the smallest number of squared integers which sum to n.

For example, given n = 13, return 2 since 13 = 32 + 22 = 9 + 4.

Given n = 27, return 3 since 27 = 32 + 32 + 32 = 9 + 9 + 9.

### Problem #671 <a id="p671"></a>

**Easy** · Facebook · 2021-10-24

#### Problem

Given a function f, and N return a debounced f of N milliseconds.

That is, as long as the debounced f continues to be invoked, f itself will not be called for N milliseconds.

### Problem #672 <a id="p672"></a>

**Easy** · Google · 2021-10-25

#### Problem

You are given an array of arrays of integers, where each array corresponds to a row in a triangle of numbers. For example, `[[1], [2, 3], [1, 5, 1]]` represents the triangle:

```
  1
 2 3
1 5 1
```

We define a path in the triangle to start at the top and go down one row at a time to an adjacent value, eventually ending with an entry on the bottom row. For example, 1 -> 3 -> 5. The weight of the path is the sum of the entries.

Write a program that returns the weight of the maximum weight path.

### Problem #673 <a id="p673"></a>

**Hard** · LinkedIn · 2021-10-26

#### Problem

Given a list of points, a central point, and an integer k, find the nearest k points from the central point.

For example, given the list of points `[(0, 0), (5, 4), (3, 1)]`, the central point (1, 2), and k = 2, return `[(0, 0), (3, 1)]`.

### Problem #674 <a id="p674"></a>

**Easy** · Google · 2021-10-27

#### Problem

A girl is walking along an apple orchard with a bag in each hand. She likes to pick apples from each tree as she goes along, but is meticulous about not putting different kinds of apples in the same bag.

Given an input describing the types of apples she will pass on her path, in order, determine the length of the longest portion of her path that consists of just two types of apple trees.

For example, given the input `[2, 1, 2, 3, 3, 1, 3, 5]`, the longest portion will involve types 1 and 3, with a length of four.

### Problem #675 <a id="p675"></a>

**Medium** · Google · 2021-10-28

#### Problem

You are given a set of synonyms, such as (big, large) and (eat, consume). Using this set, determine if two sentences with the same number of words are equivalent.

For example, the following two sentences are equivalent:

- `"He wants to eat food."`
- `"He wants to consume food."`

Note that the synonyms (a, b) and (a, c) do not necessarily imply (b, c): consider the case of (coach, bus) and (coach, teacher).

Follow-up: what if we can assume that (a, b) and (a, c) do in fact imply (b, c)?

### Problem #676 <a id="p676"></a>

**Hard** · LinkedIn · 2021-10-29

#### Problem

Given a string, return whether it represents a number. Here are the different kinds of numbers:

- `"10"`, a positive integer
- `"-10"`, a negative integer
- `"10.1"`, a positive real number
- `"-10.1"`, a negative real number
- `"1e5"`, a number in scientific notation

And here are examples of non-numbers:

- `"a"`
- `"x 1"`
- `"a -2"`
- `"-"`

### Problem #677 <a id="p677"></a>

**Easy** · Square · 2021-10-30

#### Problem

The Sieve of Eratosthenes is an algorithm used to generate all prime numbers smaller than N. The method is to take increasingly larger prime numbers, and mark their multiples as composite.

For example, to find all primes less than 100, we would first mark `[4, 6, 8, ...]` (multiples of two), then `[6, 9, 12, ...]` (multiples of three), and so on. Once we have done this for all primes less than N, the unmarked numbers that remain will be prime.

Implement this algorithm.

Bonus: Create a generator that produces primes indefinitely (that is, without taking N as an input).

### Problem #678 <a id="p678"></a>

**Easy** · IBM · 2021-10-31

#### Problem

Given an integer, find the next permutation of it in absolute order. For example, given 48975, the next permutation would be 49578.

### Problem #679 <a id="p679"></a>

**Medium** · Google · 2021-11-01

#### Problem

We can determine how `"out of order"` an array A is by counting the number of inversions it has. Two elements `A[i]` and `A[j]` form an inversion if `A[i]` > `A[j]` but i \< j. That is, a smaller element appears after a larger element.

Given an array, count the number of inversions it has. Do this faster than `O(N^2)` time.

You may assume each element in the array is distinct.

For example, a sorted list has zero inversions. The array `[2, 4, 1, 3, 5]` has three inversions: (2, 1), (4, 1), and (4, 3). The array `[5, 4, 3, 2, 1]` has ten inversions: every distinct pair forms an inversion.

### Problem #680 <a id="p680"></a>

**Medium** · Square · 2021-11-02

#### Problem

Given a list of words, return the shortest unique prefix of each word. For example, given the list:

- dog
- cat
- apple
- apricot
- fish

Return the list:

- d
- c
- app
- apr
- f

### Problem #681 <a id="p681"></a>

**Medium** · Flipkart · 2021-11-03

#### Problem

Snakes and Ladders is a game played on a 10 x 10 board, the goal of which is get from square 1 to square 100. On each turn players will roll a six-sided die and move forward a number of spaces equal to the result. If they land on a square that represents a snake or ladder, they will be transported ahead or behind, respectively, to a new square.

Find the smallest number of turns it takes to play snakes and ladders.

For convenience, here are the squares representing snakes and ladders, and their outcomes:

```
snakes = {16: 6, 48: 26, 49: 11, 56: 53, 62: 19, 64: 60, 87: 24, 93: 73, 95: 75, 98: 78}
ladders = {1: 38, 4: 14, 9: 31, 21: 42, 28: 84, 36: 44, 51: 67, 71: 91, 80: 100}
```

### Problem #682 <a id="p682"></a>

**Medium** · Squarespace · 2021-11-04

#### Problem

Write a function, `add_subtract`, which alternately adds and subtracts curried arguments. Here are some sample operations:

`add_subtract(7)` -> 7

`add_subtract(1)(2)(3)` -> 1 + 2 - 3 -> 0

`add_subtract(-5)(10)(3)(9)` -> -5 + 10 - 3 + 9 -> 11

### Problem #683 <a id="p683"></a>

**Medium** · MongoDB · 2021-11-05

#### Problem

Given a list of elements, find the majority element, which appears more than half the time (> `floor(len(lst) / 2.0)`).

You can assume that such element exists.

For example, given `[1, 2, 1, 1, 3, 4, 0]`, return 1.

### Problem #684 <a id="p684"></a>

**Medium** · Stripe · 2021-11-06

#### Problem

reduce (also known as fold) is a function that takes in an array, a combining function, and an initial value and builds up a result by calling the combining function on each element of the array, left to right. For example, we can write `sum()` in terms of reduce:

```
def add(a, b):
    return a + b
```

```
def sum(lst):
    return reduce(lst, add, 0)
```

This should call add on the initial value with the first element of the array, and then the result of that with the second element of the array, and so on until we reach the end, when we return the sum of the array.

Implement your own version of reduce.

### Problem #685 <a id="p685"></a>

**Hard** · Facebook · 2021-11-07

#### Problem

Given a string and a set of delimiters, reverse the words in the string while maintaining the relative order of the delimiters. For example, given `"hello/world:here"`, return `"here/world:hello"`

Follow-up: Does your solution work for the following cases: `"hello/world:here/"`, `"hello//world:here"`

### Problem #686 <a id="p686"></a>

**Hard** · Adobe · 2021-11-08

#### Problem

You are given a tree with an even number of nodes. Consider each connection between a parent and child node to be an `"edge"`. You would like to remove some of these edges, such that the disconnected subtrees that remain each have an even number of nodes.

For example, suppose your input was the following tree:

```
   1
  / \ 
 2   3
    / \ 
   4   5
 / | \
6  7  8
```

In this case, removing the edge (3, 4) satisfies our requirement.

Write a function that returns the maximum number of edges you can remove while still satisfying this requirement.

### Problem #687 <a id="p687"></a>

**Hard** · Airbnb · 2021-11-09

#### Problem

An 8-puzzle is a game played on a 3 x 3 board of tiles, with the ninth tile missing. The remaining tiles are labeled 1 through 8 but shuffled randomly. Tiles may slide horizontally or vertically into an empty space, but may not be removed from the board.

Design a class to represent the board, and find a series of steps to bring the board to the state `[[1, 2, 3], [4, 5, 6], [7, 8, None]]`.

### Problem #688 <a id="p688"></a>

**Hard** · Dropbox · 2021-11-10

#### Problem

A Boolean formula can be said to be satisfiable if there is a way to assign truth values to each variable such that the entire formula evaluates to true.

For example, suppose we have the following formula, where the symbol ¬ is used to denote negation:

(¬c OR b) AND (b OR c) AND (¬b OR c) AND (¬c OR ¬a)

One way to satisfy this formula would be to let a = False, b = True, and c = True.

This type of formula, with AND statements joining tuples containing exactly one OR, is known as 2-CNF.

Given a 2-CNF formula, find a way to assign truth values to satisfy it, or return False if this is impossible.

### Problem #689 <a id="p689"></a>

**Easy** · Jane Street · 2021-11-11

#### Problem

The United States uses the imperial system of weights and measures, which means that there are many different, seemingly arbitrary units to measure distance. There are 12 inches in a foot, 3 feet in a yard, 22 yards in a chain, and so on.

Create a data structure that can efficiently convert a certain quantity of one unit to the correct amount of any other unit. You should also allow for additional units to be added to the system.

### Problem #690 <a id="p690"></a>

**Hard** · Microsoft · 2021-11-12

#### Problem

Given an array of numbers, find the length of the longest increasing subsequence in the array. The subsequence does not necessarily have to be contiguous.

For example, given the array `[0, 8, 4, 12, 2, 10, 6, 14, 1, 9, 5, 13, 3, 11, 7, 15]`, the longest increasing subsequence has length 6: it is 0, 2, 6, 9, 11, 15.

### Problem #691 <a id="p691"></a>

**Medium** · Yahoo · 2021-11-13

#### Problem

Write a function that returns the bitwise AND of all integers between M and N, inclusive.

### Problem #692 <a id="p692"></a>

**Medium** · Twitter · 2021-11-14

#### Problem

Implement an autocomplete system. That is, given a query string s and a set of all possible query strings, return all strings in the set that have s as a prefix.

For example, given the query string de and the set of strings `[dog, deer, deal]`, return `[deer, deal]`.

Hint: Try preprocessing the dictionary into a more efficient data structure to speed up queries.

### Problem #693 <a id="p693"></a>

**Medium** · Cisco · 2021-11-15

#### Problem

Given an unsigned 8-bit integer, swap its even and odd bits. The 1st and 2nd bit should be swapped, the 3rd and 4th bit should be swapped, and so on.

For example, 10101010 should be 01010101. 11100010 should be 11010001.

Bonus: Can you do this in one line?

### Problem #694 <a id="p694"></a>

**Hard** · Stripe · 2021-11-16

#### Problem

Given an array of integers, find the first missing positive integer in linear time and constant space. In other words, find the lowest positive integer that does not exist in the array. The array can contain duplicates and negative numbers as well.

For example, the input `[3, 4, -1, 1]` should give 2. The input `[1, 2, 0]` should give 3.

You can modify the input array in-place.

### Problem #695 <a id="p695"></a>

**Medium** · Google · 2021-11-17

#### Problem

Given an integer n and a list of integers l, write a function that randomly generates a number from 0 to n-1 that isn't in l (uniform).

### Problem #696 <a id="p696"></a>

**Hard** · Twitter · 2021-11-18

#### Problem

You are given an array of length 24, where each element represents the number of new subscribers during the corresponding hour. Implement a data structure that efficiently supports the following:

- `update(hour: int, value: int)`: Increment the element at index hour by value.
- `query(start: int, end: int)`: Retrieve the number of subscribers that have signed up between start and end (inclusive).

You can assume that all values get cleared at the end of the day, and that you will not be asked for start and end values that wrap around midnight.

### Problem #697 <a id="p697"></a>

**Hard** · Google · 2021-11-19

#### Problem

Implement an LRU (Least Recently Used) cache. It should be able to be initialized with a cache size n, and contain the following methods:

- `set(key, value)`: sets key to value. If there are already n items in the cache and we are adding a new item, then it should also remove the least recently used item.
- `get(key)`: gets the value at key. If no such key exists, return null.

Each operation should run in `O(1)` time.

### Problem #698 <a id="p698"></a>

**Medium** · Uber · 2021-11-20

#### Problem

You are given a 2-d matrix where each cell consists of either /, \\, or an empty space. Write an algorithm that determines into how many regions the slashes divide the space.

For example, suppose the input for a three-by-six grid is the following:

```
\    /
 \  /
  \/
```

Considering the edges of the matrix as boundaries, this divides the grid into three triangles, so you should return 3.

### Problem #699 <a id="p699"></a>

**Easy** · Airbnb · 2021-11-21

#### Problem

Given a linked list and a positive integer k, rotate the list to the right by k places.

For example, given the linked list 7 -> 7 -> 3 -> 5 and k = 2, it should become 3 -> 5 -> 7 -> 7.

Given the linked list 1 -> 2 -> 3 -> 4 -> 5 and k = 3, it should become 3 -> 4 -> 5 -> 1 -> 2.

### Problem #700 <a id="p700"></a>

**Medium** · Airbnb · 2021-11-22

#### Problem

You are given a huge list of airline ticket prices between different cities around the world on a given day. These are all direct flights. Each element in the list has the format (`source_city`, destination, price).

Consider a user who is willing to take up to k connections from their origin city A to their destination B. Find the cheapest fare possible for this journey and print the itinerary for that journey.

For example, our traveler wants to go from JFK to LAX with up to 3 connections, and our input flights are as follows:

```
[
    ('JFK', 'ATL', 150),
    ('ATL', 'SFO', 400),
    ('ORD', 'LAX', 200),
    ('LAX', 'DFW', 80),
    ('JFK', 'HKG', 800),
    ('ATL', 'ORD', 90),
    ('JFK', 'LAX', 500),
]
```

Due to some improbably low flight prices, the cheapest itinerary would be JFK -> ATL -> ORD -> LAX, costing \$440.

### Problem #701 <a id="p701"></a>

**Medium** · Amazon · 2021-11-23

#### Problem

At a popular bar, each customer has a set of favorite drinks, and will happily accept any drink among this set. For example, in the following situation, customer 0 will be satisfied with drinks 0, 1, 3, or 6.

```
preferences = {
    0: [0, 1, 3, 6],
    1: [1, 4, 7],
    2: [2, 4, 7, 5],
    3: [3, 2, 5],
    4: [5, 8]
}
```

A lazy bartender working at this bar is trying to reduce his effort by limiting the drink recipes he must memorize. Given a dictionary input such as the one above, return the fewest number of drinks he must learn in order to satisfy all customers.

For the input above, the answer would be 2, as drinks 1 and 5 will satisfy everyone.

### Problem #702 <a id="p702"></a>

**Medium** · Google · 2021-11-24

#### Problem

Given the root to a binary tree, implement `serialize(root)`, which serializes the tree into a string, and deserialize(s), which deserializes the string back into the tree.

For example, given the following Node class

```
class Node:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
```

The following test should pass:

```
node = Node('root', Node('left', Node('left.left')), Node('right'))
assert deserialize(serialize(node)).left.left.val == 'left.left'
```

### Problem #703 <a id="p703"></a>

**Hard** · Two Sigma · 2021-11-25

#### Problem

A knight is placed on a given square on an 8 x 8 chessboard. It is then moved randomly several times, where each move is a standard knight move. If the knight jumps off the board at any point, however, it is not allowed to jump back on.

After k moves, what is the probability that the knight remains on the board?

### Problem #704 <a id="p704"></a>

**Medium** · Uber · 2021-11-26

#### Problem

One way to unlock an Android phone is through a pattern of swipes across a 1-9 keypad.

For a pattern to be valid, it must satisfy the following:

- All of its keys must be distinct.
- It must not connect two keys by jumping over a third key, unless that key has already been used.

For example, 4 - 2 - 1 - 7 is a valid pattern, whereas 2 - 1 - 7 is not.

Find the total number of valid unlock patterns of length N, where 1 \<= N \<= 9.

### Problem #705 <a id="p705"></a>

**Medium** · Facebook · 2021-11-27

#### Problem

You are given an array of non-negative integers that represents a two-dimensional elevation map where each element is unit-width wall and the integer is the height. Suppose it will rain and all spots between two walls get filled up.

Compute how many units of water remain trapped on the map in `O(N)` time and `O(1)` space.

For example, given the input `[2, 1, 2]`, we can hold 1 unit of water in the middle.

Given the input `[3, 0, 1, 3, 0, 5]`, we can hold 3 units in the first index, 2 in the second, and 3 in the fourth index (we cannot hold 5 since it would run off to the left), so we can trap 8 units of water.

### Problem #706 <a id="p706"></a>

**Easy** · Twitter · 2021-11-28

#### Problem

The 24 game is played as follows. You are given a list of four integers, each between 1 and 9, in a fixed order. By placing the operators +, -, \*, and / between the numbers, and grouping them with parentheses, determine whether it is possible to reach the value 24.

For example, given the input `[5, 2, 7, 8]`, you should return True, since (5 \* 2 - 7) \* 8 = 24.

Write a function that plays the 24 game.

### Problem #707 <a id="p707"></a>

**Medium** · Spotify · 2021-11-29

#### Problem

You are the technical director of WSPT radio, serving listeners nationwide. For simplicity's sake we can consider each listener to live along a horizontal line stretching from 0 (west) to 1000 (east).

Given a list of N listeners, and a list of M radio towers, each placed at various locations along this line, determine what the minimum broadcast range would have to be in order for each listener's home to be covered.

For example, suppose listeners = `[1, 5, 11, 20]`, and towers = `[4, 8, 15]`. In this case the minimum range would be 5, since that would be required for the tower at position 15 to reach the listener at position 20.

### Problem #708 <a id="p708"></a>

**Easy** · Apple · 2021-11-30

#### Problem

A fixed point in an array is an element whose value is equal to its index. Given a sorted array of distinct elements, return a fixed point, if one exists. Otherwise, return False.

For example, given `[-6, 0, 2, 40]`, you should return 2. Given `[1, 5, 7, 8]`, you should return False.

### Problem #709 <a id="p709"></a>

**Medium** · Slack · 2021-12-01

#### Problem

You are given an N by M matrix of 0s and 1s. Starting from the top left corner, how many ways are there to reach the bottom right corner?

You can only move right and down. 0 represents an empty space while 1 represents a wall you cannot walk through.

For example, given the following matrix:

```
[[0, 0, 1],
 [0, 0, 1],
 [1, 0, 0]]
```

Return two, as there are only two ways to get to the bottom right:

- Right, down, down, right
- Down, right, down, right

The top left corner and bottom right corner will always be 0.

### Problem #710 <a id="p710"></a>

**Medium** · Dropbox · 2021-12-02

#### Problem

Given a string s and a list of words words, where each word is the same length, find all starting indices of substrings in s that is a concatenation of every word in words exactly once.

For example, given s = `"dogcatcatcodecatdog"` and words = `["cat", "dog"]`, return `[0, 13]`, since `"dogcat"` starts at index 0 and `"catdog"` starts at index 13.

Given s = `"barfoobazbitbyte"` and words = `["dog", "cat"]`, return \[\] since there are no substrings composed of `"dog"` and `"cat"` in s.

The order of the indices does not matter.

### Problem #711 <a id="p711"></a>

**Medium** · Nvidia · 2021-12-03

#### Problem

You are given a list of N points (x1, y1), (x2, y2), ..., (xN, yN) representing a polygon. You can assume these points are given in order; that is, you can construct the polygon by connecting point 1 to point 2, point 2 to point 3, and so on, finally looping around to connect point N to point 1.

Determine if a new point p lies inside this polygon. (If p is on the boundary of the polygon, you should return False).

### Problem #712 <a id="p712"></a>

**Easy** · Facebook · 2021-12-04

#### Problem

Given a string of round, curly, and square open and closing brackets, return whether the brackets are balanced (well-formed).

For example, given the string `"([])[]({})"`, you should return true.

Given the string `"([)]"` or `"((()"`, you should return false.

### Problem #713 <a id="p713"></a>

**Medium** · Quora · 2021-12-05

#### Problem

Given an absolute pathname that may have . or .. as part of it, return the shortest standardized path.

For example, given `"/usr/bin/../bin/./scripts/../"`, return `"/usr/bin/"`.

### Problem #714 <a id="p714"></a>

**Easy** · Google · 2021-12-06

#### Problem

Given the head of a singly linked list, swap every two nodes and return its head.

For example, given 1 -> 2 -> 3 -> 4, return 2 -> 1 -> 4 -> 3.

### Problem #715 <a id="p715"></a>

**Easy** · Google · 2021-12-07

#### Problem

Determine whether a doubly linked list is a palindrome. What if it’s singly linked?

For example, 1 -> 4 -> 3 -> 4 -> 1 returns True while 1 -> 4 returns False.

### Problem #716 <a id="p716"></a>

**Easy** · Google · 2021-12-08

#### Problem

UTF-8 is a character encoding that maps each symbol to one, two, three, or four bytes.

For example, the Euro sign, €, corresponds to the three bytes 11100010 10000010 10101100. The rules for mapping characters are as follows:

- For a single-byte character, the first bit must be zero.
- For an n-byte character, the first byte starts with n ones and a zero. The other n - 1 bytes all start with 10.

Visually, this can be represented as follows.

```
 Bytes   |           Byte format
-----------------------------------------------
   1     | 0xxxxxxx
   2     | 110xxxxx 10xxxxxx
   3     | 1110xxxx 10xxxxxx 10xxxxxx
   4     | 11110xxx 10xxxxxx 10xxxxxx 10xxxxxx
```

Write a program that takes in an array of integers representing byte values, and returns whether it is a valid UTF-8 encoding.

### Problem #717 <a id="p717"></a>

**Medium** · Facebook · 2021-12-09

#### Problem

A builder is looking to build a row of N houses that can be of K different colors. He has a goal of minimizing cost while ensuring that no two neighboring houses are of the same color.

Given an N by K matrix where the nth row and kth column represents the cost to build the nth house with kth color, return the minimum cost which achieves this goal.

### Problem #718 <a id="p718"></a>

**Medium** · Apple · 2021-12-10

#### Problem

Gray code is a binary code where each successive value differ in only one bit, as well as when wrapping around. Gray code is common in hardware so that we don't see temporary spurious values during transitions.

Given a number of bits n, generate a possible gray code for it.

For example, for n = 2, one gray code would be `[00, 01, 11, 10]`.

### Problem #719 <a id="p719"></a>

**Easy** · Dropbox · 2021-12-11

#### Problem

Spreadsheets often use this alphabetical encoding for its columns: `"A"`, `"B"`, `"C"`, ..., `"AA"`, `"AB"`, ..., `"ZZ"`, `"AAA"`, `"AAB"`, ....

Given a column number, return its alphabetical column id. For example, given 1, return `"A"`. Given 27, return `"AA"`.

### Problem #720 <a id="p720"></a>

**Hard** · Dropbox · 2021-12-12

#### Problem

Sudoku is a puzzle where you're given a partially-filled 9 by 9 grid with digits. The objective is to fill the grid with the constraint that every row, column, and box (3 by 3 subgrid) must contain all of the digits from 1 to 9.

Implement an efficient sudoku solver.

### Problem #721 <a id="p721"></a>

**Hard** · Microsoft · 2021-12-13

#### Problem

Recall that the minimum spanning tree is the subset of edges of a tree that connect all its vertices with the smallest possible total edge weight. Given an undirected graph with weighted edges, compute the maximum weight spanning tree.

### Problem #722 <a id="p722"></a>

**Hard** · Spotify · 2021-12-14

#### Problem

There are N couples sitting in a row of length 2 \* N. They are currently ordered randomly, but would like to rearrange themselves so that each couple's partners can sit side by side.

What is the minimum number of swaps necessary for this to happen?

### Problem #723 <a id="p723"></a>

**Medium** · Google · 2021-12-15

#### Problem

Given a set of closed intervals, find the smallest set of numbers that covers all the intervals. If there are multiple smallest sets, return any of them.

For example, given the intervals `[0, 3]`, `[2, 6]`, `[3, 4]`, `[6, 9]`, one set of numbers that covers all these intervals is `{3, 6}`.

### Problem #724 <a id="p724"></a>

**Easy** · Microsoft · 2021-12-16

#### Problem

Suppose an arithmetic expression is given as a binary tree. Each leaf is an integer and each internal node is one of `'+'`, `'−'`, `'∗'`, or `'/'`.

Given the root to such a tree, write a function to evaluate it.

For example, given the following tree:

```
    *
   / \
  +    +
 / \  / \
3  2  4  5
```

You should return 45, as it is (3 + 2) \* (4 + 5).

### Problem #725 <a id="p725"></a>

**Easy** · Amazon · 2021-12-17

#### Problem

Consider the following scenario: there are N mice and N holes placed at integer points along a line. Given this, find a method that maps mice to holes such that the largest number of steps any mouse takes is minimized.

Each move consists of moving one mouse one unit to the left or right, and only one mouse can fit inside each hole.

For example, suppose the mice are positioned at `[1, 4, 9, 15]`, and the holes are located at `[10, -5, 0, 16]`. In this case, the best pairing would require us to send the mouse at 1 to the hole at -5, so our function should return 6.

### Problem #726 <a id="p726"></a>

**Medium** · Microsoft · 2021-12-18

#### Problem

Implement the singleton pattern with a twist. First, instead of storing one instance, store two instances. And in every even call of `getInstance()`, return the first instance and in every odd call of `getInstance()`, return the second instance.

### Problem #727 <a id="p727"></a>

**Easy** · Microsoft · 2021-12-19

#### Problem

Compute the running median of a sequence of numbers. That is, given a stream of numbers, print out the median of the list so far on each new element.

Recall that the median of an even-numbered list is the average of the two middle numbers.

For example, given the sequence `[2, 1, 5, 7, 2, 0, 5]`, your algorithm should print out:

```
2
1.5
2
3.5
2
2
2
```

### Problem #728 <a id="p728"></a>

**Hard** · Twitter · 2021-12-20

#### Problem

A teacher must divide a class of students into two teams to play dodgeball. Unfortunately, not all the kids get along, and several refuse to be put on the same team as that of their enemies.

Given an adjacency list of students and their enemies, write an algorithm that finds a satisfactory pair of teams, or returns False if none exists.

For example, given the following enemy graph you should return the teams `{0, 1, 4, 5}` and `{2, 3}`.

```
students = {
    0: [3],
    1: [2],
    2: [1, 4],
    3: [0, 4, 5],
    4: [2, 3],
    5: [3]
}
```

On the other hand, given the input below, you should return False.

```
students = {
    0: [3],
    1: [2],
    2: [1, 3, 4],
    3: [0, 2, 4, 5],
    4: [2, 3],
    5: [3]
}
```

### Problem #729 <a id="p729"></a>

**Medium** · Spotify · 2021-12-21

#### Problem

You have access to ranked lists of songs for various users. Each song is represented as an integer, and more preferred songs appear earlier in each list. For example, the list `[4, 1, 7]` indicates that a user likes song 4 the best, followed by songs 1 and 7.

Given a set of these ranked lists, interleave them to create a playlist that satisfies everyone's priorities.

For example, suppose your input is `{[1, 7, 3], [2, 1, 6, 7, 9], [3, 9, 5]}`. In this case a satisfactory playlist could be `[2, 1, 6, 7, 3, 9, 5]`.

### Problem #730 <a id="p730"></a>

**Medium** · Google · 2021-12-22

#### Problem

What will this code print out?

```
def make_functions():
    flist = []
```

```
for i in [1, 2, 3]:
    def print_i():
        print(i)
    flist.append(print_i)
```

```
return flist
```

```
functions = make_functions()
for f in functions:
    f()
```

How can we make it print out what we apparently want?

### Problem #731 <a id="p731"></a>

**Easy** · Facebook · 2021-12-23

#### Problem

Given a array of numbers representing the stock prices of a company in chronological order, write a function that calculates the maximum profit you could have made from buying and selling that stock once. You must buy before you can sell it.

For example, given `[9, 11, 8, 5, 7, 10]`, you should return 5, since you could buy the stock at 5 dollars and sell it at 10 dollars.

### Problem #732 <a id="p732"></a>

**Medium** · Glassdoor · 2021-12-24

#### Problem

An imminent hurricane threatens the coastal town of Codeville. If at most two people can fit in a rescue boat, and the maximum weight limit for a given boat is k, determine how many boats will be needed to save everyone.

For example, given a population with weights `[100, 200, 150, 80]` and a boat limit of 200, the smallest number of boats required will be three.

### Problem #733 <a id="p733"></a>

**Hard** · Salesforce · 2021-12-25

#### Problem

Connect 4 is a game where opponents take turns dropping red or black discs into a 7 x 6 vertically suspended grid. The game ends either when one player creates a line of four consecutive discs of their color (horizontally, vertically, or diagonally), or when there are no more spots left in the grid.

Design and implement Connect 4.

### Problem #734 <a id="p734"></a>

**Medium** · Stripe · 2021-12-26

#### Problem

Write a map implementation with a get function that lets you retrieve the value of a key at a particular time.

It should contain the following methods:

- `set(key, value, time)`: sets key to value for t = time.
- `get(key, time)`: gets the key at t = time.

The map should work like this. If we set a key at a particular time, it will maintain that value forever or until it gets set at a later time. In other words, when we get a key at a time, it should return the value that was set for that key set at the most recent time.

Consider the following examples:

```
d.set(1, 1, 0) # set key 1 to value 1 at time 0
d.set(1, 2, 2) # set key 1 to value 2 at time 2
d.get(1, 1) # get key 1 at time 1 should be 1
d.get(1, 3) # get key 1 at time 3 should be 2
```

```
d.set(1, 1, 5) # set key 1 to value 1 at time 5
d.get(1, 0) # get key 1 at time 0 should be null
d.get(1, 10) # get key 1 at time 10 should be 1
```

```
d.set(1, 1, 0) # set key 1 to value 1 at time 0
d.set(1, 2, 0) # set key 1 to value 2 at time 0
d.get(1, 0) # get key 1 at time 0 should be 2
```

### Problem #735 <a id="p735"></a>

**Easy** · Sumo Logic · 2021-12-27

#### Problem

Given an unsorted array, in which all elements are distinct, find a `"peak"` element in `O(log N)` time.

An element is considered a peak if it is greater than both its left and right neighbors. It is guaranteed that the first and last elements are lower than all others.

### Problem #736 <a id="p736"></a>

**Easy** · Amazon · 2021-12-28

#### Problem

Given a complete binary tree, count the number of nodes in faster than `O(n)` time. Recall that a complete binary tree has every level filled except the last, and the nodes in the last level are filled starting from the left.

### Problem #737 <a id="p737"></a>

**Hard** · Airbnb · 2021-12-29

#### Problem

We're given a hashmap associating each courseId key with a list of courseIds values, which represents that the prerequisites of courseId are courseIds. Return a sorted ordering of courses such that we can finish all courses.

Return null if there is no such ordering.

For example, given `{'CSC300': ['CSC100', 'CSC200'], 'CSC200': ['CSC100'], 'CSC100': []}`, should return `['CSC100', 'CSC200', 'CSCS300']`.

### Problem #738 <a id="p738"></a>

**Medium** · Amazon · 2021-12-30

#### Problem

Given a string s and an integer k, break up the string into multiple lines such that each line has a length of k or less. You must break it up so that words don't break across lines. Each line has to have the maximum possible amount of words. If there's no way to break the text up, then return null.

You can assume that there are no spaces at the ends of the string and that there is exactly one space between each word.

For example, given the string `"the quick brown fox jumps over the lazy dog"` and k = 10, you should return: `["the quick", "brown fox", "jumps over", "the lazy", "dog"]`. No string in the list has a length of more than 10.

### Problem #739 <a id="p739"></a>

**Hard** · Google · 2021-12-31

#### Problem

A quack is a data structure combining properties of both stacks and queues. It can be viewed as a list of elements written left to right such that three operations are possible:

- `push(x)`: add a new item x to the left end of the list
- `pop()`: remove and return the item on the left end of the list
- `pull()`: remove the item on the right end of the list.

Implement a quack using three stacks and `O(1)` additional memory, so that the amortized time for any push, pop, or pull operation is `O(1)`.

### Problem #740 <a id="p740"></a>

**Easy** · Google · 2022-01-01

#### Problem

A regular number in mathematics is defined as one which evenly divides some power of 60. Equivalently, we can say that a regular number is one whose only prime divisors are 2, 3, and 5.

These numbers have had many applications, from helping ancient Babylonians keep time to tuning instruments according to the diatonic scale.

Given an integer N, write a program that returns, in order, the first N regular numbers.

### Problem #741 <a id="p741"></a>

**Hard** · Facebook · 2022-01-02

#### Problem

Given a string consisting of parentheses, single digits, and positive and negative signs, convert the string into a mathematical expression to obtain the answer.

Don't use eval or a similar built-in parser.

For example, given `'-1 + (2 + 3)'`, you should return 4.

### Problem #742 <a id="p742"></a>

**Easy** · Stripe · 2022-01-03

#### Problem

Write a function to flatten a nested dictionary. Namespace the keys with a period.

For example, given the following dictionary:

```
{
    "key": 3,
    "foo": {
        "a": 5,
        "bar": {
            "baz": 8
        }
    }
}
```

it should become:

```
{
    "key": 3,
    "foo.a": 5,
    "foo.bar.baz": 8
}
```

You can assume keys do not contain dots in them, i.e. no clobbering will occur.

### Problem #743 <a id="p743"></a>

**Medium** · Mozilla · 2022-01-04

#### Problem

A bridge in a connected (undirected) graph is an edge that, if removed, causes the graph to become disconnected. Find all the bridges in a graph.

### Problem #744 <a id="p744"></a>

**Hard** · Google · 2022-01-05

#### Problem

Implement an LFU (Least Frequently Used) cache. It should be able to be initialized with a cache size n, and contain the following methods:

- `set(key, value)`: sets key to value. If there are already n items in the cache and we are adding a new item, then it should also remove the least frequently used item. If there is a tie, then the least recently used key should be removed.
- `get(key)`: gets the value at key. If no such key exists, return null.

Each operation should run in `O(1)` time.

### Problem #745 <a id="p745"></a>

**Medium** · Triplebyte · 2022-01-06

#### Problem

Implement a data structure which carries out the following operations without resizing the underlying array:

- `add(value)`: Add a value to the set of values.
- `check(value)`: Check whether a value is in the set.

The check method may return occasional false positives (in other words, incorrectly identifying an element as part of the set), but should always correctly identify a true element.

### Problem #746 <a id="p746"></a>

**Easy** · Amazon · 2022-01-07

#### Problem

Implement a stack that has the following methods:

- `push(val)`, which pushes an element onto the stack
- `pop()`, which pops off and returns the topmost element of the stack. If there are no elements in the stack, then it should throw an error or return null.
- `max()`, which returns the maximum value in the stack currently. If there are no elements in the stack, then it should throw an error or return null.

Each method should run in constant time.

### Problem #747 <a id="p747"></a>

**Easy** · Google · 2022-01-08

#### Problem

Given two strings A and B, return whether or not A can be shifted some number of times to get B.

For example, if A is abcde and B is cdeab, return true. If A is abc and B is acb, return false.

### Problem #748 <a id="p748"></a>

**Easy** · Apple · 2022-01-09

#### Problem

Given the root of a binary tree, find the most frequent subtree sum. The subtree sum of a node is the sum of all values under a node, including the node itself.

For example, given the following tree:

```
  5
 / \
2  -5
```

Return 2 as it occurs twice: once as the left leaf, and once as the sum of 2 + 5 - 5.

### Problem #749 <a id="p749"></a>

**Hard** · Amazon · 2022-01-10

#### Problem

Given a string, find the longest palindromic contiguous substring. If there are more than one with the maximum length, return any one.

For example, the longest palindromic substring of `"aabcdcb"` is `"bcdcb"`. The longest palindromic substring of `"bananas"` is `"anana"`.

### Problem #750 <a id="p750"></a>

**Medium** · Jane Street · 2022-01-11

#### Problem

Generate a finite, but an arbitrarily large binary tree quickly in `O(1)`.

That is, `generate()` should return a tree whose size is unbounded but finite.

### Problem #751 <a id="p751"></a>

**Hard** · Palantir · 2022-01-12

#### Problem

Typically, an implementation of in-order traversal of a binary tree has `O(h)` space complexity, where h is the height of the tree. Write a program to compute the in-order traversal of a binary tree using `O(1)` space.

### Problem #752 <a id="p752"></a>

**Easy** · Microsoft · 2022-01-13

#### Problem

Print the nodes in a binary tree level-wise. For example, the following should print 1, 2, 3, 4, 5.

```
  1
 / \
2   3
   / \
  4   5
```

### Problem #753 <a id="p753"></a>

**Easy** · Zillow · 2022-01-14

#### Problem

A ternary search tree is a trie-like data structure where each node may have up to three children. Here is an example which represents the words code, cob, be, ax, war, and we.

```
       c
    /  |  \
   b   o   w
 / |   |   |
a  e   d   a
|    / |   | \ 
x   b  e   r  e
```

The tree is structured according to the following rules:

- left child nodes link to words lexicographically earlier than the parent prefix
- right child nodes link to words lexicographically later than the parent prefix
- middle child nodes continue the current word

For instance, since code is the first word inserted in the tree, and cob lexicographically precedes cod, cob is represented as a left child extending from cod.

Implement insertion and search functions for a ternary search tree.

### Problem #754 <a id="p754"></a>

**Medium** · Square · 2022-01-15

#### Problem

In front of you is a row of N coins, with values v1, v1, ..., vn.

You are asked to play the following game. You and an opponent take turns choosing either the first or last coin from the row, removing it from the row, and receiving the value of the coin.

Write a program that returns the maximum amount of money you can win with certainty, if you move first, assuming your opponent plays optimally.

### Problem #755 <a id="p755"></a>

**Hard** · Google · 2022-01-16

#### Problem

In a directed graph, each node is assigned an uppercase letter. We define a path's value as the number of most frequently-occurring letter along that path. For example, if a path in the graph goes through `"ABACA"`, the value of the path is 3, since there are 3 occurrences of `'A'` on the path.

Given a graph with n nodes and m directed edges, return the largest value path of the graph. If the largest value is infinite, then return null.

The graph is represented with a string and an edge list. The i-th character represents the uppercase letter of the i-th node. Each tuple in the edge list (i, j) means there is a directed edge from the i-th node to the j-th node. Self-edges are possible, as well as multi-edges.

For example, the following input graph:

ABACA

```
[(0, 1),
 (0, 2),
 (2, 3),
 (3, 4)]
```

Would have maximum value 3 using the path of vertices `[0, 2, 3, 4]`, (A, A, C, A).

The following input graph:

A

`[(0, 0)]`

Should return null, since we have an infinite loop.

### Problem #756 <a id="p756"></a>

**Easy** · Slack · 2022-01-17

#### Problem

You are given a string formed by concatenating several words corresponding to the integers zero through nine and then anagramming.

For example, the input could be `'niesevehrtfeev'`, which is an anagram of `'threefiveseven'`. Note that there can be multiple instances of each integer.

Given this string, return the original integers in sorted order. In the example above, this would be 357.

### Problem #757 <a id="p757"></a>

**Hard** · MIT · 2022-01-18

#### Problem

Blackjack is a two player card game whose rules are as follows:

- The player and then the dealer are each given two cards.
- The player can then `"hit"`, or ask for arbitrarily many additional cards, so long as their total does not exceed 21.
- The dealer must then hit if their total is 16 or lower, otherwise pass.
- Finally, the two compare totals, and the one with the greatest sum not exceeding 21 is the winner.

For this problem, cards values are counted as follows: each card between 2 and 10 counts as their face value, face cards count as 10, and aces count as 1.

Given perfect knowledge of the sequence of cards in the deck, implement a blackjack solver that maximizes the player's score (that is, wins minus losses).

### Problem #758 <a id="p758"></a>

**Medium** · Facebook · 2022-01-19

#### Problem

Write a function that rotates a list by k elements. For example, `[1, 2, 3, 4, 5, 6]` rotated by two becomes `[3, 4, 5, 6, 1, 2]`. Try solving this without creating a copy of the list. How many swap or move operations do you need?

### Problem #759 <a id="p759"></a>

**Medium** · Snapchat · 2022-01-20

#### Problem

Given a string of digits, generate all possible valid IP address combinations.

IP addresses must follow the format A.B.C.D, where A, B, C, and D are numbers between 0 and 255. Zero-prefixed numbers, such as 01 and 065, are not allowed, except for 0 itself.

For example, given `"2542540123"`, you should return `['254.25.40.123', '254.254.0.123']`.

### Problem #760 <a id="p760"></a>

**Hard** · Apple · 2022-01-21

#### Problem

Given a linked list, uniformly shuffle the nodes. What if we want to prioritize space over time?

### Problem #761 <a id="p761"></a>

**Easy** · Two Sigma · 2022-01-22

#### Problem

Using a function `rand7()` that returns an integer from 1 to 7 (inclusive) with uniform probability, implement a function `rand5()` that returns an integer from 1 to 5 (inclusive).

### Problem #762 <a id="p762"></a>

**Hard** · 2022-01-23

#### Problem

Find an efficient algorithm to find the smallest distance (measured in number of words) between any two given words in a string.

For example, given words `"hello"`, and `"world"` and a text content of `"dog cat hello cat dog dog hello cat world"`, return 1 because there's only one word `"cat"` in between the two words.

### Problem #763 <a id="p763"></a>

**Hard** · Google · 2022-01-24

#### Problem

Given an array of integers and a number k, where 1 \<= k \<= length of the array, compute the maximum values of each subarray of length k.

For example, given array = `[10, 5, 2, 7, 8, 7]` and k = 3, we should get: `[10, 7, 8, 8]`, since:

- 10 = `max(10, 5, 2)`
- 7 = `max(5, 2, 7)`
- 8 = `max(2, 7, 8)`
- 8 = `max(7, 8, 7)`

Do this in `O(n)` time and `O(k)` space. You can modify the input array in-place and you do not need to store the results. You can simply print them out as you compute them.

### Problem #764 <a id="p764"></a>

**Medium** · Twitter · 2022-01-25

#### Problem

Given a list of numbers, create an algorithm that arranges them in order to form the largest possible integer. For example, given `[10, 7, 76, 415]`, you should return 77641510.

### Problem #765 <a id="p765"></a>

**Medium** · Google · 2022-01-26

#### Problem

Given a singly linked list and an integer k, remove the kth last element from the list. k is guaranteed to be smaller than the length of the list.

The list is very long, so making more than one pass is prohibitively expensive.

Do this in constant space and in one pass.

### Problem #766 <a id="p766"></a>

**Medium** · LinkedIn · 2022-01-27

#### Problem

You are given a string consisting of the letters x and y, such as xyxxxyxyy. In addition, you have an operation called flip, which changes a single x to y or vice versa.

Determine how many times you would need to apply this operation to ensure that all x's come before all y's. In the preceding example, it suffices to flip the second and sixth characters, so you should return 2.

### Problem #767 <a id="p767"></a>

**Hard** · Google · 2022-01-28

#### Problem

Given a word W and a string S, find all starting indices in S which are anagrams of W.

For example, given that W is `"ab"`, and S is `"abxaba"`, return 0, 3, and 4.

### Problem #768 <a id="p768"></a>

**Hard** · Facebook · 2022-01-29

#### Problem

Given an array of numbers of length N, find both the minimum and maximum using less than 2 \* (N - 2) comparisons.

### Problem #769 <a id="p769"></a>

**Hard** · Two Sigma · 2022-01-30

#### Problem

Alice wants to join her school's Probability Student Club. Membership dues are computed via one of two simple probabilistic games.

The first game: roll a die repeatedly. Stop rolling once you get a five followed by a six. Your number of rolls is the amount you pay, in dollars.

The second game: same, except that the stopping condition is a five followed by a five.

Which of the two games should Alice elect to play? Does it even matter? Write a program to simulate the two games and calculate their expected value.

### Problem #770 <a id="p770"></a>

**Hard** · Google · 2022-01-31

#### Problem

The game of Nim is played as follows. Starting with three heaps, each containing a variable number of items, two players take turns removing one or more items from a single pile. The player who eventually is forced to take the last stone loses. For example, if the initial heap sizes are 3, 4, and 5, a game could be played as shown below:

```
  A  |  B  |  C
-----------------
  3  |  4  |  5
  3  |  1  |  3
  3  |  1  |  3
  0  |  1  |  3
  0  |  1  |  0
  0  |  0  |  0
```

In other words, to start, the first player takes three items from pile B. The second player responds by removing two stones from pile C. The game continues in this way until player one takes last stone and loses.

Given a list of non-zero starting values `[a, b, c]`, and assuming optimal play, determine whether the first player has a forced win.

### Problem #771 <a id="p771"></a>

**Easy** · Bloomberg · 2022-02-01

#### Problem

Determine whether there exists a one-to-one character mapping from one string s1 to another s2.

For example, given s1 = abc and s2 = bcd, return true since we can map a to b, b to c, and c to d.

Given s1 = foo and s2 = bar, return false since the o cannot map to two characters.

### Problem #772 <a id="p772"></a>

**Easy** · Facebook · 2022-02-02

#### Problem

Boggle is a game played on a 4 x 4 grid of letters. The goal is to find as many words as possible that can be formed by a sequence of adjacent letters in the grid, using each cell at most once. Given a game board and a dictionary of valid words, implement a Boggle solver.

### Problem #773 <a id="p773"></a>

**Medium** · Google · 2022-02-03

#### Problem

We can determine how `"out of order"` an array A is by counting the number of inversions it has. Two elements `A[i]` and `A[j]` form an inversion if `A[i]` > `A[j]` but i \< j. That is, a smaller element appears after a larger element.

Given an array, count the number of inversions it has. Do this faster than `O(N^2)` time.

You may assume each element in the array is distinct.

For example, a sorted list has zero inversions. The array `[2, 4, 1, 3, 5]` has three inversions: (2, 1), (4, 1), and (4, 3). The array `[5, 4, 3, 2, 1]` has ten inversions: every distinct pair forms an inversion.

### Problem #774 <a id="p774"></a>

**Easy** · Microsoft · 2022-02-04

#### Problem

Using a `read7()` method that returns 7 characters from a file, implement `readN(n)` which reads n characters.

For example, given a file with the content “Hello world”, three `read7()` returns “Hello w”, “orld” and then “”.

### Problem #775 <a id="p775"></a>

**Easy** · Snapchat · 2022-02-05

#### Problem

Given an array of time intervals (start, end) for classroom lectures (possibly overlapping), find the minimum number of rooms required.

For example, given `[(30, 75), (0, 50), (60, 150)]`, you should return 2.

### Problem #776 <a id="p776"></a>

**Easy** · Bloomberg · 2022-02-06

#### Problem

There are N prisoners standing in a circle, waiting to be executed. The executions are carried out starting with the kth person, and removing every successive kth person going clockwise until there is no one left.

Given N and k, write an algorithm to determine where a prisoner should stand in order to be the last survivor.

For example, if N = 5 and k = 2, the order of executions would be `[2, 4, 1, 5, 3]`, so you should return 3.

Bonus: Find an `O(log N)` solution if k = 2.

### Problem #777 <a id="p777"></a>

**Easy** · Zillow · 2022-02-07

#### Problem

A ternary search tree is a trie-like data structure where each node may have up to three children. Here is an example which represents the words code, cob, be, ax, war, and we.

```
       c
    /  |  \
   b   o   w
 / |   |   |
a  e   d   a
|    / |   | \ 
x   b  e   r  e
```

The tree is structured according to the following rules:

- left child nodes link to words lexicographically earlier than the parent prefix
- right child nodes link to words lexicographically later than the parent prefix
- middle child nodes continue the current word

For instance, since code is the first word inserted in the tree, and cob lexicographically precedes cod, cob is represented as a left child extending from cod.

Implement insertion and search functions for a ternary search tree.

### Problem #778 <a id="p778"></a>

**Medium** · Spotify · 2022-02-08

#### Problem

You have access to ranked lists of songs for various users. Each song is represented as an integer, and more preferred songs appear earlier in each list. For example, the list `[4, 1, 7]` indicates that a user likes song 4 the best, followed by songs 1 and 7.

Given a set of these ranked lists, interleave them to create a playlist that satisfies everyone's priorities.

For example, suppose your input is `{[1, 7, 3], [2, 1, 6, 7, 9], [3, 9, 5]}`. In this case a satisfactory playlist could be `[2, 1, 6, 7, 3, 9, 5]`.

### Problem #779 <a id="p779"></a>

**Medium** · Goldman Sachs · 2022-02-09

#### Problem

You are given N identical eggs and access to a building with k floors. Your task is to find the lowest floor that will cause an egg to break, if dropped from that floor. Once an egg breaks, it cannot be dropped again. If an egg breaks when dropped from the xth floor, you can assume it will also break when dropped from any floor greater than x.

Write an algorithm that finds the minimum number of trial drops it will take, in the worst case, to identify this floor.

For example, if N = 1 and k = 5, we will need to try dropping the egg at every floor, beginning with the first, until we reach the fifth floor, so our solution will be 5.

### Problem #780 <a id="p780"></a>

**Hard** · Airbnb · 2022-02-10

#### Problem

We're given a hashmap associating each courseId key with a list of courseIds values, which represents that the prerequisites of courseId are courseIds. Return a sorted ordering of courses such that we can finish all courses.

Return null if there is no such ordering.

For example, given `{'CSC300': ['CSC100', 'CSC200'], 'CSC200': ['CSC100'], 'CSC100': []}`, should return `['CSC100', 'CSC200', 'CSCS300']`.

### Problem #781 <a id="p781"></a>

**Medium** · Square · 2022-02-11

#### Problem

You are given a histogram consisting of rectangles of different heights. These heights are represented in an input list, such that `[1, 3, 2, 5]` corresponds to the following diagram:

```
      x
      x  
  x   x
  x x x
x x x x
```

Determine the area of the largest rectangle that can be formed only from the bars of the histogram. For the diagram above, for example, this would be six, representing the 2 x 3 area at the bottom right.

### Problem #782 <a id="p782"></a>

**Easy** · Palantir · 2022-02-12

#### Problem

In academia, the h-index is a metric used to calculate the impact of a researcher's papers. It is calculated as follows:

A researcher has index h if at least h of her N papers have h citations each. If there are multiple h satisfying this formula, the maximum is chosen.

For example, suppose N = 5, and the respective citations of each paper are `[4, 3, 0, 1, 5]`. Then the h-index would be 3, since the researcher has 3 papers with at least 3 citations.

Given a list of paper citations of a researcher, calculate their h-index.

### Problem #783 <a id="p783"></a>

**Hard** · Microsoft · 2022-02-13

#### Problem

Given an array of numbers, find the length of the longest increasing subsequence in the array. The subsequence does not necessarily have to be contiguous.

For example, given the array `[0, 8, 4, 12, 2, 10, 6, 14, 1, 9, 5, 13, 3, 11, 7, 15]`, the longest increasing subsequence has length 6: it is 0, 2, 6, 9, 11, 15.

### Problem #784 <a id="p784"></a>

**Easy** · Microsoft · 2022-02-14

#### Problem

Given a 2D matrix of characters and a target word, write a function that returns whether the word can be found in the matrix by going left-to-right, or up-to-down.

For example, given the following matrix:

```
[['F', 'A', 'C', 'I'],
 ['O', 'B', 'Q', 'P'],
 ['A', 'N', 'O', 'B'],
 ['M', 'A', 'S', 'S']]
```

and the target word `'FOAM'`, you should return true, since it's the leftmost column. Similarly, given the target word `'MASS'`, you should return true, since it's the last row.

### Problem #785 <a id="p785"></a>

**Easy** · Facebook · 2022-02-15

#### Problem

Given a list of integers, return the largest product that can be made by multiplying any three integers.

For example, if the list is `[-10, -10, 5, 2]`, we should return 500, since that's -10 \* -10 \* 5.

You can assume the list has at least three integers.

### Problem #786 <a id="p786"></a>

**Medium** · Google · 2022-02-16

#### Problem

Implement integer exponentiation. That is, implement the `pow(x, y)` function, where x and y are integers and returns x^y.

Do this faster than the naive method of repeated multiplication.

For example, `pow(2, 10)` should return 1024.

### Problem #787 <a id="p787"></a>

**Medium** · Facebook · 2022-02-17

#### Problem

Given an integer n, find the next biggest integer with the same number of 1-bits on. For example, given the number 6 (0110 in binary), return 9 (1001).

### Problem #788 <a id="p788"></a>

**Easy** · Palantir · 2022-02-18

#### Problem

Write a program that checks whether an integer is a palindrome. For example, 121 is a palindrome, as well as 888. 678 is not a palindrome. Do not convert the integer into a string.

### Problem #789 <a id="p789"></a>

**Hard** · Palantir · 2022-02-19

#### Problem

Given a number represented by a list of digits, find the next greater permutation of a number, in terms of lexicographic ordering. If there is not greater permutation possible, return the permutation with the lowest value/ordering.

For example, the list `[1,2,3]` should return `[1,3,2]`. The list `[1,3,2]` should return `[2,1,3]`. The list `[3,2,1]` should return `[1,2,3]`.

Can you perform the operation without allocating extra memory (disregarding the input memory)?

### Problem #790 <a id="p790"></a>

**Easy** · Pivotal · 2022-02-20

#### Problem

A step word is formed by taking a given word, adding a letter, and anagramming the result. For example, starting with the word `"APPLE"`, you can add an `"A"` and anagram to get `"APPEAL"`.

Given a dictionary of words and an input word, create a function that returns all valid step words.

### Problem #791 <a id="p791"></a>

**Medium** · Spotify · 2022-02-21

#### Problem

Write a function, `throw_dice(N, faces, total)`, that determines how many ways it is possible to throw N dice with some number of faces each to get a specific total.

For example, `throw_dice(3, 6, 7)` should equal 15.

### Problem #792 <a id="p792"></a>

**Hard** · Uber · 2022-02-22

#### Problem

You have N stones in a row, and would like to create from them a pyramid. This pyramid should be constructed such that the height of each stone increases by one until reaching the tallest stone, after which the heights decrease by one. In addition, the start and end stones of the pyramid should each be one stone high.

You can change the height of any stone by paying a cost of 1 unit to lower its height by 1, as many times as necessary. Given this information, determine the lowest cost method to produce this pyramid.

For example, given the stones `[1, 1, 3, 3, 2, 1]`, the optimal solution is to pay 2 to create `[0, 1, 2, 3, 2, 1]`.

### Problem #793 <a id="p793"></a>

**Medium** · Yahoo · 2022-02-23

#### Problem

Recall that a full binary tree is one in which each node is either a leaf node, or has two children. Given a binary tree, convert it to a full one by removing nodes with only one child.

For example, given the following tree:

```
         0
      /     \
    1         2
  /            \
3                 4
  \             /   \
    5          6     7
```

You should convert it to:

```
     0
  /     \
5         4
        /   \
       6     7
```

### Problem #794 <a id="p794"></a>

**Easy** · Amazon · 2022-02-24

#### Problem

Implement a stack that has the following methods:

- `push(val)`, which pushes an element onto the stack
- `pop()`, which pops off and returns the topmost element of the stack. If there are no elements in the stack, then it should throw an error or return null.
- `max()`, which returns the maximum value in the stack currently. If there are no elements in the stack, then it should throw an error or return null.

Each method should run in constant time.

### Problem #795 <a id="p795"></a>

**Medium** · Square · 2022-02-25

#### Problem

Assume you have access to a function `toss_biased()` which returns 0 or 1 with a probability that's not 50-50 (but also not 0-100 or 100-0). You do not know the bias of the coin.

Write a function to simulate an unbiased coin toss.

### Problem #796 <a id="p796"></a>

**Medium** · Nvidia · 2022-02-26

#### Problem

You are given a list of N points (x1, y1), (x2, y2), ..., (xN, yN) representing a polygon. You can assume these points are given in order; that is, you can construct the polygon by connecting point 1 to point 2, point 2 to point 3, and so on, finally looping around to connect point N to point 1.

Determine if a new point p lies inside this polygon. (If p is on the boundary of the polygon, you should return False).

### Problem #797 <a id="p797"></a>

**Easy** · Alibaba · 2022-02-27

#### Problem

Given an even number (greater than 2), return two prime numbers whose sum will be equal to the given number.

A solution will always exist. See Goldbach’s conjecture.

Example:

```
Input: 4
Output: 2 + 2 = 4
```

If there are more than one solution possible, return the lexicographically smaller solution.

If `[a, b]` is one solution with a \<= b, and `[c, d]` is another solution with c \<= d, then

`[a, b]` \< `[c, d]`

If a \< c OR a==c AND b \< d.

### Problem #798 <a id="p798"></a>

**Medium** · Google · 2022-02-28

#### Problem

You are given a set of synonyms, such as (big, large) and (eat, consume). Using this set, determine if two sentences with the same number of words are equivalent.

For example, the following two sentences are equivalent:

- `"He wants to eat food."`
- `"He wants to consume food."`

Note that the synonyms (a, b) and (a, c) do not necessarily imply (b, c): consider the case of (coach, bus) and (coach, teacher).

Follow-up: what if we can assume that (a, b) and (a, c) do in fact imply (b, c)?

### Problem #799 <a id="p799"></a>

**Easy** · Google · 2022-03-01

#### Problem

Implement a PrefixMapSum class with the following methods:

- `insert(key: str, value: int)`: Set a given key's value in the map. If the key already exists, overwrite the value.
- `sum(prefix: str)`: Return the sum of all values of keys that begin with a given prefix.

For example, you should be able to run the following code:

```
mapsum.insert("columnar", 3)
assert mapsum.sum("col") == 3
```

```
mapsum.insert("column", 2)
assert mapsum.sum("col") == 5
```

### Problem #800 <a id="p800"></a>

**Medium** · Fitbit · 2022-03-02

#### Problem

Given a linked list, rearrange the node values such that they appear in alternating low -> high -> low -> high ... form. For example, given 1 -> 2 -> 3 -> 4 -> 5, you should return 1 -> 3 -> 2 -> 5 -> 4.

### Problem #801 <a id="p801"></a>

**Easy** · Zillow · 2022-03-03

#### Problem

Let's define a `"sevenish"` number to be one which is either a power of 7, or the sum of unique powers of 7. The first few sevenish numbers are 1, 7, 8, 49, and so on. Create an algorithm to find the nth sevenish number.

### Problem #802 <a id="p802"></a>

**Medium** · Triplebyte · 2022-03-04

#### Problem

You are given n numbers as well as n probabilities that sum up to 1. Write a function to generate one of the numbers with its corresponding probability.

For example, given the numbers `[1, 2, 3, 4]` and probabilities `[0.1, 0.5, 0.2, 0.2]`, your function should return 1 10% of the time, 2 50% of the time, and 3 and 4 20% of the time.

You can generate random numbers between 0 and 1 uniformly.

### Problem #803 <a id="p803"></a>

**Medium** · Facebook · 2022-03-05

#### Problem

Mastermind is a two-player game in which the first player attempts to guess the secret code of the second. In this version, the code may be any six-digit number with all distinct digits.

Each turn the first player guesses some number, and the second player responds by saying how many digits in this number correctly matched their location in the secret code. For example, if the secret code were 123456, then a guess of 175286 would score two, since 1 and 6 were correctly placed.

Write an algorithm which, given a sequence of guesses and their scores, determines whether there exists some secret code that could have produced them.

For example, for the following scores you should return True, since they correspond to the secret code 123456:

`{175286: 2, 293416: 3, 654321: 0}`

However, it is impossible for any key to result in the following scores, so in this case you should return False:

`{123456: 4, 345678: 4, 567890: 4}`

### Problem #804 <a id="p804"></a>

**Medium** · Airbnb · 2022-03-06

#### Problem

You are given a huge list of airline ticket prices between different cities around the world on a given day. These are all direct flights. Each element in the list has the format (`source_city`, destination, price).

Consider a user who is willing to take up to k connections from their origin city A to their destination B. Find the cheapest fare possible for this journey and print the itinerary for that journey.

For example, our traveler wants to go from JFK to LAX with up to 3 connections, and our input flights are as follows:

```
[
    ('JFK', 'ATL', 150),
    ('ATL', 'SFO', 400),
    ('ORD', 'LAX', 200),
    ('LAX', 'DFW', 80),
    ('JFK', 'HKG', 800),
    ('ATL', 'ORD', 90),
    ('JFK', 'LAX', 500),
]
```

Due to some improbably low flight prices, the cheapest itinerary would be JFK -> ATL -> ORD -> LAX, costing \$440.

### Problem #805 <a id="p805"></a>

**Easy** · Dropbox · 2022-03-07

#### Problem

Spreadsheets often use this alphabetical encoding for its columns: `"A"`, `"B"`, `"C"`, ..., `"AA"`, `"AB"`, ..., `"ZZ"`, `"AAA"`, `"AAB"`, ....

Given a column number, return its alphabetical column id. For example, given 1, return `"A"`. Given 27, return `"AA"`.

### Problem #806 <a id="p806"></a>

**Hard** · Google · 2022-03-08

#### Problem

Let A be an N by M matrix in which every row and every column is sorted.

Given i1, j1, i2, and j2, compute the number of elements of M smaller than `M[i1, j1]` and larger than `M[i2, j2]`.

For example, given the following matrix:

```
[[1, 3, 7, 10, 15, 20],
 [2, 6, 9, 14, 22, 25],
 [3, 8, 10, 15, 25, 30],
 [10, 11, 12, 23, 30, 35],
 [20, 25, 30, 35, 40, 45]]
```

And i1 = 1, j1 = 1, i2 = 3, j2 = 3, return 15 as there are 15 numbers in the matrix smaller than 6 or greater than 23.

### Problem #807 <a id="p807"></a>

**Hard** · Amazon · 2022-03-09

#### Problem

Given an integer k and a string s, find the length of the longest substring that contains at most k distinct characters.

For example, given s = `"abcba"` and k = 2, the longest substring with k distinct characters is `"bcb"`.

### Problem #808 <a id="p808"></a>

**Hard** · Palantir · 2022-03-10

#### Problem

Typically, an implementation of in-order traversal of a binary tree has `O(h)` space complexity, where h is the height of the tree. Write a program to compute the in-order traversal of a binary tree using `O(1)` space.

### Problem #809 <a id="p809"></a>

**Easy** · Facebook · 2022-03-11

#### Problem

Given a string of round, curly, and square open and closing brackets, return whether the brackets are balanced (well-formed).

For example, given the string `"([])[]({})"`, you should return true.

Given the string `"([)]"` or `"((()"`, you should return false.

### Problem #810 <a id="p810"></a>

**Easy** · Morgan Stanley · 2022-03-12

#### Problem

In Ancient Greece, it was common to write text with the first line going left to right, the second line going right to left, and continuing to go back and forth. This style was called `"boustrophedon"`.

Given a binary tree, write an algorithm to print the nodes in boustrophedon order.

For example, given the following tree:

```
       1
    /     \
  2         3
 / \       / \
4   5     6   7
```

You should return `[1, 3, 2, 4, 5, 6, 7]`.

### Problem #811 <a id="p811"></a>

**Medium** · Square · 2022-03-13

#### Problem

Given a list of words, return the shortest unique prefix of each word. For example, given the list:

- dog
- cat
- apple
- apricot
- fish

Return the list:

- d
- c
- app
- apr
- f

### Problem #812 <a id="p812"></a>

**Medium** · Facebook · 2022-03-14

#### Problem

In chess, the Elo rating system is used to calculate player strengths based on game results.

A simplified description of the Elo system is as follows. Every player begins at the same score. For each subsequent game, the loser transfers some points to the winner, where the amount of points transferred depends on how unlikely the win is. For example, a 1200-ranked player should gain much more points for beating a 2000-ranked player than for beating a 1300-ranked player.

Implement this system.

### Problem #813 <a id="p813"></a>

**Medium** · Facebook · 2022-03-15

#### Problem

Given an array of integers, write a function to determine whether the array could become non-decreasing by modifying at most 1 element.

For example, given the array `[10, 5, 7]`, you should return true, since we can modify the 10 into a 1 to make the array non-decreasing.

Given the array `[10, 5, 1]`, you should return false, since we can't modify any one element to get a non-decreasing array.

### Problem #814 <a id="p814"></a>

**Easy** · Microsoft · 2022-03-16

#### Problem

Let's represent an integer in a linked list format by having each node represent a digit in the number. The nodes make up the number in reversed order.

For example, the following linked list:

1 -> 2 -> 3 -> 4 -> 5

is the number 54321.

Given two linked lists in this format, return their sum in the same linked list format.

For example, given

9 -> 9

5 -> 2

return 124 (99 + 25) as:

4 -> 2 -> 1

### Problem #815 <a id="p815"></a>

**Medium** · Google · 2022-03-17

#### Problem

The area of a circle is defined as πr^2. Estimate π to 3 decimal places using a Monte Carlo method.

Hint: The basic equation of a circle is x2 + y2 = r2.

### Problem #816 <a id="p816"></a>

**Medium** · Facebook · 2022-03-18

#### Problem

Given three 32-bit integers x, y, and b, return x if b is 1 and y if b is 0, using only mathematical or bit operations. You can assume b can only be 1 or 0.

### Problem #817 <a id="p817"></a>

**Medium** · Microsoft · 2022-03-19

#### Problem

Given a dictionary of words and a string made up of those words (no spaces), return the original sentence in a list. If there is more than one possible reconstruction, return any of them. If there is no possible reconstruction, then return null.

For example, given the set of words `'quick'`, `'brown'`, `'the'`, `'fox'`, and the string `"thequickbrownfox"`, you should return `['the', 'quick', 'brown', 'fox']`.

Given the set of words `'bed'`, `'bath'`, `'bedbath'`, `'and'`, `'beyond'`, and the string `"bedbathandbeyond"`, return either `['bed', 'bath', 'and', 'beyond]` or `['bedbath', 'and', 'beyond']`.

### Problem #818 <a id="p818"></a>

**Easy** · Square · 2022-03-20

#### Problem

The Sieve of Eratosthenes is an algorithm used to generate all prime numbers smaller than N. The method is to take increasingly larger prime numbers, and mark their multiples as composite.

For example, to find all primes less than 100, we would first mark `[4, 6, 8, ...]` (multiples of two), then `[6, 9, 12, ...]` (multiples of three), and so on. Once we have done this for all primes less than N, the unmarked numbers that remain will be prime.

Implement this algorithm.

Bonus: Create a generator that produces primes indefinitely (that is, without taking N as an input).

### Problem #819 <a id="p819"></a>

**Easy** · Facebook · 2022-03-21

#### Problem

Given a array of numbers representing the stock prices of a company in chronological order, write a function that calculates the maximum profit you could have made from buying and selling that stock once. You must buy before you can sell it.

For example, given `[9, 11, 8, 5, 7, 10]`, you should return 5, since you could buy the stock at 5 dollars and sell it at 10 dollars.

### Problem #820 <a id="p820"></a>

**Hard** · Stripe · 2022-03-22

#### Problem

Given an array of integers, find the first missing positive integer in linear time and constant space. In other words, find the lowest positive integer that does not exist in the array. The array can contain duplicates and negative numbers as well.

For example, the input `[3, 4, -1, 1]` should give 2. The input `[1, 2, 0]` should give 3.

You can modify the input array in-place.

### Problem #821 <a id="p821"></a>

**Easy** · Apple · 2022-03-23

#### Problem

A fixed point in an array is an element whose value is equal to its index. Given a sorted array of distinct elements, return a fixed point, if one exists. Otherwise, return False.

For example, given `[-6, 0, 2, 40]`, you should return 2. Given `[1, 5, 7, 8]`, you should return False.

### Problem #822 <a id="p822"></a>

**Easy** · Snapchat · 2022-03-24

#### Problem

Given a list of possibly overlapping intervals, return a new list of intervals where all overlapping intervals have been merged.

The input list is not necessarily ordered in any way.

For example, given `[(1, 3), (5, 8), (4, 10), (20, 25)]`, you should return `[(1, 3), (4, 10), (20, 25)]`.

### Problem #823 <a id="p823"></a>

**Medium** · Amazon · 2022-03-25

#### Problem

Implement a bit array.

A bit array is a space efficient array that holds a value of 1 or 0 at each index.

- `init(size)`: initialize the array with size
- `set(i, val)`: updates index at i with val where val is either 1 or 0.
- `get(i)`: gets the value at index i.

### Problem #824 <a id="p824"></a>

**Easy** · Salesforce · 2022-03-26

#### Problem

Write a program to merge two binary trees. Each node in the new tree should hold a value equal to the sum of the values of the corresponding nodes of the input trees.

If only one input tree has a node in a given position, the corresponding node in the new tree should match that input node.

### Problem #825 <a id="p825"></a>

**Easy** · Google · 2022-03-27

#### Problem

Given a sorted list of integers, square the elements and give the output in sorted order.

For example, given `[-9, -2, 0, 2, 3]`, return `[0, 4, 4, 9, 81]`.

### Problem #826 <a id="p826"></a>

**Hard** · Google · 2022-03-28

#### Problem

Implement a file syncing algorithm for two computers over a low-bandwidth network. What if we know the files in the two computers are mostly the same?

### Problem #827 <a id="p827"></a>

**Medium** · Spotify · 2022-03-29

#### Problem

You are the technical director of WSPT radio, serving listeners nationwide. For simplicity's sake we can consider each listener to live along a horizontal line stretching from 0 (west) to 1000 (east).

Given a list of N listeners, and a list of M radio towers, each placed at various locations along this line, determine what the minimum broadcast range would have to be in order for each listener's home to be covered.

For example, suppose listeners = `[1, 5, 11, 20]`, and towers = `[4, 8, 15]`. In this case the minimum range would be 5, since that would be required for the tower at position 15 to reach the listener at position 20.

### Problem #828 <a id="p828"></a>

**Medium** · Microsoft · 2022-03-30

#### Problem

Write a program to determine how many distinct ways there are to create a max heap from a list of N given integers.

For example, if N = 3, and our integers are `[1, 2, 3]`, there are two ways, shown below.

```
  3      3
 / \    / \
1   2  2   1
```

### Problem #829 <a id="p829"></a>

**Hard** · Dropbox · 2022-03-31

#### Problem

Create a data structure that performs all the following operations in `O(1)` time:

- plus: Add a key with value 1. If the key already exists, increment its value by one.
- minus: Decrement the value of a key. If the key's value is currently 1, remove it.
- `get_max`: Return a key with the highest value.
- `get_min`: Return a key with the lowest value.

### Problem #830 <a id="p830"></a>

**Medium** · Twitter · 2022-04-01

#### Problem

Given a list of numbers, create an algorithm that arranges them in order to form the largest possible integer. For example, given `[10, 7, 76, 415]`, you should return 77641510.

### Problem #831 <a id="p831"></a>

**Medium** · Dropbox · 2022-04-02

#### Problem

Given a string s and a list of words words, where each word is the same length, find all starting indices of substrings in s that is a concatenation of every word in words exactly once.

For example, given s = `"dogcatcatcodecatdog"` and words = `["cat", "dog"]`, return `[0, 13]`, since `"dogcat"` starts at index 0 and `"catdog"` starts at index 13.

Given s = `"barfoobazbitbyte"` and words = `["dog", "cat"]`, return \[\] since there are no substrings composed of `"dog"` and `"cat"` in s.

The order of the indices does not matter.

### Problem #832 <a id="p832"></a>

**Easy** · Google · 2022-04-03

#### Problem

Given an array of elements, return the length of the longest subarray where all its elements are distinct.

For example, given the array `[5, 1, 3, 5, 2, 3, 4, 1]`, return 5 as the longest subarray of distinct elements is `[5, 2, 3, 4, 1]`.

### Problem #833 <a id="p833"></a>

**Medium** · Pinterest · 2022-04-04

#### Problem

At a party, there is a single person who everyone knows, but who does not know anyone in return (the `"celebrity"`). To help figure out who this is, you have access to an `O(1)` method called `knows(a, b)`, which returns True if person a knows person b, else False.

Given a list of N people and the above operation, find a way to identify the celebrity in `O(N)` time.

### Problem #834 <a id="p834"></a>

**Hard** · Spotify · 2022-04-05

#### Problem

There are N couples sitting in a row of length 2 \* N. They are currently ordered randomly, but would like to rearrange themselves so that each couple's partners can sit side by side.

What is the minimum number of swaps necessary for this to happen?

### Problem #835 <a id="p835"></a>

**Medium** · Square · 2022-04-06

#### Problem

Given a string and a set of characters, return the shortest substring containing all the characters in the set.

For example, given the string `"figehaeci"` and the set of characters `{a, e, i}`, you should return `"aeci"`.

If there is no substring containing all the characters in the set, return null.

### Problem #836 <a id="p836"></a>

**Medium** · Facebook · 2022-04-07

#### Problem

Given a function that generates perfectly random numbers between 1 and k (inclusive), where k is an input, write a function that shuffles a deck of cards represented as an array using only swaps.

It should run in `O(N)` time.

Hint: Make sure each one of the 52! permutations of the deck is equally likely.

### Problem #837 <a id="p837"></a>

**Medium** · Nest · 2022-04-08

#### Problem

Create a basic sentence checker that takes in a stream of characters and determines whether they form valid sentences. If a sentence is valid, the program should print it out.

We can consider a sentence valid if it conforms to the following rules:

- The sentence must start with a capital letter, followed by a lowercase letter or a space.
- All other characters must be lowercase letters, separators (,,;,:) or terminal marks (.,?,!,‽).
- There must be a single space between each word.
- The sentence must end with a terminal mark immediately following a word.

### Problem #838 <a id="p838"></a>

**Medium** · Zillow · 2022-04-09

#### Problem

You are given a 2-d matrix where each cell represents number of coins in that cell. Assuming we start at `matrix[0][0]`, and can only move right or down, find the maximum number of coins you can collect by the bottom right corner.

For example, in this matrix

```
0 3 1 1
2 0 0 4
1 5 3 1
```

The most we can collect is 0 + 2 + 1 + 5 + 3 + 1 = 12 coins.

### Problem #839 <a id="p839"></a>

**Easy** · Google · 2022-04-10

#### Problem

You are given an N by N matrix of random letters and a dictionary of words. Find the maximum number of words that can be packed on the board from the given dictionary.

A word is considered to be able to be packed on the board if:

- It can be found in the dictionary
- It can be constructed from untaken letters by other words found so far on the board
- The letters are adjacent to each other (vertically and horizontally, not diagonally).

Each tile can be visited only once by any word.

For example, given the following dictionary:

`{ 'eat', 'rain', 'in', 'rat' }`

and matrix:

```
[['e', 'a', 'n'],
 ['t', 't', 'i'],
 ['a', 'r', 'a']]
```

Your function should return 3, since we can make the words `'eat'`, `'in'`, and `'rat'` without them touching each other. We could have alternatively made `'eat'` and `'rain'`, but that would be incorrect since that's only 2 words.

### Problem #841 <a id="p841"></a>

**Easy** · Jane Street · 2022-04-12

#### Problem

Given integers M and N, write a program that counts how many positive integer pairs (a, b) satisfy the following conditions:

- a + b = M
- a XOR b = N

### Problem #842 <a id="p842"></a>

**Medium** · Google · 2022-04-13

#### Problem

Invert a binary tree.

For example, given the following tree:

```
    a
   / \
  b   c
 / \  /
d   e f
```

should become:

```
 a
/ \
c  b
\  / \
 f e  d
```

### Problem #843 <a id="p843"></a>

**Medium** · Microsoft · 2022-04-14

#### Problem

Given a string and a pattern, find the starting indices of all occurrences of the pattern in the string. For example, given the string `"abracadabra"` and the pattern `"abr"`, you should return `[0, 7]`.

### Problem #844 <a id="p844"></a>

**Hard** · Dropbox · 2022-04-15

#### Problem

A Boolean formula can be said to be satisfiable if there is a way to assign truth values to each variable such that the entire formula evaluates to true.

For example, suppose we have the following formula, where the symbol ¬ is used to denote negation:

(¬c OR b) AND (b OR c) AND (¬b OR c) AND (¬c OR ¬a)

One way to satisfy this formula would be to let a = False, b = True, and c = True.

This type of formula, with AND statements joining tuples containing exactly one OR, is known as 2-CNF.

Given a 2-CNF formula, find a way to assign truth values to satisfy it, or return False if this is impossible.

### Problem #845 <a id="p845"></a>

**Medium** · Facebook · 2022-04-16

#### Problem

Write a function that rotates a list by k elements. For example, `[1, 2, 3, 4, 5, 6]` rotated by two becomes `[3, 4, 5, 6, 1, 2]`. Try solving this without creating a copy of the list. How many swap or move operations do you need?

### Problem #846 <a id="p846"></a>

**Medium** · Jane Street · 2022-04-17

#### Problem

`cons(a, b)` constructs a pair, and `car(pair)` and `cdr(pair)` returns the first and last element of that pair. For example, `car(cons(3, 4))` returns 3, and `cdr(cons(3, 4))` returns 4.

Given this implementation of cons:

```
def cons(a, b):
    def pair(f):
        return f(a, b)
    return pair
```

Implement car and cdr.

### Problem #847 <a id="p847"></a>

**Medium** · Pinterest · 2022-04-18

#### Problem

Given an integer list where each number represents the number of hops you can make, determine whether you can reach to the last index starting at index 0.

For example, `[2, 0, 1, 0]` returns True while `[1, 1, 0, 1]` returns False.

### Problem #848 <a id="p848"></a>

**Hard** · Google · 2022-04-19

#### Problem

Implement an LRU (Least Recently Used) cache. It should be able to be initialized with a cache size n, and contain the following methods:

- `set(key, value)`: sets key to value. If there are already n items in the cache and we are adding a new item, then it should also remove the least recently used item.
- `get(key)`: gets the value at key. If no such key exists, return null.

Each operation should run in `O(1)` time.

### Problem #849 <a id="p849"></a>

**Easy** · Apple · 2022-04-20

#### Problem

A Collatz sequence in mathematics can be defined as follows. Starting with any positive integer:

- if n is even, the next number in the sequence is n / 2
- if n is odd, the next number in the sequence is 3n + 1

It is conjectured that every such sequence eventually reaches the number 1. Test this conjecture.

Bonus: What input n \<= 1000000 gives the longest sequence?

### Problem #850 <a id="p850"></a>

**Hard** · LinkedIn · 2022-04-21

#### Problem

Given a set of characters C and an integer k, a De Bruijn sequence is a cyclic sequence in which every possible k-length string of characters in C occurs exactly once.

For example, suppose C = `{0, 1}` and k = 3. Then our sequence should contain the substrings `{'000', '001', '010', '011', '100', '101', '110', '111'}`, and one possible solution would be 00010111.

Create an algorithm that finds a De Bruijn sequence.

### Problem #851 <a id="p851"></a>

**Medium** · Facebook · 2022-04-22

#### Problem

A graph is minimally-connected if it is connected and there is no edge that can be removed while still leaving the graph connected. For example, any binary tree is minimally-connected.

Given an undirected graph, check if the graph is minimally-connected. You can choose to represent the graph as either an adjacency matrix or adjacency list.

### Problem #852 <a id="p852"></a>

**Medium** · Facebook · 2022-04-23

#### Problem

Given a circular array, compute its maximum subarray sum in `O(n)` time. A subarray can be empty, and in this case the sum is 0.

For example, given `[8, -1, 3, 4]`, return 15 as we choose the numbers 3, 4, and 8 where the 8 is obtained from wrapping around.

Given `[-4, 5, 1, 0]`, return 6 as we choose the numbers 5 and 1.

### Problem #853 <a id="p853"></a>

**Hard** · 2022-04-24

#### Problem

Find an efficient algorithm to find the smallest distance (measured in number of words) between any two given words in a string.

For example, given words `"hello"`, and `"world"` and a text content of `"dog cat hello cat dog dog hello cat world"`, return 1 because there's only one word `"cat"` in between the two words.

### Problem #854 <a id="p854"></a>

**Medium** · Amazon · 2022-04-25

#### Problem

Given a string s and an integer k, break up the string into multiple lines such that each line has a length of k or less. You must break it up so that words don't break across lines. Each line has to have the maximum possible amount of words. If there's no way to break the text up, then return null.

You can assume that there are no spaces at the ends of the string and that there is exactly one space between each word.

For example, given the string `"the quick brown fox jumps over the lazy dog"` and k = 10, you should return: `["the quick", "brown fox", "jumps over", "the lazy", "dog"]`. No string in the list has a length of more than 10.

### Problem #855 <a id="p855"></a>

**Hard** · Microsoft · 2022-04-26

#### Problem

You have an N by N board. Write a function that, given N, returns the number of possible arrangements of the board where N queens can be placed on the board without threatening each other, i.e. no two queens share the same row, column, or diagonal.

### Problem #856 <a id="p856"></a>

**Medium** · Quora · 2022-04-27

#### Problem

You are given a list of (website, user) pairs that represent users visiting websites. Come up with a program that identifies the top k pairs of websites with the greatest similarity.

For example, suppose k = 1, and the list of tuples is:

```
[('a', 1), ('a', 3), ('a', 5),
 ('b', 2), ('b', 6),
 ('c', 1), ('c', 2), ('c', 3), ('c', 4), ('c', 5)
 ('d', 4), ('d', 5), ('d', 6), ('d', 7),
 ('e', 1), ('e', 3), ('e': 5), ('e', 6)]
```

Then a reasonable similarity metric would most likely conclude that a and e are the most similar, so your program should return `[('a', 'e')]`.

### Problem #857 <a id="p857"></a>

**Hard** · LinkedIn · 2022-04-28

#### Problem

You are given a binary tree in a peculiar string representation. Each node is written in the form (lr), where l corresponds to the left child and r corresponds to the right child.

If either l or r is null, it will be represented as a zero. Otherwise, it will be represented by a new (lr) pair.

Here are a few examples:

- A root node with no children: (00)
- A root node with two children: ((00)(00))
- An unbalanced tree with three consecutive left children: ((((00)0)0)0)

Given this representation, determine the depth of the tree.

### Problem #858 <a id="p858"></a>

**Easy** · Microsoft · 2022-04-29

#### Problem

Compute the running median of a sequence of numbers. That is, given a stream of numbers, print out the median of the list so far on each new element.

Recall that the median of an even-numbered list is the average of the two middle numbers.

For example, given the sequence `[2, 1, 5, 7, 2, 0, 5]`, your algorithm should print out:

```
2
1.5
2
3.5
2
2
2
```

### Problem #859 <a id="p859"></a>

**Medium** · Yelp · 2022-04-30

#### Problem

The horizontal distance of a binary tree node describes how far left or right the node will be when the tree is printed out.

More rigorously, we can define it as follows:

- The horizontal distance of the root is 0.
- The horizontal distance of a left child is `hd(parent)` - 1.
- The horizontal distance of a right child is `hd(parent)` + 1.

For example, for the following tree, `hd(1)` = -2, and `hd(6)` = 0.

```
           5
        /     \
      3         7
    /  \      /   \
  1     4    6     9
 /                /
0                8
```

The bottom view of a tree, then, consists of the lowest node at each horizontal distance. If there are two nodes at the same depth and horizontal distance, either is acceptable.

For this tree, for example, the bottom view could be `[0, 1, 3, 6, 8, 9]`.

Given the root to a binary tree, return its bottom view.

### Problem #860 <a id="p860"></a>

**Hard** · Facebook · 2022-05-01

#### Problem

Implement regular expression matching with the following special characters:

- . (period) which matches any single character
- \* (asterisk) which matches zero or more of the preceding element

That is, implement a function that takes in a string and a valid regular expression and returns whether or not the string matches the regular expression.

For example, given the regular expression `"ra."` and the string `"ray"`, your function should return true. The same regular expression on the string `"raymond"` should return false.

Given the regular expression `".*at"` and the string `"chat"`, your function should return true. The same regular expression on the string `"chats"` should return false.

### Problem #861 <a id="p861"></a>

**Hard** · Two Sigma · 2022-05-02

#### Problem

A knight is placed on a given square on an 8 x 8 chessboard. It is then moved randomly several times, where each move is a standard knight move. If the knight jumps off the board at any point, however, it is not allowed to jump back on.

After k moves, what is the probability that the knight remains on the board?

### Problem #862 <a id="p862"></a>

**Medium** · Mozilla · 2022-05-03

#### Problem

A bridge in a connected (undirected) graph is an edge that, if removed, causes the graph to become disconnected. Find all the bridges in a graph.

### Problem #863 <a id="p863"></a>

**Medium** · Jane Street · 2022-05-04

#### Problem

Generate a finite, but an arbitrarily large binary tree quickly in `O(1)`.

That is, `generate()` should return a tree whose size is unbounded but finite.

### Problem #864 <a id="p864"></a>

**Medium** · Glassdoor · 2022-05-05

#### Problem

An imminent hurricane threatens the coastal town of Codeville. If at most two people can fit in a rescue boat, and the maximum weight limit for a given boat is k, determine how many boats will be needed to save everyone.

For example, given a population with weights `[100, 200, 150, 80]` and a boat limit of 200, the smallest number of boats required will be three.

### Problem #865 <a id="p865"></a>

**Medium** · Samsung · 2022-05-06

#### Problem

A group of houses is connected to the main water plant by means of a set of pipes. A house can either be connected by a set of pipes extending directly to the plant, or indirectly by a pipe to a nearby house which is otherwise connected.

For example, here is a possible configuration, where A, B, and C are houses, and arrows represent pipes:

A \<--> B \<--> C \<--> plant

Each pipe has an associated cost, which the utility company would like to minimize. Given an undirected graph of pipe connections, return the lowest cost configuration of pipes such that each house has access to water.

In the following setup, for example, we can remove all but the pipes from plant to A, plant to B, and B to C, for a total cost of 16.

```
pipes = {
    'plant': {'A': 1, 'B': 5, 'C': 20},
    'A': {'C': 15},
    'B': {'C': 10},
    'C': {}
}
```

### Problem #866 <a id="p866"></a>

**Medium** · Facebook · 2022-05-07

#### Problem

Given an array of numbers representing the stock prices of a company in chronological order and an integer k, return the maximum profit you can make from k buys and sells. You must buy the stock before you can sell it, and you must sell the stock before you can buy it again.

For example, given k = 2 and the array `[5, 2, 4, 0, 1]`, you should return 3.

### Problem #867 <a id="p867"></a>

**Easy** · Google · 2022-05-08

#### Problem

You are given an array of arrays of integers, where each array corresponds to a row in a triangle of numbers. For example, `[[1], [2, 3], [1, 5, 1]]` represents the triangle:

```
  1
 2 3
1 5 1
```

We define a path in the triangle to start at the top and go down one row at a time to an adjacent value, eventually ending with an entry on the bottom row. For example, 1 -> 3 -> 5. The weight of the path is the sum of the entries.

Write a program that returns the weight of the maximum weight path.

### Problem #868 <a id="p868"></a>

**Easy** · Amazon · 2022-05-09

#### Problem

Given a string, determine whether any permutation of it is a palindrome.

For example, carrace should return true, since it can be rearranged to form racecar, which is a palindrome. daily should return false, since there's no rearrangement that can form a palindrome.

### Problem #869 <a id="p869"></a>

**Easy** · Google · 2022-05-10

#### Problem

Determine whether a doubly linked list is a palindrome. What if it’s singly linked?

For example, 1 -> 4 -> 3 -> 4 -> 1 returns True while 1 -> 4 returns False.

### Problem #870 <a id="p870"></a>

**Medium** · Google · 2022-05-11

#### Problem

Given an N by M matrix consisting only of 1's and 0's, find the largest rectangle containing only 1's and return its area.

For example, given the following matrix:

```
[[1, 0, 0, 0],
 [1, 0, 1, 1],
 [1, 0, 1, 1],
 [0, 1, 0, 0]]
```

Return 4.

### Problem #871 <a id="p871"></a>

**Hard** · Grammarly · 2022-05-12

#### Problem

Soundex is an algorithm used to categorize phonetically, such that two names that sound alike but are spelled differently have the same representation.

Soundex maps every name to a string consisting of one letter and three numbers, like M460.

One version of the algorithm is as follows:

- Remove consecutive consonants with the same sound (for example, change ck -> c).
- Keep the first letter. The remaining steps only apply to the rest of the string.
- Remove all vowels, including y, w, and h.
- Replace all consonants with the following digits:

- b, f, p, v → 1
- c, g, j, k, q, s, x, z → 2
- d, t → 3
- l → 4
- m, n → 5
- r → 6

- If you don't have three numbers yet, append zeros until you do. Keep the first three numbers.

Using this scheme, Jackson and Jaxen both map to J250.

Implement Soundex.

### Problem #872 <a id="p872"></a>

**Medium** · Dropbox · 2022-05-13

#### Problem

Create an algorithm to efficiently compute the approximate median of a list of numbers.

More precisely, given an unordered list of N numbers, find an element whose rank is between N / 4 and 3 \* N / 4, with a high level of certainty, in less than `O(N)` time.

### Problem #873 <a id="p873"></a>

**Hard** · Jane Street · 2022-05-14

#### Problem

Given an arithmetic expression in Reverse Polish Notation, write a program to evaluate it.

The expression is given as a list of numbers and operands. For example: `[5, 3, '+']` should return 5 + 3 = 8.

For example, `[15, 7, 1, 1, '+', '-', '/', 3, '*', 2, 1, 1, '+', '+', '-']` should return 5, since it is equivalent to ((15 / (7 - (1 + 1))) \* 3) - (2 + (1 + 1)) = 5.

You can assume the given expression is always valid.

### Problem #874 <a id="p874"></a>

**Hard** · Amazon · 2022-05-15

#### Problem

Given a string, find the longest palindromic contiguous substring. If there are more than one with the maximum length, return any one.

For example, the longest palindromic substring of `"aabcdcb"` is `"bcdcb"`. The longest palindromic substring of `"bananas"` is `"anana"`.

### Problem #875 <a id="p875"></a>

**Hard** · Google · 2022-05-16

#### Problem

Suppose we represent our file system by a string in the following manner:

The string `"dir\n\tsubdir1\n\tsubdir2\n\t\tfile.ext"` represents:

```
dir
    subdir1
    subdir2
        file.ext
```

The directory dir contains an empty sub-directory subdir1 and a sub-directory subdir2 containing a file file.ext.

The string `"dir\n\tsubdir1\n\t\tfile1.ext\n\t\tsubsubdir1\n\tsubdir2\n\t\tsubsubdir2\n\t\t\tfile2.ext"` represents:

```
dir
    subdir1
        file1.ext
        subsubdir1
    subdir2
        subsubdir2
            file2.ext
```

The directory dir contains two sub-directories subdir1 and subdir2. subdir1 contains a file file1.ext and an empty second-level sub-directory subsubdir1. subdir2 contains a second-level sub-directory subsubdir2 containing a file file2.ext.

We are interested in finding the longest (number of characters) absolute path to a file within our file system. For example, in the second example above, the longest absolute path is `"dir/subdir2/subsubdir2/file2.ext"`, and its length is 32 (not including the double quotes).

Given a string representing the file system in the above format, return the length of the longest absolute path to a file in the abstracted file system. If there is no file in the system, return 0.

Note:

The name of a file contains at least a period and an extension.

The name of a directory or sub-directory will not contain a period.

### Problem #876 <a id="p876"></a>

**Medium** · Palantir · 2022-05-17

#### Problem

You are given a list of N numbers, in which each number is located at most k places away from its sorted position. For example, if k = 1, a given element at index 4 might end up at indices 3, 4, or 5.

Come up with an algorithm that sorts this list in `O(N log k)` time.

### Problem #877 <a id="p877"></a>

**Hard** · Google · 2022-05-18

#### Problem

Given two non-empty binary trees s and t, check whether tree t has exactly the same structure and node values with a subtree of s. A subtree of s is a tree consists of a node in s and all of this node's descendants. The tree s could also be considered as a subtree of itself.

### Problem #878 <a id="p878"></a>

**Hard** · Apple · 2022-05-19

#### Problem

Given a tree, find the largest tree/subtree that is a BST.

Given a tree, return the size of the largest tree/subtree that is a BST.

### Problem #879 <a id="p879"></a>

**Easy** · Google · 2022-05-20

#### Problem

Given a list of numbers and a number k, return whether any two numbers from the list add up to k.

For example, given `[10, 15, 3, 7]` and k of 17, return true since 10 + 7 is 17.

Bonus: Can you do this in one pass?

### Problem #880 <a id="p880"></a>

**Hard** · Google · 2022-05-21

#### Problem

Given a string, split it into as few strings as possible such that each string is a palindrome.

For example, given the input string racecarannakayak, return `["racecar", "anna", "kayak"]`.

Given the input string abc, return `["a", "b", "c"]`.

### Problem #881 <a id="p881"></a>

**Medium** · Uber · 2022-05-22

#### Problem

Implement a 2D iterator class. It will be initialized with an array of arrays, and should implement the following methods:

- `next()`: returns the next element in the array of arrays. If there are no more elements, raise an exception.
- `has_next()`: returns whether or not the iterator still has elements left.

For example, given the input `[[1, 2], [3], [], [4, 5, 6]]`, calling `next()` repeatedly should output 1, 2, 3, 4, 5, 6.

Do not use flatten or otherwise clone the arrays. Some of the arrays can be empty.

### Problem #882 <a id="p882"></a>

**Hard** · Airbnb · 2022-05-23

#### Problem

An 8-puzzle is a game played on a 3 x 3 board of tiles, with the ninth tile missing. The remaining tiles are labeled 1 through 8 but shuffled randomly. Tiles may slide horizontally or vertically into an empty space, but may not be removed from the board.

Design a class to represent the board, and find a series of steps to bring the board to the state `[[1, 2, 3], [4, 5, 6], [7, 8, None]]`.

### Problem #883 <a id="p883"></a>

**Easy** · PagerDuty · 2022-05-24

#### Problem

Given a positive integer N, find the smallest number of steps it will take to reach 1.

There are two kinds of permitted steps:

- You may decrement N to N - 1.
- If a \* b = N, you may decrement N to the larger of a and b.

For example, given 100, you can reach 1 in five steps with the following route: 100 -> 10 -> 9 -> 3 -> 2 -> 1.

### Problem #884 <a id="p884"></a>

**Easy** · Two Sigma · 2022-05-25

#### Problem

Using a function `rand7()` that returns an integer from 1 to 7 (inclusive) with uniform probability, implement a function `rand5()` that returns an integer from 1 to 5 (inclusive).

### Problem #885 <a id="p885"></a>

**Hard** · Google · 2022-05-26

#### Problem

Given an array of strictly the characters `'R'`, `'G'`, and `'B'`, segregate the values of the array so that all the Rs come first, the Gs come second, and the Bs come last. You can only swap elements of the array.

Do this in linear time and in-place.

For example, given the array `['G', 'B', 'R', 'R', 'B', 'R', 'G']`, it should become `['R', 'R', 'R', 'G', 'G', 'B', 'B']`.

### Problem #886 <a id="p886"></a>

**Easy** · Google · 2022-05-27

#### Problem

The edit distance between two strings refers to the minimum number of character insertions, deletions, and substitutions required to change one string to the other. For example, the edit distance between “kitten” and “sitting” is three: substitute the “k” for “s”, substitute the “e” for “i”, and append a “g”.

Given two strings, compute the edit distance between them.

### Problem #887 <a id="p887"></a>

**Easy** · Palantir · 2022-05-28

#### Problem

The ancient Egyptians used to express fractions as a sum of several terms where each numerator is one. For example, 4 / 13 can be represented as 1 / 4 + 1 / 18 + 1 / 468.

Create an algorithm to turn an ordinary fraction a / b, where a \< b, into an Egyptian fraction.

### Problem #888 <a id="p888"></a>

**Hard** · LinkedIn · 2022-05-29

#### Problem

Given a list of points, a central point, and an integer k, find the nearest k points from the central point.

For example, given the list of points `[(0, 0), (5, 4), (3, 1)]`, the central point (1, 2), and k = 2, return `[(0, 0), (3, 1)]`.

### Problem #889 <a id="p889"></a>

**Easy** · Amazon · 2022-05-30

#### Problem

Run-length encoding is a fast and simple method of encoding strings. The basic idea is to represent repeated successive characters as a single count and character. For example, the string `"AAAABBBCCDAA"` would be encoded as `"4A3B2C1D2A"`.

Implement run-length encoding and decoding. You can assume the string to be encoded have no digits and consists solely of alphabetic characters. You can assume the string to be decoded is valid.

### Problem #890 <a id="p890"></a>

**Medium** · Facebook · 2022-05-31

#### Problem

There is an N by M matrix of zeroes. Given N and M, write a function to count the number of ways of starting at the top-left corner and getting to the bottom-right corner. You can only move right or down.

For example, given a 2 by 2 matrix, you should return 2, since there are two ways to get to the bottom-right:

- Right, then down
- Down, then right

Given a 5 by 5 matrix, there are 70 ways to get to the bottom-right.

### Problem #891 <a id="p891"></a>

**Medium** · Uber · 2022-06-01

#### Problem

You are given a 2-d matrix where each cell consists of either /, \\, or an empty space. Write an algorithm that determines into how many regions the slashes divide the space.

For example, suppose the input for a three-by-six grid is the following:

```
\    /
 \  /
  \/
```

Considering the edges of the matrix as boundaries, this divides the grid into three triangles, so you should return 3.

### Problem #892 <a id="p892"></a>

**Easy** · Google · 2022-06-02

#### Problem

The power set of a set is the set of all its subsets. Write a function that, given a set, generates its power set.

For example, given the set `{1, 2, 3}`, it should return `{{}, {1}, {2}, {3}, {1, 2}, {1, 3}, {2, 3}, {1, 2, 3}}`.

You may also use a list or array to represent a set.

### Problem #893 <a id="p893"></a>

**Medium** · Facebook · 2022-06-03

#### Problem

A builder is looking to build a row of N houses that can be of K different colors. He has a goal of minimizing cost while ensuring that no two neighboring houses are of the same color.

Given an N by K matrix where the nth row and kth column represents the cost to build the nth house with kth color, return the minimum cost which achieves this goal.

### Problem #894 <a id="p894"></a>

**Easy** · Riot Games · 2022-06-04

#### Problem

Design and implement a HitCounter class that keeps track of requests (or hits). It should support the following operations:

- `record(timestamp)`: records a hit that happened at timestamp
- `total()`: returns the total number of hits recorded
- `range(lower, upper)`: returns the number of hits that occurred between timestamps lower and upper (inclusive)

Follow-up: What if our system has limited memory?

### Problem #895 <a id="p895"></a>

**Hard** · Google · 2022-06-05

#### Problem

Given a string which we can delete at most k, return whether you can make a palindrome.

For example, given `'waterrfetawx'` and a k of 2, you could delete f and x to get `'waterretaw'`.

### Problem #896 <a id="p896"></a>

**Medium** · ContextLogic · 2022-06-06

#### Problem

Implement division of two positive integers without using the division, multiplication, or modulus operators. Return the quotient as an integer, ignoring the remainder.

### Problem #897 <a id="p897"></a>

**Medium** · Apple · 2022-06-07

#### Problem

Implement a queue using two stacks. Recall that a queue is a FIFO (first-in, first-out) data structure with the following methods: enqueue, which inserts an element into the queue, and dequeue, which removes it.

### Problem #898 <a id="p898"></a>

**Easy** · Amazon · 2022-06-08

#### Problem

Implement a stack API using only a heap. A stack implements the following methods:

- `push(item)`, which adds an element to the stack
- `pop()`, which removes and returns the most recently added element (or throws an error if there is nothing on the stack)

Recall that a heap has the following operations:

- `push(item)`, which adds a new key to the heap
- `pop()`, which removes and returns the max value of the heap

### Problem #899 <a id="p899"></a>

**Easy** · Twitter · 2022-06-09

#### Problem

The 24 game is played as follows. You are given a list of four integers, each between 1 and 9, in a fixed order. By placing the operators +, -, \*, and / between the numbers, and grouping them with parentheses, determine whether it is possible to reach the value 24.

For example, given the input `[5, 2, 7, 8]`, you should return True, since (5 \* 2 - 7) \* 8 = 24.

Write a function that plays the 24 game.

### Problem #900 <a id="p900"></a>

**Medium** · Amazon · 2022-06-10

#### Problem

Given an array of numbers, find the maximum sum of any contiguous subarray of the array.

For example, given the array `[34, -50, 42, 14, -5, 86]`, the maximum sum would be 137, since we would take elements 42, 14, -5, and 86.

Given the array `[-5, -1, -8, -9]`, the maximum sum would be 0, since we would not take any elements.

Do this in `O(N)` time.

### Problem #901 <a id="p901"></a>

**Hard** · Google · 2022-06-11

#### Problem

Design a system to crawl and copy all of Wikipedia using a distributed network of machines.

More specifically, suppose your server has access to a set of client machines. Your client machines can execute code you have written to access Wikipedia pages, download and parse their data, and write the results to a database.

Some questions you may want to consider as part of your solution are:

- How will you reach as many pages as possible?
- How can you keep track of pages that have already been visited?
- How will you deal with your client machines being blacklisted?
- How can you update your database when Wikipedia pages are added or updated?

### Problem #902 <a id="p902"></a>

**Easy** · Stripe · 2022-06-12

#### Problem

Given a collection of intervals, find the minimum number of intervals you need to remove to make the rest of the intervals non-overlapping.

Intervals can `"touch"`, such as `[0, 1]` and `[1, 2]`, but they won't be considered overlapping.

For example, given the intervals (7, 9), (2, 4), (5, 8), return 1 as the last interval can be removed and the first two won't overlap.

The intervals are not necessarily sorted in any order.

### Problem #903 <a id="p903"></a>

**Easy** · Microsoft · 2022-06-13

#### Problem

Given an array of numbers and a number k, determine if there are three entries in the array which add up to the specified number k. For example, given `[20, 303, 3, 4, 25]` and k = 49, return true as 20 + 4 + 25 = 49.

### Problem #904 <a id="p904"></a>

**Easy** · Google · 2022-06-14

#### Problem

Given the root of a binary tree, return a deepest node. For example, in the following tree, return d.

```
    a
   / \
  b   c
 /
d
```

### Problem #905 <a id="p905"></a>

**Medium** · Dropbox · 2022-06-15

#### Problem

Given a list of words, determine whether the words can be chained to form a circle. A word X can be placed in front of another word Y in a circle if the last character of X is same as the first character of Y.

For example, the words `['chair', 'height', 'racket', touch', 'tunic']` can form the following circle: chair --> racket --> touch --> height --> tunic --> chair.

### Problem #906 <a id="p906"></a>

**Easy** · Google · 2022-06-16

#### Problem

Given a set of points (x, y) on a 2D cartesian plane, find the two closest points. For example, given the points `[(1, 1), (-1, -1), (3, 4), (6, 1), (-1, -6), (-4, -3)]`, return `[(-1, -1), (1, 1)]`.

### Problem #907 <a id="p907"></a>

**Medium** · Apple · 2022-06-17

#### Problem

Gray code is a binary code where each successive value differ in only one bit, as well as when wrapping around. Gray code is common in hardware so that we don't see temporary spurious values during transitions.

Given a number of bits n, generate a possible gray code for it.

For example, for n = 2, one gray code would be `[00, 01, 11, 10]`.

### Problem #908 <a id="p908"></a>

**Medium** · Google · 2022-06-18

#### Problem

You are given an N by M 2D matrix of lowercase letters. Determine the minimum number of columns that can be removed to ensure that each row is ordered from top to bottom lexicographically. That is, the letter at each column is lexicographically later as you go down each row. It does not matter whether each row itself is ordered lexicographically.

For example, given the following table:

```
cba
daf
ghi
```

This is not ordered because of the a in the center. We can remove the second column to make it ordered:

```
ca
df
gi
```

So your function should return 1, since we only needed to remove 1 column.

As another example, given the following table:

abcdef

Your function should return 0, since the rows are already ordered (there's only one row).

As another example, given the following table:

```
zyx
wvu
tsr
```

Your function should return 3, since we would need to remove all the columns to order it.

### Problem #909 <a id="p909"></a>

**Medium** · Google · 2022-06-19

#### Problem

Given a set of closed intervals, find the smallest set of numbers that covers all the intervals. If there are multiple smallest sets, return any of them.

For example, given the intervals `[0, 3]`, `[2, 6]`, `[3, 4]`, `[6, 9]`, one set of numbers that covers all these intervals is `{3, 6}`.

### Problem #910 <a id="p910"></a>

**Medium** · Yelp · 2022-06-20

#### Problem

You are given an array of integers, where each element represents the maximum number of steps that can be jumped going forward from that element. Write a function to return the minimum number of jumps you must take in order to get from the start to the end of the array.

For example, given `[6, 2, 4, 0, 5, 1, 1, 4, 2, 9]`, you should return 2, as the optimal solution involves jumping from 6 to 5, and then from 5 to 9.

### Problem #911 <a id="p911"></a>

**Medium** · Facebook · 2022-06-21

#### Problem

Given a stream of elements too large to store in memory, pick a random element from the stream with uniform probability.

### Problem #912 <a id="p912"></a>

**Medium** · Yext · 2022-06-22

#### Problem

Two nodes in a binary tree can be called cousins if they are on the same level of the tree but have different parents. For example, in the following diagram 4 and 6 are cousins.

```
    1
   / \
  2   3
 / \   \
4   5   6
```

Given a binary tree and a particular node, find all cousins of that node.

### Problem #913 <a id="p913"></a>

**Easy** · Amazon · 2022-06-23

#### Problem

Given a complete binary tree, count the number of nodes in faster than `O(n)` time. Recall that a complete binary tree has every level filled except the last, and the nodes in the last level are filled starting from the left.

### Problem #914 <a id="p914"></a>

**Hard** · Netflix · 2022-06-24

#### Problem

Implement a queue using a set of fixed-length arrays.

The queue should support enqueue, dequeue, and `get_size` operations.

### Problem #915 <a id="p915"></a>

**Hard** · Airbnb · 2022-06-25

#### Problem

You are given an array X of floating-point numbers x1, x2, ... xn. These can be rounded up or down to create a corresponding array Y of integers y1, y2, ... yn.

Write an algorithm that finds an appropriate Y array with the following properties:

- The rounded sums of both arrays should be equal.
- The absolute pairwise difference between elements is minimized. In other words, |x1- y1| + |x2- y2| + ... + |xn- yn| should be as small as possible.

For example, suppose your input is `[1.3, 2.3, 4.4]`. In this case you cannot do better than `[1, 2, 5]`, which has an absolute difference of |1.3 - 1| + |2.3 - 2| + |4.4 - 5| = 1.

### Problem #916 <a id="p916"></a>

**Easy** · Google · 2022-06-26

#### Problem

Given the head of a singly linked list, reverse it in-place.

### Problem #917 <a id="p917"></a>

**Medium** · LinkedIn · 2022-06-27

#### Problem

Given a linked list of numbers and a pivot k, partition the linked list so that all nodes less than k come before nodes greater than or equal to k.

For example, given the linked list 5 -> 1 -> 8 -> 0 -> 3 and k = 3, the solution could be 1 -> 0 -> 5 -> 8 -> 3.

### Problem #918 <a id="p918"></a>

**Hard** · Google · 2022-06-28

#### Problem

PageRank is an algorithm used by Google to rank the importance of different websites. While there have been changes over the years, the central idea is to assign each site a score based on the importance of other pages that link to that page.

More mathematically, suppose there are N sites, and each site i has a certain count Ci of outgoing links. Then the score for a particular site Sj is defined as :

`score(Sj)` = (1 - d) / N + d \* (`score(Sx)` / Cx+ `score(Sy)` / Cy+ ... + `score(Sz)` / Cz))

Here, Sx, Sy, ..., Sz denote the scores of all the other sites that have outgoing links to Sj, and d is a damping factor, usually set to around 0.85, used to model the probability that a user will stop searching.

Given a directed graph of links between various websites, write a program that calculates each site's page rank.

### Problem #919 <a id="p919"></a>

**Medium** · LinkedIn · 2022-06-29

#### Problem

A wall consists of several rows of bricks of various integer lengths and uniform height. Your goal is to find a vertical line going from the top to the bottom of the wall that cuts through the fewest number of bricks. If the line goes through the edge between two bricks, this does not count as a cut.

For example, suppose the input is as follows, where values in each row represent the lengths of bricks in that row:

```
[[3, 5, 1, 1],
 [2, 3, 3, 2],
 [5, 5],
 [4, 4, 2],
 [1, 3, 3, 3],
 [1, 1, 6, 1, 1]]
```

The best we can we do here is to draw a line after the eighth brick, which will only require cutting through the bricks in the third and fifth row.

Given an input consisting of brick lengths for each row such as the one above, return the fewest number of bricks that must be cut to create a vertical line.

### Problem #920 <a id="p920"></a>

**Medium** · Triplebyte · 2022-06-30

#### Problem

Implement a data structure which carries out the following operations without resizing the underlying array:

- `add(value)`: Add a value to the set of values.
- `check(value)`: Check whether a value is in the set.

The check method may return occasional false positives (in other words, incorrectly identifying an element as part of the set), but should always correctly identify a true element.

### Problem #921 <a id="p921"></a>

**Hard** · Google · 2022-07-01

#### Problem

An XOR linked list is a more memory efficient doubly linked list. Instead of each node holding next and prev fields, it holds a field named both, which is an XOR of the next node and the previous node. Implement an XOR linked list; it has an `add(element)` which adds the element to the end, and a `get(index)` which returns the node at index.

If using a language that has no pointers (such as Python), you can assume you have access to `get_pointer` and `dereference_pointer` functions that converts between nodes and memory addresses.

### Problem #922 <a id="p922"></a>

**Medium** · Google · 2022-07-02

#### Problem

Implement locking in a binary tree. A binary tree node can be locked or unlocked only if all of its descendants or ancestors are not locked.

Design a binary tree node class with the following methods:

- `is_locked`, which returns whether the node is locked
- lock, which attempts to lock the node. If it cannot be locked, then it should return false. Otherwise, it should lock it and return true.
- unlock, which unlocks the node. If it cannot be unlocked, then it should return false. Otherwise, it should unlock it and return true.

You may augment the node to add parent pointers or any other property you would like. You may assume the class is used in a single-threaded program, so there is no need for actual locks or mutexes. Each method should run in `O(h)`, where h is the height of the tree.

### Problem #923 <a id="p923"></a>

**Medium** · Indeed · 2022-07-03

#### Problem

Given a 32-bit positive integer N, determine whether it is a power of four in faster than `O(log N)` time.

### Problem #924 <a id="p924"></a>

**Easy** · WhatsApp · 2022-07-04

#### Problem

Given an array of integers out of order, determine the bounds of the smallest window that must be sorted in order for the entire array to be sorted. For example, given `[3, 7, 5, 6, 9]`, you should return (1, 3).

### Problem #925 <a id="p925"></a>

**Easy** · Stripe · 2022-07-05

#### Problem

Write a function to flatten a nested dictionary. Namespace the keys with a period.

For example, given the following dictionary:

```
{
    "key": 3,
    "foo": {
        "a": 5,
        "bar": {
            "baz": 8
        }
    }
}
```

it should become:

```
{
    "key": 3,
    "foo.a": 5,
    "foo.bar.baz": 8
}
```

You can assume keys do not contain dots in them, i.e. no clobbering will occur.

### Problem #926 <a id="p926"></a>

**Easy** · Apple · 2022-07-06

#### Problem

Given the root of a binary tree, find the most frequent subtree sum. The subtree sum of a node is the sum of all values under a node, including the node itself.

For example, given the following tree:

```
  5
 / \
2  -5
```

Return 2 as it occurs twice: once as the left leaf, and once as the sum of 2 + 5 - 5.

### Problem #927 <a id="p927"></a>

**Easy** · Airbnb · 2022-07-07

#### Problem

Given a linked list and a positive integer k, rotate the list to the right by k places.

For example, given the linked list 7 -> 7 -> 3 -> 5 and k = 2, it should become 3 -> 5 -> 7 -> 7.

Given the linked list 1 -> 2 -> 3 -> 4 -> 5 and k = 3, it should become 3 -> 4 -> 5 -> 1 -> 2.

### Problem #928 <a id="p928"></a>

**Easy** · Microsoft · 2022-07-08

#### Problem

A number is considered perfect if its digits sum up to exactly 10.

Given a positive integer n, return the n-th perfect number.

For example, given 1, you should return 19. Given 2, you should return 28.

### Problem #929 <a id="p929"></a>

**Medium** · Square · 2022-07-09

#### Problem

A competitive runner would like to create a route that starts and ends at his house, with the condition that the route goes entirely uphill at first, and then entirely downhill.

Given a dictionary of places of the form `{location: elevation}`, and a dictionary mapping paths between some of these locations to their corresponding distances, find the length of the shortest route satisfying the condition above. Assume the runner's home is location 0.

For example, suppose you are given the following input:

```
elevations = {0: 5, 1: 25, 2: 15, 3: 20, 4: 10}
paths = {
    (0, 1): 10,
    (0, 2): 8,
    (0, 3): 15,
    (1, 3): 12,
    (2, 4): 10,
    (3, 4): 5,
    (3, 0): 17,
    (4, 0): 10
}
```

In this case, the shortest valid path would be 0 -> 2 -> 4 -> 0, with a distance of 28.

### Problem #930 <a id="p930"></a>

**Medium** · Google · 2022-07-10

#### Problem

Given a linked list, sort it in `O(n log n)` time and constant space.

For example, the linked list 4 -> 1 -> -3 -> 99 should become -3 -> 1 -> 4 -> 99.

### Problem #931 <a id="p931"></a>

**Easy** · Amazon · 2022-07-11

#### Problem

Given n numbers, find the greatest common denominator between them.

For example, given the numbers `[42, 56, 14]`, return 14.

### Problem #932 <a id="p932"></a>

**Hard** · Twitch · 2022-07-12

#### Problem

Describe what happens when you type a URL into your browser and press Enter.

### Problem #933 <a id="p933"></a>

**Medium** · Pinterest · 2022-07-13

#### Problem

The sequence `[0, 1, ..., N]` has been jumbled, and the only clue you have for its order is an array representing whether each number is larger or smaller than the last. Given this information, reconstruct an array that is consistent with it. For example, given `[None, +, +, -, +]`, you could return `[1, 2, 3, 0, 4]`.

### Problem #934 <a id="p934"></a>

**Easy** · Google · 2022-07-14

#### Problem

Given a string, return the first recurring character in it, or null if there is no recurring character.

For example, given the string `"acbbac"`, return `"b"`. Given the string `"abcdef"`, return null.

### Problem #936 <a id="p936"></a>

**Medium** · Google · 2022-07-16

#### Problem

Given a binary search tree and a range `[a, b]` (inclusive), return the sum of the elements of the binary search tree within the range.

For example, given the following tree:

```
    5
   / \
  3   8
 / \ / \
2  4 6  10
```

and the range `[4, 9]`, return 23 (5 + 4 + 6 + 8).

### Problem #937 <a id="p937"></a>

**Hard** · Google · 2022-07-17

#### Problem

You're given a string consisting solely of (, ), and \*. \* can represent either a (, ), or an empty string. Determine whether the parentheses are balanced.

For example, (()\* and (\*) are balanced. )\*( is not balanced.

### Problem #938 <a id="p938"></a>

**Hard** · Citrix · 2022-07-18

#### Problem

You are given a circular lock with three wheels, each of which display the numbers 0 through 9 in order. Each of these wheels rotate clockwise and counterclockwise.

In addition, the lock has a certain number of `"dead ends"`, meaning that if you turn the wheels to one of these combinations, the lock becomes stuck in that state and cannot be opened.

Let us consider a `"move"` to be a rotation of a single wheel by one digit, in either direction. Given a lock initially set to 000, a target combination, and a list of dead ends, write a function that returns the minimum number of moves required to reach the target state, or None if this is impossible.

### Problem #939 <a id="p939"></a>

**Easy** · Amazon · 2022-07-19

#### Problem

Given a N by M matrix of numbers, print out the matrix in a clockwise spiral.

For example, given the following matrix:

```
[[1,  2,  3,  4,  5],
 [6,  7,  8,  9,  10],
 [11, 12, 13, 14, 15],
 [16, 17, 18, 19, 20]]
```

You should print out the following:

```
1
2
3
4
5
10
15
20
19
18
17
16
11
6
7
8
9
14
13
12
```

### Problem #940 <a id="p940"></a>

**Medium** · Twitter · 2022-07-20

#### Problem

A network consists of nodes labeled 0 to N. You are given a list of edges (a, b, t), describing the time t it takes for a message to be sent from node a to node b. Whenever a node receives a message, it immediately passes the message on to a neighboring node, if possible.

Assuming all nodes are connected, determine how long it will take for every node to receive a message that begins at node 0.

For example, given N = 5, and the following edges:

```
edges = [
    (0, 1, 5),
    (0, 2, 3),
    (0, 5, 4),
    (1, 3, 8),
    (2, 3, 1),
    (3, 5, 10),
    (3, 4, 5)
]
```

You should return 9, because propagating the message from 0 -> 2 -> 3 -> 4 will take that much time.

### Problem #941 <a id="p941"></a>

**Hard** · Jane Street · 2022-07-21

#### Problem

Suppose you are given a table of currency exchange rates, represented as a 2D array. Determine whether there is a possible arbitrage: that is, whether there is some sequence of trades you can make, starting with some amount A of any currency, so that you can end up with some amount greater than A of that currency.

There are no transaction costs and you can trade fractional quantities.

### Problem #942 <a id="p942"></a>

**Medium** · Google · 2022-07-22

#### Problem

Given a string of parentheses, write a function to compute the minimum number of parentheses to be removed to make the string valid (i.e. each open parenthesis is eventually closed).

For example, given the string `"()())()"`, you should return 1. Given the string `")("`, you should return 2, since we must remove all of them.

### Problem #943 <a id="p943"></a>

**Easy** · Wayfair · 2022-07-23

#### Problem

You are given a 2 x N board, and instructed to completely cover the board with the following shapes:

- Dominoes, or 2 x 1 rectangles.
- Trominoes, or L-shapes.

For example, if N = 4, here is one possible configuration, where A is a domino, and B and C are trominoes.

```
A B B C
A B C C
```

Given an integer N, determine in how many ways this task is possible.

### Problem #944 <a id="p944"></a>

**Easy** · IBM · 2022-07-24

#### Problem

Given an integer, find the next permutation of it in absolute order. For example, given 48975, the next permutation would be 49578.

### Problem #945 <a id="p945"></a>

**Hard** · Uber · 2022-07-25

#### Problem

Given a tree where each edge has a weight, compute the length of the longest path in the tree.

For example, given the following tree:

```
  a
 /|\
b c d
   / \
  e   f
 / \
g   h
```

and the weights: a-b: 3, a-c: 5, a-d: 8, d-e: 2, d-f: 4, e-g: 1, e-h: 1, the longest path would be c -> a -> d -> f, with a length of 17.

The path does not have to pass through the root, and each node can have any amount of children.

### Problem #946 <a id="p946"></a>

**Medium** · Stitch Fix · 2022-07-26

#### Problem

Pascal's triangle is a triangular array of integers constructed with the following formula:

- The first row consists of the number 1.
- For each subsequent row, each element is the sum of the numbers directly above it, on either side.

For example, here are the first few rows:

```
    1
   1 1
  1 2 1
 1 3 3 1
1 4 6 4 1
```

Given an input k, return the kth row of Pascal's triangle.

Bonus: Can you do this using only `O(k)` space?

### Problem #947 <a id="p947"></a>

**Hard** · Oracle · 2022-07-27

#### Problem

We say a number is sparse if there are no adjacent ones in its binary representation. For example, 21 (10101) is sparse, but 22 (10110) is not. For a given input N, find the smallest sparse number greater than or equal to N.

Do this in faster than `O(N log N)` time.

### Problem #948 <a id="p948"></a>

**Medium** · 2022-07-28

#### Problem

The Tower of Hanoi is a puzzle game with three rods and n disks, each a different size.

All the disks start off on the first rod in a stack. They are ordered by size, with the largest disk on the bottom and the smallest one at the top.

The goal of this puzzle is to move all the disks from the first rod to the last rod while following these rules:

- You can only move one disk at a time.
- A move consists of taking the uppermost disk from one of the stacks and placing it on top of another stack.
- You cannot place a larger disk on top of a smaller disk.

Write a function that prints out all the steps necessary to complete the Tower of Hanoi. You should assume that the rods are numbered, with the first rod being 1, the second (auxiliary) rod being 2, and the last (goal) rod being 3.

For example, with n = 3, we can do this in 7 moves:

```
Move 1 to 3
Move 1 to 2
Move 3 to 2
Move 1 to 3
Move 2 to 1
Move 2 to 3
Move 1 to 3
```

### Problem #949 <a id="p949"></a>

**Medium** · Twitter · 2022-07-29

#### Problem

Implement an autocomplete system. That is, given a query string s and a set of all possible query strings, return all strings in the set that have s as a prefix.

For example, given the query string de and the set of strings `[dog, deer, deal]`, return `[deer, deal]`.

Hint: Try preprocessing the dictionary into a more efficient data structure to speed up queries.

### Problem #950 <a id="p950"></a>

**Easy** · Amazon · 2022-07-30

#### Problem

You are given a list of data entries that represent entries and exits of groups of people into a building. An entry looks like this:

`{"timestamp": 1526579928, count: 3, "type": "enter"}`

This means 3 people entered the building. An exit looks like this:

`{"timestamp": 1526580382, count: 2, "type": "exit"}`

This means that 2 people exited the building. timestamp is in Unix time.

Find the busiest period in the building, that is, the time with the most people in the building. Return it as a pair of (start, end) timestamps. You can assume the building always starts off and ends up empty, i.e. with 0 people inside.

### Problem #951 <a id="p951"></a>

**Medium** · Google · 2022-07-31

#### Problem

Given a stack of N elements, interleave the first half of the stack with the second half reversed using only one other queue. This should be done in-place.

Recall that you can only push or pop from a stack, and enqueue or dequeue from a queue.

For example, if the stack is `[1, 2, 3, 4, 5]`, it should become `[1, 5, 2, 4, 3]`. If the stack is `[1, 2, 3, 4]`, it should become `[1, 4, 2, 3]`.

Hint: Try working backwards from the end state.

### Problem #952 <a id="p952"></a>

**Easy** · Slack · 2022-08-01

#### Problem

You are given a string formed by concatenating several words corresponding to the integers zero through nine and then anagramming.

For example, the input could be `'niesevehrtfeev'`, which is an anagram of `'threefiveseven'`. Note that there can be multiple instances of each integer.

Given this string, return the original integers in sorted order. In the example above, this would be 357.

### Problem #953 <a id="p953"></a>

**Hard** · Airbnb · 2022-08-02

#### Problem

Given a list of integers, write a function that returns the largest sum of non-adjacent numbers. Numbers can be 0 or negative.

For example, `[2, 4, 6, 2, 5]` should return 13, since we pick 2, 6, and 5. `[5, 1, 1, 5]` should return 10, since we pick 5 and 5.

Follow-up: Can you do this in `O(N)` time and constant space?

### Problem #954 <a id="p954"></a>

**Easy** · Google · 2022-08-03

#### Problem

A unival tree (which stands for `"universal value"`) is a tree where all nodes under it have the same value.

Given the root to a binary tree, count the number of unival subtrees.

For example, the following tree has 5 unival subtrees:

```
  0
 / \
1   0
   / \
  1   0
 / \
1   1
```

### Problem #955 <a id="p955"></a>

**Easy** · Microsoft · 2022-08-04

#### Problem

Suppose an arithmetic expression is given as a binary tree. Each leaf is an integer and each internal node is one of `'+'`, `'−'`, `'∗'`, or `'/'`.

Given the root to such a tree, write a function to evaluate it.

For example, given the following tree:

```
    *
   / \
  +    +
 / \  / \
3  2  4  5
```

You should return 45, as it is (3 + 2) \* (4 + 5).

### Problem #956 <a id="p956"></a>

**Medium** · Google · 2022-08-05

#### Problem

Given k sorted singly linked lists, write a function to merge all the lists into one sorted singly linked list.

### Problem #957 <a id="p957"></a>

**Hard** · VMware · 2022-08-06

#### Problem

The skyline of a city is composed of several buildings of various widths and heights, possibly overlapping one another when viewed from a distance. We can represent the buildings using an array of (left, right, height) tuples, which tell us where on an imaginary x-axis a building begins and ends, and how tall it is. The skyline itself can be described by a list of (x, height) tuples, giving the locations at which the height visible to a distant observer changes, and each new height.

Given an array of buildings as described above, create a function that returns the skyline.

For example, suppose the input consists of the buildings `[(0, 15, 3), (4, 11, 5), (19, 23, 4)]`. In aggregate, these buildings would create a skyline that looks like the one below.

### Problem #958 <a id="p958"></a>

**Medium** · Facebook · 2022-08-07

#### Problem

Given an unordered list of flights taken by someone, each represented as (origin, destination) pairs, and a starting airport, compute the person's itinerary. If no such itinerary exists, return null. If there are multiple possible itineraries, return the lexicographically smallest one. All flights must be used in the itinerary.

For example, given the list of flights `[('SFO', 'HKO'), ('YYZ', 'SFO'), ('YUL', 'YYZ'), ('HKO', 'ORD')]` and starting airport `'YUL'`, you should return the list `['YUL', 'YYZ', 'SFO', 'HKO', 'ORD']`.

Given the list of flights `[('SFO', 'COM'), ('COM', 'YYZ')]` and starting airport `'COM'`, you should return null.

Given the list of flights `[('A', 'B'), ('A', 'C'), ('B', 'C'), ('C', 'A')]` and starting airport `'A'`, you should return the list `['A', 'B', 'C', 'A', 'C']` even though `['A', 'C', 'A', 'B', 'C']` is also a valid itinerary. However, the first one is lexicographically smaller.

### Problem #959 <a id="p959"></a>

**Easy** · Pivotal · 2022-08-08

#### Problem

Write an algorithm that finds the total number of set bits in all integers between 1 and N.

### Problem #960 <a id="p960"></a>

**Medium** · Google · 2022-08-09

#### Problem

You are given an array of nonnegative integers. Let's say you start at the beginning of the array and are trying to advance to the end. You can advance at most, the number of steps that you're currently on. Determine whether you can get to the end of the array.

For example, given the array `[1, 3, 1, 2, 0, 1]`, we can go from indices 0 -> 1 -> 3 -> 5, so return true.

Given the array `[1, 2, 1, 0, 0]`, we can't reach the end, so return false.

### Problem #961 <a id="p961"></a>

**Medium** · Quora · 2022-08-10

#### Problem

Given an absolute pathname that may have . or .. as part of it, return the shortest standardized path.

For example, given `"/usr/bin/../bin/./scripts/../"`, return `"/usr/bin/"`.

### Problem #962 <a id="p962"></a>

**Hard** · Google · 2022-08-11

#### Problem

A knight's tour is a sequence of moves by a knight on a chessboard such that all squares are visited once.

Given N, write a function to return the number of knight's tours on an N by N chessboard.

### Problem #963 <a id="p963"></a>

**Easy** · Google · 2022-08-12

#### Problem

Given two singly linked lists that intersect at some point, find the intersecting node. The lists are non-cyclical.

For example, given A = 3 -> 7 -> 8 -> 10 and B = 99 -> 1 -> 8 -> 10, return the node with value 8.

In this example, assume nodes with the same value are the exact same node objects.

Do this in `O(M + N)` time (where M and N are the lengths of the lists) and constant space.

### Problem #964 <a id="p964"></a>

**Easy** · Twitter · 2022-08-13

#### Problem

You run an e-commerce website and want to record the last N order ids in a log. Implement a data structure to accomplish this, with the following API:

- `record(order_id)`: adds the `order_id` to the log
- `get_last(i)`: gets the ith last element from the log. i is guaranteed to be smaller than or equal to N.

You should be as efficient with time and space as possible.

### Problem #965 <a id="p965"></a>

**Easy** · Google · 2022-08-14

#### Problem

UTF-8 is a character encoding that maps each symbol to one, two, three, or four bytes.

For example, the Euro sign, €, corresponds to the three bytes 11100010 10000010 10101100. The rules for mapping characters are as follows:

- For a single-byte character, the first bit must be zero.
- For an n-byte character, the first byte starts with n ones and a zero. The other n - 1 bytes all start with 10.

Visually, this can be represented as follows.

```
 Bytes   |           Byte format
-----------------------------------------------
   1     | 0xxxxxxx
   2     | 110xxxxx 10xxxxxx
   3     | 1110xxxx 10xxxxxx 10xxxxxx
   4     | 11110xxx 10xxxxxx 10xxxxxx 10xxxxxx
```

Write a program that takes in an array of integers representing byte values, and returns whether it is a valid UTF-8 encoding.

### Problem #966 <a id="p966"></a>

**Medium** · Snapchat · 2022-08-15

#### Problem

Given the head to a singly linked list, where each node also has a “random” pointer that points to anywhere in the linked list, deep clone the list.

### Problem #967 <a id="p967"></a>

**Medium** · Google · 2022-08-16

#### Problem

You are given an array of length n + 1 whose elements belong to the set `{1, 2, ..., n}`. By the pigeonhole principle, there must be a duplicate. Find it in linear time and space.

### Problem #968 <a id="p968"></a>

**Medium** · Flipkart · 2022-08-17

#### Problem

Starting from 0 on a number line, you would like to make a series of jumps that lead to the integer N.

On the ith jump, you may move exactly i places to the left or right.

Find a path with the fewest number of jumps required to get from 0 to N.

### Problem #969 <a id="p969"></a>

**Medium** · Amazon · 2022-08-18

#### Problem

Given a matrix of 1s and 0s, return the number of `"islands"` in the matrix. A 1 represents land and 0 represents water, so an island is a group of 1s that are neighboring whose perimeter is surrounded by water.

For example, this matrix has 4 islands.

```
1 0 0 0 0
0 0 1 1 0
0 1 1 0 0
0 0 0 0 0
1 1 0 0 1
1 1 0 0 1
```

### Problem #970 <a id="p970"></a>

**Easy** · Facebook · 2022-08-19

#### Problem

You have a large array with most of the elements as zero.

Use a more space-efficient data structure, SparseArray, that implements the same interface:

- `init(arr, size)`: initialize with the original large array and size.
- `set(i, val)`: updates index at i with val.
- `get(i)`: gets the value at index i.

### Problem #971 <a id="p971"></a>

**Medium** · Facebook · 2022-08-20

#### Problem

Given an N by N matrix, rotate it by 90 degrees clockwise.

For example, given the following matrix:

```
[[1, 2, 3],
 [4, 5, 6],
 [7, 8, 9]]
```

you should return:

```
[[7, 4, 1],
 [8, 5, 2],
 [9, 6, 3]]
```

Follow-up: What if you couldn't use any extra space?

### Problem #972 <a id="p972"></a>

**Easy** · IBM · 2022-08-21

#### Problem

Given a string with repeated characters, rearrange the string so that no two adjacent characters are the same. If this is not possible, return None.

For example, given `"aaabbc"`, you could return `"ababac"`. Given `"aaab"`, return None.

### Problem #973 <a id="p973"></a>

**Medium** · Facebook · 2022-08-22

#### Problem

Given the mapping a = 1, b = 2, ... z = 26, and an encoded message, count the number of ways it can be decoded.

For example, the message `'111'` would give 3, since it could be decoded as `'aaa'`, `'ka'`, and `'ak'`.

You can assume that the messages are decodable. For example, `'001'` is not allowed.

### Problem #974 <a id="p974"></a>

**Hard** · Facebook · 2022-08-23

#### Problem

Given a string consisting of parentheses, single digits, and positive and negative signs, convert the string into a mathematical expression to obtain the answer.

Don't use eval or a similar built-in parser.

For example, given `'-1 + (2 + 3)'`, you should return 4.

### Problem #975 <a id="p975"></a>

**Hard** · Apple · 2022-08-24

#### Problem

You are going on a road trip, and would like to create a suitable music playlist. The trip will require N songs, though you only have M songs downloaded, where M \< N. A valid playlist should select each song at least once, and guarantee a buffer of B songs between repeats.

Given N, M, and B, determine the number of valid playlists.

### Problem #976 <a id="p976"></a>

**Medium** · Google · 2022-08-25

#### Problem

A cryptarithmetic puzzle is a mathematical game where the digits of some numbers are represented by letters. Each letter represents a unique digit.

For example, a puzzle of the form:

```
  SEND
+ MORE
--------
 MONEY
```

may have the solution:

`{'S': 9, 'E': 5, 'N': 6, 'D': 7, 'M': 1, 'O', 0, 'R': 8, 'Y': 2}`

Given a three-word puzzle like the one above, create an algorithm that finds a solution.

### Problem #977 <a id="p977"></a>

**Easy** · Microsoft · 2022-08-26

#### Problem

Implement a URL shortener with the following methods:

- `shorten(url)`, which shortens the url into a six-character alphanumeric string, such as zLg6wl.
- `restore(short)`, which expands the shortened string into the original url. If no such shortened string exists, return null.

Hint: What if we enter the same URL twice?

### Problem #978 <a id="p978"></a>

**Hard** · LinkedIn · 2022-08-27

#### Problem

Given a string, return whether it represents a number. Here are the different kinds of numbers:

- `"10"`, a positive integer
- `"-10"`, a negative integer
- `"10.1"`, a positive real number
- `"-10.1"`, a negative real number
- `"1e5"`, a number in scientific notation

And here are examples of non-numbers:

- `"a"`
- `"x 1"`
- `"a -2"`
- `"-"`

### Problem #979 <a id="p979"></a>

**Medium** · Dropbox · 2022-08-28

#### Problem

Conway's Game of Life takes place on an infinite two-dimensional board of square cells. Each cell is either dead or alive, and at each tick, the following rules apply:

- Any live cell with less than two live neighbours dies.
- Any live cell with two or three live neighbours remains living.
- Any live cell with more than three live neighbours dies.
- Any dead cell with exactly three live neighbours becomes a live cell.

A cell neighbours another cell if it is horizontally, vertically, or diagonally adjacent.

Implement Conway's Game of Life. It should be able to be initialized with a starting list of live cell coordinates and the number of steps it should run for. Once initialized, it should print out the board state at each step. Since it's an infinite board, print out only the relevant coordinates, i.e. from the top-leftmost live cell to bottom-rightmost live cell.

You can represent a live cell with an asterisk (\*) and a dead cell with a dot (.).

### Problem #980 <a id="p980"></a>

**Medium** · Google · 2022-08-29

#### Problem

On our special chessboard, two bishops attack each other if they share the same diagonal. This includes bishops that have another bishop located between them, i.e. bishops can attack through pieces.

You are given N bishops, represented as (row, column) tuples on a M by M chessboard. Write a function to count the number of pairs of bishops that attack each other. The ordering of the pair doesn't matter: (1, 2) is considered the same as (2, 1).

For example, given M = 5 and the list of bishops:

- (0, 0)
- (1, 2)
- (2, 2)
- (4, 0)

The board would look like this:

```
[b 0 0 0 0]
[0 0 b 0 0]
[0 0 b 0 0]
[0 0 0 0 0]
[b 0 0 0 0]
```

You should return 2, since bishops 1 and 3 attack each other, as well as bishops 3 and 4.

### Problem #981 <a id="p981"></a>

**Medium** · Facebook · 2022-08-30

#### Problem

Describe an algorithm to compute the longest increasing subsequence of an array of numbers in `O(n log n)` time.

### Problem #982 <a id="p982"></a>

**Easy** · Twitter · 2022-08-31

#### Problem

A strobogrammatic number is a positive number that appears the same after being rotated 180 degrees. For example, 16891 is strobogrammatic.

Create a program that finds all strobogrammatic numbers with N digits.

### Problem #983 <a id="p983"></a>

**Medium** · Facebook · 2022-09-01

#### Problem

Given a binary tree, return all paths from the root to leaves.

For example, given the tree:

```
  1
 / \
2   3
   / \
  4   5
```

Return `[[1, 2], [1, 3, 4], [1, 3, 5]]`.

### Problem #984 <a id="p984"></a>

**Hard** · Facebook · 2022-09-02

#### Problem

Given a string of parentheses, find the balanced string that can be produced from it using the minimum number of insertions and deletions. If there are multiple solutions, return any of them.

For example, given `"(()"`, you could return `"(())"`. Given `"))()("`, you could return `"()()()()"`.

### Problem #985 <a id="p985"></a>

**Medium** · Uber · 2022-09-03

#### Problem

One way to unlock an Android phone is through a pattern of swipes across a 1-9 keypad.

For a pattern to be valid, it must satisfy the following:

- All of its keys must be distinct.
- It must not connect two keys by jumping over a third key, unless that key has already been used.

For example, 4 - 2 - 1 - 7 is a valid pattern, whereas 2 - 1 - 7 is not.

Find the total number of valid unlock patterns of length N, where 1 \<= N \<= 9.

### Problem #986 <a id="p986"></a>

**Easy** · Stripe · 2022-09-04

#### Problem

Given an integer n, return the length of the longest consecutive run of 1s in its binary representation.

For example, given 156, you should return 3.

### Problem #987 <a id="p987"></a>

**Medium** · Google · 2022-09-05

#### Problem

Given an array of numbers and an index i, return the index of the nearest larger number of the number at index i, where distance is measured in array indices.

For example, given `[4, 1, 3, 5, 6]` and index 0, you should return 3.

If two distances to larger numbers are the equal, then return any one of them. If the array at i doesn't have a nearest larger integer, then return null.

Follow-up: If you can preprocess the array, can you do this in constant time?

### Problem #988 <a id="p988"></a>

**Medium** · Facebook · 2022-09-06

#### Problem

Given a positive integer n, find the smallest number of squared integers which sum to n.

For example, given n = 13, return 2 since 13 = 32 + 22 = 9 + 4.

Given n = 27, return 3 since 27 = 32 + 32 + 32 = 9 + 9 + 9.

### Problem #989 <a id="p989"></a>

**Medium** · Snapchat · 2022-09-07

#### Problem

You are given an array of length N, where each element i represents the number of ways we can produce i units of change. For example, `[1, 0, 1, 1, 2]` would indicate that there is only one way to make 0, 2, or 3 units, and two ways of making 4 units.

Given such an array, determine the denominations that must be in use. In the case above, for example, there must be coins with value 2, 3, and 4.

### Problem #990 <a id="p990"></a>

**Hard** · Amazon · 2022-09-08

#### Problem

There exists a staircase with N steps, and you can climb up either 1 or 2 steps at a time. Given N, write a function that returns the number of unique ways you can climb the staircase. The order of the steps matters.

For example, if N is 4, then there are 5 unique ways:

- 1, 1, 1, 1
- 2, 1, 1
- 1, 2, 1
- 1, 1, 2
- 2, 2

What if, instead of being able to climb 1 or 2 steps at a time, you could climb any number from a set of positive integers X? For example, if X = `{1, 3, 5}`, you could climb 1, 3, or 5 steps at a time.

### Problem #991 <a id="p991"></a>

**Medium** · Google · 2022-09-09

#### Problem

What will this code print out?

```
def make_functions():
    flist = []
```

```
for i in [1, 2, 3]:
    def print_i():
        print(i)
    flist.append(print_i)
```

```
return flist
```

```
functions = make_functions()
for f in functions:
    f()
```

How can we make it print out what we apparently want?

### Problem #992 <a id="p992"></a>

**Medium** · Dropbox · 2022-09-10

#### Problem

Given the root to a binary search tree, find the second largest node in the tree.

### Problem #993 <a id="p993"></a>

**Medium** · MongoDB · 2022-09-11

#### Problem

Given a list of elements, find the majority element, which appears more than half the time (> `floor(len(lst) / 2.0)`).

You can assume that such element exists.

For example, given `[1, 2, 1, 1, 3, 4, 0]`, return 1.

### Problem #994 <a id="p994"></a>

**Easy** · Microsoft · 2022-09-12

#### Problem

Print the nodes in a binary tree level-wise. For example, the following should print 1, 2, 3, 4, 5.

```
  1
 / \
2   3
   / \
  4   5
```

### Problem #995 <a id="p995"></a>

**Hard** · Uber · 2022-09-13

#### Problem

Given an array of integers, return a new array such that each element at index i of the new array is the product of all the numbers in the original array except the one at i.

For example, if our input was `[1, 2, 3, 4, 5]`, the expected output would be `[120, 60, 40, 30, 24]`. If our input was `[3, 2, 1]`, the expected output would be `[2, 3, 6]`.

Follow-up: what if you can't use division?

### Problem #996 <a id="p996"></a>

**Hard** · Microsoft · 2022-09-14

#### Problem

Recall that the minimum spanning tree is the subset of edges of a tree that connect all its vertices with the smallest possible total edge weight. Given an undirected graph with weighted edges, compute the maximum weight spanning tree.

### Problem #997 <a id="p997"></a>

**Medium** · Google · 2022-09-15

#### Problem

Given the root to a binary tree, implement `serialize(root)`, which serializes the tree into a string, and deserialize(s), which deserializes the string back into the tree.

For example, given the following Node class

```
class Node:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
```

The following test should pass:

```
node = Node('root', Node('left', Node('left.left')), Node('right'))
assert deserialize(serialize(node)).left.left.val == 'left.left'
```

### Problem #998 <a id="p998"></a>

**Medium** · Google · 2022-09-16

#### Problem

Given an undirected graph represented as an adjacency matrix and an integer k, write a function to determine whether each vertex in the graph can be colored such that no two adjacent vertices share the same color using at most k colors.

### Problem #999 <a id="p999"></a>

**Medium** · Apple · 2022-09-17

#### Problem

Suppose you have a multiplication table that is N by N. That is, a 2D array where the value at the i-th row and j-th column is (i + 1) \* (j + 1) (if 0-indexed) or i \* j (if 1-indexed).

Given integers N and X, write a function that returns the number of times X appears as a value in an N by N multiplication table.

For example, given N = 6 and X = 12, you should return 4, since the multiplication table looks like this:

| 1 | 2 | 3 | 4 | 5 | 6 |

| 2 | 4 | 6 | 8 | 10 | 12 |

| 3 | 6 | 9 | 12 | 15 | 18 |

| 4 | 8 | 12 | 16 | 20 | 24 |

| 5 | 10 | 15 | 20 | 25 | 30 |

| 6 | 12 | 18 | 24 | 30 | 36 |

And there are 4 12's in the table.

### Problem #1000 <a id="p1000"></a>

**Medium** · Uber · 2022-09-18

#### Problem

Suppose an array sorted in ascending order is rotated at some pivot unknown to you beforehand. Find the minimum element in `O(log N)` time. You may assume the array does not contain duplicates.

For example, given `[5, 7, 10, 3, 4]`, return 3.

### Problem #1001 <a id="p1001"></a>

**Medium** · LinkedIn · 2022-09-19

#### Problem

Determine whether a tree is a valid binary search tree.

A binary search tree is a tree with two children, left and right, and satisfies the constraint that the key in the left child must be less than or equal to the root and the key in the right child must be greater than or equal to the root.

### Problem #1002 <a id="p1002"></a>

**Easy** · Amazon · 2022-09-20

#### Problem

Given a sorted array, find the smallest positive integer that is not the sum of a subset of the array.

For example, for the input `[1, 2, 3, 10]`, you should return 7.

Do this in `O(N)` time.

### Problem #1003 <a id="p1003"></a>

**Easy** · Microsoft · 2022-09-21

#### Problem

The transitive closure of a graph is a measure of which vertices are reachable from other vertices. It can be represented as a matrix M, where `M[i][j]` == 1 if there is a path between vertices i and j, and otherwise 0.

For example, suppose we are given the following graph in adjacency list form:

```
graph = [
    [0, 1, 3],
    [1, 2],
    [2],
    [3]
]
```

The transitive closure of this graph would be:

```
[1, 1, 1, 1]
[0, 1, 1, 0]
[0, 0, 1, 0]
[0, 0, 0, 1]
```

Given a graph, find its transitive closure.

### Problem #1004 <a id="p1004"></a>

**Hard** · Goldman Sachs · 2022-09-22

#### Problem

Given a list of numbers L, implement a method `sum(i, j)` which returns the sum from the sublist `L[i:j]` (including i, excluding j).

For example, given L = `[1, 2, 3, 4, 5]`, `sum(1, 3)` should return `sum([2, 3])`, which is 5.

You can assume that you can do some pre-processing. `sum()` should be optimized over the pre-processing step.

### Problem #1005 <a id="p1005"></a>

**Hard** · MIT · 2022-09-23

#### Problem

Blackjack is a two player card game whose rules are as follows:

- The player and then the dealer are each given two cards.
- The player can then `"hit"`, or ask for arbitrarily many additional cards, so long as their total does not exceed 21.
- The dealer must then hit if their total is 16 or lower, otherwise pass.
- Finally, the two compare totals, and the one with the greatest sum not exceeding 21 is the winner.

For this problem, cards values are counted as follows: each card between 2 and 10 counts as their face value, face cards count as 10, and aces count as 1.

Given perfect knowledge of the sequence of cards in the deck, implement a blackjack solver that maximizes the player's score (that is, wins minus losses).

### Problem #1006 <a id="p1006"></a>

**Easy** · Facebook · 2022-09-24

#### Problem

Suppose you are given two lists of n points, one list p1, p2, ..., pn on the line y = 0 and the other list q1, q2, ..., qn on the line y = 1. Imagine a set of n line segments connecting each point pi to qi. Write an algorithm to determine how many pairs of the line segments intersect.

### Problem #1007 <a id="p1007"></a>

**Hard** · Oracle · 2022-09-25

#### Problem

You are presented with an 8 by 8 matrix representing the positions of pieces on a chess board. The only pieces on the board are the black king and various white pieces. Given this matrix, determine whether the king is in check.

For details on how each piece moves, see here.

For example, given the following matrix:

```
...K....
........
.B......
......P.
.......R
..N.....
........
.....Q..
```

You should return True, since the bishop is attacking the king diagonally.

### Problem #1008 <a id="p1008"></a>

**Hard** · Salesforce · 2022-09-26

#### Problem

Given an array of integers, find the maximum XOR of any two elements.

### Problem #1009 <a id="p1009"></a>

**Easy** · Coursera · 2022-09-27

#### Problem

Given a 2D board of characters and a word, find if the word exists in the grid.

The word can be constructed from letters of sequentially adjacent cell, where `"adjacent"` cells are those horizontally or vertically neighboring. The same letter cell may not be used more than once.

For example, given the following board:

```
[
  ['A','B','C','E'],
  ['S','F','C','S'],
  ['A','D','E','E']
]
```

`exists(board, "ABCCED")` returns true, `exists(board, "SEE")` returns true, `exists(board, "ABCB")` returns false.

### Problem #1010 <a id="p1010"></a>

**Easy** · Jane Street · 2022-09-28

#### Problem

The United States uses the imperial system of weights and measures, which means that there are many different, seemingly arbitrary units to measure distance. There are 12 inches in a foot, 3 feet in a yard, 22 yards in a chain, and so on.

Create a data structure that can efficiently convert a certain quantity of one unit to the correct amount of any other unit. You should also allow for additional units to be added to the system.

### Problem #1011 <a id="p1011"></a>

**Hard** · Nvidia · 2022-09-29

#### Problem

Find the maximum of two numbers without using any if-else statements, branching, or direct comparisons.

### Problem #1012 <a id="p1012"></a>

**Medium** · 2022-09-30

#### Problem

Given a real number n, find the square root of n. For example, given n = 9, return 3.

### Problem #1013 <a id="p1013"></a>

**Medium** · Yahoo · 2022-10-01

#### Problem

Write a function that returns the bitwise AND of all integers between M and N, inclusive.

### Problem #1014 <a id="p1014"></a>

**Medium** · Facebook · 2022-10-02

#### Problem

Given a start word, an end word, and a dictionary of valid words, find the shortest transformation sequence from start to end such that only one letter is changed at each step of the sequence, and each transformed word exists in the dictionary. If there is no possible transformation, return null. Each word in the dictionary have the same length as start and end and is lowercase.

For example, given start = `"dog"`, end = `"cat"`, and dictionary = `{"dot", "dop", "dat", "cat"}`, return `["dog", "dot", "dat", "cat"]`.

Given start = `"dog"`, end = `"cat"`, and dictionary = `{"dot", "tod", "dat", "dar"}`, return null as there is no possible transformation from dog to cat.

### Problem #1015 <a id="p1015"></a>

**Hard** · Google · 2022-10-03

#### Problem

In a directed graph, each node is assigned an uppercase letter. We define a path's value as the number of most frequently-occurring letter along that path. For example, if a path in the graph goes through `"ABACA"`, the value of the path is 3, since there are 3 occurrences of `'A'` on the path.

Given a graph with n nodes and m directed edges, return the largest value path of the graph. If the largest value is infinite, then return null.

The graph is represented with a string and an edge list. The i-th character represents the uppercase letter of the i-th node. Each tuple in the edge list (i, j) means there is a directed edge from the i-th node to the j-th node. Self-edges are possible, as well as multi-edges.

For example, the following input graph:

ABACA

```
[(0, 1),
 (0, 2),
 (2, 3),
 (3, 4)]
```

Would have maximum value 3 using the path of vertices `[0, 2, 3, 4]`, (A, A, C, A).

The following input graph:

A

`[(0, 0)]`

Should return null, since we have an infinite loop.

### Problem #1016 <a id="p1016"></a>

**Easy** · Amazon · 2022-10-04

#### Problem

Huffman coding is a method of encoding characters based on their frequency. Each letter is assigned a variable-length binary string, such as 0101 or 111110, where shorter lengths correspond to more common letters. To accomplish this, a binary tree is built such that the path from the root to any leaf uniquely maps to a character. When traversing the path, descending to a left child corresponds to a 0 in the prefix, while descending right corresponds to 1.

Here is an example tree (note that only the leaf nodes have letters):

```
        *
      /   \
    *       *
   / \     / \
  *   a   t   *
 /             \
c               s
```

With this encoding, cats would be represented as 0000110111.

Given a dictionary of character frequencies, build a Huffman tree, and use it to determine a mapping between characters and their encoded binary strings.

### Problem #1017 <a id="p1017"></a>

**Hard** · 2022-10-05

#### Problem

Given a list, sort it using this method: `reverse(lst, i, j)`, which reverses lst from i to j.

### Problem #1018 <a id="p1018"></a>

**Easy** · Microsoft · 2022-10-06

#### Problem

Given a clock time in hh:mm format, determine, to the nearest degree, the angle between the hour and the minute hands.

Bonus: When, during the course of a day, will the angle be zero?

### Problem #1019 <a id="p1019"></a>

**Easy** · Google · 2022-10-07

#### Problem

Given two rectangles on a 2D graph, return the area of their intersection. If the rectangles don't intersect, return 0.

For example, given the following rectangles:

```
{
    "top_left": (1, 4),
    "dimensions": (3, 3) # width, height
}
```

and

```
{
    "top_left": (0, 5),
    "dimensions": (4, 3) # width, height
}
```

return 6.

### Problem #1020 <a id="p1020"></a>

**Easy** · Google · 2022-10-08

#### Problem

Given the root of a binary search tree, and a target K, return two nodes in the tree whose sum equals K.

For example, given the following tree and K of 20

```
   10
  /   \
5      15
      /  \
    11    15
```

Return the nodes 5 and 15.

### Problem #1021 <a id="p1021"></a>

**Medium** · Cisco · 2022-10-09

#### Problem

Given an unsigned 8-bit integer, swap its even and odd bits. The 1st and 2nd bit should be swapped, the 3rd and 4th bit should be swapped, and so on.

For example, 10101010 should be 01010101. 11100010 should be 11010001.

Bonus: Can you do this in one line?

### Problem #1022 <a id="p1022"></a>

**Hard** · Google · 2022-10-10

#### Problem

Given an array of integers where every integer occurs three times except for one integer, which only occurs once, find and return the non-duplicated integer.

For example, given `[6, 1, 3, 3, 3, 6, 6]`, return 1. Given `[13, 19, 13, 13]`, return 19.

Do this in `O(N)` time and `O(1)` space.

### Problem #1023 <a id="p1023"></a>

**Hard** · Airbnb · 2022-10-11

#### Problem

You come across a dictionary of sorted words in a language you've never seen before. Write a program that returns the correct order of letters in this language.

For example, given `['xww', 'wxyz', 'wxyw', 'ywx', 'ywz']`, you should return `['x', 'z', 'w', 'y']`.

### Problem #1024 <a id="p1024"></a>

**Easy** · Facebook · 2022-10-12

#### Problem

Given a 32-bit integer, return the number with its bits reversed.

For example, given the binary number 1111 0000 1111 0000 1111 0000 1111 0000, return 0000 1111 0000 1111 0000 1111 0000 1111.

### Problem #1025 <a id="p1025"></a>

**Easy** · Amazon · 2022-10-13

#### Problem

Given a linked list, remove all consecutive nodes that sum to zero. Print out the remaining nodes.

For example, suppose you are given the input 3 -> 4 -> -7 -> 5 -> -6 -> 6. In this case, you should first remove 3 -> 4 -> -7, then -6 -> 6, leaving only 5.

### Problem #1026 <a id="p1026"></a>

**Medium** · Palantir · 2022-10-14

#### Problem

Write an algorithm to justify text. Given a sequence of words and an integer line length k, return a list of strings which represents each line, fully justified.

More specifically, you should have as many words as possible in each line. There should be at least one space between each word. Pad extra spaces when necessary so that each line has exactly length k. Spaces should be distributed as equally as possible, with the extra spaces, if any, distributed starting from the left.

If you can only fit one word on a line, then you should pad the right-hand side with spaces.

Each word is guaranteed not to be longer than k.

For example, given the list of words `["the", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]` and k = 16, you should return the following:

```
["the  quick brown", # 1 extra space on the left
"fox  jumps  over", # 2 extra spaces distributed evenly
"the   lazy   dog"] # 4 extra spaces distributed evenly
```

### Problem #1027 <a id="p1027"></a>

**Medium** · Microsoft · 2022-10-15

#### Problem

Given an unsorted array of integers, find the length of the longest consecutive elements sequence.

For example, given `[100, 4, 200, 1, 3, 2]`, the longest consecutive element sequence is `[1, 2, 3, 4]`. Return its length: 4.

Your algorithm should run in `O(n)` complexity.

### Problem #1028 <a id="p1028"></a>

**Medium** · Salesforce · 2022-10-16

#### Problem

The number 6174 is known as Kaprekar's contant, after the mathematician who discovered an associated property: for all four-digit numbers with at least two distinct digits, repeatedly applying a simple procedure eventually results in this value. The procedure is as follows:

- For a given input x, create two new numbers that consist of the digits in x in ascending and descending order.
- Subtract the smaller number from the larger number.

For example, this algorithm terminates in three steps when starting from 1234:

- 4321 - 1234 = 3087
- 8730 - 0378 = 8352
- 8532 - 2358 = 6174

Write a function that returns how many steps this will take for a given input N.

### Problem #1029 <a id="p1029"></a>

**Hard** · Google · 2022-10-17

#### Problem

Find the minimum number of coins required to make n cents.

You can use standard American denominations, that is, 1¢, 5¢, 10¢, and 25¢.

For example, given n = 16, return 3 since we can make it with a 10¢, a 5¢, and a 1¢.

### Problem #1030 <a id="p1030"></a>

**Easy** · Facebook · 2022-10-18

#### Problem

On a mysterious island there are creatures known as Quxes which come in three colors: red, green, and blue. One power of the Qux is that if two of them are standing next to each other, they can transform into a single creature of the third color.

Given N Quxes standing in a line, determine the smallest number of them remaining after any possible sequence of such transformations.

For example, given the input `['R', 'G', 'B', 'G', 'B']`, it is possible to end up with a single Qux through the following steps:

```
        Arrangement       |   Change
----------------------------------------
['R', 'G', 'B', 'G', 'B'] | (R, G) -> B
['B', 'B', 'G', 'B']      | (B, G) -> R
['B', 'R', 'B']           | (R, B) -> G
['B', 'G']                | (B, G) -> R
['R']                     |
```

### Problem #1031 <a id="p1031"></a>

**Medium** · Flipkart · 2022-10-19

#### Problem

Snakes and Ladders is a game played on a 10 x 10 board, the goal of which is get from square 1 to square 100. On each turn players will roll a six-sided die and move forward a number of spaces equal to the result. If they land on a square that represents a snake or ladder, they will be transported ahead or behind, respectively, to a new square.

Find the smallest number of turns it takes to play snakes and ladders.

For convenience, here are the squares representing snakes and ladders, and their outcomes:

```
snakes = {16: 6, 48: 26, 49: 11, 56: 53, 62: 19, 64: 60, 87: 24, 93: 73, 95: 75, 98: 78}
ladders = {1: 38, 4: 14, 9: 31, 21: 42, 28: 84, 36: 44, 51: 67, 71: 91, 80: 100}
```

### Problem #1032 <a id="p1032"></a>

**Easy** · Google · 2022-10-20

#### Problem

In linear algebra, a Toeplitz matrix is one in which the elements on any given diagonal from top left to bottom right are identical.

Here is an example:

```
1 2 3 4 8
5 1 2 3 4
4 5 1 2 3
7 4 5 1 2
```

Write a program to determine whether a given input is a Toeplitz matrix.

### Problem #1033 <a id="p1033"></a>

**Hard** · Microsoft · 2022-10-21

#### Problem

Given an array of positive integers, divide the array into two subsets such that the difference between the sum of the subsets is as small as possible.

For example, given `[5, 10, 15, 20, 25]`, return the sets `{10, 25}` and `{5, 15, 20}`, which has a difference of 5, which is the smallest possible difference.

### Problem #1034 <a id="p1034"></a>

**Easy** · Microsoft · 2022-10-22

#### Problem

You have n fair coins and you flip them all at the same time. Any that come up tails you set aside. The ones that come up heads you flip again. How many rounds do you expect to play before only one coin remains?

Write a function that, given n, returns the number of rounds you'd expect to play until one coin remains.

### Problem #1035 <a id="p1035"></a>

**Easy** · Atlassian · 2022-10-23

#### Problem

MegaCorp wants to give bonuses to its employees based on how many lines of codes they have written. They would like to give the smallest positive amount to each worker consistent with the constraint that if a developer has written more lines of code than their neighbor, they should receive more money.

Given an array representing a line of seats of employees at MegaCorp, determine how much each one should get paid.

For example, given `[10, 40, 200, 1000, 60, 30]`, you should return `[1, 2, 3, 4, 2, 1]`.

### Problem #1036 <a id="p1036"></a>

**Medium** · Google · 2022-10-24

#### Problem

Given the sequence of keys visited by a postorder traversal of a binary search tree, reconstruct the tree.

For example, given the sequence 2, 4, 3, 8, 7, 5, you should construct the following tree:

```
    5
   / \
  3   7
 / \   \
2   4   8
```

### Problem #1037 <a id="p1037"></a>

**Hard** · Etsy · 2022-10-25

#### Problem

Given a sorted array, convert it into a height-balanced binary search tree.

### Problem #1038 <a id="p1038"></a>

**Medium** · Google · 2022-10-26

#### Problem

Given a string of words delimited by spaces, reverse the words in string. For example, given `"hello world here"`, return `"here world hello"`

Follow-up: given a mutable string representation, can you perform this operation in-place?

### Problem #1039 <a id="p1039"></a>

**Hard** · Twitter · 2022-10-27

#### Problem

You are given an array of length 24, where each element represents the number of new subscribers during the corresponding hour. Implement a data structure that efficiently supports the following:

- `update(hour: int, value: int)`: Increment the element at index hour by value.
- `query(start: int, end: int)`: Retrieve the number of subscribers that have signed up between start and end (inclusive).

You can assume that all values get cleared at the end of the day, and that you will not be asked for start and end values that wrap around midnight.

### Problem #1040 <a id="p1040"></a>

**Hard** · Dropbox · 2022-10-28

#### Problem

Implement an efficient string matching algorithm.

That is, given a string of length N and a pattern of length k, write a program that searches for the pattern in the string with less than `O(N * k)` worst-case time complexity.

If the pattern is found, return the start index of its location. If not, return False.

### Problem #1041 <a id="p1041"></a>

**Medium** · Facebook · 2022-10-29

#### Problem

You are given an array of non-negative integers that represents a two-dimensional elevation map where each element is unit-width wall and the integer is the height. Suppose it will rain and all spots between two walls get filled up.

Compute how many units of water remain trapped on the map in `O(N)` time and `O(1)` space.

For example, given the input `[2, 1, 2]`, we can hold 1 unit of water in the middle.

Given the input `[3, 0, 1, 3, 0, 5]`, we can hold 3 units in the first index, 2 in the second, and 3 in the fourth index (we cannot hold 5 since it would run off to the left), so we can trap 8 units of water.

### Problem #1042 <a id="p1042"></a>

**Hard** · YouTube · 2022-10-30

#### Problem

Write a program that computes the length of the longest common subsequence of three given strings. For example, given `"epidemiologist"`, `"refrigeration"`, and `"supercalifragilisticexpialodocious"`, it should return 5, since the longest common subsequence is `"eieio"`.

### Problem #1043 <a id="p1043"></a>

**Medium** · Google · 2022-10-31

#### Problem

Given an iterator with methods `next()` and `hasNext()`, create a wrapper iterator, PeekableInterface, which also implements `peek()`. peek shows the next element that would be returned on `next()`.

Here is the interface:

```
class PeekableInterface(object):
    def __init__(self, iterator):
        pass
```

```
def peek(self):
    pass
```

```
def next(self):
    pass
```

```
def hasNext(self):
    pass
```

### Problem #1044 <a id="p1044"></a>

**Hard** · Facebook · 2022-11-01

#### Problem

Given a string and a set of delimiters, reverse the words in the string while maintaining the relative order of the delimiters. For example, given `"hello/world:here"`, return `"here/world:hello"`

Follow-up: Does your solution work for the following cases: `"hello/world:here/"`, `"hello//world:here"`

### Problem #1045 <a id="p1045"></a>

**Medium** · Microsoft · 2022-11-02

#### Problem

Describe and give an example of each of the following types of polymorphism:

- Ad-hoc polymorphism
- Parametric polymorphism
- Subtype polymorphism

### Problem #1046 <a id="p1046"></a>

**Medium** · Quora · 2022-11-03

#### Problem

Given a string, find the palindrome that can be made by inserting the fewest number of characters as possible anywhere in the word. If there is more than one palindrome of minimum length that can be made, return the lexicographically earliest one (the first one alphabetically).

For example, given the string `"race"`, you should return `"ecarace"`, since we can add three letters to it (which is the smallest amount to make a palindrome). There are seven other palindromes that can be made from `"race"` by adding three letters, but `"ecarace"` comes first alphabetically.

As another example, given the string `"google"`, you should return `"elgoogle"`.

### Problem #1047 <a id="p1047"></a>

**Medium** · Stripe · 2022-11-04

#### Problem

Write a map implementation with a get function that lets you retrieve the value of a key at a particular time.

It should contain the following methods:

- `set(key, value, time)`: sets key to value for t = time.
- `get(key, time)`: gets the key at t = time.

The map should work like this. If we set a key at a particular time, it will maintain that value forever or until it gets set at a later time. In other words, when we get a key at a time, it should return the value that was set for that key set at the most recent time.

Consider the following examples:

```
d.set(1, 1, 0) # set key 1 to value 1 at time 0
d.set(1, 2, 2) # set key 1 to value 2 at time 2
d.get(1, 1) # get key 1 at time 1 should be 1
d.get(1, 3) # get key 1 at time 3 should be 2
```

```
d.set(1, 1, 5) # set key 1 to value 1 at time 5
d.get(1, 0) # get key 1 at time 0 should be null
d.get(1, 10) # get key 1 at time 10 should be 1
```

```
d.set(1, 1, 0) # set key 1 to value 1 at time 0
d.set(1, 2, 0) # set key 1 to value 2 at time 0
d.get(1, 0) # get key 1 at time 0 should be 2
```

### Problem #1048 <a id="p1048"></a>

**Hard** · Netflix · 2022-11-05

#### Problem

A Cartesian tree with sequence S is a binary tree defined by the following two properties:

- It is heap-ordered, so that each parent value is strictly less than that of its children.
- An in-order traversal of the tree produces nodes with values that correspond exactly to S.

For example, given the sequence `[3, 2, 6, 1, 9]`, the resulting Cartesian tree would be:

```
      1
    /   \   
  2       9
 / \
3   6
```

Given a sequence S, construct the corresponding Cartesian tree.

### Problem #1049 <a id="p1049"></a>

**Medium** · Amazon · 2022-11-06

#### Problem

Given an array of a million integers between zero and a billion, out of order, how can you efficiently sort it? Assume that you cannot store an array of a billion elements in memory.

### Problem #1050 <a id="p1050"></a>

**Easy** · Twitter · 2022-11-07

#### Problem

A permutation can be specified by an array P, where `P[i]` represents the location of the element at i in the permutation. For example, `[2, 1, 0]` represents the permutation where elements at the index 0 and 2 are swapped.

Given an array and a permutation, apply the permutation to the array. For example, given the array `["a", "b", "c"]` and the permutation `[2, 1, 0]`, return `["c", "b", "a"]`.

### Problem #1051 <a id="p1051"></a>

**Medium** · Amazon · 2022-11-08

#### Problem

An sorted array of integers was rotated an unknown number of times.

Given such an array, find the index of the element in the array in faster than linear time. If the element doesn't exist in the array, return null.

For example, given the array `[13, 18, 25, 2, 8, 10]` and the element 8, return 4 (the index of 8 in the array).

You can assume all the integers in the array are unique.

### Problem #1052 <a id="p1052"></a>

**Hard** · Twitter · 2022-11-09

#### Problem

A teacher must divide a class of students into two teams to play dodgeball. Unfortunately, not all the kids get along, and several refuse to be put on the same team as that of their enemies.

Given an adjacency list of students and their enemies, write an algorithm that finds a satisfactory pair of teams, or returns False if none exists.

For example, given the following enemy graph you should return the teams `{0, 1, 4, 5}` and `{2, 3}`.

```
students = {
    0: [3],
    1: [2],
    2: [1, 4],
    3: [0, 4, 5],
    4: [2, 3],
    5: [3]
}
```

On the other hand, given the input below, you should return False.

```
students = {
    0: [3],
    1: [2],
    2: [1, 3, 4],
    3: [0, 2, 4, 5],
    4: [2, 3],
    5: [3]
}
```

### Problem #1053 <a id="p1053"></a>

**Hard** · Uber · 2022-11-10

#### Problem

A rule looks like this:

A NE B

This means this means point A is located northeast of point B.

A SW C

means that point A is southwest of C.

Given a list of rules, check if the sum of the rules validate. For example:

```
A N B
B NE C
C N A
```

does not validate, since A cannot be both north and south of C.

```
A NW B
A N B
```

is considered valid.

### Problem #1054 <a id="p1054"></a>

**Medium** · Microsoft · 2022-11-11

#### Problem

Implement the singleton pattern with a twist. First, instead of storing one instance, store two instances. And in every even call of `getInstance()`, return the first instance and in every odd call of `getInstance()`, return the second instance.

### Problem #1055 <a id="p1055"></a>

**Medium** · Amazon · 2022-11-12

#### Problem

Given a node in a binary search tree, return the next bigger element, also known as the inorder successor.

For example, the inorder successor of 22 is 30.

```
  10
 /  \
5    30
    /  \
  22    35
```

You can assume each node has a parent pointer.

### Problem #1056 <a id="p1056"></a>

**Medium** · Epic · 2022-11-13

#### Problem

The `"look and say"` sequence is defined as follows: beginning with the term 1, each subsequent term visually describes the digits appearing in the previous term. The first few terms are as follows:

```
1
11
21
1211
111221
```

As an example, the fourth term is 1211, since the third term consists of one 2 and one 1.

Given an integer N, print the Nth term of this sequence.

### Problem #1057 <a id="p1057"></a>

**Easy** · Amazon · 2022-11-14

#### Problem

Given an integer N, construct all possible binary search trees with N nodes.

### Problem #1058 <a id="p1058"></a>

**Medium** · Dropbox · 2022-11-15

#### Problem

Given an undirected graph G, check whether it is bipartite. Recall that a graph is bipartite if its vertices can be divided into two independent sets, U and V, such that no edge connects vertices of the same set.

### Problem #1059 <a id="p1059"></a>

**Hard** · Quora · 2022-11-16

#### Problem

Word sense disambiguation is the problem of determining which sense a word takes on in a particular setting, if that word has multiple meanings. For example, in the sentence `"I went to get money from the bank"`, bank probably means the place where people deposit money, not the land beside a river or lake.

Suppose you are given a list of meanings for several words, formatted like so:

```
{
    "word_1": ["meaning one", "meaning two", ...],
    ...
    "word_n": ["meaning one", "meaning two", ...]
}
```

Given a sentence, most of whose words are contained in the meaning list above, create an algorithm that determines the likely sense of each possibly ambiguous word.

### Problem #1060 <a id="p1060"></a>

**Easy** · Facebook · 2022-11-17

#### Problem

Given a function f, and N return a debounced f of N milliseconds.

That is, as long as the debounced f continues to be invoked, f itself will not be called for N milliseconds.

### Problem #1061 <a id="p1061"></a>

**Easy** · Google · 2022-11-18

#### Problem

You are given an M by N matrix consisting of booleans that represents a board. Each True boolean represents a wall. Each False boolean represents a tile you can walk on.

Given this matrix, a start coordinate, and an end coordinate, return the minimum number of steps required to reach the end coordinate from the start. If there is no possible path, then return null. You can move up, left, down, and right. You cannot move through walls. You cannot wrap around the edges of the board.

For example, given the following board:

```
[[f, f, f, f],
[t, t, f, t],
[f, f, f, f],
[f, f, f, f]]
```

and start = (3, 0) (bottom left) and end = (0, 0) (top left), the minimum number of steps required to reach the end is 7, since we would need to go through (1, 2) because there is a wall everywhere else on the second row.

### Problem #1062 <a id="p1062"></a>

**Easy** · Google · 2022-11-19

#### Problem

Given the head of a singly linked list, swap every two nodes and return its head.

For example, given 1 -> 2 -> 3 -> 4, return 2 -> 1 -> 4 -> 3.

### Problem #1063 <a id="p1063"></a>

**Easy** · Sumo Logic · 2022-11-20

#### Problem

Given a array that's sorted but rotated at some unknown pivot, in which all elements are distinct, find a `"peak"` element in `O(log N)` time.

An element is considered a peak if it is greater than both its left and right neighbors. It is guaranteed that the first and last elements are lower than all others.

### Problem #1064 <a id="p1064"></a>

**Easy** · Two Sigma · 2022-11-21

#### Problem

Using a function `rand5()` that returns an integer from 1 to 5 (inclusive) with uniform probability, implement a function `rand7()` that returns an integer from 1 to 7 (inclusive).

### Problem #1065 <a id="p1065"></a>

**Medium** · Yahoo · 2022-11-22

#### Problem

Write an algorithm that computes the reversal of a directed graph. For example, if a graph consists of A -> B -> C, it should become A \<- B \<- C.

### Problem #1066 <a id="p1066"></a>

**Hard** · Amazon · 2022-11-23

#### Problem

The stable marriage problem is defined as follows:

Suppose you have N men and N women, and each person has ranked their prospective opposite-sex partners in order of preference.

For example, if N = 3, the input could be something like this:

```
guy_preferences = {
    'andrew': ['caroline', 'abigail', 'betty'],
    'bill': ['caroline', 'betty', 'abigail'],
    'chester': ['betty', 'caroline', 'abigail'],
}
```

```
gal_preferences = {
    'abigail': ['andrew', 'bill', 'chester'],
    'betty': ['bill', 'andrew', 'chester'],
    'caroline': ['bill', 'chester', 'andrew']
}
```

Write an algorithm that pairs the men and women together in such a way that no two people of opposite sex would both rather be with each other than with their current partners.

### Problem #1067 <a id="p1067"></a>

**Medium** · BufferBox · 2022-11-24

#### Problem

Given a binary tree where all nodes are either 0 or 1, prune the tree so that subtrees containing all 0s are removed.

For example, given the following tree:

```
  0
 / \
1   0
   / \
  1   0
 / \
0   0
```

should be pruned to:

```
  0
 / \
1   0
   /
  1
```

We do not remove the tree at the root or its left child because it still has a 1 as a descendant.

### Problem #1068 <a id="p1068"></a>

**Easy** · Microsoft · 2022-11-25

#### Problem

You are given an string representing the initial conditions of some dominoes. Each element can take one of three values:

- L, meaning the domino has just been pushed to the left,
- R, meaning the domino has just been pushed to the right, or
- ., meaning the domino is standing still.

Determine the orientation of each tile when the dominoes stop falling. Note that if a domino receives a force from the left and right side simultaneously, it will remain upright.

For example, given the string .L.R....L, you should return LL.RRRLLL.

Given the string ..R...L.L, you should return ..RR.LLLL.

### Problem #1069 <a id="p1069"></a>

**Medium** · Square · 2022-11-26

#### Problem

In front of you is a row of N coins, with values v1, v1, ..., vn.

You are asked to play the following game. You and an opponent take turns choosing either the first or last coin from the row, removing it from the row, and receiving the value of the coin.

Write a program that returns the maximum amount of money you can win with certainty, if you move first, assuming your opponent plays optimally.

### Problem #1070 <a id="p1070"></a>

**Easy** · Google · 2022-11-27

#### Problem

A girl is walking along an apple orchard with a bag in each hand. She likes to pick apples from each tree as she goes along, but is meticulous about not putting different kinds of apples in the same bag.

Given an input describing the types of apples she will pass on her path, in order, determine the length of the longest portion of her path that consists of just two types of apple trees.

For example, given the input `[2, 1, 2, 3, 3, 1, 3, 5]`, the longest portion will involve types 1 and 3, with a length of four.

### Problem #1071 <a id="p1071"></a>

**Hard** · Google · 2022-11-28

#### Problem

Given a list of integers S and a target number k, write a function that returns a subset of S that adds up to k. If such a subset cannot be made, then return null.

Integers can appear more than once in the list. You may assume all numbers in the list are positive.

For example, given S = `[12, 1, 61, 5, 9, 2]` and k = 24, return `[12, 9, 2, 1]` since it sums up to 24.

### Problem #1072 <a id="p1072"></a>

**Easy** · Facebook · 2022-11-29

#### Problem

Given a binary tree, return the level of the tree with minimum sum.

### Problem #1073 <a id="p1073"></a>

**Medium** · Apple · 2022-11-30

#### Problem

Implement a job scheduler which takes in a function f and an integer n, and calls f after n milliseconds.

### Problem #1074 <a id="p1074"></a>

**Hard** · Twitter · 2022-12-01

#### Problem

Given a binary tree, find the lowest common ancestor (LCA) of two given nodes in the tree. Assume that each node in the tree also has a pointer to its parent.

According to the definition of LCA on Wikipedia: “The lowest common ancestor is defined between two nodes v and w as the lowest node in T that has both v and w as descendants (where we allow a node to be a descendant of itself).”

### Problem #1075 <a id="p1075"></a>

**Easy** · Apple · 2022-12-01

#### Problem

Implement the function `fib(n)`, which returns the nth number in the Fibonacci sequence, using only `O(1)` space.

### Problem #1076 <a id="p1076"></a>

**Medium** · Squarespace · 2022-12-02

#### Problem

Write a function, `add_subtract`, which alternately adds and subtracts curried arguments. Here are some sample operations:

`add_subtract(7)` -> 7

`add_subtract(1)(2)(3)` -> 1 + 2 - 3 -> 0

`add_subtract(-5)(10)(3)(9)` -> -5 + 10 - 3 + 9 -> 11

### Problem #1077 <a id="p1077"></a>

**Easy** · Amazon · 2022-12-03

#### Problem

A tree is symmetric if its data and shape remain unchanged when it is reflected about the root node. The following tree is an example:

```
        4
      / | \
    3   5   3
  /           \
9              9
```

Given a k-ary tree, determine whether it is symmetric.

### Problem #1078 <a id="p1078"></a>

**Easy** · Google · 2022-12-04

#### Problem

Given a binary tree of integers, find the maximum path sum between two nodes. The path must go through at least one node, and does not need to go through the root.

### Problem #1079 <a id="p1079"></a>

**Medium** · Facebook · 2022-12-05

#### Problem

Given an array of integers in which two elements appear exactly once and all other elements appear exactly twice, find the two elements that appear only once.

For example, given the array `[2, 4, 6, 8, 10, 2, 6, 10]`, return 4 and 8. The order does not matter.

Follow-up: Can you do this in linear time and constant space?

### Problem #1080 <a id="p1080"></a>

**Easy** · Google · 2022-12-06

#### Problem

You are given a starting state start, a list of transition probabilities for a Markov chain, and a number of steps `num_steps`. Run the Markov chain starting from start for `num_steps` and compute the number of times we visited each state.

For example, given the starting state a, number of steps 5000, and the following transition probabilities:

```
[
  ('a', 'a', 0.9),
  ('a', 'b', 0.075),
  ('a', 'c', 0.025),
  ('b', 'a', 0.15),
  ('b', 'b', 0.8),
  ('b', 'c', 0.05),
  ('c', 'a', 0.25),
  ('c', 'b', 0.25),
  ('c', 'c', 0.5)
]
```

One instance of running this Markov chain might produce `{ 'a': 3012, 'b': 1656, 'c': 332 }`.

### Problem #1081 <a id="p1081"></a>

**Medium** · 2022-12-07

#### Problem

Given a 2-D matrix representing an image, a location of a pixel in the screen and a color C, replace the color of the given pixel and all adjacent same colored pixels with C.

For example, given the following matrix, and location pixel of (2, 2), and `'G'` for green:

```
B B W
W W W
W W W
B B B
```

Becomes

```
B B G
G G G
G G G
B B B
```

### Problem #1082 <a id="p1082"></a>

**Hard** · Salesforce · 2022-12-08

#### Problem

Connect 4 is a game where opponents take turns dropping red or black discs into a 7 x 6 vertically suspended grid. The game ends either when one player creates a line of four consecutive discs of their color (horizontally, vertically, or diagonally), or when there are no more spots left in the grid.

Design and implement Connect 4.

### Problem #1083 <a id="p1083"></a>

**Medium** · Facebook · 2022-12-09

#### Problem

Given a number in Roman numeral format, convert it to decimal.

The values of Roman numerals are as follows:

```
{
    'M': 1000,
    'D': 500,
    'C': 100,
    'L': 50,
    'X': 10,
    'V': 5,
    'I': 1
}
```

In addition, note that the Roman numeral system uses subtractive notation for numbers such as IV and XL.

For the input XIV, for instance, you should return 14.

### Problem #1084 <a id="p1084"></a>

**Medium** · Stripe · 2022-12-10

#### Problem

reduce (also known as fold) is a function that takes in an array, a combining function, and an initial value and builds up a result by calling the combining function on each element of the array, left to right. For example, we can write `sum()` in terms of reduce:

```
def add(a, b):
    return a + b
```

```
def sum(lst):
    return reduce(lst, add, 0)
```

This should call add on the initial value with the first element of the array, and then the result of that with the second element of the array, and so on until we reach the end, when we return the sum of the array.

Implement your own version of reduce.

### Problem #1085 <a id="p1085"></a>

**Medium** · Snapchat · 2022-12-11

#### Problem

Given a string of digits, generate all possible valid IP address combinations.

IP addresses must follow the format A.B.C.D, where A, B, C, and D are numbers between 0 and 255. Zero-prefixed numbers, such as 01 and 065, are not allowed, except for 0 itself.

For example, given `"2542540123"`, you should return `['254.25.40.123', '254.254.0.123']`.

### Problem #1086 <a id="p1086"></a>

**Hard** · Dropbox · 2022-12-12

#### Problem

Sudoku is a puzzle where you're given a partially-filled 9 by 9 grid with digits. The objective is to fill the grid with the constraint that every row, column, and box (3 by 3 subgrid) must contain all of the digits from 1 to 9.

Implement an efficient sudoku solver.

### Problem #1087 <a id="p1087"></a>

**Easy** · Google · 2022-12-13

#### Problem

Given two strings A and B, return whether or not A can be shifted some number of times to get B.

For example, if A is abcde and B is cdeab, return true. If A is abc and B is acb, return false.

### Problem #1088 <a id="p1088"></a>

**Hard** · Quantcast · 2022-12-14

#### Problem

You are presented with an array representing a Boolean expression. The elements are of two kinds:

- T and F, representing the values True and False.
- &, |, and ^, representing the bitwise operators for AND, OR, and XOR.

Determine the number of ways to group the array elements using parentheses so that the entire expression evaluates to True.

For example, suppose the input is `['F', '|', 'T', '&', 'T']`. In this case, there are two acceptable groupings: (F | T) & T and F | (T & T).

### Problem #1089 <a id="p1089"></a>

**Hard** · Google · 2022-12-15

#### Problem

Given an array of integers and a number k, where 1 \<= k \<= length of the array, compute the maximum values of each subarray of length k.

For example, given array = `[10, 5, 2, 7, 8, 7]` and k = 3, we should get: `[10, 7, 8, 8]`, since:

- 10 = `max(10, 5, 2)`
- 7 = `max(5, 2, 7)`
- 8 = `max(2, 7, 8)`
- 8 = `max(7, 8, 7)`

Do this in `O(n)` time and `O(k)` space. You can modify the input array in-place and you do not need to store the results. You can simply print them out as you compute them.

### Problem #1090 <a id="p1090"></a>

**Hard** · Airbnb · 2022-12-16

#### Problem

Given a list of words, find all pairs of unique indices such that the concatenation of the two words is a palindrome.

For example, given the list `["code", "edoc", "da", "d"]`, return `[(0, 1), (1, 0), (2, 3)]`.

### Problem #1091 <a id="p1091"></a>

**Easy** · Twitter · 2022-12-17

#### Problem

A classroom consists of N students, whose friendships can be represented in an adjacency list. For example, the following descibes a situation where 0 is friends with 1 and 2, 3 is friends with 6, and so on.

```
{0: [1, 2],
 1: [0, 5],
 2: [0],
 3: [6],
 4: [],
 5: [1],
 6: [3]}
```

Each student can be placed in a friend group, which can be defined as the transitive closure of that student's friendship relations. In other words, this is the smallest set such that no student in the group has any friends outside this group. For the example above, the friend groups would be `{0, 1, 2, 5}`, `{3, 6}`, `{4}`.

Given a friendship list such as the one above, determine the number of friend groups in the class.

### Problem #1092 <a id="p1092"></a>

**Easy** · Google · 2022-12-18

#### Problem

A regular number in mathematics is defined as one which evenly divides some power of 60. Equivalently, we can say that a regular number is one whose only prime divisors are 2, 3, and 5.

These numbers have had many applications, from helping ancient Babylonians keep time to tuning instruments according to the diatonic scale.

Given an integer N, write a program that returns, in order, the first N regular numbers.

### Problem #1093 <a id="p1093"></a>

**Medium** · Google · 2022-12-19

#### Problem

Given an array of integers, return a new array where each element in the new array is the number of smaller elements to the right of that element in the original input array.

For example, given the array `[3, 4, 9, 6, 1]`, return `[1, 1, 2, 1, 0]`, since:

- There is 1 smaller element to the right of 3
- There is 1 smaller element to the right of 4
- There are 2 smaller elements to the right of 9
- There is 1 smaller element to the right of 6
- There are no smaller elements to the right of 1

### Problem #1094 <a id="p1094"></a>

**Hard** · Apple · 2022-12-20

#### Problem

Given a linked list, uniformly shuffle the nodes. What if we want to prioritize space over time?

### Problem #1095 <a id="p1095"></a>

**Easy** · Google · 2022-12-21

#### Problem

You are given given a list of rectangles represented by min and max x- and y-coordinates. Compute whether or not a pair of rectangles overlap each other. If one rectangle completely covers another, it is considered overlapping.

For example, given the following rectangles:

```
{
    "top_left": (1, 4),
    "dimensions": (3, 3) # width, height
},
{
    "top_left": (-1, 3),
    "dimensions": (2, 1)
},
{
    "top_left": (0, 5),
    "dimensions": (4, 3)
}
```

return true as the first and third rectangle overlap each other.

### Problem #1096 <a id="p1096"></a>

**Hard** · Google · 2022-12-22

#### Problem

Implement an LFU (Least Frequently Used) cache. It should be able to be initialized with a cache size n, and contain the following methods:

- `set(key, value)`: sets key to value. If there are already n items in the cache and we are adding a new item, then it should also remove the least frequently used item. If there is a tie, then the least recently used key should be removed.
- `get(key)`: gets the value at key. If no such key exists, return null.

Each operation should run in `O(1)` time.

### Problem #1097 <a id="p1097"></a>

**Easy** · Yahoo · 2022-12-23

#### Problem

You are given a string of length N and a parameter k. The string can be manipulated by taking one of the first k letters and moving it to the end.

Write a program to determine the lexicographically smallest string that can be created after an unlimited number of moves.

For example, suppose we are given the string daily and k = 1. The best we can create in this case is ailyd.

### Problem #1098 <a id="p1098"></a>

**Easy** · Oracle · 2022-12-24

#### Problem

Given a binary search tree, find the floor and ceiling of a given integer. The floor is the highest element in the tree less than or equal to an integer, while the ceiling is the lowest element in the tree greater than or equal to an integer.

If either value does not exist, return None.

### Problem #1099 <a id="p1099"></a>

**Easy** · Amazon · 2022-12-25

#### Problem

Given an array and a number k that's smaller than the length of the array, rotate the array to the right k elements in-place.

### Problem #1100 <a id="p1100"></a>

**Hard** · Netflix · 2022-12-26

#### Problem

Given a sorted list of integers of length N, determine if an element x is in the list without performing any multiplication, division, or bit-shift operations.

Do this in `O(log N)` time.

### Problem #1101 <a id="p1101"></a>

**Easy** · Amazon · 2022-12-27

#### Problem

Consider the following scenario: there are N mice and N holes placed at integer points along a line. Given this, find a method that maps mice to holes such that the largest number of steps any mouse takes is minimized.

Each move consists of moving one mouse one unit to the left or right, and only one mouse can fit inside each hole.

For example, suppose the mice are positioned at `[1, 4, 9, 15]`, and the holes are located at `[10, -5, 0, 16]`. In this case, the best pairing would require us to send the mouse at 1 to the hole at -5, so our function should return 6.

### Problem #1102 <a id="p1102"></a>

**Medium** · Walmart Labs · 2022-12-28

#### Problem

There are M people sitting in a row of N seats, where M \< N. Your task is to redistribute people such that there are no gaps between any of them, while keeping overall movement to a minimum.

For example, suppose you are faced with an input of `[0, 1, 1, 0, 1, 0, 0, 0, 1]`, where 0 represents an empty seat and 1 represents a person. In this case, one solution would be to place the person on the right in the fourth seat. We can consider the cost of a solution to be the sum of the absolute distance each person must move, so that the cost here would be five.

Given an input such as the one above, return the lowest possible cost of moving people to remove all gaps.

### Problem #1103 <a id="p1103"></a>

**Medium** · Slack · 2022-12-29

#### Problem

You are given an N by M matrix of 0s and 1s. Starting from the top left corner, how many ways are there to reach the bottom right corner?

You can only move right and down. 0 represents an empty space while 1 represents a wall you cannot walk through.

For example, given the following matrix:

```
[[0, 0, 1],
 [0, 0, 1],
 [1, 0, 0]]
```

Return two, as there are only two ways to get to the bottom right:

- Right, down, down, right
- Down, right, down, right

The top left corner and bottom right corner will always be 0.

### Problem #1104 <a id="p1104"></a>

**Easy** · Yelp · 2022-12-30

#### Problem

Given a mapping of digits to letters (as in a phone number), and a digit string, return all possible letters the number could represent. You can assume each valid number in the mapping is a single digit.

For example if `{“2”: [“a”, “b”, “c”], 3: [“d”, “e”, “f”], …}` then “23” should return \[“ad”, “ae”, “af”, “bd”, “be”, “bf”, “cd”, “ce”, “cf"\].

### Problem #1105 <a id="p1105"></a>

**Medium** · Facebook · 2022-12-31

#### Problem

Given a multiset of integers, return whether it can be partitioned into two subsets whose sums are the same.

For example, given the multiset `{15, 5, 20, 10, 35, 15, 10}`, it would return true, since we can split it up into `{15, 5, 10, 15, 10}` and `{20, 35}`, which both add up to 55.

Given the multiset `{15, 5, 20, 10, 35}`, it would return false, since we can't split it up into two subsets that add up to the same sum.

### Problem #1106 <a id="p1106"></a>

**Medium** · Mailchimp · 2023-01-01

#### Problem

You are given an array representing the heights of neighboring buildings on a city street, from east to west. The city assessor would like you to write an algorithm that returns how many of these buildings have a view of the setting sun, in order to properly value the street.

For example, given the array `[3, 7, 8, 3, 6, 1]`, you should return 3, since the top floors of the buildings with heights 8, 6, and 1 all have an unobstructed view to the west.

Can you do this using just one forward pass through the array?

### Problem #1107 <a id="p1107"></a>

**Hard** · Adobe · 2023-01-02

#### Problem

You are given a tree with an even number of nodes. Consider each connection between a parent and child node to be an `"edge"`. You would like to remove some of these edges, such that the disconnected subtrees that remain each have an even number of nodes.

For example, suppose your input was the following tree:

```
   1
  / \ 
 2   3
    / \ 
   4   5
 / | \
6  7  8
```

In this case, removing the edge (3, 4) satisfies our requirement.

Write a function that returns the maximum number of edges you can remove while still satisfying this requirement.

### Problem #1108 <a id="p1108"></a>

**Easy** · Pandora · 2023-01-03

#### Problem

Given an undirected graph, determine if it contains a cycle.

### Problem #1109 <a id="p1109"></a>

**Medium** · Amazon · 2023-01-04

#### Problem

Given a pivot x, and a list lst, partition the list into three parts.

- The first part contains all elements in lst that are less than x
- The second part contains all elements in lst that are equal to x
- The third part contains all elements in lst that are larger than x

Ordering within a part can be arbitrary.

For example, given x = 10 and lst = `[9, 12, 3, 5, 14, 10, 10]`, one partition may be `[9, 3, 5, 10, 10, 12, 14]`.

### Problem #1110 <a id="p1110"></a>

**Easy** · Netflix · 2023-01-05

#### Problem

Given an array of integers, determine whether it contains a Pythagorean triplet. Recall that a Pythogorean triplet (a, b, c) is defined by the equation a2+ b2= c2.

### Problem #1111 <a id="p1111"></a>

**Medium** · Amazon · 2023-01-06

#### Problem

At a popular bar, each customer has a set of favorite drinks, and will happily accept any drink among this set. For example, in the following situation, customer 0 will be satisfied with drinks 0, 1, 3, or 6.

```
preferences = {
    0: [0, 1, 3, 6],
    1: [1, 4, 7],
    2: [2, 4, 7, 5],
    3: [3, 2, 5],
    4: [5, 8]
}
```

A lazy bartender working at this bar is trying to reduce his effort by limiting the drink recipes he must memorize. Given a dictionary input such as the one above, return the fewest number of drinks he must learn in order to satisfy all customers.

For the input above, the answer would be 2, as drinks 1 and 5 will satisfy everyone.

### Problem #1112 <a id="p1112"></a>

**Easy** · Apple · 2023-01-07

#### Problem

Given a binary tree, find a minimum path sum from root to a leaf.

For example, the minimum path in this tree is `[10, 5, 1, -1]`, which has sum 15.

```
  10
 /  \
5    5
 \     \
   2    1
       /
     -1
```

### Problem #1113 <a id="p1113"></a>

**Easy** · Palantir · 2023-01-08

#### Problem

A typical American-style crossword puzzle grid is an N x N matrix with black and white squares, which obeys the following rules:

- Every white square must be part of an `"across"` word and a `"down"` word.
- No word can be fewer than three letters long.
- Every white square must be reachable from every other white square.
- The grid is rotationally symmetric (for example, the colors of the top left and bottom right squares must match).

Write a program to determine whether a given matrix qualifies as a crossword grid.

### Problem #1114 <a id="p1114"></a>

**Easy** · Uber · 2023-01-09

#### Problem

On election day, a voting machine writes data in the form (`voter_id`, `candidate_id`) to a text file. Write a program that reads this file as a stream and returns the top 3 candidates at any given time. If you find a voter voting more than once, report this as fraud.

### Problem #1115 <a id="p1115"></a>

**Medium** · Google · 2023-01-10

#### Problem

Given an integer n and a list of integers l, write a function that randomly generates a number from 0 to n-1 that isn't in l (uniform).

### Problem #1116 <a id="p1116"></a>

**Hard** · Google · 2023-01-11

#### Problem

A quack is a data structure combining properties of both stacks and queues. It can be viewed as a list of elements written left to right such that three operations are possible:

- `push(x)`: add a new item x to the left end of the list
- `pop()`: remove and return the item on the left end of the list
- `pull()`: remove the item on the right end of the list.

Implement a quack using three stacks and `O(1)` additional memory, so that the amortized time for any push, pop, or pull operation is `O(1)`.

### Problem #1117 <a id="p1117"></a>

**Hard** · Microsoft · 2023-01-12

#### Problem

Implement 3 stacks using a single list:

```
class Stack:
    def __init__(self):
        self.list = []
```

```
def pop(self, stack_number):
    pass
```

```
def push(self, item, stack_number):
    pass
```

### Problem #1118 <a id="p1118"></a>

**Easy** · Dropbox · 2023-01-13

#### Problem

What does the below code snippet print out? How can we fix the anonymous functions to behave as we'd expect?

```
functions = []
for i in range(10):
    functions.append(lambda : i)
```

```
for f in functions:
    print(f())
```

### Problem #1119 <a id="p1119"></a>

**Medium** · Etsy · 2023-01-14

#### Problem

Given an array of numbers N and an integer k, your task is to split N into k partitions such that the maximum sum of any partition is minimized. Return this sum.

For example, given N = `[5, 1, 2, 7, 3, 4]` and k = 3, you should return 8, since the optimal partition is `[5, 1, 2]`, `[7]`, `[3, 4]`.

### Problem #1120 <a id="p1120"></a>

**Easy** · Google · 2023-01-15

#### Problem

You are in an infinite 2D grid where you can move in any of the 8 directions:

```
(x,y) to
   (x+1, y),
   (x - 1, y),
   (x, y+1),
   (x, y-1),
   (x-1, y-1),
   (x+1,y+1),
   (x-1,y+1),
   (x+1,y-1)
```

You are given a sequence of points and the order in which you need to cover the points. Give the minimum number of steps in which you can achieve it. You start from the first point.

Example:

```
Input: [(0, 0), (1, 1), (1, 2)]
Output: 2
```

It takes 1 step to move from (0, 0) to (1, 1). It takes one more step to move from (1, 1) to (1, 2).

### Problem #1121 <a id="p1121"></a>

**Hard** · Affirm · 2023-01-16

#### Problem

Given a array of numbers representing the stock prices of a company in chronological order, write a function that calculates the maximum profit you could have made from buying and selling that stock. You're also given a number fee that represents a transaction fee for each buy and sell transaction.

You must buy before you can sell the stock, but you can make as many transactions as you like.

For example, given `[1, 3, 2, 8, 4, 10]` and fee = 2, you should return 9, since you could buy the stock at 1 dollar, and sell at 8 dollars, and then buy it at 4 dollars and sell it at 10 dollars. Since we did two transactions, there is a 4 dollar fee, so we have 7 + 6 = 13 profit minus 4 dollars of fees.

### Problem #1122 <a id="p1122"></a>

**Hard** · Two Sigma · 2023-01-17

#### Problem

Ghost is a two-person word game where players alternate appending letters to a word. The first person who spells out a word, or creates a prefix for which there is no possible continuation, loses. Here is a sample game:

- Player 1: g
- Player 2: h
- Player 1: o
- Player 2: s
- Player 1: t \[loses\]

Given a dictionary of words, determine the letters the first player should start with, such that with optimal play they cannot lose.

For example, if the dictionary is `["cat", "calf", "dog", "bear"]`, the only winning start letter would be b.

### Problem #1123 <a id="p1123"></a>

**Medium** · Google · 2023-01-18

#### Problem

Given a set of distinct positive integers, find the largest subset such that every pair of elements in the subset (i, j) satisfies either i % j = 0 or j % i = 0.

For example, given the set `[3, 5, 10, 20, 21]`, you should return `[5, 10, 20]`. Given `[1, 3, 6, 24]`, return `[1, 3, 6, 24]`.

### Problem #1124 <a id="p1124"></a>

**Hard** · Microsoft · 2023-01-19

#### Problem

Let X be a set of n intervals on the real line. We say that a set of points P `"stabs"` X if every interval in X contains at least one point in P. Compute the smallest set of points that stabs X.

For example, given the intervals `[(1, 4), (4, 5), (7, 9), (9, 12)]`, you should return `[4, 9]`.

### Problem #1125 <a id="p1125"></a>

**Medium** · Lyft · 2023-01-20

#### Problem

Given a list of integers and a number K, return which contiguous elements of the list sum to K.

For example, if the list is `[1, 2, 3, 4, 5]` and K is 9, then it should return `[2, 3, 4]`, since 2 + 3 + 4 = 9.

### Problem #1126 <a id="p1126"></a>

**Medium** · Uber · 2023-01-21

#### Problem

Write a program that determines the smallest number of perfect squares that sum up to N.

Here are a few examples:

- Given N = 4, return 1 (4)
- Given N = 17, return 2 (16 + 1)
- Given N = 18, return 2 (9 + 9)

### Problem #1127 <a id="p1127"></a>

**Medium** · Amazon · 2023-01-22

#### Problem

Given a string, find the length of the smallest window that contains every distinct character. Characters may appear more than once in the window.

For example, given `"jiujitsu"`, you should return 5, corresponding to the final five letters.

### Problem #1128 <a id="p1128"></a>

**Easy** · Microsoft · 2023-01-23

#### Problem

Given a number in the form of a list of digits, return all possible permutations.

For example, given `[1,2,3]`, return `[[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]`.

### Problem #1129 <a id="p1129"></a>

**Medium** · Facebook · 2023-01-24

#### Problem

In chess, the Elo rating system is used to calculate player strengths based on game results.

A simplified description of the Elo system is as follows. Every player begins at the same score. For each subsequent game, the loser transfers some points to the winner, where the amount of points transferred depends on how unlikely the win is. For example, a 1200-ranked player should gain much more points for beating a 2000-ranked player than for beating a 1300-ranked player.

Implement this system.

### Problem #1130 <a id="p1130"></a>

**Hard** · Apple · 2023-01-25

#### Problem

Given a tree, find the largest tree/subtree that is a BST.

Given a tree, return the size of the largest tree/subtree that is a BST.

### Problem #1131 <a id="p1131"></a>

**Hard** · MIT · 2023-01-26

#### Problem

Blackjack is a two player card game whose rules are as follows:

- The player and then the dealer are each given two cards.
- The player can then `"hit"`, or ask for arbitrarily many additional cards, so long as their total does not exceed 21.
- The dealer must then hit if their total is 16 or lower, otherwise pass.
- Finally, the two compare totals, and the one with the greatest sum not exceeding 21 is the winner.

For this problem, cards values are counted as follows: each card between 2 and 10 counts as their face value, face cards count as 10, and aces count as 1.

Given perfect knowledge of the sequence of cards in the deck, implement a blackjack solver that maximizes the player's score (that is, wins minus losses).

### Problem #1132 <a id="p1132"></a>

**Medium** · Facebook · 2023-01-27

#### Problem

Write a function that rotates a list by k elements. For example, `[1, 2, 3, 4, 5, 6]` rotated by two becomes `[3, 4, 5, 6, 1, 2]`. Try solving this without creating a copy of the list. How many swap or move operations do you need?

### Problem #1133 <a id="p1133"></a>

**Easy** · Yelp · 2023-01-28

#### Problem

Given a mapping of digits to letters (as in a phone number), and a digit string, return all possible letters the number could represent. You can assume each valid number in the mapping is a single digit.

For example if `{“2”: [“a”, “b”, “c”], 3: [“d”, “e”, “f”], …}` then “23” should return \[“ad”, “ae”, “af”, “bd”, “be”, “bf”, “cd”, “ce”, “cf"\].

### Problem #1134 <a id="p1134"></a>

**Easy** · Microsoft · 2023-01-29

#### Problem

Compute the running median of a sequence of numbers. That is, given a stream of numbers, print out the median of the list so far on each new element.

Recall that the median of an even-numbered list is the average of the two middle numbers.

For example, given the sequence `[2, 1, 5, 7, 2, 0, 5]`, your algorithm should print out:

```
2
1.5
2
3.5
2
2
2
```

### Problem #1135 <a id="p1135"></a>

**Easy** · Google · 2023-01-30

#### Problem

The power set of a set is the set of all its subsets. Write a function that, given a set, generates its power set.

For example, given the set `{1, 2, 3}`, it should return `{{}, {1}, {2}, {3}, {1, 2}, {1, 3}, {2, 3}, {1, 2, 3}}`.

You may also use a list or array to represent a set.

### Problem #1136 <a id="p1136"></a>

**Medium** · Yelp · 2023-01-31

#### Problem

The horizontal distance of a binary tree node describes how far left or right the node will be when the tree is printed out.

More rigorously, we can define it as follows:

- The horizontal distance of the root is 0.
- The horizontal distance of a left child is `hd(parent)` - 1.
- The horizontal distance of a right child is `hd(parent)` + 1.

For example, for the following tree, `hd(1)` = -2, and `hd(6)` = 0.

```
           5
        /     \
      3         7
    /  \      /   \
  1     4    6     9
 /                /
0                8
```

The bottom view of a tree, then, consists of the lowest node at each horizontal distance. If there are two nodes at the same depth and horizontal distance, either is acceptable.

For this tree, for example, the bottom view could be `[0, 1, 3, 6, 8, 9]`.

Given the root to a binary tree, return its bottom view.

### Problem #1137 <a id="p1137"></a>

**Medium** · Jane Street · 2023-02-01

#### Problem

`cons(a, b)` constructs a pair, and `car(pair)` and `cdr(pair)` returns the first and last element of that pair. For example, `car(cons(3, 4))` returns 3, and `cdr(cons(3, 4))` returns 4.

Given this implementation of cons:

```
def cons(a, b):
    def pair(f):
        return f(a, b)
    return pair
```

Implement car and cdr.

### Problem #1138 <a id="p1138"></a>

**Medium** · Amazon · 2023-02-02

#### Problem

An sorted array of integers was rotated an unknown number of times.

Given such an array, find the index of the element in the array in faster than linear time. If the element doesn't exist in the array, return null.

For example, given the array `[13, 18, 25, 2, 8, 10]` and the element 8, return 4 (the index of 8 in the array).

You can assume all the integers in the array are unique.

### Problem #1139 <a id="p1139"></a>

**Hard** · Twitter · 2023-02-03

#### Problem

Given a binary tree, find the lowest common ancestor (LCA) of two given nodes in the tree. Assume that each node in the tree also has a pointer to its parent.

According to the definition of LCA on Wikipedia: “The lowest common ancestor is defined between two nodes v and w as the lowest node in T that has both v and w as descendants (where we allow a node to be a descendant of itself).”

### Problem #1140 <a id="p1140"></a>

**Hard** · LinkedIn · 2023-02-04

#### Problem

Given a set of characters C and an integer k, a De Bruijn sequence is a cyclic sequence in which every possible k-length string of characters in C occurs exactly once.

For example, suppose C = `{0, 1}` and k = 3. Then our sequence should contain the substrings `{'000', '001', '010', '011', '100', '101', '110', '111'}`, and one possible solution would be 00010111.

Create an algorithm that finds a De Bruijn sequence.

### Problem #1141 <a id="p1141"></a>

**Medium** · Walmart Labs · 2023-02-05

#### Problem

There are M people sitting in a row of N seats, where M \< N. Your task is to redistribute people such that there are no gaps between any of them, while keeping overall movement to a minimum.

For example, suppose you are faced with an input of `[0, 1, 1, 0, 1, 0, 0, 0, 1]`, where 0 represents an empty seat and 1 represents a person. In this case, one solution would be to place the person on the right in the fourth seat. We can consider the cost of a solution to be the sum of the absolute distance each person must move, so that the cost here would be five.

Given an input such as the one above, return the lowest possible cost of moving people to remove all gaps.

### Problem #1142 <a id="p1142"></a>

**Easy** · Zillow · 2023-02-06

#### Problem

Let's define a `"sevenish"` number to be one which is either a power of 7, or the sum of unique powers of 7. The first few sevenish numbers are 1, 7, 8, 49, and so on. Create an algorithm to find the nth sevenish number.

### Problem #1143 <a id="p1143"></a>

**Medium** · Google · 2023-02-07

#### Problem

Given k sorted singly linked lists, write a function to merge all the lists into one sorted singly linked list.

### Problem #1144 <a id="p1144"></a>

**Easy** · Jane Street · 2023-02-08

#### Problem

Given integers M and N, write a program that counts how many positive integer pairs (a, b) satisfy the following conditions:

- a + b = M
- a XOR b = N

### Problem #1145 <a id="p1145"></a>

**Medium** · Triplebyte · 2023-02-09

#### Problem

Implement a data structure which carries out the following operations without resizing the underlying array:

- `add(value)`: Add a value to the set of values.
- `check(value)`: Check whether a value is in the set.

The check method may return occasional false positives (in other words, incorrectly identifying an element as part of the set), but should always correctly identify a true element.

### Problem #1146 <a id="p1146"></a>

**Easy** · Microsoft · 2023-02-10

#### Problem

You are given an string representing the initial conditions of some dominoes. Each element can take one of three values:

- L, meaning the domino has just been pushed to the left,
- R, meaning the domino has just been pushed to the right, or
- ., meaning the domino is standing still.

Determine the orientation of each tile when the dominoes stop falling. Note that if a domino receives a force from the left and right side simultaneously, it will remain upright.

For example, given the string .L.R....L, you should return LL.RRRLLL.

Given the string ..R...L.L, you should return ..RR.LLLL.

### Problem #1147 <a id="p1147"></a>

**Medium** · Google · 2023-02-11

#### Problem

Implement locking in a binary tree. A binary tree node can be locked or unlocked only if all of its descendants or ancestors are not locked.

Design a binary tree node class with the following methods:

- `is_locked`, which returns whether the node is locked
- lock, which attempts to lock the node. If it cannot be locked, then it should return false. Otherwise, it should lock it and return true.
- unlock, which unlocks the node. If it cannot be unlocked, then it should return false. Otherwise, it should unlock it and return true.

You may augment the node to add parent pointers or any other property you would like. You may assume the class is used in a single-threaded program, so there is no need for actual locks or mutexes. Each method should run in `O(h)`, where h is the height of the tree.

### Problem #1148 <a id="p1148"></a>

**Easy** · Airbnb · 2023-02-12

#### Problem

Given a linked list and a positive integer k, rotate the list to the right by k places.

For example, given the linked list 7 -> 7 -> 3 -> 5 and k = 2, it should become 3 -> 5 -> 7 -> 7.

Given the linked list 1 -> 2 -> 3 -> 4 -> 5 and k = 3, it should become 3 -> 4 -> 5 -> 1 -> 2.

### Problem #1149 <a id="p1149"></a>

**Medium** · Amazon · 2023-02-13

#### Problem

Given a matrix of 1s and 0s, return the number of `"islands"` in the matrix. A 1 represents land and 0 represents water, so an island is a group of 1s that are neighboring whose perimeter is surrounded by water.

For example, this matrix has 4 islands.

```
1 0 0 0 0
0 0 1 1 0
0 1 1 0 0
0 0 0 0 0
1 1 0 0 1
1 1 0 0 1
```

### Problem #1150 <a id="p1150"></a>

**Hard** · VMware · 2023-02-14

#### Problem

The skyline of a city is composed of several buildings of various widths and heights, possibly overlapping one another when viewed from a distance. We can represent the buildings using an array of (left, right, height) tuples, which tell us where on an imaginary x-axis a building begins and ends, and how tall it is. The skyline itself can be described by a list of (x, height) tuples, giving the locations at which the height visible to a distant observer changes, and each new height.

Given an array of buildings as described above, create a function that returns the skyline.

For example, suppose the input consists of the buildings `[(0, 15, 3), (4, 11, 5), (19, 23, 4)]`. In aggregate, these buildings would create a skyline that looks like the one below.

### Problem #1151 <a id="p1151"></a>

**Easy** · Google · 2023-02-15

#### Problem

Determine whether a doubly linked list is a palindrome. What if it’s singly linked?

For example, 1 -> 4 -> 3 -> 4 -> 1 returns True while 1 -> 4 returns False.

### Problem #1152 <a id="p1152"></a>

**Medium** · Quora · 2023-02-16

#### Problem

Given an absolute pathname that may have . or .. as part of it, return the shortest standardized path.

For example, given `"/usr/bin/../bin/./scripts/../"`, return `"/usr/bin/"`.

### Problem #1153 <a id="p1153"></a>

**Easy** · Google · 2023-02-17

#### Problem

You are in an infinite 2D grid where you can move in any of the 8 directions:

```
(x,y) to
   (x+1, y),
   (x - 1, y),
   (x, y+1),
   (x, y-1),
   (x-1, y-1),
   (x+1,y+1),
   (x-1,y+1),
   (x+1,y-1)
```

You are given a sequence of points and the order in which you need to cover the points. Give the minimum number of steps in which you can achieve it. You start from the first point.

Example:

```
Input: [(0, 0), (1, 1), (1, 2)]
Output: 2
```

It takes 1 step to move from (0, 0) to (1, 1). It takes one more step to move from (1, 1) to (1, 2).

### Problem #1154 <a id="p1154"></a>

**Easy** · Apple · 2023-02-18

#### Problem

Given a binary tree, find a minimum path sum from root to a leaf.

For example, the minimum path in this tree is `[10, 5, 1, -1]`, which has sum 15.

```
  10
 /  \
5    5
 \     \
   2    1
       /
     -1
```

### Problem #1155 <a id="p1155"></a>

**Hard** · Google · 2023-02-19

#### Problem

Given an array of integers and a number k, where 1 \<= k \<= length of the array, compute the maximum values of each subarray of length k.

For example, given array = `[10, 5, 2, 7, 8, 7]` and k = 3, we should get: `[10, 7, 8, 8]`, since:

- 10 = `max(10, 5, 2)`
- 7 = `max(5, 2, 7)`
- 8 = `max(2, 7, 8)`
- 8 = `max(7, 8, 7)`

Do this in `O(n)` time and `O(k)` space. You can modify the input array in-place and you do not need to store the results. You can simply print them out as you compute them.

### Problem #1156 <a id="p1156"></a>

**Easy** · Microsoft · 2023-02-20

#### Problem

Print the nodes in a binary tree level-wise. For example, the following should print 1, 2, 3, 4, 5.

```
  1
 / \
2   3
   / \
  4   5
```

### Problem #1157 <a id="p1157"></a>

**Easy** · Palantir · 2023-02-21

#### Problem

In academia, the h-index is a metric used to calculate the impact of a researcher's papers. It is calculated as follows:

A researcher has index h if at least h of her N papers have h citations each. If there are multiple h satisfying this formula, the maximum is chosen.

For example, suppose N = 5, and the respective citations of each paper are `[4, 3, 0, 1, 5]`. Then the h-index would be 3, since the researcher has 3 papers with at least 3 citations.

Given a list of paper citations of a researcher, calculate their h-index.

### Problem #1158 <a id="p1158"></a>

**Medium** · Google · 2023-02-22

#### Problem

Given a linked list, sort it in `O(n log n)` time and constant space.

For example, the linked list 4 -> 1 -> -3 -> 99 should become -3 -> 1 -> 4 -> 99.

### Problem #1159 <a id="p1159"></a>

**Easy** · Alibaba · 2023-02-23

#### Problem

Given an even number (greater than 2), return two prime numbers whose sum will be equal to the given number.

A solution will always exist. See Goldbach’s conjecture.

Example:

```
Input: 4
Output: 2 + 2 = 4
```

If there are more than one solution possible, return the lexicographically smaller solution.

If `[a, b]` is one solution with a \<= b, and `[c, d]` is another solution with c \<= d, then

`[a, b]` \< `[c, d]`

If a \< c OR a==c AND b \< d.

### Problem #1160 <a id="p1160"></a>

**Easy** · Slack · 2023-02-24

#### Problem

You are given a string formed by concatenating several words corresponding to the integers zero through nine and then anagramming.

For example, the input could be `'niesevehrtfeev'`, which is an anagram of `'threefiveseven'`. Note that there can be multiple instances of each integer.

Given this string, return the original integers in sorted order. In the example above, this would be 357.

### Problem #1161 <a id="p1161"></a>

**Hard** · Airbnb · 2023-02-25

#### Problem

An 8-puzzle is a game played on a 3 x 3 board of tiles, with the ninth tile missing. The remaining tiles are labeled 1 through 8 but shuffled randomly. Tiles may slide horizontally or vertically into an empty space, but may not be removed from the board.

Design a class to represent the board, and find a series of steps to bring the board to the state `[[1, 2, 3], [4, 5, 6], [7, 8, None]]`.

### Problem #1162 <a id="p1162"></a>

**Hard** · Adobe · 2023-02-26

#### Problem

You are given a tree with an even number of nodes. Consider each connection between a parent and child node to be an `"edge"`. You would like to remove some of these edges, such that the disconnected subtrees that remain each have an even number of nodes.

For example, suppose your input was the following tree:

```
   1
  / \ 
 2   3
    / \ 
   4   5
 / | \
6  7  8
```

In this case, removing the edge (3, 4) satisfies our requirement.

Write a function that returns the maximum number of edges you can remove while still satisfying this requirement.

### Problem #1163 <a id="p1163"></a>

**Easy** · Stripe · 2023-02-27

#### Problem

Given a collection of intervals, find the minimum number of intervals you need to remove to make the rest of the intervals non-overlapping.

Intervals can `"touch"`, such as `[0, 1]` and `[1, 2]`, but they won't be considered overlapping.

For example, given the intervals (7, 9), (2, 4), (5, 8), return 1 as the last interval can be removed and the first two won't overlap.

The intervals are not necessarily sorted in any order.

### Problem #1164 <a id="p1164"></a>

**Easy** · Microsoft · 2023-02-28

#### Problem

You have n fair coins and you flip them all at the same time. Any that come up tails you set aside. The ones that come up heads you flip again. How many rounds do you expect to play before only one coin remains?

Write a function that, given n, returns the number of rounds you'd expect to play until one coin remains.

### Problem #1165 <a id="p1165"></a>

**Hard** · Two Sigma · 2023-03-01

#### Problem

A knight is placed on a given square on an 8 x 8 chessboard. It is then moved randomly several times, where each move is a standard knight move. If the knight jumps off the board at any point, however, it is not allowed to jump back on.

After k moves, what is the probability that the knight remains on the board?

### Problem #1166 <a id="p1166"></a>

**Medium** · 2023-03-02

#### Problem

Given a 2-D matrix representing an image, a location of a pixel in the screen and a color C, replace the color of the given pixel and all adjacent same colored pixels with C.

For example, given the following matrix, and location pixel of (2, 2), and `'G'` for green:

```
B B W
W W W
W W W
B B B
```

Becomes

```
B B G
G G G
G G G
B B B
```

### Problem #1167 <a id="p1167"></a>

**Medium** · Google · 2023-03-03

#### Problem

Implement integer exponentiation. That is, implement the `pow(x, y)` function, where x and y are integers and returns x^y.

Do this faster than the naive method of repeated multiplication.

For example, `pow(2, 10)` should return 1024.

### Problem #1168 <a id="p1168"></a>

**Easy** · Twitter · 2023-03-04

#### Problem

The 24 game is played as follows. You are given a list of four integers, each between 1 and 9, in a fixed order. By placing the operators +, -, \*, and / between the numbers, and grouping them with parentheses, determine whether it is possible to reach the value 24.

For example, given the input `[5, 2, 7, 8]`, you should return True, since (5 \* 2 - 7) \* 8 = 24.

Write a function that plays the 24 game.

### Problem #1169 <a id="p1169"></a>

**Hard** · LinkedIn · 2023-03-05

#### Problem

Given a list of points, a central point, and an integer k, find the nearest k points from the central point.

For example, given the list of points `[(0, 0), (5, 4), (3, 1)]`, the central point (1, 2), and k = 2, return `[(0, 0), (3, 1)]`.

### Problem #1170 <a id="p1170"></a>

**Hard** · Google · 2023-03-06

#### Problem

Design a system to crawl and copy all of Wikipedia using a distributed network of machines.

More specifically, suppose your server has access to a set of client machines. Your client machines can execute code you have written to access Wikipedia pages, download and parse their data, and write the results to a database.

Some questions you may want to consider as part of your solution are:

- How will you reach as many pages as possible?
- How can you keep track of pages that have already been visited?
- How will you deal with your client machines being blacklisted?
- How can you update your database when Wikipedia pages are added or updated?

### Problem #1171 <a id="p1171"></a>

**Easy** · Palantir · 2023-03-07

#### Problem

A typical American-style crossword puzzle grid is an N x N matrix with black and white squares, which obeys the following rules:

- Every white square must be part of an `"across"` word and a `"down"` word.
- No word can be fewer than three letters long.
- Every white square must be reachable from every other white square.
- The grid is rotationally symmetric (for example, the colors of the top left and bottom right squares must match).

Write a program to determine whether a given matrix qualifies as a crossword grid.

### Problem #1172 <a id="p1172"></a>

**Medium** · Google · 2023-03-08

#### Problem

Given pre-order and in-order traversals of a binary tree, write a function to reconstruct the tree.

For example, given the following preorder traversal:

`[a, b, d, e, c, f, g]`

And the following inorder traversal:

`[d, b, e, a, f, c, g]`

You should return the following tree:

```
    a
   / \
  b   c
 / \ / \
d  e f  g
```

### Problem #1173 <a id="p1173"></a>

**Hard** · Netflix · 2023-03-09

#### Problem

A Cartesian tree with sequence S is a binary tree defined by the following two properties:

- It is heap-ordered, so that each parent value is strictly less than that of its children.
- An in-order traversal of the tree produces nodes with values that correspond exactly to S.

For example, given the sequence `[3, 2, 6, 1, 9]`, the resulting Cartesian tree would be:

```
      1
    /   \   
  2       9
 / \
3   6
```

Given a sequence S, construct the corresponding Cartesian tree.

### Problem #1174 <a id="p1174"></a>

**Hard** · LinkedIn · 2023-03-10

#### Problem

Given a string, return whether it represents a number. Here are the different kinds of numbers:

- `"10"`, a positive integer
- `"-10"`, a negative integer
- `"10.1"`, a positive real number
- `"-10.1"`, a negative real number
- `"1e5"`, a number in scientific notation

And here are examples of non-numbers:

- `"a"`
- `"x 1"`
- `"a -2"`
- `"-"`

### Problem #1175 <a id="p1175"></a>

**Medium** · Facebook · 2023-03-11

#### Problem

Given a circular array, compute its maximum subarray sum in `O(n)` time. A subarray can be empty, and in this case the sum is 0.

For example, given `[8, -1, 3, 4]`, return 15 as we choose the numbers 3, 4, and 8 where the 8 is obtained from wrapping around.

Given `[-4, 5, 1, 0]`, return 6 as we choose the numbers 5 and 1.

### Problem #1176 <a id="p1176"></a>

**Easy** · Coursera · 2023-03-12

#### Problem

Given a 2D board of characters and a word, find if the word exists in the grid.

The word can be constructed from letters of sequentially adjacent cell, where `"adjacent"` cells are those horizontally or vertically neighboring. The same letter cell may not be used more than once.

For example, given the following board:

```
[
  ['A','B','C','E'],
  ['S','F','C','S'],
  ['A','D','E','E']
]
```

`exists(board, "ABCCED")` returns true, `exists(board, "SEE")` returns true, `exists(board, "ABCB")` returns false.

### Problem #1177 <a id="p1177"></a>

**Hard** · Google · 2023-03-14

#### Problem

Given an array of integers where every integer occurs three times except for one integer, which only occurs once, find and return the non-duplicated integer.

For example, given `[6, 1, 3, 3, 3, 6, 6]`, return 1. Given `[13, 19, 13, 13]`, return 19.

Do this in `O(N)` time and `O(1)` space.

### Problem #1178 <a id="p1178"></a>

**Easy** · Apple · 2023-03-15

#### Problem

A Collatz sequence in mathematics can be defined as follows. Starting with any positive integer:

- if n is even, the next number in the sequence is n / 2
- if n is odd, the next number in the sequence is 3n + 1

It is conjectured that every such sequence eventually reaches the number 1. Test this conjecture.

Bonus: What input n \<= 1000000 gives the longest sequence?

### Problem #1179 <a id="p1179"></a>

**Easy** · Facebook · 2023-03-16

#### Problem

Given a function f, and N return a debounced f of N milliseconds.

That is, as long as the debounced f continues to be invoked, f itself will not be called for N milliseconds.

### Problem #1180 <a id="p1180"></a>

**Easy** · Google · 2023-03-17

#### Problem

Given the head of a singly linked list, swap every two nodes and return its head.

For example, given 1 -> 2 -> 3 -> 4, return 2 -> 1 -> 4 -> 3.

### Problem #1181 <a id="p1181"></a>

**Medium** · Uber · 2023-03-18

#### Problem

Suppose an array sorted in ascending order is rotated at some pivot unknown to you beforehand. Find the minimum element in `O(log N)` time. You may assume the array does not contain duplicates.

For example, given `[5, 7, 10, 3, 4]`, return 3.

### Problem #1182 <a id="p1182"></a>

**Medium** · Etsy · 2023-03-19

#### Problem

Given an array of numbers N and an integer k, your task is to split N into k partitions such that the maximum sum of any partition is minimized. Return this sum.

For example, given N = `[5, 1, 2, 7, 3, 4]` and k = 3, you should return 8, since the optimal partition is `[5, 1, 2]`, `[7]`, `[3, 4]`.

### Problem #1183 <a id="p1183"></a>

**Easy** · Twitter · 2023-03-20

#### Problem

A strobogrammatic number is a positive number that appears the same after being rotated 180 degrees. For example, 16891 is strobogrammatic.

Create a program that finds all strobogrammatic numbers with N digits.

### Problem #1184 <a id="p1184"></a>

**Medium** · Spotify · 2023-03-21

#### Problem

You have access to ranked lists of songs for various users. Each song is represented as an integer, and more preferred songs appear earlier in each list. For example, the list `[4, 1, 7]` indicates that a user likes song 4 the best, followed by songs 1 and 7.

Given a set of these ranked lists, interleave them to create a playlist that satisfies everyone's priorities.

For example, suppose your input is `{[1, 7, 3], [2, 1, 6, 7, 9], [3, 9, 5]}`. In this case a satisfactory playlist could be `[2, 1, 6, 7, 3, 9, 5]`.

### Problem #1185 <a id="p1185"></a>

**Hard** · 2023-03-22

#### Problem

Find an efficient algorithm to find the smallest distance (measured in number of words) between any two given words in a string.

For example, given words `"hello"`, and `"world"` and a text content of `"dog cat hello cat dog dog hello cat world"`, return 1 because there's only one word `"cat"` in between the two words.

### Problem #1186 <a id="p1186"></a>

**Easy** · Facebook · 2023-03-23

#### Problem

Suppose you are given two lists of n points, one list p1, p2, ..., pn on the line y = 0 and the other list q1, q2, ..., qn on the line y = 1. Imagine a set of n line segments connecting each point pi to qi. Write an algorithm to determine how many pairs of the line segments intersect.

### Problem #1187 <a id="p1187"></a>

**Easy** · Google · 2023-03-24

#### Problem

You are given a starting state start, a list of transition probabilities for a Markov chain, and a number of steps `num_steps`. Run the Markov chain starting from start for `num_steps` and compute the number of times we visited each state.

For example, given the starting state a, number of steps 5000, and the following transition probabilities:

```
[
  ('a', 'a', 0.9),
  ('a', 'b', 0.075),
  ('a', 'c', 0.025),
  ('b', 'a', 0.15),
  ('b', 'b', 0.8),
  ('b', 'c', 0.05),
  ('c', 'a', 0.25),
  ('c', 'b', 0.25),
  ('c', 'c', 0.5)
]
```

One instance of running this Markov chain might produce `{ 'a': 3012, 'b': 1656, 'c': 332 }`.

### Problem #1188 <a id="p1188"></a>

**Medium** · Google · 2023-03-25

#### Problem

We can determine how `"out of order"` an array A is by counting the number of inversions it has. Two elements `A[i]` and `A[j]` form an inversion if `A[i]` > `A[j]` but i \< j. That is, a smaller element appears after a larger element.

Given an array, count the number of inversions it has. Do this faster than `O(N^2)` time.

You may assume each element in the array is distinct.

For example, a sorted list has zero inversions. The array `[2, 4, 1, 3, 5]` has three inversions: (2, 1), (4, 1), and (4, 3). The array `[5, 4, 3, 2, 1]` has ten inversions: every distinct pair forms an inversion.

### Problem #1189 <a id="p1189"></a>

**Easy** · Microsoft · 2023-03-26

#### Problem

Given an array of numbers and a number k, determine if there are three entries in the array which add up to the specified number k. For example, given `[20, 303, 3, 4, 25]` and k = 49, return true as 20 + 4 + 25 = 49.

### Problem #1190 <a id="p1190"></a>

**Medium** · Twitter · 2023-03-27

#### Problem

A network consists of nodes labeled 0 to N. You are given a list of edges (a, b, t), describing the time t it takes for a message to be sent from node a to node b. Whenever a node receives a message, it immediately passes the message on to a neighboring node, if possible.

Assuming all nodes are connected, determine how long it will take for every node to receive a message that begins at node 0.

For example, given N = 5, and the following edges:

```
edges = [
    (0, 1, 5),
    (0, 2, 3),
    (0, 5, 4),
    (1, 3, 8),
    (2, 3, 1),
    (3, 5, 10),
    (3, 4, 5)
]
```

You should return 9, because propagating the message from 0 -> 2 -> 3 -> 4 will take that much time.

### Problem #1191 <a id="p1191"></a>

**Medium** · Dropbox · 2023-03-28

#### Problem

Create an algorithm to efficiently compute the approximate median of a list of numbers.

More precisely, given an unordered list of N numbers, find an element whose rank is between N / 4 and 3 \* N / 4, with a high level of certainty, in less than `O(N)` time.

### Problem #1192 <a id="p1192"></a>

**Easy** · Pivotal · 2023-03-29

#### Problem

A step word is formed by taking a given word, adding a letter, and anagramming the result. For example, starting with the word `"APPLE"`, you can add an `"A"` and anagram to get `"APPEAL"`.

Given a dictionary of words and an input word, create a function that returns all valid step words.

### Problem #1193 <a id="p1193"></a>

**Hard** · Two Sigma · 2023-03-30

#### Problem

Ghost is a two-person word game where players alternate appending letters to a word. The first person who spells out a word, or creates a prefix for which there is no possible continuation, loses. Here is a sample game:

- Player 1: g
- Player 2: h
- Player 1: o
- Player 2: s
- Player 1: t \[loses\]

Given a dictionary of words, determine the letters the first player should start with, such that with optimal play they cannot lose.

For example, if the dictionary is `["cat", "calf", "dog", "bear"]`, the only winning start letter would be b.

### Problem #1194 <a id="p1194"></a>

**Hard** · Dropbox · 2023-03-31

#### Problem

A Boolean formula can be said to be satisfiable if there is a way to assign truth values to each variable such that the entire formula evaluates to true.

For example, suppose we have the following formula, where the symbol ¬ is used to denote negation:

(¬c OR b) AND (b OR c) AND (¬b OR c) AND (¬c OR ¬a)

One way to satisfy this formula would be to let a = False, b = True, and c = True.

This type of formula, with AND statements joining tuples containing exactly one OR, is known as 2-CNF.

Given a 2-CNF formula, find a way to assign truth values to satisfy it, or return False if this is impossible.

### Problem #1195 <a id="p1195"></a>

**Medium** · Amazon · 2023-04-01

#### Problem

Given a string, find the length of the smallest window that contains every distinct character. Characters may appear more than once in the window.

For example, given `"jiujitsu"`, you should return 5, corresponding to the final five letters.

### Problem #1196 <a id="p1196"></a>

**Hard** · Dropbox · 2023-04-02

#### Problem

Create a data structure that performs all the following operations in `O(1)` time:

- plus: Add a key with value 1. If the key already exists, increment its value by one.
- minus: Decrement the value of a key. If the key's value is currently 1, remove it.
- `get_max`: Return a key with the highest value.
- `get_min`: Return a key with the lowest value.

### Problem #1197 <a id="p1197"></a>

**Easy** · Amazon · 2023-04-03

#### Problem

Implement a stack that has the following methods:

- `push(val)`, which pushes an element onto the stack
- `pop()`, which pops off and returns the topmost element of the stack. If there are no elements in the stack, then it should throw an error or return null.
- `max()`, which returns the maximum value in the stack currently. If there are no elements in the stack, then it should throw an error or return null.

Each method should run in constant time.

### Problem #1198 <a id="p1198"></a>

**Medium** · Snapchat · 2023-04-04

#### Problem

You are given an array of length N, where each element i represents the number of ways we can produce i units of change. For example, `[1, 0, 1, 1, 2]` would indicate that there is only one way to make 0, 2, or 3 units, and two ways of making 4 units.

Given such an array, determine the denominations that must be in use. In the case above, for example, there must be coins with value 2, 3, and 4.

### Problem #1199 <a id="p1199"></a>

**Medium** · Apple · 2023-04-05

#### Problem

Implement a job scheduler which takes in a function f and an integer n, and calls f after n milliseconds.

### Problem #1200 <a id="p1200"></a>

**Easy** · Facebook · 2023-04-06

#### Problem

Given a 32-bit integer, return the number with its bits reversed.

For example, given the binary number 1111 0000 1111 0000 1111 0000 1111 0000, return 0000 1111 0000 1111 0000 1111 0000 1111.

### Problem #1201 <a id="p1201"></a>

**Medium** · Yahoo · 2023-04-07

#### Problem

Write an algorithm that computes the reversal of a directed graph. For example, if a graph consists of A -> B -> C, it should become A \<- B \<- C.

### Problem #1202 <a id="p1202"></a>

**Hard** · Spotify · 2023-04-08

#### Problem

There are N couples sitting in a row of length 2 \* N. They are currently ordered randomly, but would like to rearrange themselves so that each couple's partners can sit side by side.

What is the minimum number of swaps necessary for this to happen?

### Problem #1203 <a id="p1203"></a>

**Hard** · Airbnb · 2023-04-09

#### Problem

You come across a dictionary of sorted words in a language you've never seen before. Write a program that returns the correct order of letters in this language.

For example, given `['xww', 'wxyz', 'wxyw', 'ywx', 'ywz']`, you should return `['x', 'z', 'w', 'y']`.

### Problem #1204 <a id="p1204"></a>

**Easy** · Amazon · 2023-04-10

#### Problem

Given n numbers, find the greatest common denominator between them.

For example, given the numbers `[42, 56, 14]`, return 14.

### Problem #1205 <a id="p1205"></a>

**Easy** · Microsoft · 2023-04-11

#### Problem

Let's represent an integer in a linked list format by having each node represent a digit in the number. The nodes make up the number in reversed order.

For example, the following linked list:

1 -> 2 -> 3 -> 4 -> 5

is the number 54321.

Given two linked lists in this format, return their sum in the same linked list format.

For example, given

9 -> 9

5 -> 2

return 124 (99 + 25) as:

4 -> 2 -> 1

### Problem #1206 <a id="p1206"></a>

**Hard** · Uber · 2023-04-12

#### Problem

A rule looks like this:

A NE B

This means this means point A is located northeast of point B.

A SW C

means that point A is southwest of C.

Given a list of rules, check if the sum of the rules validate. For example:

```
A N B
B NE C
C N A
```

does not validate, since A cannot be both north and south of C.

```
A NW B
A N B
```

is considered valid.

### Problem #1207 <a id="p1207"></a>

**Medium** · Google · 2023-04-13

#### Problem

Given a singly linked list and an integer k, remove the kth last element from the list. k is guaranteed to be smaller than the length of the list.

The list is very long, so making more than one pass is prohibitively expensive.

Do this in constant space and in one pass.

### Problem #1208 <a id="p1208"></a>

**Medium** · Uber · 2023-04-14

#### Problem

Write a program that determines the smallest number of perfect squares that sum up to N.

Here are a few examples:

- Given N = 4, return 1 (4)
- Given N = 17, return 2 (16 + 1)
- Given N = 18, return 2 (9 + 9)

### Problem #1209 <a id="p1209"></a>

**Medium** · Google · 2023-04-15

#### Problem

Given a set of distinct positive integers, find the largest subset such that every pair of elements in the subset (i, j) satisfies either i % j = 0 or j % i = 0.

For example, given the set `[3, 5, 10, 20, 21]`, you should return `[5, 10, 20]`. Given `[1, 3, 6, 24]`, return `[1, 3, 6, 24]`.

### Problem #1210 <a id="p1210"></a>

**Easy** · Oracle · 2023-04-16

#### Problem

Given a binary search tree, find the floor and ceiling of a given integer. The floor is the highest element in the tree less than or equal to an integer, while the ceiling is the lowest element in the tree greater than or equal to an integer.

If either value does not exist, return None.

### Problem #1211 <a id="p1211"></a>

**Medium** · Google · 2023-04-17

#### Problem

Given a stack of N elements, interleave the first half of the stack with the second half reversed using only one other queue. This should be done in-place.

Recall that you can only push or pop from a stack, and enqueue or dequeue from a queue.

For example, if the stack is `[1, 2, 3, 4, 5]`, it should become `[1, 5, 2, 4, 3]`. If the stack is `[1, 2, 3, 4]`, it should become `[1, 4, 2, 3]`.

Hint: Try working backwards from the end state.

### Problem #1212 <a id="p1212"></a>

**Medium** · Amazon · 2023-04-18

#### Problem

Implement a bit array.

A bit array is a space efficient array that holds a value of 1 or 0 at each index.

- `init(size)`: initialize the array with size
- `set(i, val)`: updates index at i with val where val is either 1 or 0.
- `get(i)`: gets the value at index i.

### Problem #1213 <a id="p1213"></a>

**Hard** · Amazon · 2023-04-19

#### Problem

The stable marriage problem is defined as follows:

Suppose you have N men and N women, and each person has ranked their prospective opposite-sex partners in order of preference.

For example, if N = 3, the input could be something like this:

```
guy_preferences = {
    'andrew': ['caroline', 'abigail', 'betty'],
    'bill': ['caroline', 'betty', 'abigail'],
    'chester': ['betty', 'caroline', 'abigail'],
}
```

```
gal_preferences = {
    'abigail': ['andrew', 'bill', 'chester'],
    'betty': ['bill', 'andrew', 'chester'],
    'caroline': ['bill', 'chester', 'andrew']
}
```

Write an algorithm that pairs the men and women together in such a way that no two people of opposite sex would both rather be with each other than with their current partners.

### Problem #1214 <a id="p1214"></a>

**Medium** · Facebook · 2023-04-20

#### Problem

Given an unordered list of flights taken by someone, each represented as (origin, destination) pairs, and a starting airport, compute the person's itinerary. If no such itinerary exists, return null. If there are multiple possible itineraries, return the lexicographically smallest one. All flights must be used in the itinerary.

For example, given the list of flights `[('SFO', 'HKO'), ('YYZ', 'SFO'), ('YUL', 'YYZ'), ('HKO', 'ORD')]` and starting airport `'YUL'`, you should return the list `['YUL', 'YYZ', 'SFO', 'HKO', 'ORD']`.

Given the list of flights `[('SFO', 'COM'), ('COM', 'YYZ')]` and starting airport `'COM'`, you should return null.

Given the list of flights `[('A', 'B'), ('A', 'C'), ('B', 'C'), ('C', 'A')]` and starting airport `'A'`, you should return the list `['A', 'B', 'C', 'A', 'C']` even though `['A', 'C', 'A', 'B', 'C']` is also a valid itinerary. However, the first one is lexicographically smaller.

### Problem #1215 <a id="p1215"></a>

**Easy** · PagerDuty · 2023-04-21

#### Problem

Given a positive integer N, find the smallest number of steps it will take to reach 1.

There are two kinds of permitted steps:

- You may decrement N to N - 1.
- If a \* b = N, you may decrement N to the larger of a and b.

For example, given 100, you can reach 1 in five steps with the following route: 100 -> 10 -> 9 -> 3 -> 2 -> 1.

### Problem #1216 <a id="p1216"></a>

**Medium** · Google · 2023-04-22

#### Problem

You are given an N by M 2D matrix of lowercase letters. Determine the minimum number of columns that can be removed to ensure that each row is ordered from top to bottom lexicographically. That is, the letter at each column is lexicographically later as you go down each row. It does not matter whether each row itself is ordered lexicographically.

For example, given the following table:

```
cba
daf
ghi
```

This is not ordered because of the a in the center. We can remove the second column to make it ordered:

```
ca
df
gi
```

So your function should return 1, since we only needed to remove 1 column.

As another example, given the following table:

abcdef

Your function should return 0, since the rows are already ordered (there's only one row).

As another example, given the following table:

```
zyx
wvu
tsr
```

Your function should return 3, since we would need to remove all the columns to order it.

### Problem #1217 <a id="p1217"></a>

**Easy** · Pivotal · 2023-04-23

#### Problem

Write an algorithm that finds the total number of set bits in all integers between 1 and N.

### Problem #1218 <a id="p1218"></a>

**Hard** · Microsoft · 2023-04-24

#### Problem

Implement 3 stacks using a single list:

```
class Stack:
    def __init__(self):
        self.list = []
```

```
def pop(self, stack_number):
    pass
```

```
def push(self, item, stack_number):
    pass
```

### Problem #1219 <a id="p1219"></a>

**Easy** · Dropbox · 2023-04-25

#### Problem

What does the below code snippet print out? How can we fix the anonymous functions to behave as we'd expect?

```
functions = []
for i in range(10):
    functions.append(lambda : i)
```

```
for f in functions:
    print(f())
```

### Problem #1220 <a id="p1220"></a>

**Easy** · Salesforce · 2023-04-26

#### Problem

Write a program to merge two binary trees. Each node in the new tree should hold a value equal to the sum of the values of the corresponding nodes of the input trees.

If only one input tree has a node in a given position, the corresponding node in the new tree should match that input node.

### Problem #1221 <a id="p1221"></a>

**Easy** · Apple · 2023-04-27

#### Problem

Implement the function `fib(n)`, which returns the nth number in the Fibonacci sequence, using only `O(1)` space.

### Problem #1222 <a id="p1222"></a>

**Easy** · Apple · 2023-04-28

#### Problem

Given the root of a binary tree, find the most frequent subtree sum. The subtree sum of a node is the sum of all values under a node, including the node itself.

For example, given the following tree:

```
  5
 / \
2  -5
```

Return 2 as it occurs twice: once as the left leaf, and once as the sum of 2 + 5 - 5.

### Problem #1223 <a id="p1223"></a>

**Hard** · LinkedIn · 2023-04-29

#### Problem

You are given a binary tree in a peculiar string representation. Each node is written in the form (lr), where l corresponds to the left child and r corresponds to the right child.

If either l or r is null, it will be represented as a zero. Otherwise, it will be represented by a new (lr) pair.

Here are a few examples:

- A root node with no children: (00)
- A root node with two children: ((00)(00))
- An unbalanced tree with three consecutive left children: ((((00)0)0)0)

Given this representation, determine the depth of the tree.

### Problem #1224 <a id="p1224"></a>

**Medium** · Google · 2023-04-30

#### Problem

Given a binary search tree and a range `[a, b]` (inclusive), return the sum of the elements of the binary search tree within the range.

For example, given the following tree:

```
    5
   / \
  3   8
 / \ / \
2  4 6  10
```

and the range `[4, 9]`, return 23 (5 + 4 + 6 + 8).

### Problem #1225 <a id="p1225"></a>

**Medium** · Google · 2023-05-01

#### Problem

Given a string of parentheses, write a function to compute the minimum number of parentheses to be removed to make the string valid (i.e. each open parenthesis is eventually closed).

For example, given the string `"()())()"`, you should return 1. Given the string `")("`, you should return 2, since we must remove all of them.

### Problem #1226 <a id="p1226"></a>

**Medium** · Yext · 2023-05-02

#### Problem

Two nodes in a binary tree can be called cousins if they are on the same level of the tree but have different parents. For example, in the following diagram 4 and 6 are cousins.

```
    1
   / \
  2   3
 / \   \
4   5   6
```

Given a binary tree and a particular node, find all cousins of that node.

### Problem #1227 <a id="p1227"></a>

**Medium** · Google · 2023-05-03

#### Problem

Given an undirected graph represented as an adjacency matrix and an integer k, write a function to determine whether each vertex in the graph can be colored such that no two adjacent vertices share the same color using at most k colors.

### Problem #1228 <a id="p1228"></a>

**Easy** · Snapchat · 2023-05-04

#### Problem

Given a list of possibly overlapping intervals, return a new list of intervals where all overlapping intervals have been merged.

The input list is not necessarily ordered in any way.

For example, given `[(1, 3), (5, 8), (4, 10), (20, 25)]`, you should return `[(1, 3), (4, 10), (20, 25)]`.

### Problem #1229 <a id="p1229"></a>

**Easy** · Google · 2023-05-05

#### Problem

A unival tree (which stands for `"universal value"`) is a tree where all nodes under it have the same value.

Given the root to a binary tree, count the number of unival subtrees.

For example, the following tree has 5 unival subtrees:

```
  0
 / \
1   0
   / \
  1   0
 / \
1   1
```

### Problem #1230 <a id="p1230"></a>

**Medium** · Square · 2023-05-06

#### Problem

Given a list of words, return the shortest unique prefix of each word. For example, given the list:

- dog
- cat
- apple
- apricot
- fish

Return the list:

- d
- c
- app
- apr
- f

### Problem #1231 <a id="p1231"></a>

**Medium** · Google · 2023-05-07

#### Problem

The area of a circle is defined as πr^2. Estimate π to 3 decimal places using a Monte Carlo method.

Hint: The basic equation of a circle is x2 + y2 = r2.

### Problem #1232 <a id="p1232"></a>

**Medium** · Uber · 2023-05-08

#### Problem

You are given a 2-d matrix where each cell consists of either /, \\, or an empty space. Write an algorithm that determines into how many regions the slashes divide the space.

For example, suppose the input for a three-by-six grid is the following:

```
\    /
 \  /
  \/
```

Considering the edges of the matrix as boundaries, this divides the grid into three triangles, so you should return 3.

### Problem #1233 <a id="p1233"></a>

**Hard** · Google · 2023-05-09

#### Problem

Given an array of strictly the characters `'R'`, `'G'`, and `'B'`, segregate the values of the array so that all the Rs come first, the Gs come second, and the Bs come last. You can only swap elements of the array.

Do this in linear time and in-place.

For example, given the array `['G', 'B', 'R', 'R', 'B', 'R', 'G']`, it should become `['R', 'R', 'R', 'G', 'G', 'B', 'B']`.

### Problem #1234 <a id="p1234"></a>

**Easy** · Facebook · 2023-05-10

#### Problem

On a mysterious island there are creatures known as Quxes which come in three colors: red, green, and blue. One power of the Qux is that if two of them are standing next to each other, they can transform into a single creature of the third color.

Given N Quxes standing in a line, determine the smallest number of them remaining after any possible sequence of such transformations.

For example, given the input `['R', 'G', 'B', 'G', 'B']`, it is possible to end up with a single Qux through the following steps:

```
        Arrangement       |   Change
----------------------------------------
['R', 'G', 'B', 'G', 'B'] | (R, G) -> B
['B', 'B', 'G', 'B']      | (B, G) -> R
['B', 'R', 'B']           | (R, B) -> G
['B', 'G']                | (B, G) -> R
['R']                     |
```

### Problem #1235 <a id="p1235"></a>

**Medium** · Google · 2023-05-11

#### Problem

Given the root to a binary tree, implement `serialize(root)`, which serializes the tree into a string, and deserialize(s), which deserializes the string back into the tree.

For example, given the following Node class

```
class Node:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
```

The following test should pass:

```
node = Node('root', Node('left', Node('left.left')), Node('right'))
assert deserialize(serialize(node)).left.left.val == 'left.left'
```

### Problem #1236 <a id="p1236"></a>

**Medium** · Spotify · 2023-05-12

#### Problem

Write a function, `throw_dice(N, faces, total)`, that determines how many ways it is possible to throw N dice with some number of faces each to get a specific total.

For example, `throw_dice(3, 6, 7)` should equal 15.

### Problem #1237 <a id="p1237"></a>

**Easy** · Twitter · 2023-05-13

#### Problem

A classroom consists of N students, whose friendships can be represented in an adjacency list. For example, the following descibes a situation where 0 is friends with 1 and 2, 3 is friends with 6, and so on.

```
{0: [1, 2],
 1: [0, 5],
 2: [0],
 3: [6],
 4: [],
 5: [1],
 6: [3]}
```

Each student can be placed in a friend group, which can be defined as the transitive closure of that student's friendship relations. In other words, this is the smallest set such that no student in the group has any friends outside this group. For the example above, the friend groups would be `{0, 1, 2, 5}`, `{3, 6}`, `{4}`.

Given a friendship list such as the one above, determine the number of friend groups in the class.

### Problem #1238 <a id="p1238"></a>

**Medium** · Quora · 2023-05-14

#### Problem

Given a string, find the palindrome that can be made by inserting the fewest number of characters as possible anywhere in the word. If there is more than one palindrome of minimum length that can be made, return the lexicographically earliest one (the first one alphabetically).

For example, given the string `"race"`, you should return `"ecarace"`, since we can add three letters to it (which is the smallest amount to make a palindrome). There are seven other palindromes that can be made from `"race"` by adding three letters, but `"ecarace"` comes first alphabetically.

As another example, given the string `"google"`, you should return `"elgoogle"`.

### Problem #1239 <a id="p1239"></a>

**Medium** · ContextLogic · 2023-05-15

#### Problem

Implement division of two positive integers without using the division, multiplication, or modulus operators. Return the quotient as an integer, ignoring the remainder.

### Problem #1240 <a id="p1240"></a>

**Medium** · Apple · 2023-05-16

#### Problem

Gray code is a binary code where each successive value differ in only one bit, as well as when wrapping around. Gray code is common in hardware so that we don't see temporary spurious values during transitions.

Given a number of bits n, generate a possible gray code for it.

For example, for n = 2, one gray code would be `[00, 01, 11, 10]`.

### Problem #1241 <a id="p1241"></a>

**Medium** · Pinterest · 2023-05-17

#### Problem

At a party, there is a single person who everyone knows, but who does not know anyone in return (the `"celebrity"`). To help figure out who this is, you have access to an `O(1)` method called `knows(a, b)`, which returns True if person a knows person b, else False.

Given a list of N people and the above operation, find a way to identify the celebrity in `O(N)` time.

### Problem #1242 <a id="p1242"></a>

**Medium** · Facebook · 2023-05-18

#### Problem

Given a start word, an end word, and a dictionary of valid words, find the shortest transformation sequence from start to end such that only one letter is changed at each step of the sequence, and each transformed word exists in the dictionary. If there is no possible transformation, return null. Each word in the dictionary have the same length as start and end and is lowercase.

For example, given start = `"dog"`, end = `"cat"`, and dictionary = `{"dot", "dop", "dat", "cat"}`, return `["dog", "dot", "dat", "cat"]`.

Given start = `"dog"`, end = `"cat"`, and dictionary = `{"dot", "tod", "dat", "dar"}`, return null as there is no possible transformation from dog to cat.

### Problem #1243 <a id="p1243"></a>

**Easy** · Google · 2023-05-19

#### Problem

Given two strings A and B, return whether or not A can be shifted some number of times to get B.

For example, if A is abcde and B is cdeab, return true. If A is abc and B is acb, return false.

### Problem #1244 <a id="p1244"></a>

**Medium** · Glassdoor · 2023-05-20

#### Problem

An imminent hurricane threatens the coastal town of Codeville. If at most two people can fit in a rescue boat, and the maximum weight limit for a given boat is k, determine how many boats will be needed to save everyone.

For example, given a population with weights `[100, 200, 150, 80]` and a boat limit of 200, the smallest number of boats required will be three.

### Problem #1245 <a id="p1245"></a>

**Medium** · Palantir · 2023-05-21

#### Problem

Write an algorithm to justify text. Given a sequence of words and an integer line length k, return a list of strings which represents each line, fully justified.

More specifically, you should have as many words as possible in each line. There should be at least one space between each word. Pad extra spaces when necessary so that each line has exactly length k. Spaces should be distributed as equally as possible, with the extra spaces, if any, distributed starting from the left.

If you can only fit one word on a line, then you should pad the right-hand side with spaces.

Each word is guaranteed not to be longer than k.

For example, given the list of words `["the", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]` and k = 16, you should return the following:

```
["the  quick brown", # 1 extra space on the left
"fox  jumps  over", # 2 extra spaces distributed evenly
"the   lazy   dog"] # 4 extra spaces distributed evenly
```

### Problem #1246 <a id="p1246"></a>

**Hard** · Microsoft · 2023-05-22

#### Problem

Recall that the minimum spanning tree is the subset of edges of a tree that connect all its vertices with the smallest possible total edge weight. Given an undirected graph with weighted edges, compute the maximum weight spanning tree.

### Problem #1247 <a id="p1247"></a>

**Easy** · Google · 2023-05-23

#### Problem

Given a sorted list of integers, square the elements and give the output in sorted order.

For example, given `[-9, -2, 0, 2, 3]`, return `[0, 4, 4, 9, 81]`.

### Problem #1248 <a id="p1248"></a>

**Medium** · Dropbox · 2023-05-24

#### Problem

Given a string s and a list of words words, where each word is the same length, find all starting indices of substrings in s that is a concatenation of every word in words exactly once.

For example, given s = `"dogcatcatcodecatdog"` and words = `["cat", "dog"]`, return `[0, 13]`, since `"dogcat"` starts at index 0 and `"catdog"` starts at index 13.

Given s = `"barfoobazbitbyte"` and words = `["dog", "cat"]`, return \[\] since there are no substrings composed of `"dog"` and `"cat"` in s.

The order of the indices does not matter.

### Problem #1249 <a id="p1249"></a>

**Medium** · LinkedIn · 2023-05-25

#### Problem

Determine whether a tree is a valid binary search tree.

A binary search tree is a tree with two children, left and right, and satisfies the constraint that the key in the left child must be less than or equal to the root and the key in the right child must be greater than or equal to the root.

### Problem #1250 <a id="p1250"></a>

**Easy** · Amazon · 2023-05-26

#### Problem

Given a string, determine whether any permutation of it is a palindrome.

For example, carrace should return true, since it can be rearranged to form racecar, which is a palindrome. daily should return false, since there's no rearrangement that can form a palindrome.

### Problem #1251 <a id="p1251"></a>

**Hard** · Facebook · 2023-05-27

#### Problem

Implement regular expression matching with the following special characters:

- . (period) which matches any single character
- \* (asterisk) which matches zero or more of the preceding element

That is, implement a function that takes in a string and a valid regular expression and returns whether or not the string matches the regular expression.

For example, given the regular expression `"ra."` and the string `"ray"`, your function should return true. The same regular expression on the string `"raymond"` should return false.

Given the regular expression `".*at"` and the string `"chat"`, your function should return true. The same regular expression on the string `"chats"` should return false.

### Problem #1252 <a id="p1252"></a>

**Easy** · Microsoft · 2023-05-28

#### Problem

Given a clock time in hh:mm format, determine, to the nearest degree, the angle between the hour and the minute hands.

Bonus: When, during the course of a day, will the angle be zero?

### Problem #1253 <a id="p1253"></a>

**Easy** · Facebook · 2023-05-29

#### Problem

Given a string of round, curly, and square open and closing brackets, return whether the brackets are balanced (well-formed).

For example, given the string `"([])[]({})"`, you should return true.

Given the string `"([)]"` or `"((()"`, you should return false.

### Problem #1254 <a id="p1254"></a>

**Hard** · Microsoft · 2023-05-30

#### Problem

Let X be a set of n intervals on the real line. We say that a set of points P `"stabs"` X if every interval in X contains at least one point in P. Compute the smallest set of points that stabs X.

For example, given the intervals `[(1, 4), (4, 5), (7, 9), (9, 12)]`, you should return `[4, 9]`.

### Problem #1255 <a id="p1255"></a>

**Easy** · Google · 2023-05-31

#### Problem

Given a list of numbers and a number k, return whether any two numbers from the list add up to k.

For example, given `[10, 15, 3, 7]` and k of 17, return true since 10 + 7 is 17.

Bonus: Can you do this in one pass?

### Problem #1256 <a id="p1256"></a>

**Medium** · Amazon · 2023-06-01

#### Problem

Given an array of numbers, find the maximum sum of any contiguous subarray of the array.

For example, given the array `[34, -50, 42, 14, -5, 86]`, the maximum sum would be 137, since we would take elements 42, 14, -5, and 86.

Given the array `[-5, -1, -8, -9]`, the maximum sum would be 0, since we would not take any elements.

Do this in `O(N)` time.

### Problem #1257 <a id="p1257"></a>

**Medium** · Spotify · 2023-06-02

#### Problem

You are the technical director of WSPT radio, serving listeners nationwide. For simplicity's sake we can consider each listener to live along a horizontal line stretching from 0 (west) to 1000 (east).

Given a list of N listeners, and a list of M radio towers, each placed at various locations along this line, determine what the minimum broadcast range would have to be in order for each listener's home to be covered.

For example, suppose listeners = `[1, 5, 11, 20]`, and towers = `[4, 8, 15]`. In this case the minimum range would be 5, since that would be required for the tower at position 15 to reach the listener at position 20.

### Problem #1258 <a id="p1258"></a>

**Medium** · Google · 2023-06-03

#### Problem

Given an iterator with methods `next()` and `hasNext()`, create a wrapper iterator, PeekableInterface, which also implements `peek()`. peek shows the next element that would be returned on `next()`.

Here is the interface:

```
class PeekableInterface(object):
    def __init__(self, iterator):
        pass
```

```
def peek(self):
    pass
```

```
def next(self):
    pass
```

```
def hasNext(self):
    pass
```

### Problem #1259 <a id="p1259"></a>

**Medium** · LinkedIn · 2023-06-04

#### Problem

You are given a string consisting of the letters x and y, such as xyxxxyxyy. In addition, you have an operation called flip, which changes a single x to y or vice versa.

Determine how many times you would need to apply this operation to ensure that all x's come before all y's. In the preceding example, it suffices to flip the second and sixth characters, so you should return 2.

### Problem #1261 <a id="p1261"></a>

**Hard** · Airbnb · 2023-06-06

#### Problem

Given a list of words, find all pairs of unique indices such that the concatenation of the two words is a palindrome.

For example, given the list `["code", "edoc", "da", "d"]`, return `[(0, 1), (1, 0), (2, 3)]`.

### Problem #1262 <a id="p1262"></a>

**Medium** · Google · 2023-06-07

#### Problem

What will this code print out?

```
def make_functions():
    flist = []
```

```
for i in [1, 2, 3]:
    def print_i():
        print(i)
    flist.append(print_i)
```

```
return flist
```

```
functions = make_functions()
for f in functions:
    f()
```

How can we make it print out what we apparently want?

### Problem #1263 <a id="p1263"></a>

**Medium** · Facebook · 2023-06-08

#### Problem

Given a binary tree, return all paths from the root to leaves.

For example, given the tree:

```
  1
 / \
2   3
   / \
  4   5
```

Return `[[1, 2], [1, 3, 4], [1, 3, 5]]`.

### Problem #1264 <a id="p1264"></a>

**Medium** · Google · 2023-06-09

#### Problem

Given an N by M matrix consisting only of 1's and 0's, find the largest rectangle containing only 1's and return its area.

For example, given the following matrix:

```
[[1, 0, 0, 0],
 [1, 0, 1, 1],
 [1, 0, 1, 1],
 [0, 1, 0, 0]]
```

Return 4.

### Problem #1265 <a id="p1265"></a>

**Medium** · Microsoft · 2023-06-10

#### Problem

Given a dictionary of words and a string made up of those words (no spaces), return the original sentence in a list. If there is more than one possible reconstruction, return any of them. If there is no possible reconstruction, then return null.

For example, given the set of words `'quick'`, `'brown'`, `'the'`, `'fox'`, and the string `"thequickbrownfox"`, you should return `['the', 'quick', 'brown', 'fox']`.

Given the set of words `'bed'`, `'bath'`, `'bedbath'`, `'and'`, `'beyond'`, and the string `"bedbathandbeyond"`, return either `['bed', 'bath', 'and', 'beyond]` or `['bedbath', 'and', 'beyond']`.

### Problem #1266 <a id="p1266"></a>

**Medium** · Twitter · 2023-06-11

#### Problem

Given a list of numbers, create an algorithm that arranges them in order to form the largest possible integer. For example, given `[10, 7, 76, 415]`, you should return 77641510.

### Problem #1267 <a id="p1267"></a>

**Hard** · Goldman Sachs · 2023-06-12

#### Problem

Given a list of numbers L, implement a method `sum(i, j)` which returns the sum from the sublist `L[i:j]` (including i, excluding j).

For example, given L = `[1, 2, 3, 4, 5]`, `sum(1, 3)` should return `sum([2, 3])`, which is 5.

You can assume that you can do some pre-processing. `sum()` should be optimized over the pre-processing step.

### Problem #1268 <a id="p1268"></a>

**Medium** · Facebook · 2023-06-13

#### Problem

Given three 32-bit integers x, y, and b, return x if b is 1 and y if b is 0, using only mathematical or bit operations. You can assume b can only be 1 or 0.

### Problem #1269 <a id="p1269"></a>

**Medium** · Facebook · 2023-06-14

#### Problem

Given an N by N matrix, rotate it by 90 degrees clockwise.

For example, given the following matrix:

```
[[1, 2, 3],
 [4, 5, 6],
 [7, 8, 9]]
```

you should return:

```
[[7, 4, 1],
 [8, 5, 2],
 [9, 6, 3]]
```

Follow-up: What if you couldn't use any extra space?

### Problem #1270 <a id="p1270"></a>

**Hard** · Google · 2023-06-15

#### Problem

Given a word W and a string S, find all starting indices in S which are anagrams of W.

For example, given that W is `"ab"`, and S is `"abxaba"`, return 0, 3, and 4.

### Problem #1271 <a id="p1271"></a>

**Easy** · Two Sigma · 2023-06-16

#### Problem

Using a function `rand7()` that returns an integer from 1 to 7 (inclusive) with uniform probability, implement a function `rand5()` that returns an integer from 1 to 5 (inclusive).

### Problem #1272 <a id="p1272"></a>

**Easy** · Google · 2023-06-17

#### Problem

In linear algebra, a Toeplitz matrix is one in which the elements on any given diagonal from top left to bottom right are identical.

Here is an example:

```
1 2 3 4 8
5 1 2 3 4
4 5 1 2 3
7 4 5 1 2
```

Write a program to determine whether a given input is a Toeplitz matrix.

### Problem #1273 <a id="p1273"></a>

**Easy** · Google · 2023-06-18

#### Problem

Implement a PrefixMapSum class with the following methods:

- `insert(key: str, value: int)`: Set a given key's value in the map. If the key already exists, overwrite the value.
- `sum(prefix: str)`: Return the sum of all values of keys that begin with a given prefix.

For example, you should be able to run the following code:

```
mapsum.insert("columnar", 3)
assert mapsum.sum("col") == 3
```

```
mapsum.insert("column", 2)
assert mapsum.sum("col") == 5
```

### Problem #1274 <a id="p1274"></a>

**Medium** · BufferBox · 2023-06-19

#### Problem

Given a binary tree where all nodes are either 0 or 1, prune the tree so that subtrees containing all 0s are removed.

For example, given the following tree:

```
  0
 / \
1   0
   / \
  1   0
 / \
0   0
```

should be pruned to:

```
  0
 / \
1   0
   /
  1
```

We do not remove the tree at the root or its left child because it still has a 1 as a descendant.

### Problem #1275 <a id="p1275"></a>

**Hard** · Amazon · 2023-06-20

#### Problem

Given a string, find the longest palindromic contiguous substring. If there are more than one with the maximum length, return any one.

For example, the longest palindromic substring of `"aabcdcb"` is `"bcdcb"`. The longest palindromic substring of `"bananas"` is `"anana"`.

### Problem #1276 <a id="p1276"></a>

**Medium** · Facebook · 2023-06-21

#### Problem

Given a multiset of integers, return whether it can be partitioned into two subsets whose sums are the same.

For example, given the multiset `{15, 5, 20, 10, 35, 15, 10}`, it would return true, since we can split it up into `{15, 5, 10, 15, 10}` and `{20, 35}`, which both add up to 55.

Given the multiset `{15, 5, 20, 10, 35}`, it would return false, since we can't split it up into two subsets that add up to the same sum.

### Problem #1277 <a id="p1277"></a>

**Medium** · Squarespace · 2023-06-22

#### Problem

Write a function, `add_subtract`, which alternately adds and subtracts curried arguments. Here are some sample operations:

`add_subtract(7)` -> 7

`add_subtract(1)(2)(3)` -> 1 + 2 - 3 -> 0

`add_subtract(-5)(10)(3)(9)` -> -5 + 10 - 3 + 9 -> 11

### Problem #1278 <a id="p1278"></a>

**Hard** · Dropbox · 2023-06-23

#### Problem

Sudoku is a puzzle where you're given a partially-filled 9 by 9 grid with digits. The objective is to fill the grid with the constraint that every row, column, and box (3 by 3 subgrid) must contain all of the digits from 1 to 9.

Implement an efficient sudoku solver.

### Problem #1279 <a id="p1279"></a>

**Medium** · Amazon · 2023-06-24

#### Problem

At a popular bar, each customer has a set of favorite drinks, and will happily accept any drink among this set. For example, in the following situation, customer 0 will be satisfied with drinks 0, 1, 3, or 6.

```
preferences = {
    0: [0, 1, 3, 6],
    1: [1, 4, 7],
    2: [2, 4, 7, 5],
    3: [3, 2, 5],
    4: [5, 8]
}
```

A lazy bartender working at this bar is trying to reduce his effort by limiting the drink recipes he must memorize. Given a dictionary input such as the one above, return the fewest number of drinks he must learn in order to satisfy all customers.

For the input above, the answer would be 2, as drinks 1 and 5 will satisfy everyone.

### Problem #1280 <a id="p1280"></a>

**Medium** · Cisco · 2023-06-25

#### Problem

Given an unsigned 8-bit integer, swap its even and odd bits. The 1st and 2nd bit should be swapped, the 3rd and 4th bit should be swapped, and so on.

For example, 10101010 should be 01010101. 11100010 should be 11010001.

Bonus: Can you do this in one line?

### Problem #1281 <a id="p1281"></a>

**Easy** · Google · 2023-06-26

#### Problem

Given two rectangles on a 2D graph, return the area of their intersection. If the rectangles don't intersect, return 0.

For example, given the following rectangles:

```
{
    "top_left": (1, 4),
    "dimensions": (3, 3) # width, height
}
```

and

```
{
    "top_left": (0, 5),
    "dimensions": (4, 3) # width, height
}
```

return 6.

### Problem #1282 <a id="p1282"></a>

**Easy** · Microsoft · 2023-06-27

#### Problem

A number is considered perfect if its digits sum up to exactly 10.

Given a positive integer n, return the n-th perfect number.

For example, given 1, you should return 19. Given 2, you should return 28.

### Problem #1283 <a id="p1283"></a>

**Medium** · Google · 2023-06-28

#### Problem

Given an array of integers, return a new array where each element in the new array is the number of smaller elements to the right of that element in the original input array.

For example, given the array `[3, 4, 9, 6, 1]`, return `[1, 1, 2, 1, 0]`, since:

- There is 1 smaller element to the right of 3
- There is 1 smaller element to the right of 4
- There are 2 smaller elements to the right of 9
- There is 1 smaller element to the right of 6
- There are no smaller elements to the right of 1

### Problem #1284 <a id="p1284"></a>

**Hard** · Google · 2023-06-29

#### Problem

Let A be an N by M matrix in which every row and every column is sorted.

Given i1, j1, i2, and j2, compute the number of elements of M smaller than `M[i1, j1]` and larger than `M[i2, j2]`.

For example, given the following matrix:

```
[[1, 3, 7, 10, 15, 20],
 [2, 6, 9, 14, 22, 25],
 [3, 8, 10, 15, 25, 30],
 [10, 11, 12, 23, 30, 35],
 [20, 25, 30, 35, 40, 45]]
```

And i1 = 1, j1 = 1, i2 = 3, j2 = 3, return 15 as there are 15 numbers in the matrix smaller than 6 or greater than 23.

### Problem #1285 <a id="p1285"></a>

**Medium** · PayPal · 2023-06-30

#### Problem

Given a string and a number of lines k, print the string in zigzag form. In zigzag, characters are printed out diagonally from top left to bottom right until reaching the kth line, then back up to top right, and so on.

For example, given the sentence `"thisisazigzag"` and k = 4, you should print:

```
t     a     g
 h   s z   a
  i i   i z
   s     g
```

### Problem #1286 <a id="p1286"></a>

**Medium** · Mozilla · 2023-07-01

#### Problem

A bridge in a connected (undirected) graph is an edge that, if removed, causes the graph to become disconnected. Find all the bridges in a graph.

### Problem #1287 <a id="p1287"></a>

**Easy** · Atlassian · 2023-07-02

#### Problem

MegaCorp wants to give bonuses to its employees based on how many lines of codes they have written. They would like to give the smallest positive amount to each worker consistent with the constraint that if a developer has written more lines of code than their neighbor, they should receive more money.

Given an array representing a line of seats of employees at MegaCorp, determine how much each one should get paid.

For example, given `[10, 40, 200, 1000, 60, 30]`, you should return `[1, 2, 3, 4, 2, 1]`.

### Problem #1288 <a id="p1288"></a>

**Hard** · Facebook · 2023-07-03

#### Problem

Given a string of parentheses, find the balanced string that can be produced from it using the minimum number of insertions and deletions. If there are multiple solutions, return any of them.

For example, given `"(()"`, you could return `"(())"`. Given `"))()("`, you could return `"()()()()"`.

### Problem #1289 <a id="p1289"></a>

**Hard** · Google · 2023-07-04

#### Problem

PageRank is an algorithm used by Google to rank the importance of different websites. While there have been changes over the years, the central idea is to assign each site a score based on the importance of other pages that link to that page.

More mathematically, suppose there are N sites, and each site i has a certain count Ci of outgoing links. Then the score for a particular site Sj is defined as :

`score(Sj)` = (1 - d) / N + d \* (`score(Sx)` / Cx+ `score(Sy)` / Cy+ ... + `score(Sz)` / Cz))

Here, Sx, Sy, ..., Sz denote the scores of all the other sites that have outgoing links to Sj, and d is a damping factor, usually set to around 0.85, used to model the probability that a user will stop searching.

Given a directed graph of links between various websites, write a program that calculates each site's page rank.

### Problem #1290 <a id="p1290"></a>

**Medium** · Nvidia · 2023-07-05

#### Problem

You are given a list of N points (x1, y1), (x2, y2), ..., (xN, yN) representing a polygon. You can assume these points are given in order; that is, you can construct the polygon by connecting point 1 to point 2, point 2 to point 3, and so on, finally looping around to connect point N to point 1.

Determine if a new point p lies inside this polygon. (If p is on the boundary of the polygon, you should return False).

### Problem #1291 <a id="p1291"></a>

**Easy** · IBM · 2023-07-06

#### Problem

Given an integer, find the next permutation of it in absolute order. For example, given 48975, the next permutation would be 49578.

### Problem #1292 <a id="p1292"></a>

**Medium** · Stitch Fix · 2023-07-07

#### Problem

Pascal's triangle is a triangular array of integers constructed with the following formula:

- The first row consists of the number 1.
- For each subsequent row, each element is the sum of the numbers directly above it, on either side.

For example, here are the first few rows:

```
    1
   1 1
  1 2 1
 1 3 3 1
1 4 6 4 1
```

Given an input k, return the kth row of Pascal's triangle.

Bonus: Can you do this using only `O(k)` space?

### Problem #1293 <a id="p1293"></a>

**Hard** · Apple · 2023-07-08

#### Problem

You are going on a road trip, and would like to create a suitable music playlist. The trip will require N songs, though you only have M songs downloaded, where M \< N. A valid playlist should select each song at least once, and guarantee a buffer of B songs between repeats.

Given N, M, and B, determine the number of valid playlists.

### Problem #1294 <a id="p1294"></a>

**Easy** · Amazon · 2023-07-09

#### Problem

Run-length encoding is a fast and simple method of encoding strings. The basic idea is to represent repeated successive characters as a single count and character. For example, the string `"AAAABBBCCDAA"` would be encoded as `"4A3B2C1D2A"`.

Implement run-length encoding and decoding. You can assume the string to be encoded have no digits and consists solely of alphabetic characters. You can assume the string to be decoded is valid.

### Problem #1295 <a id="p1295"></a>

**Easy** · Twitter · 2023-07-10

#### Problem

You run an e-commerce website and want to record the last N order ids in a log. Implement a data structure to accomplish this, with the following API:

- `record(order_id)`: adds the `order_id` to the log
- `get_last(i)`: gets the ith last element from the log. i is guaranteed to be smaller than or equal to N.

You should be as efficient with time and space as possible.

### Problem #1296 <a id="p1296"></a>

**Easy** · Amazon · 2023-07-11

#### Problem

Given a sorted array, find the smallest positive integer that is not the sum of a subset of the array.

For example, for the input `[1, 2, 3, 10]`, you should return 7.

Do this in `O(N)` time.

#### Solution

##### Smallest positive integer not representable as subset sum (O(N))

Given a sorted array of integers (typically non‑negative), you can find the smallest positive integer that cannot be formed as a sum of a subset in linear time using this invariant:

- Maintain `reachable`, the smallest positive integer in the range `[1, reachable)` that is guaranteed constructible from a subset seen so far.
- Initialize `reachable = 1` (we can build nothing yet, so we know we can’t reach 1 unless a 1 appears).
- Iterate through the array (skip any non‑positive values):
  - If the current value `x` is greater than `reachable`, then `reachable` cannot be formed (and is the answer), because all subset sums from the previous elements are strictly less than `reachable`, and adding `x` jumps beyond it.
  - Otherwise (i.e., `x <= reachable`), extend the constructible range by `x`: set `reachable += x`.
- If you finish the loop, the answer is `reachable`.

This works because if you have already constructed all sums in `[1, reachable)`, and the next number `x` is at most `reachable`, then by including or excluding `x` you can now construct everything up to `reachable + x - 1`, thus expanding the covered interval continuously.

For the example `[1, 2, 3, 10]`:
- Start `reachable = 1`.
- See `1` (<= 1): `reachable = 2` (now can make [1]).
- See `2` (<= 2): `reachable = 4` (now can make [1..3]).
- See `3` (<= 4): `reachable = 7` (now can make [1..6]).
- See `10` (> 7): cannot make `7` → answer is `7`.

Time: O(N), Space: O(1). Requires array to be sorted non‑decreasing; otherwise, sort first for O(N log N).


In [1]:
def smallest_nonconstructible_value(arr):
    # If array might contain non-positives, skip them
    reachable = 1
    for x in arr:
        if x <= 0:
            continue
        if x > reachable:
            return reachable
        reachable += x
    return reachable

# Examples
print(smallest_nonconstructible_value([1, 2, 3, 10]))  # 7
print(smallest_nonconstructible_value([1, 1, 1, 1]))   # 5
print(smallest_nonconstructible_value([2, 3, 4]))      # 1
print(smallest_nonconstructible_value([1, 2, 5, 10]))  # 4

7
5
1
4


### Problem #1297 <a id="p1297"></a>

**Easy** · Microsoft · 2023-07-12

#### Problem

Using a `read7()` method that returns 7 characters from a file, implement `readN(n)` which reads n characters.

For example, given a file with the content “Hello world”, three `read7()` returns “Hello w”, “orld” and then “”.

#### Solution

##### Implement `readN(n)` using `read7()`

Task: Build `readN(n)` that reads exactly `n` characters from a file-like source, given only `read7()` which returns up to 7 characters per call.  
Requirement: Preserve leftover characters between calls to `readN`, since `read7()` may over-read.

In [1]:
from collections import deque


class Reader:
    """
    Wraps a file-like source that only supports read7(), and provides readN(n).
    Maintains an internal buffer of leftover characters between calls.
    """

    def __init__(self, text: str):
        self._text = text
        self._i = 0
        self._buf = deque()  # leftover chars from previous read7 calls

    def read7(self) -> str:
        if self._i >= len(self._text):
            return ""
        chunk = self._text[self._i:self._i + 7]
        self._i += len(chunk)
        return chunk

    def readN(self, n: int) -> str:
        if n <= 0:
            return ""

        out = []

        # Use any buffered leftovers first
        while self._buf and len(out) < n:
            out.append(self._buf.popleft())

        # Pull more from read7 until we have n or hit EOF
        while len(out) < n:
            chunk = self.read7()
            if chunk == "":
                break  # EOF
            needed = n - len(out)
            out.extend(chunk[:needed])
            # store leftovers
            for ch in chunk[needed:]:
                self._buf.append(ch)

        return "".join(out)


# Demo (matches prompt idea)
r = Reader("Hello world")
print(r.read7())  # "Hello w"
print(r.read7())  # "orld"
print(r.read7())  # ""

# Demo for readN correctness across calls
r = Reader("Hello world")
print(r.readN(5))  # "Hello"
print(r.readN(4))  # " wor"
print(r.readN(100))  # "ld"
print(r.readN(1))  # ""


Hello w
orld

Hello
 wor
ld



### Problem #1298 <a id="p1298"></a>

**Medium** · Salesforce · 2023-07-13

#### Problem

The number 6174 is known as Kaprekar's contant, after the mathematician who discovered an associated property: for all four-digit numbers with at least two distinct digits, repeatedly applying a simple procedure eventually results in this value. The procedure is as follows:

- For a given input x, create two new numbers that consist of the digits in x in ascending and descending order.
- Subtract the smaller number from the larger number.

For example, this algorithm terminates in three steps when starting from 1234:

- 4321 - 1234 = 3087
- 8730 - 0378 = 8352
- 8532 - 2358 = 6174

Write a function that returns how many steps this will take for a given input N.

### Problem #1299 <a id="p1299"></a>

**Hard** · Amazon · 2023-07-14

#### Problem

There exists a staircase with N steps, and you can climb up either 1 or 2 steps at a time. Given N, write a function that returns the number of unique ways you can climb the staircase. The order of the steps matters.

For example, if N is 4, then there are 5 unique ways:

- 1, 1, 1, 1
- 2, 1, 1
- 1, 2, 1
- 1, 1, 2
- 2, 2

What if, instead of being able to climb 1 or 2 steps at a time, you could climb any number from a set of positive integers X? For example, if X = `{1, 3, 5}`, you could climb 1, 3, or 5 steps at a time.

### Problem #1300 <a id="p1300"></a>

**Medium** · Lyft · 2023-07-15

#### Problem

Given a list of integers and a number K, return which contiguous elements of the list sum to K.

For example, if the list is `[1, 2, 3, 4, 5]` and K is 9, then it should return `[2, 3, 4]`, since 2 + 3 + 4 = 9.

### Problem #1301 <a id="p1301"></a>

**Medium** · Flipkart · 2023-07-16

#### Problem

Starting from 0 on a number line, you would like to make a series of jumps that lead to the integer N.

On the ith jump, you may move exactly i places to the left or right.

Find a path with the fewest number of jumps required to get from 0 to N.

### Problem #1302 <a id="p1302"></a>

**Hard** · Stripe · 2023-07-17

#### Problem

Given an array of integers, find the first missing positive integer in linear time and constant space. In other words, find the lowest positive integer that does not exist in the array. The array can contain duplicates and negative numbers as well.

For example, the input `[3, 4, -1, 1]` should give 2. The input `[1, 2, 0]` should give 3.

You can modify the input array in-place.

### Problem #1303 <a id="p1303"></a>

**Medium** · Facebook · 2023-07-18

#### Problem

Given an integer n, find the next biggest integer with the same number of 1-bits on. For example, given the number 6 (0110 in binary), return 9 (1001).

### Problem #1304 <a id="p1304"></a>

**Medium** · Google · 2023-07-19

#### Problem

Given an integer n and a list of integers l, write a function that randomly generates a number from 0 to n-1 that isn't in l (uniform).

### Problem #1305 <a id="p1305"></a>

**Hard** · YouTube · 2023-07-20

#### Problem

Write a program that computes the length of the longest common subsequence of three given strings. For example, given `"epidemiologist"`, `"refrigeration"`, and `"supercalifragilisticexpialodocious"`, it should return 5, since the longest common subsequence is `"eieio"`.

### Problem #1306 <a id="p1306"></a>

**Hard** · Google · 2023-07-21

#### Problem

Suppose we represent our file system by a string in the following manner:

The string `"dir\n\tsubdir1\n\tsubdir2\n\t\tfile.ext"` represents:

```
dir
    subdir1
    subdir2
        file.ext
```

The directory dir contains an empty sub-directory subdir1 and a sub-directory subdir2 containing a file file.ext.

The string `"dir\n\tsubdir1\n\t\tfile1.ext\n\t\tsubsubdir1\n\tsubdir2\n\t\tsubsubdir2\n\t\t\tfile2.ext"` represents:

```
dir
    subdir1
        file1.ext
        subsubdir1
    subdir2
        subsubdir2
            file2.ext
```

The directory dir contains two sub-directories subdir1 and subdir2. subdir1 contains a file file1.ext and an empty second-level sub-directory subsubdir1. subdir2 contains a second-level sub-directory subsubdir2 containing a file file2.ext.

We are interested in finding the longest (number of characters) absolute path to a file within our file system. For example, in the second example above, the longest absolute path is `"dir/subdir2/subsubdir2/file2.ext"`, and its length is 32 (not including the double quotes).

Given a string representing the file system in the above format, return the length of the longest absolute path to a file in the abstracted file system. If there is no file in the system, return 0.

Note:

The name of a file contains at least a period and an extension.

The name of a directory or sub-directory will not contain a period.

### Problem #1307 <a id="p1307"></a>

**Medium** · Facebook · 2023-07-22

#### Problem

Given a function that generates perfectly random numbers between 1 and k (inclusive), where k is an input, write a function that shuffles a deck of cards represented as an array using only swaps.

It should run in `O(N)` time.

Hint: Make sure each one of the 52! permutations of the deck is equally likely.

### Problem #1308 <a id="p1308"></a>

**Medium** · Samsung · 2023-07-23

#### Problem

A group of houses is connected to the main water plant by means of a set of pipes. A house can either be connected by a set of pipes extending directly to the plant, or indirectly by a pipe to a nearby house which is otherwise connected.

For example, here is a possible configuration, where A, B, and C are houses, and arrows represent pipes:

A \<--> B \<--> C \<--> plant

Each pipe has an associated cost, which the utility company would like to minimize. Given an undirected graph of pipe connections, return the lowest cost configuration of pipes such that each house has access to water.

In the following setup, for example, we can remove all but the pipes from plant to A, plant to B, and B to C, for a total cost of 16.

```
pipes = {
    'plant': {'A': 1, 'B': 5, 'C': 20},
    'A': {'C': 15},
    'B': {'C': 10},
    'C': {}
}
```

### Problem #1309 <a id="p1309"></a>

**Hard** · Grammarly · 2023-07-24

#### Problem

Soundex is an algorithm used to categorize phonetically, such that two names that sound alike but are spelled differently have the same representation.

Soundex maps every name to a string consisting of one letter and three numbers, like M460.

One version of the algorithm is as follows:

1. Remove consecutive consonants with the same sound (for example, change ck -> c).
2. Keep the first letter. The remaining steps only apply to the rest of the string.
3. Remove all vowels, including y, w, and h.
4. Replace all consonants with the following digits:

    - b, f, p, v → 1
    - c, g, j, k, q, s, x, z → 2
    - d, t → 3
    - l → 4
    - m, n → 5
    - r → 6

5. If you don't have three numbers yet, append zeros until you do. Keep the first three numbers.

Using this scheme, Jackson and Jaxen both map to J250.

Implement Soundex.

### Problem #1310 <a id="p1310"></a>

**Medium** · Fitbit · 2023-07-25

#### Problem

Given a linked list, rearrange the node values such that they appear in alternating low -> high -> low -> high ... form. For example, given 1 -> 2 -> 3 -> 4 -> 5, you should return 1 -> 3 -> 2 -> 5 -> 4.

### Problem #1311 <a id="p1311"></a>

**Hard** · Google · 2023-07-26

#### Problem

A quack is a data structure combining properties of both stacks and queues. It can be viewed as a list of elements written left to right such that three operations are possible:

- `push(x)`: add a new item x to the left end of the list
- `pop()`: remove and return the item on the left end of the list
- `pull()`: remove the item on the right end of the list.

Implement a quack using three stacks and `O(1)` additional memory, so that the amortized time for any push, pop, or pull operation is `O(1)`.

### Problem #1312 <a id="p1312"></a>

**Easy** · Two Sigma · 2023-07-27

#### Problem

Using a function `rand5()` that returns an integer from 1 to 5 (inclusive) with uniform probability, implement a function `rand7()` that returns an integer from 1 to 7 (inclusive).

### Problem #1313 <a id="p1313"></a>

**Medium** · Yahoo · 2023-07-28

#### Problem

Write a function that returns the bitwise AND of all integers between M and N, inclusive.

### Problem #1314 <a id="p1314"></a>

**Easy** · Palantir · 2023-07-29

#### Problem

The ancient Egyptians used to express fractions as a sum of several terms where each numerator is one. For example, 4 / 13 can be represented as 1 / 4 + 1 / 18 + 1 / 468.

Create an algorithm to turn an ordinary fraction a / b, where a \< b, into an Egyptian fraction.

### Problem #1315 <a id="p1315"></a>

**Medium** · Epic · 2023-07-30

#### Problem

The `"look and say"` sequence is defined as follows: beginning with the term 1, each subsequent term visually describes the digits appearing in the previous term. The first few terms are as follows:

```
1
11
21
1211
111221
```

As an example, the fourth term is 1211, since the third term consists of one 2 and one 1.

Given an integer N, print the Nth term of this sequence.

### Problem #1316 <a id="p1316"></a>

**Hard** · Oracle · 2023-07-31

#### Problem

We say a number is sparse if there are no adjacent ones in its binary representation. For example, 21 (10101) is sparse, but 22 (10110) is not. For a given input N, find the smallest sparse number greater than or equal to N.

Do this in faster than `O(N log N)` time.

### Problem #1317 <a id="p1317"></a>

**Medium** · Pinterest · 2023-08-01

#### Problem

The sequence `[0, 1, ..., N]` has been jumbled, and the only clue you have for its order is an array representing whether each number is larger or smaller than the last. Given this information, reconstruct an array that is consistent with it. For example, given `[None, +, +, -, +]`, you could return `[1, 2, 3, 0, 4]`.

### Problem #1318 <a id="p1318"></a>

**Easy** · Yahoo · 2023-08-02

#### Problem

You are given a string of length N and a parameter k. The string can be manipulated by taking one of the first k letters and moving it to the end.

Write a program to determine the lexicographically smallest string that can be created after an unlimited number of moves.

For example, suppose we are given the string daily and k = 1. The best we can create in this case is ailyd.

### Problem #1319 <a id="p1319"></a>

**Medium** · 2023-08-03

#### Problem

Given a real number n, find the square root of n. For example, given n = 9, return 3.

### Problem #1320 <a id="p1320"></a>

**Hard** · Amazon · 2023-08-04

#### Problem

Given an integer k and a string s, find the length of the longest substring that contains at most k distinct characters.

For example, given s = `"abcba"` and k = 2, the longest substring with k distinct characters is `"bcb"`.

### Problem #1321 <a id="p1321"></a>

**Medium** · Facebook · 2023-08-05

#### Problem

Given a number in Roman numeral format, convert it to decimal.

The values of Roman numerals are as follows:

```
{
    'M': 1000,
    'D': 500,
    'C': 100,
    'L': 50,
    'X': 10,
    'V': 5,
    'I': 1
}
```

In addition, note that the Roman numeral system uses subtractive notation for numbers such as IV and XL.

For the input XIV, for instance, you should return 14.

### Problem #1322 <a id="p1322"></a>

**Hard** · Google · 2023-08-06

#### Problem

The game of Nim is played as follows. Starting with three heaps, each containing a variable number of items, two players take turns removing one or more items from a single pile. The player who eventually is forced to take the last stone loses. For example, if the initial heap sizes are 3, 4, and 5, a game could be played as shown below:

```
  A  |  B  |  C
-----------------
  3  |  4  |  5
  3  |  1  |  3
  3  |  1  |  3
  0  |  1  |  3
  0  |  1  |  0
  0  |  0  |  0
```

In other words, to start, the first player takes three items from pile B. The second player responds by removing two stones from pile C. The game continues in this way until player one takes last stone and loses.

Given a list of non-zero starting values `[a, b, c]`, and assuming optimal play, determine whether the first player has a forced win.

### Problem #1323 <a id="p1323"></a>

**Hard** · Uber · 2023-08-07

#### Problem

You have N stones in a row, and would like to create from them a pyramid. This pyramid should be constructed such that the height of each stone increases by one until reaching the tallest stone, after which the heights decrease by one. In addition, the start and end stones of the pyramid should each be one stone high.

You can change the height of any stone by paying a cost of 1 unit to lower its height by 1, as many times as necessary. Given this information, determine the lowest cost method to produce this pyramid.

For example, given the stones `[1, 1, 3, 3, 2, 1]`, the optimal solution is to pay 2 to create `[0, 1, 2, 3, 2, 1]`.

### Problem #1324 <a id="p1324"></a>

**Hard** · Twitter · 2023-08-08

#### Problem

You are given an array of length 24, where each element represents the number of new subscribers during the corresponding hour. Implement a data structure that efficiently supports the following:

- `update(hour: int, value: int)`: Increment the element at index hour by value.
- `query(start: int, end: int)`: Retrieve the number of subscribers that have signed up between start and end (inclusive).

You can assume that all values get cleared at the end of the day, and that you will not be asked for start and end values that wrap around midnight.

### Problem #1325 <a id="p1325"></a>

**Hard** · Etsy · 2023-08-09

#### Problem

Given a sorted array, convert it into a height-balanced binary search tree.

### Problem #1326 <a id="p1326"></a>

**Medium** · Stripe · 2023-08-10

#### Problem

reduce (also known as fold) is a function that takes in an array, a combining function, and an initial value and builds up a result by calling the combining function on each element of the array, left to right. For example, we can write `sum()` in terms of reduce:

```
def add(a, b):
    return a + b
```

```
def sum(lst):
    return reduce(lst, add, 0)
```

This should call add on the initial value with the first element of the array, and then the result of that with the second element of the array, and so on until we reach the end, when we return the sum of the array.

Implement your own version of reduce.

### Problem #1327 <a id="p1327"></a>

**Hard** · Netflix · 2023-08-11

#### Problem

Implement a queue using a set of fixed-length arrays.

The queue should support enqueue, dequeue, and `get_size` operations.

### Problem #1328 <a id="p1328"></a>

**Hard** · Google · 2023-08-12

#### Problem

Given a string, split it into as few strings as possible such that each string is a palindrome.

For example, given the input string racecarannakayak, return `["racecar", "anna", "kayak"]`.

Given the input string abc, return `["a", "b", "c"]`.

### Problem #1329 <a id="p1329"></a>

**Easy** · Google · 2023-08-13

#### Problem

Given a set of points (x, y) on a 2D cartesian plane, find the two closest points. For example, given the points `[(1, 1), (-1, -1), (3, 4), (6, 1), (-1, -6), (-4, -3)]`, return `[(-1, -1), (1, 1)]`.

### Problem #1330 <a id="p1330"></a>

**Easy** · Dropbox · 2023-08-14

#### Problem

Spreadsheets often use this alphabetical encoding for its columns: `"A"`, `"B"`, `"C"`, ..., `"AA"`, `"AB"`, ..., `"ZZ"`, `"AAA"`, `"AAB"`, ....

Given a column number, return its alphabetical column id. For example, given 1, return `"A"`. Given 27, return `"AA"`.

### Problem #1331 <a id="p1331"></a>

**Easy** · Google · 2023-08-15

#### Problem

You are given given a list of rectangles represented by min and max x- and y-coordinates. Compute whether or not a pair of rectangles overlap each other. If one rectangle completely covers another, it is considered overlapping.

For example, given the following rectangles:

```
{
    "top_left": (1, 4),
    "dimensions": (3, 3) # width, height
},
{
    "top_left": (-1, 3),
    "dimensions": (2, 1)
},
{
    "top_left": (0, 5),
    "dimensions": (4, 3)
}
```

return true as the first and third rectangle overlap each other.

### Problem #1332 <a id="p1332"></a>

**Hard** · Airbnb · 2023-08-16

#### Problem

You are given an array X of floating-point numbers x1, x2, ... xn. These can be rounded up or down to create a corresponding array Y of integers y1, y2, ... yn.

Write an algorithm that finds an appropriate Y array with the following properties:

- The rounded sums of both arrays should be equal.
- The absolute pairwise difference between elements is minimized. In other words, |x1- y1| + |x2- y2| + ... + |xn- yn| should be as small as possible.

For example, suppose your input is `[1.3, 2.3, 4.4]`. In this case you cannot do better than `[1, 2, 5]`, which has an absolute difference of |1.3 - 1| + |2.3 - 2| + |4.4 - 5| = 1.

### Problem #1333 <a id="p1333"></a>

**Medium** · Slack · 2023-08-17

#### Problem

You are given an N by M matrix of 0s and 1s. Starting from the top left corner, how many ways are there to reach the bottom right corner?

You can only move right and down. 0 represents an empty space while 1 represents a wall you cannot walk through.

For example, given the following matrix:

```
[[0, 0, 1],
 [0, 0, 1],
 [1, 0, 0]]
```

Return two, as there are only two ways to get to the bottom right:

- Right, down, down, right
- Down, right, down, right

The top left corner and bottom right corner will always be 0.

### Problem #1334 <a id="p1334"></a>

**Easy** · Google · 2023-08-18

#### Problem

The edit distance between two strings refers to the minimum number of character insertions, deletions, and substitutions required to change one string to the other. For example, the edit distance between “kitten” and “sitting” is three: substitute the “k” for “s”, substitute the “e” for “i”, and append a “g”.

Given two strings, compute the edit distance between them.

### Problem #1335 <a id="p1335"></a>

**Medium** · Facebook · 2023-08-19

#### Problem

Given the mapping a = 1, b = 2, ... z = 26, and an encoded message, count the number of ways it can be decoded.

For example, the message `'111'` would give 3, since it could be decoded as `'aaa'`, `'ka'`, and `'ak'`.

You can assume that the messages are decodable. For example, `'001'` is not allowed.

### Problem #1336 <a id="p1336"></a>

**Hard** · Microsoft · 2023-08-20

#### Problem

You have an N by N board. Write a function that, given N, returns the number of possible arrangements of the board where N queens can be placed on the board without threatening each other, i.e. no two queens share the same row, column, or diagonal.

### Problem #1337 <a id="p1337"></a>

**Easy** · Square · 2023-08-21

#### Problem

The Sieve of Eratosthenes is an algorithm used to generate all prime numbers smaller than N. The method is to take increasingly larger prime numbers, and mark their multiples as composite.

For example, to find all primes less than 100, we would first mark `[4, 6, 8, ...]` (multiples of two), then `[6, 9, 12, ...]` (multiples of three), and so on. Once we have done this for all primes less than N, the unmarked numbers that remain will be prime.

Implement this algorithm.

Bonus: Create a generator that produces primes indefinitely (that is, without taking N as an input).

### Problem #1338 <a id="p1338"></a>

**Medium** · Microsoft · 2023-08-22

#### Problem

Given an unsorted array of integers, find the length of the longest consecutive elements sequence.

For example, given `[100, 4, 200, 1, 3, 2]`, the longest consecutive element sequence is `[1, 2, 3, 4]`. Return its length: 4.

Your algorithm should run in `O(n)` complexity.

### Problem #1339 <a id="p1339"></a>

**Medium** · Google · 2023-08-23

#### Problem

On our special chessboard, two bishops attack each other if they share the same diagonal. This includes bishops that have another bishop located between them, i.e. bishops can attack through pieces.

You are given N bishops, represented as (row, column) tuples on a M by M chessboard. Write a function to count the number of pairs of bishops that attack each other. The ordering of the pair doesn't matter: (1, 2) is considered the same as (2, 1).

For example, given M = 5 and the list of bishops:

- (0, 0)
- (1, 2)
- (2, 2)
- (4, 0)

The board would look like this:

```
[b 0 0 0 0]
[0 0 b 0 0]
[0 0 b 0 0]
[0 0 0 0 0]
[b 0 0 0 0]
```

You should return 2, since bishops 1 and 3 attack each other, as well as bishops 3 and 4.

### Problem #1340 <a id="p1340"></a>

**Medium** · Facebook · 2023-08-24

#### Problem

There is an N by M matrix of zeroes. Given N and M, write a function to count the number of ways of starting at the top-left corner and getting to the bottom-right corner. You can only move right or down.

For example, given a 2 by 2 matrix, you should return 2, since there are two ways to get to the bottom-right:

- Right, then down
- Down, then right

Given a 5 by 5 matrix, there are 70 ways to get to the bottom-right.

### Problem #1341 <a id="p1341"></a>

**Easy** · PayPal · 2023-08-25

#### Problem

Given a binary tree, determine whether or not it is height-balanced. A height-balanced binary tree can be defined as one in which the heights of the two subtrees of any node never differ by more than one.

### Problem #1342 <a id="p1342"></a>

**Easy** · Google · 2023-08-26

#### Problem

You are given an N by N matrix of random letters and a dictionary of words. Find the maximum number of words that can be packed on the board from the given dictionary.

A word is considered to be able to be packed on the board if:

- It can be found in the dictionary
- It can be constructed from untaken letters by other words found so far on the board
- The letters are adjacent to each other (vertically and horizontally, not diagonally).

Each tile can be visited only once by any word.

For example, given the following dictionary:

`{ 'eat', 'rain', 'in', 'rat' }`

and matrix:

```
[['e', 'a', 'n'],
 ['t', 't', 'i'],
 ['a', 'r', 'a']]
```

Your function should return 3, since we can make the words `'eat'`, `'in'`, and `'rat'` without them touching each other. We could have alternatively made `'eat'` and `'rain'`, but that would be incorrect since that's only 2 words.

### Problem #1343 <a id="p1343"></a>

**Hard** · Google · 2023-08-27

#### Problem

Find the minimum number of coins required to make n cents.

You can use standard American denominations, that is, 1¢, 5¢, 10¢, and 25¢.

For example, given n = 16, return 3 since we can make it with a 10¢, a 5¢, and a 1¢.

### Problem #1344 <a id="p1344"></a>

**Hard** · Twitter · 2023-08-28

#### Problem

A teacher must divide a class of students into two teams to play dodgeball. Unfortunately, not all the kids get along, and several refuse to be put on the same team as that of their enemies.

Given an adjacency list of students and their enemies, write an algorithm that finds a satisfactory pair of teams, or returns False if none exists.

For example, given the following enemy graph you should return the teams `{0, 1, 4, 5}` and `{2, 3}`.

```
students = {
    0: [3],
    1: [2],
    2: [1, 4],
    3: [0, 4, 5],
    4: [2, 3],
    5: [3]
}
```

On the other hand, given the input below, you should return False.

```
students = {
    0: [3],
    1: [2],
    2: [1, 3, 4],
    3: [0, 2, 4, 5],
    4: [2, 3],
    5: [3]
}
```

### Problem #1345 <a id="p1345"></a>

**Easy** · WhatsApp · 2023-08-29

#### Problem

Given an array of integers out of order, determine the bounds of the smallest window that must be sorted in order for the entire array to be sorted. For example, given `[3, 7, 5, 6, 9]`, you should return (1, 3).

### Problem #1346 <a id="p1346"></a>

**Medium** · Google · 2023-08-30

#### Problem

Given an array of numbers and an index i, return the index of the nearest larger number of the number at index i, where distance is measured in array indices.

For example, given `[4, 1, 3, 5, 6]` and index 0, you should return 3.

If two distances to larger numbers are the equal, then return any one of them. If the array at i doesn't have a nearest larger integer, then return null.

Follow-up: If you can preprocess the array, can you do this in constant time?

### Problem #1347 <a id="p1347"></a>

**Easy** · IBM · 2023-08-31

#### Problem

Given a string with repeated characters, rearrange the string so that no two adjacent characters are the same. If this is not possible, return None.

For example, given `"aaabbc"`, you could return `"ababac"`. Given `"aaab"`, return None.

### Problem #1348 <a id="p1348"></a>

**Hard** · Uber · 2023-09-01

#### Problem

Given a tree where each edge has a weight, compute the length of the longest path in the tree.

For example, given the following tree:

```
  a
 /|\
b c d
   / \
  e   f
 / \
g   h
```

and the weights: a-b: 3, a-c: 5, a-d: 8, d-e: 2, d-f: 4, e-g: 1, e-h: 1, the longest path would be c -> a -> d -> f, with a length of 17.

The path does not have to pass through the root, and each node can have any amount of children.

### Problem #1349 <a id="p1349"></a>

**Medium** · Google · 2023-09-02

#### Problem

Given the sequence of keys visited by a postorder traversal of a binary search tree, reconstruct the tree.

For example, given the sequence 2, 4, 3, 8, 7, 5, you should construct the following tree:

```
    5
   / \
  3   7
 / \   \
2   4   8
```

### Problem #1350 <a id="p1350"></a>

**Easy** · Amazon · 2023-09-03

#### Problem

Given a linked list, remove all consecutive nodes that sum to zero. Print out the remaining nodes.

For example, suppose you are given the input 3 -> 4 -> -7 -> 5 -> -6 -> 6. In this case, you should first remove 3 -> 4 -> -7, then -6 -> 6, leaving only 5.

### Problem #1351 <a id="p1351"></a>

**Hard** · Facebook · 2023-09-04

#### Problem

Given an array of numbers of length N, find both the minimum and maximum using less than 2 \* (N - 2) comparisons.

### Problem #1352 <a id="p1352"></a>

**Hard** · Jane Street · 2023-09-05

#### Problem

Suppose you are given a table of currency exchange rates, represented as a 2D array. Determine whether there is a possible arbitrage: that is, whether there is some sequence of trades you can make, starting with some amount A of any currency, so that you can end up with some amount greater than A of that currency.

There are no transaction costs and you can trade fractional quantities.

### Problem #1353 <a id="p1353"></a>

**Medium** · Microsoft · 2023-09-06

#### Problem

Write a program to determine how many distinct ways there are to create a max heap from a list of N given integers.

For example, if N = 3, and our integers are `[1, 2, 3]`, there are two ways, shown below.

```
  3      3
 / \    / \
1   2  2   1
```

### Problem #1354 <a id="p1354"></a>

**Medium** · Yelp · 2023-09-07

#### Problem

You are given an array of integers, where each element represents the maximum number of steps that can be jumped going forward from that element. Write a function to return the minimum number of jumps you must take in order to get from the start to the end of the array.

For example, given `[6, 2, 4, 0, 5, 1, 1, 4, 2, 9]`, you should return 2, as the optimal solution involves jumping from 6 to 5, and then from 5 to 9.

### Problem #1355 <a id="p1355"></a>

**Easy** · Stripe · 2023-09-08

#### Problem

Given an integer n, return the length of the longest consecutive run of 1s in its binary representation.

For example, given 156, you should return 3.

### Problem #1356 <a id="p1356"></a>

**Easy** · Morgan Stanley · 2023-09-09

#### Problem

In Ancient Greece, it was common to write text with the first line going left to right, the second line going right to left, and continuing to go back and forth. This style was called `"boustrophedon"`.

Given a binary tree, write an algorithm to print the nodes in boustrophedon order.

For example, given the following tree:

```
       1
    /     \
  2         3
 / \       / \
4   5     6   7
```

You should return `[1, 3, 2, 4, 5, 6, 7]`.

### Problem #1357 <a id="p1357"></a>

**Hard** · Google · 2023-09-10

#### Problem

An XOR linked list is a more memory efficient doubly linked list. Instead of each node holding next and prev fields, it holds a field named both, which is an XOR of the next node and the previous node. Implement an XOR linked list; it has an `add(element)` which adds the element to the end, and a `get(index)` which returns the node at index.

If using a language that has no pointers (such as Python), you can assume you have access to `get_pointer` and `dereference_pointer` functions that converts between nodes and memory addresses.

### Problem #1358 <a id="p1358"></a>

**Medium** · Facebook · 2023-09-11

#### Problem

Given an array of numbers representing the stock prices of a company in chronological order and an integer k, return the maximum profit you can make from k buys and sells. You must buy the stock before you can sell it, and you must sell the stock before you can buy it again.

For example, given k = 2 and the array `[5, 2, 4, 0, 1]`, you should return 3.

### Problem #1359 <a id="p1359"></a>

**Medium** · Facebook · 2023-09-12

#### Problem

Given a stream of elements too large to store in memory, pick a random element from the stream with uniform probability.

### Problem #1360 <a id="p1360"></a>

**Medium** · Uber · 2023-09-13

#### Problem

One way to unlock an Android phone is through a pattern of swipes across a 1-9 keypad.

For a pattern to be valid, it must satisfy the following:

- All of its keys must be distinct.
- It must not connect two keys by jumping over a third key, unless that key has already been used.

For example, 4 - 2 - 1 - 7 is a valid pattern, whereas 2 - 1 - 7 is not.

Find the total number of valid unlock patterns of length N, where 1 \<= N \<= 9.

### Problem #1361 <a id="p1361"></a>

**Hard** · Affirm · 2023-09-14

#### Problem

Given a array of numbers representing the stock prices of a company in chronological order, write a function that calculates the maximum profit you could have made from buying and selling that stock. You're also given a number fee that represents a transaction fee for each buy and sell transaction.

You must buy before you can sell the stock, but you can make as many transactions as you like.

For example, given `[1, 3, 2, 8, 4, 10]` and fee = 2, you should return 9, since you could buy the stock at 1 dollar, and sell at 8 dollars, and then buy it at 4 dollars and sell it at 10 dollars. Since we did two transactions, there is a 4 dollar fee, so we have 7 + 6 = 13 profit minus 4 dollars of fees.

### Problem #1362 <a id="p1362"></a>

**Easy** · Google · 2023-09-15

#### Problem

A girl is walking along an apple orchard with a bag in each hand. She likes to pick apples from each tree as she goes along, but is meticulous about not putting different kinds of apples in the same bag.

Given an input describing the types of apples she will pass on her path, in order, determine the length of the longest portion of her path that consists of just two types of apple trees.

For example, given the input `[2, 1, 2, 3, 3, 1, 3, 5]`, the longest portion will involve types 1 and 3, with a length of four.

### Problem #1363 <a id="p1363"></a>

**Easy** · Uber · 2023-09-16

#### Problem

On election day, a voting machine writes data in the form (`voter_id`, `candidate_id`) to a text file. Write a program that reads this file as a stream and returns the top 3 candidates at any given time. If you find a voter voting more than once, report this as fraud.

### Problem #1364 <a id="p1364"></a>

**Hard** · Google · 2023-09-17

#### Problem

In a directed graph, each node is assigned an uppercase letter. We define a path's value as the number of most frequently-occurring letter along that path. For example, if a path in the graph goes through `"ABACA"`, the value of the path is 3, since there are 3 occurrences of `'A'` on the path.

Given a graph with n nodes and m directed edges, return the largest value path of the graph. If the largest value is infinite, then return null.

The graph is represented with a string and an edge list. The i-th character represents the uppercase letter of the i-th node. Each tuple in the edge list (i, j) means there is a directed edge from the i-th node to the j-th node. Self-edges are possible, as well as multi-edges.

For example, the following input graph:

ABACA

```
[(0, 1),
 (0, 2),
 (2, 3),
 (3, 4)]
```

Would have maximum value 3 using the path of vertices `[0, 2, 3, 4]`, (A, A, C, A).

The following input graph:

A

`[(0, 0)]`

Should return null, since we have an infinite loop.

### Problem #1365 <a id="p1365"></a>

**Easy** · Amazon · 2023-09-18

#### Problem

Given a complete binary tree, count the number of nodes in faster than `O(n)` time. Recall that a complete binary tree has every level filled except the last, and the nodes in the last level are filled starting from the left.

### Problem #1366 <a id="p1366"></a>

**Medium** · Dropbox · 2023-09-19

#### Problem

Given an undirected graph G, check whether it is bipartite. Recall that a graph is bipartite if its vertices can be divided into two independent sets, U and V, such that no edge connects vertices of the same set.

### Problem #1367 <a id="p1367"></a>

**Medium** · Google · 2023-09-20

#### Problem

A cryptarithmetic puzzle is a mathematical game where the digits of some numbers are represented by letters. Each letter represents a unique digit.

For example, a puzzle of the form:

```
  SEND
+ MORE
--------
 MONEY
```

may have the solution:

`{'S': 9, 'E': 5, 'N': 6, 'D': 7, 'M': 1, 'O', 0, 'R': 8, 'Y': 2}`

Given a three-word puzzle like the one above, create an algorithm that finds a solution.

### Problem #1368 <a id="p1368"></a>

**Hard** · Twitch · 2023-09-21

#### Problem

Describe what happens when you type a URL into your browser and press Enter.

### Problem #1369 <a id="p1369"></a>

**Medium** · Snapchat · 2023-09-22

#### Problem

Given a string of digits, generate all possible valid IP address combinations.

IP addresses must follow the format A.B.C.D, where A, B, C, and D are numbers between 0 and 255. Zero-prefixed numbers, such as 01 and 065, are not allowed, except for 0 itself.

For example, given `"2542540123"`, you should return `['254.25.40.123', '254.254.0.123']`.

### Problem #1370 <a id="p1370"></a>

**Hard** · Google · 2023-09-23

#### Problem

Given a string which we can delete at most k, return whether you can make a palindrome.

For example, given `'waterrfetawx'` and a k of 2, you could delete f and x to get `'waterretaw'`.

### Problem #1371 <a id="p1371"></a>

**Easy** · Amazon · 2023-09-24

#### Problem

Consider the following scenario: there are N mice and N holes placed at integer points along a line. Given this, find a method that maps mice to holes such that the largest number of steps any mouse takes is minimized.

Each move consists of moving one mouse one unit to the left or right, and only one mouse can fit inside each hole.

For example, suppose the mice are positioned at `[1, 4, 9, 15]`, and the holes are located at `[10, -5, 0, 16]`. In this case, the best pairing would require us to send the mouse at 1 to the hole at -5, so our function should return 6.

### Problem #1372 <a id="p1372"></a>

**Easy** · Google · 2023-09-25

#### Problem

You are given an M by N matrix consisting of booleans that represents a board. Each True boolean represents a wall. Each False boolean represents a tile you can walk on.

Given this matrix, a start coordinate, and an end coordinate, return the minimum number of steps required to reach the end coordinate from the start. If there is no possible path, then return null. You can move up, left, down, and right. You cannot move through walls. You cannot wrap around the edges of the board.

For example, given the following board:

```
[[f, f, f, f],
[t, t, f, t],
[f, f, f, f],
[f, f, f, f]]
```

and start = (3, 0) (bottom left) and end = (0, 0) (top left), the minimum number of steps required to reach the end is 7, since we would need to go through (1, 2) because there is a wall everywhere else on the second row.

### Problem #1373 <a id="p1373"></a>

**Medium** · Uber · 2023-09-26

#### Problem

Implement a 2D iterator class. It will be initialized with an array of arrays, and should implement the following methods:

- `next()`: returns the next element in the array of arrays. If there are no more elements, raise an exception.
- `has_next()`: returns whether or not the iterator still has elements left.

For example, given the input `[[1, 2], [3], [], [4, 5, 6]]`, calling `next()` repeatedly should output 1, 2, 3, 4, 5, 6.

Do not use flatten or otherwise clone the arrays. Some of the arrays can be empty.

### Problem #1374 <a id="p1374"></a>

**Medium** · Pinterest · 2023-09-27

#### Problem

Given an integer list where each number represents the number of hops you can make, determine whether you can reach to the last index starting at index 0.

For example, `[2, 0, 1, 0]` returns True while `[1, 1, 0, 1]` returns False.

### Problem #1375 <a id="p1375"></a>

**Medium** · Dropbox · 2023-09-28

#### Problem

Given a list of words, determine whether the words can be chained to form a circle. A word X can be placed in front of another word Y in a circle if the last character of X is same as the first character of Y.

For example, the words `['chair', 'height', 'racket', touch', 'tunic']` can form the following circle: chair --> racket --> touch --> height --> tunic --> chair.

### Problem #1376 <a id="p1376"></a>

**Hard** · Apple · 2023-09-29

#### Problem

Given a linked list, uniformly shuffle the nodes. What if we want to prioritize space over time?

### Problem #1377 <a id="p1377"></a>

**Hard** · Google · 2023-09-30

#### Problem

You're given a string consisting solely of (, ), and \*. \* can represent either a (, ), or an empty string. Determine whether the parentheses are balanced.

For example, (()\* and (\*) are balanced. )\*( is not balanced.

### Problem #1378 <a id="p1378"></a>

**Easy** · Google · 2023-10-01

#### Problem

Given a binary tree of integers, find the maximum path sum between two nodes. The path must go through at least one node, and does not need to go through the root.

### Problem #1379 <a id="p1379"></a>

**Hard** · Nvidia · 2023-10-02

#### Problem

Find the maximum of two numbers without using any if-else statements, branching, or direct comparisons.

### Problem #1380 <a id="p1380"></a>

**Hard** · Google · 2023-10-03

#### Problem

Given a list of integers S and a target number k, write a function that returns a subset of S that adds up to k. If such a subset cannot be made, then return null.

Integers can appear more than once in the list. You may assume all numbers in the list are positive.

For example, given S = `[12, 1, 61, 5, 9, 2]` and k = 24, return `[12, 9, 2, 1]` since it sums up to 24.

### Problem #1381 <a id="p1381"></a>

**Medium** · Dropbox · 2023-10-04

#### Problem

Conway's Game of Life takes place on an infinite two-dimensional board of square cells. Each cell is either dead or alive, and at each tick, the following rules apply:

- Any live cell with less than two live neighbours dies.
- Any live cell with two or three live neighbours remains living.
- Any live cell with more than three live neighbours dies.
- Any dead cell with exactly three live neighbours becomes a live cell.

A cell neighbours another cell if it is horizontally, vertically, or diagonally adjacent.

Implement Conway's Game of Life. It should be able to be initialized with a starting list of live cell coordinates and the number of steps it should run for. Once initialized, it should print out the board state at each step. Since it's an infinite board, print out only the relevant coordinates, i.e. from the top-leftmost live cell to bottom-rightmost live cell.

You can represent a live cell with an asterisk (\*) and a dead cell with a dot (.).

### Problem #1382 <a id="p1382"></a>

**Medium** · Facebook · 2023-10-05

#### Problem

Describe an algorithm to compute the longest increasing subsequence of an array of numbers in `O(n log n)` time.

### Problem #1383 <a id="p1383"></a>

**Medium** · Triplebyte · 2023-10-06

#### Problem

You are given n numbers as well as n probabilities that sum up to 1. Write a function to generate one of the numbers with its corresponding probability.

For example, given the numbers `[1, 2, 3, 4]` and probabilities `[0.1, 0.5, 0.2, 0.2]`, your function should return 1 10% of the time, 2 50% of the time, and 3 and 4 20% of the time.

You can generate random numbers between 0 and 1 uniformly.

### Problem #1384 <a id="p1384"></a>

**Easy** · Riot Games · 2023-10-07

#### Problem

Design and implement a HitCounter class that keeps track of requests (or hits). It should support the following operations:

- `record(timestamp)`: records a hit that happened at timestamp
- `total()`: returns the total number of hits recorded
- `range(lower, upper)`: returns the number of hits that occurred between timestamps lower and upper (inclusive)

Follow-up: What if our system has limited memory?

### Problem #1385 <a id="p1385"></a>

**Easy** · Amazon · 2023-10-08

#### Problem

Given a N by M matrix of numbers, print out the matrix in a clockwise spiral.

For example, given the following matrix:

```
[[1,  2,  3,  4,  5],
 [6,  7,  8,  9,  10],
 [11, 12, 13, 14, 15],
 [16, 17, 18, 19, 20]]
```

You should print out the following:

```
1
2
3
4
5
10
15
20
19
18
17
16
11
6
7
8
9
14
13
12
```

### Problem #1386 <a id="p1386"></a>

**Medium** · Dropbox · 2023-10-09

#### Problem

Given the root to a binary search tree, find the second largest node in the tree.

### Problem #1387 <a id="p1387"></a>

**Easy** · Google · 2023-10-10

#### Problem

Given the root of a binary search tree, and a target K, return two nodes in the tree whose sum equals K.

For example, given the following tree and K of 20

```
   10
  /   \
5      15
      /  \
    11    15
```

Return the nodes 5 and 15.

### Problem #1388 <a id="p1388"></a>

**Medium** · Apple · 2023-10-11

#### Problem

Implement a queue using two stacks. Recall that a queue is a FIFO (first-in, first-out) data structure with the following methods: enqueue, which inserts an element into the queue, and dequeue, which removes it.

### Problem #1389 <a id="p1389"></a>

**Medium** · Quora · 2023-10-12

#### Problem

You are given a list of (website, user) pairs that represent users visiting websites. Come up with a program that identifies the top k pairs of websites with the greatest similarity.

For example, suppose k = 1, and the list of tuples is:

```
[('a', 1), ('a', 3), ('a', 5),
 ('b', 2), ('b', 6),
 ('c', 1), ('c', 2), ('c', 3), ('c', 4), ('c', 5)
 ('d', 4), ('d', 5), ('d', 6), ('d', 7),
 ('e', 1), ('e', 3), ('e': 5), ('e', 6)]
```

Then a reasonable similarity metric would most likely conclude that a and e are the most similar, so your program should return `[('a', 'e')]`.

### Problem #1390 <a id="p1390"></a>

**Medium** · Zillow · 2023-10-13

#### Problem

You are given a 2-d matrix where each cell represents number of coins in that cell. Assuming we start at `matrix[0][0]`, and can only move right or down, find the maximum number of coins you can collect by the bottom right corner.

For example, in this matrix

```
0 3 1 1
2 0 0 4
1 5 3 1
```

The most we can collect is 0 + 2 + 1 + 5 + 3 + 1 = 12 coins.

### Problem #1391 <a id="p1391"></a>

**Medium** · Square · 2023-10-14

#### Problem

Assume you have access to a function `toss_biased()` which returns 0 or 1 with a probability that's not 50-50 (but also not 0-100 or 100-0). You do not know the bias of the coin.

Write a function to simulate an unbiased coin toss.

### Problem #1392 <a id="p1392"></a>

**Hard** · Citrix · 2023-10-15

#### Problem

You are given a circular lock with three wheels, each of which display the numbers 0 through 9 in order. Each of these wheels rotate clockwise and counterclockwise.

In addition, the lock has a certain number of `"dead ends"`, meaning that if you turn the wheels to one of these combinations, the lock becomes stuck in that state and cannot be opened.

Let us consider a `"move"` to be a rotation of a single wheel by one digit, in either direction. Given a lock initially set to 000, a target combination, and a list of dead ends, write a function that returns the minimum number of moves required to reach the target state, or None if this is impossible.

### Problem #1393 <a id="p1393"></a>

**Easy** · Bloomberg · 2023-10-16

#### Problem

There are N prisoners standing in a circle, waiting to be executed. The executions are carried out starting with the kth person, and removing every successive kth person going clockwise until there is no one left.

Given N and k, write an algorithm to determine where a prisoner should stand in order to be the last survivor.

For example, if N = 5 and k = 2, the order of executions would be `[2, 4, 1, 5, 3]`, so you should return 3.

Bonus: Find an `O(log N)` solution if k = 2.

### Problem #1394 <a id="p1394"></a>

**Easy** · Google · 2023-10-17

#### Problem

Given the head of a singly linked list, reverse it in-place.

### Problem #1395 <a id="p1395"></a>

**Hard** · Google · 2023-10-18

#### Problem

Implement an LRU (Least Recently Used) cache. It should be able to be initialized with a cache size n, and contain the following methods:

- `set(key, value)`: sets key to value. If there are already n items in the cache and we are adding a new item, then it should also remove the least recently used item.
- `get(key)`: gets the value at key. If no such key exists, return null.

Each operation should run in `O(1)` time.

### Problem #1396 <a id="p1396"></a>

**Hard** · Dropbox · 2023-10-19

#### Problem

Implement an efficient string matching algorithm.

That is, given a string of length N and a pattern of length k, write a program that searches for the pattern in the string with less than `O(N * k)` worst-case time complexity.

If the pattern is found, return the start index of its location. If not, return False.

### Problem #1397 <a id="p1397"></a>

**Easy** · Google · 2023-10-20

#### Problem

Given two singly linked lists that intersect at some point, find the intersecting node. The lists are non-cyclical.

For example, given A = 3 -> 7 -> 8 -> 10 and B = 99 -> 1 -> 8 -> 10, return the node with value 8.

In this example, assume nodes with the same value are the exact same node objects.

Do this in `O(M + N)` time (where M and N are the lengths of the lists) and constant space.

### Problem #1398 <a id="p1398"></a>

**Medium** · Facebook · 2023-10-21

#### Problem

Given an array of integers, write a function to determine whether the array could become non-decreasing by modifying at most 1 element.

For example, given the array `[10, 5, 7]`, you should return true, since we can modify the 10 into a 1 to make the array non-decreasing.

Given the array `[10, 5, 1]`, you should return false, since we can't modify any one element to get a non-decreasing array.

### Problem #1399 <a id="p1399"></a>

**Medium** · Facebook · 2023-10-22

#### Problem

A builder is looking to build a row of N houses that can be of K different colors. He has a goal of minimizing cost while ensuring that no two neighboring houses are of the same color.

Given an N by K matrix where the nth row and kth column represents the cost to build the nth house with kth color, return the minimum cost which achieves this goal.

### Problem #1400 <a id="p1400"></a>

**Medium** · Indeed · 2023-10-23

#### Problem

Given a 32-bit positive integer N, determine whether it is a power of four in faster than `O(log N)` time.

### Problem #1401 <a id="p1401"></a>

**Hard** · Google · 2023-10-24

#### Problem

Implement a file syncing algorithm for two computers over a low-bandwidth network. What if we know the files in the two computers are mostly the same?

### Problem #1402 <a id="p1402"></a>

**Easy** · Amazon · 2023-10-25

#### Problem

Huffman coding is a method of encoding characters based on their frequency. Each letter is assigned a variable-length binary string, such as 0101 or 111110, where shorter lengths correspond to more common letters. To accomplish this, a binary tree is built such that the path from the root to any leaf uniquely maps to a character. When traversing the path, descending to a left child corresponds to a 0 in the prefix, while descending right corresponds to 1.

Here is an example tree (note that only the leaf nodes have letters):

```
        *
      /   \
    *       *
   / \     / \
  *   a   t   *
 /             \
c               s
```

With this encoding, cats would be represented as 0000110111.

Given a dictionary of character frequencies, build a Huffman tree, and use it to determine a mapping between characters and their encoded binary strings.

### Problem #1403 <a id="p1403"></a>

**Hard** · Two Sigma · 2023-10-26

#### Problem

Alice wants to join her school's Probability Student Club. Membership dues are computed via one of two simple probabilistic games.

The first game: roll a die repeatedly. Stop rolling once you get a five followed by a six. Your number of rolls is the amount you pay, in dollars.

The second game: same, except that the stopping condition is a five followed by a five.

Which of the two games should Alice elect to play? Does it even matter? Write a program to simulate the two games and calculate their expected value.

### Problem #1404 <a id="p1404"></a>

**Medium** · Microsoft · 2023-10-27

#### Problem

Given a string and a pattern, find the starting indices of all occurrences of the pattern in the string. For example, given the string `"abracadabra"` and the pattern `"abr"`, you should return `[0, 7]`.

### Problem #1405 <a id="p1405"></a>

**Hard** · Quantcast · 2023-10-28

#### Problem

You are presented with an array representing a Boolean expression. The elements are of two kinds:

- T and F, representing the values True and False.
- &, |, and ^, representing the bitwise operators for AND, OR, and XOR.

Determine the number of ways to group the array elements using parentheses so that the entire expression evaluates to True.

For example, suppose the input is `['F', '|', 'T', '&', 'T']`. In this case, there are two acceptable groupings: (F | T) & T and F | (T & T).

### Problem #1406 <a id="p1406"></a>

**Easy** · Facebook · 2023-10-29

#### Problem

Given a binary tree, return the level of the tree with minimum sum.

### Problem #1407 <a id="p1407"></a>

**Medium** · Mailchimp · 2023-10-30

#### Problem

You are given an array representing the heights of neighboring buildings on a city street, from east to west. The city assessor would like you to write an algorithm that returns how many of these buildings have a view of the setting sun, in order to properly value the street.

For example, given the array `[3, 7, 8, 3, 6, 1]`, you should return 3, since the top floors of the buildings with heights 8, 6, and 1 all have an unobstructed view to the west.

Can you do this using just one forward pass through the array?

### Problem #1408 <a id="p1408"></a>

**Hard** · 2023-10-31

#### Problem

Given a list, sort it using this method: `reverse(lst, i, j)`, which reverses lst from i to j.

### Problem #1409 <a id="p1409"></a>

**Easy** · Microsoft · 2023-11-01

#### Problem

The transitive closure of a graph is a measure of which vertices are reachable from other vertices. It can be represented as a matrix M, where `M[i][j]` == 1 if there is a path between vertices i and j, and otherwise 0.

For example, suppose we are given the following graph in adjacency list form:

```
graph = [
    [0, 1, 3],
    [1, 2],
    [2],
    [3]
]
```

The transitive closure of this graph would be:

```
[1, 1, 1, 1]
[0, 1, 1, 0]
[0, 0, 1, 0]
[0, 0, 0, 1]
```

Given a graph, find its transitive closure.

### Problem #1410 <a id="p1410"></a>

**Hard** · Salesforce · 2023-11-02

#### Problem

Connect 4 is a game where opponents take turns dropping red or black discs into a 7 x 6 vertically suspended grid. The game ends either when one player creates a line of four consecutive discs of their color (horizontally, vertically, or diagonally), or when there are no more spots left in the grid.

Design and implement Connect 4.

### Problem #1411 <a id="p1411"></a>

**Hard** · Google · 2023-11-03

#### Problem

Given two non-empty binary trees s and t, check whether tree t has exactly the same structure and node values with a subtree of s. A subtree of s is a tree consists of a node in s and all of this node's descendants. The tree s could also be considered as a subtree of itself.

### Problem #1412 <a id="p1412"></a>

**Medium** · Jane Street · 2023-11-04

#### Problem

Generate a finite, but an arbitrarily large binary tree quickly in `O(1)`.

That is, `generate()` should return a tree whose size is unbounded but finite.

### Problem #1413 <a id="p1413"></a>

**Medium** · Square · 2023-11-05

#### Problem

Given a string and a set of characters, return the shortest substring containing all the characters in the set.

For example, given the string `"figehaeci"` and the set of characters `{a, e, i}`, you should return `"aeci"`.

If there is no substring containing all the characters in the set, return null.

### Problem #1414 <a id="p1414"></a>

**Easy** · Bloomberg · 2023-11-07

#### Problem

Determine whether there exists a one-to-one character mapping from one string s1 to another s2.

For example, given s1 = abc and s2 = bcd, return true since we can map a to b, b to c, and c to d.

Given s1 = foo and s2 = bar, return false since the o cannot map to two characters.

### Problem #1415 <a id="p1415"></a>

**Easy** · Stripe · 2023-11-08

#### Problem

Write a function to flatten a nested dictionary. Namespace the keys with a period.

For example, given the following dictionary:

```
{
    "key": 3,
    "foo": {
        "a": 5,
        "bar": {
            "baz": 8
        }
    }
}
```

it should become:

```
{
    "key": 3,
    "foo.a": 5,
    "foo.bar.baz": 8
}
```

You can assume keys do not contain dots in them, i.e. no clobbering will occur.

### Problem #1416 <a id="p1416"></a>

**Hard** · Facebook · 2023-11-09

#### Problem

Given a string consisting of parentheses, single digits, and positive and negative signs, convert the string into a mathematical expression to obtain the answer.

Don't use eval or a similar built-in parser.

For example, given `'-1 + (2 + 3)'`, you should return 4.

### Problem #1417 <a id="p1417"></a>

**Easy** · Wayfair · 2023-11-10

#### Problem

You are given a 2 x N board, and instructed to completely cover the board with the following shapes:

- Dominoes, or 2 x 1 rectangles.
- Trominoes, or L-shapes.

For example, if N = 4, here is one possible configuration, where A is a domino, and B and C are trominoes.

```
A B B C
A B C C
```

Given an integer N, determine in how many ways this task is possible.

### Problem #1418 <a id="p1418"></a>

**Medium** · Amazon · 2023-11-11

#### Problem

Given a node in a binary search tree, return the next bigger element, also known as the inorder successor.

For example, the inorder successor of 22 is 30.

```
  10
 /  \
5    30
    /  \
  22    35
```

You can assume each node has a parent pointer.

### Problem #1419 <a id="p1419"></a>

**Medium** · Google · 2023-11-12

#### Problem

Given a string of words delimited by spaces, reverse the words in string. For example, given `"hello world here"`, return `"here world hello"`

Follow-up: given a mutable string representation, can you perform this operation in-place?

### Problem #1420 <a id="p1420"></a>

**Medium** · Amazon · 2023-11-13

#### Problem

Given a string s and an integer k, break up the string into multiple lines such that each line has a length of k or less. You must break it up so that words don't break across lines. Each line has to have the maximum possible amount of words. If there's no way to break the text up, then return null.

You can assume that there are no spaces at the ends of the string and that there is exactly one space between each word.

For example, given the string `"the quick brown fox jumps over the lazy dog"` and k = 10, you should return: `["the quick", "brown fox", "jumps over", "the lazy", "dog"]`. No string in the list has a length of more than 10.

### Problem #1421 <a id="p1421"></a>

**Hard** · Uber · 2023-11-14

#### Problem

Given an array of integers, return a new array such that each element at index i of the new array is the product of all the numbers in the original array except the one at i.

For example, if our input was `[1, 2, 3, 4, 5]`, the expected output would be `[120, 60, 40, 30, 24]`. If our input was `[3, 2, 1]`, the expected output would be `[2, 3, 6]`.

Follow-up: what if you can't use division?

### Problem #1422 <a id="p1422"></a>

**Hard** · Airbnb · 2023-11-15

#### Problem

Given a list of integers, write a function that returns the largest sum of non-adjacent numbers. Numbers can be 0 or negative.

For example, `[2, 4, 6, 2, 5]` should return 13, since we pick 2, 6, and 5. `[5, 1, 1, 5]` should return 10, since we pick 5 and 5.

Follow-up: Can you do this in `O(N)` time and constant space?

### Problem #1423 <a id="p1423"></a>

**Medium** · Google · 2023-11-16

#### Problem

You are given an array of nonnegative integers. Let's say you start at the beginning of the array and are trying to advance to the end. You can advance at most, the number of steps that you're currently on. Determine whether you can get to the end of the array.

For example, given the array `[1, 3, 1, 2, 0, 1]`, we can go from indices 0 -> 1 -> 3 -> 5, so return true.

Given the array `[1, 2, 1, 0, 0]`, we can't reach the end, so return false.

### Problem #1424 <a id="p1424"></a>

**Easy** · Microsoft · 2023-11-17

#### Problem

Implement a URL shortener with the following methods:

- `shorten(url)`, which shortens the url into a six-character alphanumeric string, such as zLg6wl.
- `restore(short)`, which expands the shortened string into the original url. If no such shortened string exists, return null.

Hint: What if we enter the same URL twice?

### Problem #1425 <a id="p1425"></a>

**Easy** · Microsoft · 2023-11-18

#### Problem

Suppose an arithmetic expression is given as a binary tree. Each leaf is an integer and each internal node is one of `'+'`, `'−'`, `'∗'`, or `'/'`.

Given the root to such a tree, write a function to evaluate it.

For example, given the following tree:

```
    *
   / \
  +    +
 / \  / \
3  2  4  5
```

You should return 45, as it is (3 + 2) \* (4 + 5).

### Problem #1426 <a id="p1426"></a>

**Hard** · Salesforce · 2023-11-19

#### Problem

Given an array of integers, find the maximum XOR of any two elements.

### Problem #1427 <a id="p1427"></a>

**Easy** · Amazon · 2023-11-20

#### Problem

Given an array and a number k that's smaller than the length of the array, rotate the array to the right k elements in-place.

### Problem #1428 <a id="p1428"></a>

**Hard** · Microsoft · 2023-11-21

#### Problem

Given an array of positive integers, divide the array into two subsets such that the difference between the sum of the subsets is as small as possible.

For example, given `[5, 10, 15, 20, 25]`, return the sets `{10, 25}` and `{5, 15, 20}`, which has a difference of 5, which is the smallest possible difference.

### Problem #1429 <a id="p1429"></a>

**Easy** · Sumo Logic · 2023-11-22

#### Problem

Given a array that's sorted but rotated at some unknown pivot, in which all elements are distinct, find a `"peak"` element in `O(log N)` time.

An element is considered a peak if it is greater than both its left and right neighbors. It is guaranteed that the first and last elements are lower than all others.

### Problem #1430 <a id="p1430"></a>

**Easy** · Facebook · 2023-11-23

#### Problem

You have a large array with most of the elements as zero.

Use a more space-efficient data structure, SparseArray, that implements the same interface:

- `init(arr, size)`: initialize with the original large array and size.
- `set(i, val)`: updates index at i with val.
- `get(i)`: gets the value at index i.

### Problem #1431 <a id="p1431"></a>

**Medium** · MongoDB · 2023-11-24

#### Problem

Given a list of elements, find the majority element, which appears more than half the time (> `floor(len(lst) / 2.0)`).

You can assume that such element exists.

For example, given `[1, 2, 1, 1, 3, 4, 0]`, return 1.

### Problem #1432 <a id="p1432"></a>

**Medium** · Snapchat · 2023-11-25

#### Problem

Given the head to a singly linked list, where each node also has a “random” pointer that points to anywhere in the linked list, deep clone the list.

### Problem #1433 <a id="p1433"></a>

**Medium** · Nest · 2023-11-26

#### Problem

Create a basic sentence checker that takes in a stream of characters and determines whether they form valid sentences. If a sentence is valid, the program should print it out.

We can consider a sentence valid if it conforms to the following rules:

1. The sentence must start with a capital letter, followed by a lowercase letter or a space.
2. All other characters must be lowercase letters, separators (,,;,:) or terminal marks (.,?,!,‽).
3. There must be a single space between each word.
4. The sentence must end with a terminal mark immediately following a word.

### Problem #1434 <a id="p1434"></a>

**Medium** · Airbnb · 2023-11-27

#### Problem

You are given a huge list of airline ticket prices between different cities around the world on a given day. These are all direct flights. Each element in the list has the format (`source_city`, destination, price).

Consider a user who is willing to take up to k connections from their origin city A to their destination B. Find the cheapest fare possible for this journey and print the itinerary for that journey.

For example, our traveler wants to go from JFK to LAX with up to 3 connections, and our input flights are as follows:

```
[
    ('JFK', 'ATL', 150),
    ('ATL', 'SFO', 400),
    ('ORD', 'LAX', 200),
    ('LAX', 'DFW', 80),
    ('JFK', 'HKG', 800),
    ('ATL', 'ORD', 90),
    ('JFK', 'LAX', 500),
]
```

Due to some improbably low flight prices, the cheapest itinerary would be JFK -> ATL -> ORD -> LAX, costing \$440.

### Problem #1435 <a id="p1435"></a>

**Medium** · Microsoft · 2023-11-28

#### Problem

Implement the singleton pattern with a twist. First, instead of storing one instance, store two instances. And in every even call of `getInstance()`, return the first instance and in every odd call of `getInstance()`, return the second instance.

### Problem #1436 <a id="p1436"></a>

**Easy** · Google · 2023-11-29

#### Problem

Given an array of elements, return the length of the longest subarray where all its elements are distinct.

For example, given the array `[5, 1, 3, 5, 2, 3, 4, 1]`, return 5 as the longest subarray of distinct elements is `[5, 2, 3, 4, 1]`.

### Problem #1437 <a id="p1437"></a>

**Hard** · Microsoft · 2023-11-30

#### Problem

Given an array of numbers, find the length of the longest increasing subsequence in the array. The subsequence does not necessarily have to be contiguous.

For example, given the array `[0, 8, 4, 12, 2, 10, 6, 14, 1, 9, 5, 13, 3, 11, 7, 15]`, the longest increasing subsequence has length 6: it is 0, 2, 6, 9, 11, 15.

### Problem #1438 <a id="p1438"></a>

**Medium** · Square · 2023-12-01

#### Problem

You are given a histogram consisting of rectangles of different heights. These heights are represented in an input list, such that `[1, 3, 2, 5]` corresponds to the following diagram:

```
      x
      x
  x   x
  x x x
x x x x
```

Determine the area of the largest rectangle that can be formed only from the bars of the histogram. For the diagram above, for example, this would be six, representing the 2 x 3 area at the bottom right.

### Problem #1439 <a id="p1439"></a>

**Easy** · Microsoft · 2023-12-02

#### Problem

Given a 2D matrix of characters and a target word, write a function that returns whether the word can be found in the matrix by going left-to-right, or up-to-down.

For example, given the following matrix:

```
[['F', 'A', 'C', 'I'],
 ['O', 'B', 'Q', 'P'],
 ['A', 'N', 'O', 'B'],
 ['M', 'A', 'S', 'S']]
```

and the target word `'FOAM'`, you should return true, since it's the leftmost column. Similarly, given the target word `'MASS'`, you should return true, since it's the last row.

### Problem #1440 <a id="p1440"></a>

**Easy** · Zillow · 2023-12-03

#### Problem

A ternary search tree is a trie-like data structure where each node may have up to three children. Here is an example which represents the words code, cob, be, ax, war, and we.

```
       c
    /  |  \
   b   o   w
 / |   |   |
a  e   d   a
|    / |   | \
x   b  e   r  e
```

The tree is structured according to the following rules:

- left child nodes link to words lexicographically earlier than the parent prefix
- right child nodes link to words lexicographically later than the parent prefix
- middle child nodes continue the current word

For instance, since code is the first word inserted in the tree, and cob lexicographically precedes cod, cob is represented as a left child extending from cod.

Implement insertion and search functions for a ternary search tree.

### Problem #1441 <a id="p1441"></a>

**Easy** · Google · 2023-12-04

#### Problem

UTF-8 is a character encoding that maps each symbol to one, two, three, or four bytes.

For example, the Euro sign, €, corresponds to the three bytes 11100010 10000010 10101100. The rules for mapping characters are as follows:

- For a single-byte character, the first bit must be zero.
- For an n-byte character, the first byte starts with n ones and a zero. The other n - 1 bytes all start with 10.

Visually, this can be represented as follows.

```
 Bytes   |           Byte format
-----------------------------------------------
   1     | 0xxxxxxx
   2     | 110xxxxx 10xxxxxx
   3     | 1110xxxx 10xxxxxx 10xxxxxx
   4     | 11110xxx 10xxxxxx 10xxxxxx 10xxxxxx
```

Write a program that takes in an array of integers representing byte values, and returns whether it is a valid UTF-8 encoding.

### Problem #1442 <a id="p1442"></a>

**Easy** · Amazon · 2023-12-05

#### Problem

Implement a stack API using only a heap. A stack implements the following methods:

- `push(item)`, which adds an element to the stack
- `pop()`, which removes and returns the most recently added element (or throws an error if there is nothing on the stack)

Recall that a heap has the following operations:

- `push(item)`, which adds a new key to the heap
- `pop()`, which removes and returns the max value of the heap

### Problem #1443 <a id="p1443"></a>

**Hard** · Quora · 2023-12-06

#### Problem

Word sense disambiguation is the problem of determining which sense a word takes on in a particular setting, if that word has multiple meanings. For example, in the sentence `"I went to get money from the bank"`, bank probably means the place where people deposit money, not the land beside a river or lake.

Suppose you are given a list of meanings for several words, formatted like so:

```
{
    "word_1": ["meaning one", "meaning two", ...],
    ...
    "word_n": ["meaning one", "meaning two", ...]
}
```

Given a sentence, most of whose words are contained in the meaning list above, create an algorithm that determines the likely sense of each possibly ambiguous word.

### Problem #1444 <a id="p1444"></a>

**Medium** · Yahoo · 2023-12-07

#### Problem

Recall that a full binary tree is one in which each node is either a leaf node, or has two children. Given a binary tree, convert it to a full one by removing nodes with only one child.

For example, given the following tree:

```
         0
      /     \
    1         2
  /            \
3                 4
  \             /   \
    5          6     7
```

You should convert it to:

```
     0
  /     \
5         4
        /   \
       6     7
```

### Problem #1445 <a id="p1445"></a>

**Easy** · Jane Street · 2023-12-08

#### Problem

The United States uses the imperial system of weights and measures, which means that there are many different, seemingly arbitrary units to measure distance. There are 12 inches in a foot, 3 feet in a yard, 22 yards in a chain, and so on .

Create a data structure that can efficiently convert a certain quantity of one unit to the correct amount of any other unit. You should also allow for additional units to be added to the system.

### Problem #1446 <a id="p1446"></a>

**Medium** · Google · 2023-12-09

#### Problem

Given a set of closed intervals, find the smallest set of numbers that covers all the intervals. If there are multiple smallest sets, return any of them.

For example, given the intervals `[0, 3]`, `[2, 6]`, `[3, 4]`, `[6, 9]`, one set of numbers that covers all these intervals is `{3, 6}`.

### Problem #1447 <a id="p1447"></a>

**Medium** · Facebook · 2023-12-10

#### Problem

Mastermind is a two-player game in which the first player attempts to guess the secret code of the second. In this version, the code may be any six-digit number with all distinct digits.

Each turn the first player guesses some number, and the second player responds by saying how many digits in this number correctly matched their location in the secret code. For example, if the secret code were 123456, then a guess of 175286 would score two, since 1 and 6 were correctly placed.

Write an algorithm which, given a sequence of guesses and their scores, determines whether there exists some secret code that could have produced them.

For example, for the following scores you should return True, since they correspond to the secret code 123456:

`{175286: 2, 293416: 3, 654321: 0}`

However, it is impossible for any key to result in the following scores, so in this case you should return False:

`{123456: 4, 345678: 4, 567890: 4}`

### Problem #1448 <a id="p1448"></a>

**Medium** · LinkedIn · 2023-12-11

#### Problem

A wall consists of several rows of bricks of various integer lengths and uniform height. Your goal is to find a vertical line going from the top to the bottom of the wall that cuts through the fewest number of bricks. If the line goes through the edge between two bricks, this does not count as a cut.

For example, suppose the input is as follows, where values in each row represent the lengths of bricks in that row:

```
[[3, 5, 1, 1],
 [2, 3, 3, 2],
 [5, 5],
 [4, 4, 2],
 [1, 3, 3, 3],
 [1, 1, 6, 1, 1]]
```

The best we can we do here is to draw a line after the eighth brick, which will only require cutting through the bricks in the third and fifth row.

Given an input consisting of brick lengths for each row such as the one above, return the fewest number of bricks that must be cut to create a vertical line.

### Problem #1449 <a id="p1449"></a>

**Medium** · Facebook · 2023-12-12

#### Problem

You are given an array of non-negative integers that represents a two-dimensional elevation map where each element is unit-width wall and the integer is the height. Suppose it will rain and all spots between two walls get filled up.

Compute how many units of water remain trapped on the map in `O(N)` time and `O(1)` space.

For example, given the input `[2, 1, 2]`, we can hold 1 unit of water in the middle.

Given the input `[3, 0, 1, 3, 0, 5]`, we can hold 3 units in the first index, 2 in the second, and 3 in the fourth index (we cannot hold 5 since it would run off to the left), so we can trap 8 units of water.

### Problem #1450 <a id="p1450"></a>

**Medium** · 2023-12-13

#### Problem

The Tower of Hanoi is a puzzle game with three rods and n disks, each a different size.

All the disks start off on the first rod in a stack. They are ordered by size, with the largest disk on the bottom and the smallest one at the top.

The goal of this puzzle is to move all the disks from the first rod to the last rod while following these rules:

- You can only move one disk at a time.
- A move consists of taking the uppermost disk from one of the stacks and placing it on top of another stack.
- You cannot place a larger disk on top of a smaller disk.

Write a function that prints out all the steps necessary to complete the Tower of Hanoi. You should assume that the rods are numbered, with the first rod being 1, the second (auxiliary) rod being 2, and the last (goal) rod being 3.

For example, with n = 3, we can do this in 7 moves:

```
Move 1 to 3
Move 1 to 2
Move 3 to 2
Move 1 to 3
Move 2 to 1
Move 2 to 3
Move 1 to 3
```

### Problem #1451 <a id="p1451"></a>

**Hard** · Palantir · 2023-12-14

#### Problem

Given a number represented by a list of digits, find the next greater permutation of a number, in terms of lexicographic ordering. If there is not greater permutation possible, return the permutation with the lowest value/ordering.

For example, the list `[1,2,3]` should return `[1,3,2]`. The list `[1,3,2]` should return `[2,1,3]`. The list `[3,2,1]` should return `[1,2,3]`.

Can you perform the operation without allocating extra memory (disregarding the input memory)?

### Problem #1452 <a id="p1452"></a>

**Medium** · Palantir · 2023-12-15

#### Problem

You are given a list of N numbers, in which each number is located at most k places away from its sorted position. For example, if k = 1, a given element at index 4 might end up at indices 3, 4, or 5.

Come up with an algorithm that sorts this list in `O(N log k)` time.

### Problem #1453 <a id="p1453"></a>

**Easy** · Twitter · 2023-12-16

#### Problem

A permutation can be specified by an array P, where `P[i]` represents the location of the element at i in the permutation. For example, `[2, 1, 0]` represents the permutation where elements at the index 0 and 2 are swapped.

Given an array and a permutation, apply the permutation to the array. For example, given the array `["a", "b", "c"]` and the permutation `[2, 1, 0]`, return `["c", "b", "a"]`.

### Problem #1454 <a id="p1454"></a>

**Medium** · Facebook · 2023-12-17

#### Problem

A graph is minimally-connected if it is connected and there is no edge that can be removed while still leaving the graph connected. For example, any binary tree is minimally-connected.

Given an undirected graph, check if the graph is minimally-connected. You can choose to represent the graph as either an adjacency matrix or adjacency list.

### Problem #1455 <a id="p1455"></a>

**Easy** · Google · 2023-12-18

#### Problem

You are given an array of arrays of integers, where each array corresponds to a row in a triangle of numbers. For example, `[[1], [2, 3], [1, 5, 1]]` represents the triangle:

```
  1
 2 3
1 5 1
```

We define a path in the triangle to start at the top and go down one row at a time to an adjacent value, eventually ending with an entry on the bottom row. For example, 1 -> 3 -> 5. The weight of the path is the sum of the entries.

Write a program that returns the weight of the maximum weight path.

### Problem #1456 <a id="p1456"></a>

**Easy** · Google · 2023-12-19

#### Problem

Given a string, return the first recurring character in it, or null if there is no recurring character.

For example, given the string `"acbbac"`, return `"b"`. Given the string `"abcdef"`, return null.

### Problem #1457 <a id="p1457"></a>

**Hard** · Google · 2023-12-20

#### Problem

Implement an LFU (Least Frequently Used) cache. It should be able to be initialized with a cache size n, and contain the following methods:

- `set(key, value)`: sets key to value. If there are already n items in the cache and we are adding a new item, then it should also remove the least frequently used item. If there is a tie, then the least recently used key should be removed.
- `get(key)`: gets the value at key. If no such key exists, return null.

Each operation should run in `O(1)` time.

### Problem #1458 <a id="p1458"></a>

**Medium** · Flipkart · 2023-12-21

#### Problem

Snakes and Ladders is a game played on a 10 x 10 board, the goal of which is get from square 1 to square 100. On each turn players will roll a six-sided die and move forward a number of spaces equal to the result. If they land on a square that represents a snake or ladder, they will be transported ahead or behind, respectively, to a new square.

Find the smallest number of turns it takes to play snakes and ladders.

For convenience, here are the squares representing snakes and ladders, and their outcomes:

```
snakes = {16: 6, 48: 26, 49: 11, 56: 53, 62: 19, 64: 60, 87: 24, 93: 73, 95: 75, 98: 78}
ladders = {1: 38, 4: 14, 9: 31, 21: 42, 28: 84, 36: 44, 51: 67, 71: 91, 80: 100}
```

### Problem #1459 <a id="p1459"></a>

**Medium** · Google · 2023-12-22

#### Problem

Invert a binary tree.

For example, given the following tree:

```
    a
   / \
  b   c
 / \  /
d   e f
```

should become:

```
 a
/ \
c  b
\  / \
 f e  d
```

### Problem #1460 <a id="p1460"></a>

**Easy** · Apple · 2023-12-23

#### Problem

A fixed point in an array is an element whose value is equal to its index. Given a sorted array of distinct elements, return a fixed point, if one exists. Otherwise, return False.

For example, given `[-6, 0, 2, 40]`, you should return 2. Given `[1, 5, 7, 8]`, you should return False.

### Problem #1461 <a id="p1461"></a>

**Medium** · Goldman Sachs · 2023-12-24

#### Problem

You are given N identical eggs and access to a building with k floors. Your task is to find the lowest floor that will cause an egg to break, if dropped from that floor. Once an egg breaks, it cannot be dropped again. If an egg breaks when dropped from the xth floor, you can assume it will also break when dropped from any floor greater than x.

Write an algorithm that finds the minimum number of trial drops it will take, in the worst case, to identify this floor.

For example, if N = 1 and k = 5, we will need to try dropping the egg at every floor, beginning with the first, until we reach the fifth floor, so our solution will be 5.

### Problem #1462 <a id="p1462"></a>

**Hard** · Jane Street · 2023-12-25

#### Problem

Given an arithmetic expression in Reverse Polish Notation , write a program to evaluate it.

The expression is given as a list of numbers and operands. For example: `[5, 3, '+']` should return 5 + 3 = 8.

For example, `[15, 7, 1, 1, '+', '-', '/', 3, '*', 2, 1, 1, '+', '+', '-']` should return 5, since it is equivalent to ((15 / (7 - (1 + 1))) \* 3) - (2 + (1 + 1)) = 5.

You can assume the given expression is always valid.

### Problem #1463 <a id="p1463"></a>

**Easy** · Netflix · 2023-12-26

#### Problem

Given an array of integers, determine whether it contains a Pythagorean triplet. Recall that a Pythogorean triplet (a, b, c) is defined by the equation a2+ b2= c2.

### Problem #1464 <a id="p1464"></a>

**Medium** · Facebook · 2023-12-27

#### Problem

Given an array of integers in which two elements appear exactly once and all other elements appear exactly twice, find the two elements that appear only once.

For example, given the array `[2, 4, 6, 8, 10, 2, 6, 10]`, return 4 and 8. The order does not matter.

Follow-up: Can you do this in linear time and constant space?

### Problem #1465 <a id="p1465"></a>

**Medium** · Square · 2023-12-28

#### Problem

In front of you is a row of N coins, with values v1, v1, ..., vn.

You are asked to play the following game. You and an opponent take turns choosing either the first or last coin from the row, removing it from the row, and receiving the value of the coin.

Write a program that returns the maximum amount of money you can win with certainty, if you move first, assuming your opponent plays optimally.

### Problem #1466 <a id="p1466"></a>

**Medium** · Google · 2023-12-29

#### Problem

You are given a set of synonyms, such as (big, large) and (eat, consume). Using this set, determine if two sentences with the same number of words are equivalent.

For example, the following two sentences are equivalent:

- `"He wants to eat food."`
- `"He wants to consume food."`

Note that the synonyms (a, b) and (a, c) do not necessarily imply (b, c): consider the case of (coach, bus) and (coach, teacher).

Follow-up: what if we can assume that (a, b) and (a, c) do in fact imply (b, c)?

### Problem #1467 <a id="p1467"></a>

**Easy** · Microsoft · 2023-12-30

#### Problem

Given a number in the form of a list of digits, return all possible permutations.

For example, given `[1,2,3]`, return `[[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]`.

### Problem #1468 <a id="p1468"></a>

**Hard** · Google · 2023-12-31

#### Problem

A knight's tour is a sequence of moves by a knight on a chessboard such that all squares are visited once.

Given N, write a function to return the number of knight's tours on an N by N chessboard.

### Problem #1469 <a id="p1469"></a>

**Hard** · Oracle · 2024-01-01

#### Problem

You are presented with an 8 by 8 matrix representing the positions of pieces on a chess board. The only pieces on the board are the black king and various white pieces. Given this matrix, determine whether the king is in check.

For details on how each piece moves, see here .

For example, given the following matrix:

```
...K....
........
.B......
......P.
.......R
..N.....
........
.....Q..
```

You should return True, since the bishop is attacking the king diagonally.

### Problem #1470 <a id="p1470"></a>

**Medium** · Facebook · 2024-01-02

#### Problem

Given a positive integer n, find the smallest number of squared integers which sum to n.

For example, given n = 13, return 2 since 13 = 32 + 22 = 9 + 4.

Given n = 27, return 3 since 27 = 32 + 32 + 32 = 9 + 9 + 9.

### Problem #1471 <a id="p1471"></a>

**Hard** · Netflix · 2024-01-03

#### Problem

Given a sorted list of integers of length N, determine if an element x is in the list without performing any multiplication, division, or bit-shift operations.

Do this in `O(log N)` time.

### Problem #1472 <a id="p1472"></a>

**Easy** · Facebook · 2024-01-04

#### Problem

Given a list of integers, return the largest product that can be made by multiplying any three integers.

For example, if the list is `[-10, -10, 5, 2]`, we should return 500, since that's -10 \* -10 \* 5.

You can assume the list has at least three integers.

### Problem #1473 <a id="p1473"></a>

**Medium** · Apple · 2024-01-05

#### Problem

Suppose you have a multiplication table that is N by N. That is, a 2D array where the value at the i-th row and j-th column is (i + 1) \* (j + 1) (if 0-indexed) or i \* j (if 1-indexed).

Given integers N and X, write a function that returns the number of times X appears as a value in an N by N multiplication table.

For example, given N = 6 and X = 12, you should return 4, since the multiplication table looks like this:

| 1 | 2 | 3 | 4 | 5 | 6 |

| 2 | 4 | 6 | 8 | 10 | 12 |

| 3 | 6 | 9 | 12 | 15 | 18 |

| 4 | 8 | 12 | 16 | 20 | 24 |

| 5 | 10 | 15 | 20 | 25 | 30 |

| 6 | 12 | 18 | 24 | 30 | 36 |

And there are 4 12's in the table.

### Problem #1474 <a id="p1474"></a>

**Easy** · Facebook · 2024-01-06

#### Problem

Boggle is a game played on a 4 x 4 grid of letters. The goal is to find as many words as possible that can be formed by a sequence of adjacent letters in the grid, using each cell at most once. Given a game board and a dictionary of valid words, implement a Boggle solver.

### Problem #1475 <a id="p1475"></a>

**Medium** · Twitter · 2024-01-07

#### Problem

Implement an autocomplete system. That is, given a query string s and a set of all possible query strings, return all strings in the set that have s as a prefix.

For example, given the query string de and the set of strings `[dog, deer, deal]`, return `[deer, deal]`.

Hint: Try preprocessing the dictionary into a more efficient data structure to speed up queries.

### Problem #1476 <a id="p1476"></a>

**Easy** · Facebook · 2024-01-08

#### Problem

Given a array of numbers representing the stock prices of a company in chronological order, write a function that calculates the maximum profit you could have made from buying and selling that stock once. You must buy before you can sell it.

For example, given `[9, 11, 8, 5, 7, 10]`, you should return 5, since you could buy the stock at 5 dollars and sell it at 10 dollars.

### Problem #1477 <a id="p1477"></a>

**Easy** · Google · 2024-01-09

#### Problem

A regular number in mathematics is defined as one which evenly divides some power of 60. Equivalently, we can say that a regular number is one whose only prime divisors are 2, 3, and 5.

These numbers have had many applications, from helping ancient Babylonians keep time to tuning instruments according to the diatonic scale.

Given an integer N, write a program that returns, in order, the first N regular numbers.

### Problem #1478 <a id="p1478"></a>

**Easy** · Google · 2024-01-10

#### Problem

Given the root of a binary tree, return a deepest node. For example, in the following tree, return d.

```
    a
   / \
  b   c
 /
d
```

### Problem #1479 <a id="p1479"></a>

**Medium** · Amazon · 2024-01-11

#### Problem

Given a pivot x, and a list lst, partition the list into three parts.

- The first part contains all elements in lst that are less than x
- The second part contains all elements in lst that are equal to x
- The third part contains all elements in lst that are larger than x

Ordering within a part can be arbitrary.

For example, given x = 10 and lst = `[9, 12, 3, 5, 14, 10, 10]`, one partition may be `[9, 3, 5, 10, 10, 12, 14]`.

### Problem #1480 <a id="p1480"></a>

**Medium** · Amazon · 2024-01-12

#### Problem

Given an array of a million integers between zero and a billion, out of order, how can you efficiently sort it? Assume that you cannot store an array of a billion elements in memory.

### Problem #1481 <a id="p1481"></a>

**Hard** · Facebook · 2024-01-13

#### Problem

Given a string and a set of delimiters, reverse the words in the string while maintaining the relative order of the delimiters. For example, given `"hello/world:here"`, return `"here/world:hello"`

Follow-up: Does your solution work for the following cases: `"hello/world:here/"`, `"hello//world:here"`

### Problem #1482 <a id="p1482"></a>

**Easy** · Palantir · 2024-01-14

#### Problem

Write a program that checks whether an integer is a palindrome. For example, 121 is a palindrome, as well as 888. 678 is not a palindrome. Do not convert the integer into a string.

### Problem #1483 <a id="p1483"></a>

**Medium** · Square · 2024-01-15

#### Problem

A competitive runner would like to create a route that starts and ends at his house, with the condition that the route goes entirely uphill at first, and then entirely downhill.

Given a dictionary of places of the form `{location: elevation}`, and a dictionary mapping paths between some of these locations to their corresponding distances, find the length of the shortest route satisfying the condition above. Assume the runner's home is location 0.

For example, suppose you are given the following input:

```
elevations = {0: 5, 1: 25, 2: 15, 3: 20, 4: 10}
paths = {
    (0, 1): 10,
    (0, 2): 8,
    (0, 3): 15,
    (1, 3): 12,
    (2, 4): 10,
    (3, 4): 5,
    (3, 0): 17,
    (4, 0): 10
}
```

In this case, the shortest valid path would be 0 -> 2 -> 4 -> 0, with a distance of 28.

### Problem #1484 <a id="p1484"></a>

**Easy** · Amazon · 2024-01-16

#### Problem

Given an integer N, construct all possible binary search trees with N nodes.

### Problem #1485 <a id="p1485"></a>

**Easy** · Amazon · 2024-01-17

#### Problem

A tree is symmetric if its data and shape remain unchanged when it is reflected about the root node. The following tree is an example:

```
        4
      / | \
    3   5   3
  /           \
9              9
```

Given a k-ary tree, determine whether it is symmetric.

### Problem #1486 <a id="p1486"></a>

**Medium** · LinkedIn · 2024-01-18

#### Problem

Given a linked list of numbers and a pivot k, partition the linked list so that all nodes less than k come before nodes greater than or equal to k.

For example, given the linked list 5 -> 1 -> 8 -> 0 -> 3 and k = 3, the solution could be 1 -> 0 -> 5 -> 8 -> 3.

### Problem #1487 <a id="p1487"></a>

**Hard** · Palantir · 2024-01-19

#### Problem

Typically, an implementation of in-order traversal of a binary tree has `O(h)` space complexity, where h is the height of the tree. Write a program to compute the in-order traversal of a binary tree using `O(1)` space.

### Problem #1488 <a id="p1488"></a>

**Hard** · Airbnb · 2024-01-20

#### Problem

We're given a hashmap associating each courseId key with a list of courseIds values, which represents that the prerequisites of courseId are courseIds. Return a sorted ordering of courses such that we can finish all courses.

Return null if there is no such ordering.

For example, given `{'CSC300': ['CSC100', 'CSC200'], 'CSC200': ['CSC100'], 'CSC100': []}`, should return `['CSC100', 'CSC200', 'CSCS300']`.

### Problem #1489 <a id="p1489"></a>

**Medium** · Stripe · 2024-01-21

#### Problem

Write a map implementation with a get function that lets you retrieve the value of a key at a particular time.

It should contain the following methods:

- `set(key, value, time)`: sets key to value for t = time.
- `get(key, time)`: gets the key at t = time.

The map should work like this. If we set a key at a particular time, it will maintain that value forever or until it gets set at a later time. In other words, when we get a key at a time, it should return the value that was set for that key set at the most recent time.

Consider the following examples:

```
d.set(1, 1, 0) # set key 1 to value 1 at time 0
d.set(1, 2, 2) # set key 1 to value 2 at time 2
d.get(1, 1) # get key 1 at time 1 should be 1
d.get(1, 3) # get key 1 at time 3 should be 2
```

```
d.set(1, 1, 5) # set key 1 to value 1 at time 5
d.get(1, 0) # get key 1 at time 0 should be null
d.get(1, 10) # get key 1 at time 10 should be 1
```

```
d.set(1, 1, 0) # set key 1 to value 1 at time 0
d.set(1, 2, 0) # set key 1 to value 2 at time 0
d.get(1, 0) # get key 1 at time 0 should be 2
```

### Problem #1490 <a id="p1490"></a>

**Medium** · Google · 2024-01-22

#### Problem

You are given an array of length n + 1 whose elements belong to the set `{1, 2, ..., n}`. By the pigeonhole principle, there must be a duplicate. Find it in linear time and space.

### Problem #1491 <a id="p1491"></a>

**Medium** · Microsoft · 2024-01-23

#### Problem

Describe and give an example of each of the following types of polymorphism:

- Ad-hoc polymorphism
- Parametric polymorphism
- Subtype polymorphism

### Problem #1492 <a id="p1492"></a>

**Easy** · Snapchat · 2024-01-24

#### Problem

Given an array of time intervals (start, end) for classroom lectures (possibly overlapping), find the minimum number of rooms required.

For example, given `[(30, 75), (0, 50), (60, 150)]`, you should return 2.

### Problem #1493 <a id="p1493"></a>

**Easy** · Netflix · 2024-01-25

#### Problem

Given an array of integers, determine whether it contains a Pythagorean triplet. Recall that a Pythogorean triplet (a, b, c) is defined by the equation a2+ b2= c2.

### Problem #1494 <a id="p1494"></a>

**Medium** · Yahoo · 2024-01-26

#### Problem

Write an algorithm that computes the reversal of a directed graph. For example, if a graph consists of A -> B -> C, it should become A \<- B \<- C.

### Problem #1495 <a id="p1495"></a>

**Hard** · Netflix · 2024-01-27

#### Problem

A Cartesian tree with sequence S is a binary tree defined by the following two properties:

- It is heap-ordered, so that each parent value is strictly less than that of its children.
- An in-order traversal of the tree produces nodes with values that correspond exactly to S.

For example, given the sequence `[3, 2, 6, 1, 9]`, the resulting Cartesian tree would be:

```
      1
    /   \
  2       9
 / \
3   6
```

Given a sequence S, construct the corresponding Cartesian tree.

### Problem #1496 <a id="p1496"></a>

**Medium** · Facebook · 2024-01-28

#### Problem

A builder is looking to build a row of N houses that can be of K different colors. He has a goal of minimizing cost while ensuring that no two neighboring houses are of the same color.

Given an N by K matrix where the nth row and kth column represents the cost to build the nth house with kth color, return the minimum cost which achieves this goal.

### Problem #1497 <a id="p1497"></a>

**Easy** · Pivotal · 2024-01-29

#### Problem

A step word is formed by taking a given word, adding a letter, and anagramming the result. For example, starting with the word `"APPLE"`, you can add an `"A"` and anagram to get `"APPEAL"`.

Given a dictionary of words and an input word, create a function that returns all valid step words.

### Problem #1498 <a id="p1498"></a>

**Medium** · Dropbox · 2024-01-30

#### Problem

Conway's Game of Life takes place on an infinite two-dimensional board of square cells. Each cell is either dead or alive, and at each tick, the following rules apply:

- Any live cell with less than two live neighbours dies.
- Any live cell with two or three live neighbours remains living.
- Any live cell with more than three live neighbours dies.
- Any dead cell with exactly three live neighbours becomes a live cell.

A cell neighbours another cell if it is horizontally, vertically, or diagonally adjacent.

Implement Conway's Game of Life. It should be able to be initialized with a starting list of live cell coordinates and the number of steps it should run for. Once initialized, it should print out the board state at each step. Since it's an infinite board, print out only the relevant coordinates, i.e. from the top-leftmost live cell to bottom-rightmost live cell.

You can represent a live cell with an asterisk (\*) and a dead cell with a dot (.).

### Problem #1499 <a id="p1499"></a>

**Medium** · Apple · 2024-01-31

#### Problem

Implement a job scheduler which takes in a function f and an integer n, and calls f after n milliseconds.

### Problem #1500 <a id="p1500"></a>

**Medium** · Amazon · 2024-02-01

#### Problem

Given a matrix of 1s and 0s, return the number of `"islands"` in the matrix. A 1 represents land and 0 represents water, so an island is a group of 1s that are neighboring whose perimeter is surrounded by water.

For example, this matrix has 4 islands.

```
1 0 0 0 0
0 0 1 1 0
0 1 1 0 0
0 0 0 0 0
1 1 0 0 1
1 1 0 0 1
```

### Problem #1501 <a id="p1501"></a>

**Easy** · Apple · 2024-02-02

#### Problem

Given the root of a binary tree, find the most frequent subtree sum. The subtree sum of a node is the sum of all values under a node, including the node itself.

For example, given the following tree:

```
  5
 / \
2  -5
```

Return 2 as it occurs twice: once as the left leaf, and once as the sum of 2 + 5 - 5.

### Problem #1502 <a id="p1502"></a>

**Hard** · Palantir · 2024-02-03

#### Problem

Given a number represented by a list of digits, find the next greater permutation of a number, in terms of lexicographic ordering. If there is not greater permutation possible, return the permutation with the lowest value/ordering.

For example, the list `[1,2,3]` should return `[1,3,2]`. The list `[1,3,2]` should return `[2,1,3]`. The list `[3,2,1]` should return `[1,2,3]`.

Can you perform the operation without allocating extra memory (disregarding the input memory)?

### Problem #1503 <a id="p1503"></a>

**Easy** · Google · 2024-02-04

#### Problem

The power set of a set is the set of all its subsets. Write a function that, given a set, generates its power set.

For example, given the set `{1, 2, 3}`, it should return `{{}, {1}, {2}, {3}, {1, 2}, {1, 3}, {2, 3}, {1, 2, 3}}`.

You may also use a list or array to represent a set.

### Problem #1504 <a id="p1504"></a>

**Hard** · Dropbox · 2024-02-05

#### Problem

Implement an efficient string matching algorithm.

That is, given a string of length N and a pattern of length k, write a program that searches for the pattern in the string with less than `O(N * k)` worst-case time complexity.

If the pattern is found, return the start index of its location. If not, return False.

### Problem #1505 <a id="p1505"></a>

**Medium** · Pinterest · 2024-02-06

#### Problem

Given an integer list where each number represents the number of hops you can make, determine whether you can reach to the last index starting at index 0.

For example, `[2, 0, 1, 0]` returns True while `[1, 1, 0, 1]` returns False.

### Problem #1506 <a id="p1506"></a>

**Hard** · Salesforce · 2024-02-07

#### Problem

Given an array of integers, find the maximum XOR of any two elements.

### Problem #1507 <a id="p1507"></a>

**Medium** · Google · 2024-02-08

#### Problem

Given the sequence of keys visited by a postorder traversal of a binary search tree, reconstruct the tree.

For example, given the sequence 2, 4, 3, 8, 7, 5, you should construct the following tree:

```
    5
   / \
  3   7
 / \   \
2   4   8
```

### Problem #1508 <a id="p1508"></a>

**Hard** · Uber · 2024-02-09

#### Problem

Given an array of integers, return a new array such that each element at index i of the new array is the product of all the numbers in the original array except the one at i.

For example, if our input was `[1, 2, 3, 4, 5]`, the expected output would be `[120, 60, 40, 30, 24]`. If our input was `[3, 2, 1]`, the expected output would be `[2, 3, 6]`.

Follow-up: what if you can't use division?

### Problem #1509 <a id="p1509"></a>

**Medium** · Facebook · 2024-02-10

#### Problem

A graph is minimally-connected if it is connected and there is no edge that can be removed while still leaving the graph connected. For example, any binary tree is minimally-connected.

Given an undirected graph, check if the graph is minimally-connected. You can choose to represent the graph as either an adjacency matrix or adjacency list.

### Problem #1510 <a id="p1510"></a>

**Hard** · Microsoft · 2024-02-11

#### Problem

Implement 3 stacks using a single list:

```
class Stack:
    def __init__(self):
        self.list = []
```

```
def pop(self, stack_number):
    pass
```

```
def push(self, item, stack_number):
    pass
```

### Problem #1511 <a id="p1511"></a>

**Medium** · Google · 2024-02-12

#### Problem

Given a binary search tree and a range `[a, b]` (inclusive), return the sum of the elements of the binary search tree within the range.

For example, given the following tree:

```
    5
   / \
  3   8
 / \ / \
2  4 6  10
```

and the range `[4, 9]`, return 23 (5 + 4 + 6 + 8).

### Problem #1512 <a id="p1512"></a>

**Easy** · Google · 2024-02-13

#### Problem

Given a string, return the first recurring character in it, or null if there is no recurring character.

For example, given the string `"acbbac"`, return `"b"`. Given the string `"abcdef"`, return null.

### Problem #1513 <a id="p1513"></a>

**Medium** · Microsoft · 2024-02-14

#### Problem

Given a string and a pattern, find the starting indices of all occurrences of the pattern in the string. For example, given the string `"abracadabra"` and the pattern `"abr"`, you should return `[0, 7]`.

### Problem #1514 <a id="p1514"></a>

**Easy** · Google · 2024-02-15

#### Problem

Given an array of elements, return the length of the longest subarray where all its elements are distinct.

For example, given the array `[5, 1, 3, 5, 2, 3, 4, 1]`, return 5 as the longest subarray of distinct elements is `[5, 2, 3, 4, 1]`.

### Problem #1515 <a id="p1515"></a>

**Medium** · Twitter · 2024-02-16

#### Problem

Given a list of numbers, create an algorithm that arranges them in order to form the largest possible integer. For example, given `[10, 7, 76, 415]`, you should return 77641510.

### Problem #1516 <a id="p1516"></a>

**Medium** · Google · 2024-02-17

#### Problem

Given a string of words delimited by spaces, reverse the words in string. For example, given `"hello world here"`, return `"here world hello"`

Follow-up: given a mutable string representation, can you perform this operation in-place?

### Problem #1517 <a id="p1517"></a>

**Easy** · Google · 2024-02-18

#### Problem

A unival tree (which stands for `"universal value"`) is a tree where all nodes under it have the same value.

Given the root to a binary tree, count the number of unival subtrees.

For example, the following tree has 5 unival subtrees:

```
  0
 / \
1   0
   / \
  1   0
 / \
1   1
```

### Problem #1518 <a id="p1518"></a>

**Easy** · Amazon · 2024-02-19

#### Problem

Huffman coding is a method of encoding characters based on their frequency. Each letter is assigned a variable-length binary string, such as 0101 or 111110, where shorter lengths correspond to more common letters. To accomplish this, a binary tree is built such that the path from the root to any leaf uniquely maps to a character. When traversing the path, descending to a left child corresponds to a 0 in the prefix, while descending right corresponds to 1.

Here is an example tree (note that only the leaf nodes have letters):

```
        *
      /   \
    *       *
   / \     / \
  *   a   t   *
 /             \
c               s
```

With this encoding, cats would be represented as 0000110111.

Given a dictionary of character frequencies, build a Huffman tree, and use it to determine a mapping between characters and their encoded binary strings.

### Problem #1519 <a id="p1519"></a>

**Hard** · Two Sigma · 2024-02-20

#### Problem

Alice wants to join her school's Probability Student Club. Membership dues are computed via one of two simple probabilistic games.

The first game: roll a die repeatedly. Stop rolling once you get a five followed by a six. Your number of rolls is the amount you pay, in dollars.

The second game: same, except that the stopping condition is a five followed by a five.

Which of the two games should Alice elect to play? Does it even matter? Write a program to simulate the two games and calculate their expected value.

### Problem #1520 <a id="p1520"></a>

**Hard** · Google · 2024-02-21

#### Problem

Given an array of integers where every integer occurs three times except for one integer, which only occurs once, find and return the non-duplicated integer.

For example, given `[6, 1, 3, 3, 3, 6, 6]`, return 1. Given `[13, 19, 13, 13]`, return 19.

Do this in `O(N)` time and `O(1)` space.

### Problem #1521 <a id="p1521"></a>

**Hard** · Google · 2024-02-22

#### Problem

Given a word W and a string S, find all starting indices in S which are anagrams of W.

For example, given that W is `"ab"`, and S is `"abxaba"`, return 0, 3, and 4.

### Problem #1522 <a id="p1522"></a>

**Medium** · Dropbox · 2024-02-23

#### Problem

Given a string s and a list of words words, where each word is the same length, find all starting indices of substrings in s that is a concatenation of every word in words exactly once.

For example, given s = `"dogcatcatcodecatdog"` and words = `["cat", "dog"]`, return `[0, 13]`, since `"dogcat"` starts at index 0 and `"catdog"` starts at index 13.

Given s = `"barfoobazbitbyte"` and words = `["dog", "cat"]`, return \[\] since there are no substrings composed of `"dog"` and `"cat"` in s.

The order of the indices does not matter.

### Problem #1523 <a id="p1523"></a>

**Medium** · Dropbox · 2024-02-24

#### Problem

Given the root to a binary search tree, find the second largest node in the tree.

### Problem #1524 <a id="p1524"></a>

**Hard** · Stripe · 2024-02-25

#### Problem

Given an array of integers, find the first missing positive integer in linear time and constant space. In other words, find the lowest positive integer that does not exist in the array. The array can contain duplicates and negative numbers as well.

For example, the input `[3, 4, -1, 1]` should give 2. The input `[1, 2, 0]` should give 3.

You can modify the input array in-place.

### Problem #1525 <a id="p1525"></a>

**Easy** · Google · 2024-02-26

#### Problem

UTF-8 is a character encoding that maps each symbol to one, two, three, or four bytes.

For example, the Euro sign, €, corresponds to the three bytes 11100010 10000010 10101100. The rules for mapping characters are as follows:

- For a single-byte character, the first bit must be zero.
- For an n-byte character, the first byte starts with n ones and a zero. The other n - 1 bytes all start with 10.

Visually, this can be represented as follows.

```
 Bytes   |           Byte format
-----------------------------------------------
   1     | 0xxxxxxx
   2     | 110xxxxx 10xxxxxx
   3     | 1110xxxx 10xxxxxx 10xxxxxx
   4     | 11110xxx 10xxxxxx 10xxxxxx 10xxxxxx
```

Write a program that takes in an array of integers representing byte values, and returns whether it is a valid UTF-8 encoding.

### Problem #1526 <a id="p1526"></a>

**Hard** · Google · 2024-02-27

#### Problem

PageRank is an algorithm used by Google to rank the importance of different websites. While there have been changes over the years, the central idea is to assign each site a score based on the importance of other pages that link to that page.

More mathematically, suppose there are N sites, and each site i has a certain count Ci of outgoing links. Then the score for a particular site Sj is defined as :

`score(Sj)` = (1 - d) / N + d \* (`score(Sx)` / Cx+ `score(Sy)` / Cy+ ... + `score(Sz)` / Cz))

Here, Sx, Sy, ..., Sz denote the scores of all the other sites that have outgoing links to Sj, and d is a damping factor, usually set to around 0.85, used to model the probability that a user will stop searching.

Given a directed graph of links between various websites, write a program that calculates each site's page rank.

### Problem #1527 <a id="p1527"></a>

**Medium** · Apple · 2024-02-28

#### Problem

Suppose you have a multiplication table that is N by N. That is, a 2D array where the value at the i-th row and j-th column is (i + 1) \* (j + 1) (if 0-indexed) or i \* j (if 1-indexed).

Given integers N and X, write a function that returns the number of times X appears as a value in an N by N multiplication table.

For example, given N = 6 and X = 12, you should return 4, since the multiplication table looks like this:

| 1 | 2 | 3 | 4 | 5 | 6 |

| 2 | 4 | 6 | 8 | 10 | 12 |

| 3 | 6 | 9 | 12 | 15 | 18 |

| 4 | 8 | 12 | 16 | 20 | 24 |

| 5 | 10 | 15 | 20 | 25 | 30 |

| 6 | 12 | 18 | 24 | 30 | 36 |

And there are 4 12's in the table.

### Problem #1528 <a id="p1528"></a>

**Medium** · Glassdoor · 2024-02-29

#### Problem

An imminent hurricane threatens the coastal town of Codeville. If at most two people can fit in a rescue boat, and the maximum weight limit for a given boat is k, determine how many boats will be needed to save everyone.

For example, given a population with weights `[100, 200, 150, 80]` and a boat limit of 200, the smallest number of boats required will be three.

### Problem #1529 <a id="p1529"></a>

**Medium** · BufferBox · 2024-03-01

#### Problem

Given a binary tree where all nodes are either 0 or 1, prune the tree so that subtrees containing all 0s are removed.

For example, given the following tree:

```
  0
 / \
1   0
   / \
  1   0
 / \
0   0
```

should be pruned to:

```
  0
 / \
1   0
   /
  1
```

We do not remove the tree at the root or its left child because it still has a 1 as a descendant.

### Problem #1530 <a id="p1530"></a>

**Easy** · Morgan Stanley · 2024-03-02

#### Problem

In Ancient Greece, it was common to write text with the first line going left to right, the second line going right to left, and continuing to go back and forth. This style was called `"boustrophedon"`.

Given a binary tree, write an algorithm to print the nodes in boustrophedon order.

For example, given the following tree:

```
       1
    /     \
  2         3
 / \       / \
4   5     6   7
```

You should return `[1, 3, 2, 4, 5, 6, 7]`.

### Problem #1531 <a id="p1531"></a>

**Easy** · IBM · 2024-03-03

#### Problem

Given a string with repeated characters, rearrange the string so that no two adjacent characters are the same. If this is not possible, return None.

For example, given `"aaabbc"`, you could return `"ababac"`. Given `"aaab"`, return None.

### Problem #1532 <a id="p1532"></a>

**Medium** · Google · 2024-03-04

#### Problem

Invert a binary tree.

For example, given the following tree:

```
    a
   / \
  b   c
 / \  /
d   e f
```

should become:

```
 a
/ \
c  b
\  / \
 f e  d
```

### Problem #1533 <a id="p1533"></a>

**Medium** · Samsung · 2024-03-05

#### Problem

A group of houses is connected to the main water plant by means of a set of pipes. A house can either be connected by a set of pipes extending directly to the plant, or indirectly by a pipe to a nearby house which is otherwise connected.

For example, here is a possible configuration, where A, B, and C are houses, and arrows represent pipes:

A \<--> B \<--> C \<--> plant

Each pipe has an associated cost, which the utility company would like to minimize. Given an undirected graph of pipe connections, return the lowest cost configuration of pipes such that each house has access to water.

In the following setup, for example, we can remove all but the pipes from plant to A, plant to B, and B to C, for a total cost of 16.

```
pipes = {
    'plant': {'A': 1, 'B': 5, 'C': 20},
    'A': {'C': 15},
    'B': {'C': 10},
    'C': {}
}
```

### Problem #1534 <a id="p1534"></a>

**Medium** · Amazon · 2024-03-06

#### Problem

Implement a bit array.

A bit array is a space efficient array that holds a value of 1 or 0 at each index.

- `init(size)`: initialize the array with size
- `set(i, val)`: updates index at i with val where val is either 1 or 0.
- `get(i)`: gets the value at index i.

### Problem #1535 <a id="p1535"></a>

**Easy** · Google · 2024-03-07

#### Problem

Given a list of numbers and a number k, return whether any two numbers from the list add up to k.

For example, given `[10, 15, 3, 7]` and k of 17, return true since 10 + 7 is 17.

Bonus: Can you do this in one pass?

### Problem #1536 <a id="p1536"></a>

**Medium** · Amazon · 2024-03-08

#### Problem

Given an array of numbers, find the maximum sum of any contiguous subarray of the array.

For example, given the array `[34, -50, 42, 14, -5, 86]`, the maximum sum would be 137, since we would take elements 42, 14, -5, and 86.

Given the array `[-5, -1, -8, -9]`, the maximum sum would be 0, since we would not take any elements.

Do this in `O(N)` time.

### Problem #1537 <a id="p1537"></a>

**Easy** · Google · 2024-03-09

#### Problem

You are given a starting state start, a list of transition probabilities for a Markov chain, and a number of steps `num_steps`. Run the Markov chain starting from start for `num_steps` and compute the number of times we visited each state.

For example, given the starting state a, number of steps 5000, and the following transition probabilities:

```
[
  ('a', 'a', 0.9),
  ('a', 'b', 0.075),
  ('a', 'c', 0.025),
  ('b', 'a', 0.15),
  ('b', 'b', 0.8),
  ('b', 'c', 0.05),
  ('c', 'a', 0.25),
  ('c', 'b', 0.25),
  ('c', 'c', 0.5)
]
```

One instance of running this Markov chain might produce `{ 'a': 3012, 'b': 1656, 'c': 332 }`.

### Problem #1538 <a id="p1538"></a>

**Easy** · Snapchat · 2024-03-10

#### Problem

Given an array of time intervals (start, end) for classroom lectures (possibly overlapping), find the minimum number of rooms required.

For example, given `[(30, 75), (0, 50), (60, 150)]`, you should return 2.

### Problem #1539 <a id="p1539"></a>

**Easy** · Amazon · 2024-03-12

#### Problem

Given a linked list, remove all consecutive nodes that sum to zero. Print out the remaining nodes.

For example, suppose you are given the input 3 -> 4 -> -7 -> 5 -> -6 -> 6. In this case, you should first remove 3 -> 4 -> -7, then -6 -> 6, leaving only 5.

### Problem #1540 <a id="p1540"></a>

**Easy** · Facebook · 2024-03-13

#### Problem

Given a list of integers, return the largest product that can be made by multiplying any three integers.

For example, if the list is `[-10, -10, 5, 2]`, we should return 500, since that's -10 \* -10 \* 5.

You can assume the list has at least three integers.

### Problem #1541 <a id="p1541"></a>

**Medium** · Facebook · 2024-03-14

#### Problem

Given a function that generates perfectly random numbers between 1 and k (inclusive), where k is an input, write a function that shuffles a deck of cards represented as an array using only swaps.

It should run in `O(N)` time.

Hint: Make sure each one of the 52! permutations of the deck is equally likely.

### Problem #1542 <a id="p1542"></a>

**Easy** · Facebook · 2024-03-15

#### Problem

Given a string of round, curly, and square open and closing brackets, return whether the brackets are balanced (well-formed).

For example, given the string `"([])[]({})"`, you should return true.

Given the string `"([)]"` or `"((()"`, you should return false.

### Problem #1543 <a id="p1543"></a>

**Medium** · Google · 2024-03-16

#### Problem

The area of a circle is defined as πr^2. Estimate π to 3 decimal places using a Monte Carlo method.

Hint: The basic equation of a circle is x2 + y2 = r2.

### Problem #1544 <a id="p1544"></a>

**Hard** · Google · 2024-03-17

#### Problem

Suppose we represent our file system by a string in the following manner:

The string `"dir\n\tsubdir1\n\tsubdir2\n\t\tfile.ext"` represents:

```
dir
    subdir1
    subdir2
        file.ext
```

The directory dir contains an empty sub-directory subdir1 and a sub-directory subdir2 containing a file file.ext.

The string `"dir\n\tsubdir1\n\t\tfile1.ext\n\t\tsubsubdir1\n\tsubdir2\n\t\tsubsubdir2\n\t\t\tfile2.ext"` represents:

```
dir
    subdir1
        file1.ext
        subsubdir1
    subdir2
        subsubdir2
            file2.ext
```

The directory dir contains two sub-directories subdir1 and subdir2. subdir1 contains a file file1.ext and an empty second-level sub-directory subsubdir1. subdir2 contains a second-level sub-directory subsubdir2 containing a file file2.ext.

We are interested in finding the longest (number of characters) absolute path to a file within our file system. For example, in the second example above, the longest absolute path is `"dir/subdir2/subsubdir2/file2.ext"`, and its length is 32 (not including the double quotes).

Given a string representing the file system in the above format, return the length of the longest absolute path to a file in the abstracted file system. If there is no file in the system, return 0.

Note:

The name of a file contains at least a period and an extension.

The name of a directory or sub-directory will not contain a period.

### Problem #1545 <a id="p1545"></a>

**Hard** · Uber · 2024-03-18

#### Problem

You have N stones in a row, and would like to create from them a pyramid. This pyramid should be constructed such that the height of each stone increases by one until reaching the tallest stone, after which the heights decrease by one. In addition, the start and end stones of the pyramid should each be one stone high.

You can change the height of any stone by paying a cost of 1 unit to lower its height by 1, as many times as necessary. Given this information, determine the lowest cost method to produce this pyramid.

For example, given the stones `[1, 1, 3, 3, 2, 1]`, the optimal solution is to pay 2 to create `[0, 1, 2, 3, 2, 1]`.

### Problem #1546 <a id="p1546"></a>

**Medium** · LinkedIn · 2024-03-19

#### Problem

Given a linked list of numbers and a pivot k, partition the linked list so that all nodes less than k come before nodes greater than or equal to k.

For example, given the linked list 5 -> 1 -> 8 -> 0 -> 3 and k = 3, the solution could be 1 -> 0 -> 5 -> 8 -> 3.

### Problem #1547 <a id="p1547"></a>

**Hard** · Google · 2024-03-20

#### Problem

An XOR linked list is a more memory efficient doubly linked list. Instead of each node holding next and prev fields, it holds a field named both, which is an XOR of the next node and the previous node. Implement an XOR linked list; it has an `add(element)` which adds the element to the end, and a `get(index)` which returns the node at index.

If using a language that has no pointers (such as Python), you can assume you have access to `get_pointer` and `dereference_pointer` functions that converts between nodes and memory addresses.

### Problem #1548 <a id="p1548"></a>

**Easy** · Atlassian · 2024-03-21

#### Problem

MegaCorp wants to give bonuses to its employees based on how many lines of codes they have written. They would like to give the smallest positive amount to each worker consistent with the constraint that if a developer has written more lines of code than their neighbor, they should receive more money.

Given an array representing a line of seats of employees at MegaCorp, determine how much each one should get paid.

For example, given `[10, 40, 200, 1000, 60, 30]`, you should return `[1, 2, 3, 4, 2, 1]`.

### Problem #1549 <a id="p1549"></a>

**Easy** · Google · 2024-03-22

#### Problem

In linear algebra, a Toeplitz matrix is one in which the elements on any given diagonal from top left to bottom right are identical.

Here is an example:

```
1 2 3 4 8
5 1 2 3 4
4 5 1 2 3
7 4 5 1 2
```

Write a program to determine whether a given input is a Toeplitz matrix.

### Problem #1550 <a id="p1550"></a>

**Easy** · Dropbox · 2024-03-23

#### Problem

What does the below code snippet print out? How can we fix the anonymous functions to behave as we'd expect?

```
functions = []
for i in range(10):
    functions.append(lambda : i)
```

```
for f in functions:
    print(f())
```

### Problem #1551 <a id="p1551"></a>

**Medium** · Nest · 2024-03-24

#### Problem

Create a basic sentence checker that takes in a stream of characters and determines whether they form valid sentences. If a sentence is valid, the program should print it out.

We can consider a sentence valid if it conforms to the following rules:

1. The sentence must start with a capital letter, followed by a lowercase letter or a space.
2. All other characters must be lowercase letters, separators (,,;,:) or terminal marks (.,?,!,‽).
3. There must be a single space between each word.
4. The sentence must end with a terminal mark immediately following a word.

### Problem #1552 <a id="p1552"></a>

**Easy** · Facebook · 2024-03-25

#### Problem

Given a array of numbers representing the stock prices of a company in chronological order, write a function that calculates the maximum profit you could have made from buying and selling that stock once. You must buy before you can sell it.

For example, given `[9, 11, 8, 5, 7, 10]`, you should return 5, since you could buy the stock at 5 dollars and sell it at 10 dollars.

### Problem #1553 <a id="p1553"></a>

**Hard** · Airbnb · 2024-03-26

#### Problem

You come across a dictionary of sorted words in a language you've never seen before. Write a program that returns the correct order of letters in this language.

For example, given `['xww', 'wxyz', 'wxyw', 'ywx', 'ywz']`, you should return `['x', 'z', 'w', 'y']`.

### Problem #1554 <a id="p1554"></a>

**Easy** · Amazon · 2024-03-27

#### Problem

Given a string, determine whether any permutation of it is a palindrome.

For example, carrace should return true, since it can be rearranged to form racecar, which is a palindrome. daily should return false, since there's no rearrangement that can form a palindrome.

### Problem #1555 <a id="p1555"></a>

**Medium** · Jane Street · 2024-03-28

#### Problem

Generate a finite, but an arbitrarily large binary tree quickly in `O(1)`.

That is, `generate()` should return a tree whose size is unbounded but finite.

### Problem #1556 <a id="p1556"></a>

**Easy** · Salesforce · 2024-03-29

#### Problem

Write a program to merge two binary trees. Each node in the new tree should hold a value equal to the sum of the values of the corresponding nodes of the input trees.

If only one input tree has a node in a given position, the corresponding node in the new tree should match that input node.

### Problem #1557 <a id="p1557"></a>

**Medium** · Zillow · 2024-03-30

#### Problem

You are given a 2-d matrix where each cell represents number of coins in that cell. Assuming we start at `matrix[0][0]`, and can only move right or down, find the maximum number of coins you can collect by the bottom right corner.

For example, in this matrix

```
0 3 1 1
2 0 0 4
1 5 3 1
```

The most we can collect is 0 + 2 + 1 + 5 + 3 + 1 = 12 coins.

### Problem #1558 <a id="p1558"></a>

**Easy** · Twitter · 2024-03-31

#### Problem

A classroom consists of N students, whose friendships can be represented in an adjacency list. For example, the following descibes a situation where 0 is friends with 1 and 2, 3 is friends with 6, and so on.

```
{0: [1, 2],
 1: [0, 5],
 2: [0],
 3: [6],
 4: [],
 5: [1],
 6: [3]}
```

Each student can be placed in a friend group, which can be defined as the transitive closure of that student's friendship relations. In other words, this is the smallest set such that no student in the group has any friends outside this group. For the example above, the friend groups would be `{0, 1, 2, 5}`, `{3, 6}`, `{4}`.

Given a friendship list such as the one above, determine the number of friend groups in the class.

### Problem #1559 <a id="p1559"></a>

**Hard** · LinkedIn · 2024-04-01

#### Problem

You are given a binary tree in a peculiar string representation. Each node is written in the form (lr), where l corresponds to the left child and r corresponds to the right child.

If either l or r is null, it will be represented as a zero. Otherwise, it will be represented by a new (lr) pair.

Here are a few examples:

- A root node with no children: (00)
- A root node with two children: ((00)(00))
- An unbalanced tree with three consecutive left children: ((((00)0)0)0)

Given this representation, determine the depth of the tree.

### Problem #1560 <a id="p1560"></a>

**Medium** · Google · 2024-04-02

#### Problem

Given a set of closed intervals, find the smallest set of numbers that covers all the intervals. If there are multiple smallest sets, return any of them.

For example, given the intervals `[0, 3]`, `[2, 6]`, `[3, 4]`, `[6, 9]`, one set of numbers that covers all these intervals is `{3, 6}`.

### Problem #1561 <a id="p1561"></a>

**Easy** · Microsoft · 2024-04-03

#### Problem

Print the nodes in a binary tree level-wise. For example, the following should print 1, 2, 3, 4, 5.

```
  1
 / \
2   3
   / \
  4   5
```

### Problem #1562 <a id="p1562"></a>

**Easy** · Amazon · 2024-04-04

#### Problem

Consider the following scenario: there are N mice and N holes placed at integer points along a line. Given this, find a method that maps mice to holes such that the largest number of steps any mouse takes is minimized.

Each move consists of moving one mouse one unit to the left or right, and only one mouse can fit inside each hole.

For example, suppose the mice are positioned at `[1, 4, 9, 15]`, and the holes are located at `[10, -5, 0, 16]`. In this case, the best pairing would require us to send the mouse at 1 to the hole at -5, so our function should return 6.

### Problem #1563 <a id="p1563"></a>

**Medium** · Facebook · 2024-04-05

#### Problem

There is an N by M matrix of zeroes. Given N and M, write a function to count the number of ways of starting at the top-left corner and getting to the bottom-right corner. You can only move right or down.

For example, given a 2 by 2 matrix, you should return 2, since there are two ways to get to the bottom-right:

- Right, then down
- Down, then right

Given a 5 by 5 matrix, there are 70 ways to get to the bottom-right.

### Problem #1564 <a id="p1564"></a>

**Hard** · Google · 2024-04-06

#### Problem

Let A be an N by M matrix in which every row and every column is sorted.

Given i1, j1, i2, and j2, compute the number of elements of M smaller than `M[i1, j1]` and larger than `M[i2, j2]`.

For example, given the following matrix:

```
[[1, 3, 7, 10, 15, 20],
 [2, 6, 9, 14, 22, 25],
 [3, 8, 10, 15, 25, 30],
 [10, 11, 12, 23, 30, 35],
 [20, 25, 30, 35, 40, 45]]
```

And i1 = 1, j1 = 1, i2 = 3, j2 = 3, return 15 as there are 15 numbers in the matrix smaller than 6 or greater than 23.

### Problem #1565 <a id="p1565"></a>

**Medium** · Facebook · 2024-04-07

#### Problem

Given a multiset of integers, return whether it can be partitioned into two subsets whose sums are the same.

For example, given the multiset `{15, 5, 20, 10, 35, 15, 10}`, it would return true, since we can split it up into `{15, 5, 10, 15, 10}` and `{20, 35}`, which both add up to 55.

Given the multiset `{15, 5, 20, 10, 35}`, it would return false, since we can't split it up into two subsets that add up to the same sum.

### Problem #1566 <a id="p1566"></a>

**Hard** · Amazon · 2024-04-08

#### Problem

Given a string, find the longest palindromic contiguous substring. If there are more than one with the maximum length, return any one.

For example, the longest palindromic substring of `"aabcdcb"` is `"bcdcb"`. The longest palindromic substring of `"bananas"` is `"anana"`.

### Problem #1567 <a id="p1567"></a>

**Easy** · Amazon · 2024-04-09

#### Problem

You are given a list of data entries that represent entries and exits of groups of people into a building. An entry looks like this:

`{"timestamp": 1526579928, count: 3, "type": "enter"}`

This means 3 people entered the building. An exit looks like this:

`{"timestamp": 1526580382, count: 2, "type": "exit"}`

This means that 2 people exited the building. timestamp is in Unix time .

Find the busiest period in the building, that is, the time with the most people in the building. Return it as a pair of (start, end) timestamps. You can assume the building always starts off and ends up empty, i.e. with 0 people inside.

### Problem #1568 <a id="p1568"></a>

**Easy** · Stripe · 2024-04-10

#### Problem

Given an integer n, return the length of the longest consecutive run of 1s in its binary representation.

For example, given 156, you should return 3.

### Problem #1569 <a id="p1569"></a>

**Easy** · Microsoft · 2024-04-11

#### Problem

Implement a URL shortener with the following methods:

- `shorten(url)`, which shortens the url into a six-character alphanumeric string, such as zLg6wl.
- `restore(short)`, which expands the shortened string into the original url. If no such shortened string exists, return null.

Hint: What if we enter the same URL twice?

### Problem #1570 <a id="p1570"></a>

**Medium** · Epic · 2024-04-12

#### Problem

The `"look and say"` sequence is defined as follows: beginning with the term 1, each subsequent term visually describes the digits appearing in the previous term. The first few terms are as follows:

```
1
11
21
1211
111221
```

As an example, the fourth term is 1211, since the third term consists of one 2 and one 1.

Given an integer N, print the Nth term of this sequence.

### Problem #1571 <a id="p1571"></a>

**Medium** · Microsoft · 2024-04-13

#### Problem

Given a dictionary of words and a string made up of those words (no spaces), return the original sentence in a list. If there is more than one possible reconstruction, return any of them. If there is no possible reconstruction, then return null.

For example, given the set of words `'quick'`, `'brown'`, `'the'`, `'fox'`, and the string `"thequickbrownfox"`, you should return `['the', 'quick', 'brown', 'fox']`.

Given the set of words `'bed'`, `'bath'`, `'bedbath'`, `'and'`, `'beyond'`, and the string `"bedbathandbeyond"`, return either `['bed', 'bath', 'and', 'beyond]` or `['bedbath', 'and', 'beyond']`.

### Problem #1572 <a id="p1572"></a>

**Easy** · Yelp · 2024-04-14

#### Problem

Given a mapping of digits to letters (as in a phone number), and a digit string, return all possible letters the number could represent. You can assume each valid number in the mapping is a single digit.

For example if `{“2”: [“a”, “b”, “c”], 3: [“d”, “e”, “f”], …}` then “23” should return \[“ad”, “ae”, “af”, “bd”, “be”, “bf”, “cd”, “ce”, “cf"\].

### Problem #1573 <a id="p1573"></a>

**Easy** · Sumo Logic · 2024-04-15

#### Problem

Given a array that's sorted but rotated at some unknown pivot, in which all elements are distinct, find a `"peak"` element in `O(log N)` time.

An element is considered a peak if it is greater than both its left and right neighbors. It is guaranteed that the first and last elements are lower than all others.

### Problem #1574 <a id="p1574"></a>

**Medium** · Square · 2024-04-16

#### Problem

Given a list of words, return the shortest unique prefix of each word. For example, given the list:

- dog
- cat
- apple
- apricot
- fish

Return the list:

- d
- c
- app
- apr
- f

### Problem #1575 <a id="p1575"></a>

**Medium** · Google · 2024-04-17

#### Problem

A cryptarithmetic puzzle is a mathematical game where the digits of some numbers are represented by letters. Each letter represents a unique digit.

For example, a puzzle of the form:

```
  SEND
+ MORE
--------
 MONEY
```

may have the solution:

`{'S': 9, 'E': 5, 'N': 6, 'D': 7, 'M': 1, 'O', 0, 'R': 8, 'Y': 2}`

Given a three-word puzzle like the one above, create an algorithm that finds a solution.

### Problem #1576 <a id="p1576"></a>

**Hard** · MIT · 2024-04-18

#### Problem

Blackjack is a two player card game whose rules are as follows:

- The player and then the dealer are each given two cards.
- The player can then `"hit"`, or ask for arbitrarily many additional cards, so long as their total does not exceed 21.
- The dealer must then hit if their total is 16 or lower, otherwise pass.
- Finally, the two compare totals, and the one with the greatest sum not exceeding 21 is the winner.

For this problem, cards values are counted as follows: each card between 2 and 10 counts as their face value, face cards count as 10, and aces count as 1.

Given perfect knowledge of the sequence of cards in the deck, implement a blackjack solver that maximizes the player's score (that is, wins minus losses).

### Problem #1577 <a id="p1577"></a>

**Hard** · Oracle · 2024-04-19

#### Problem

We say a number is sparse if there are no adjacent ones in its binary representation. For example, 21 (10101) is sparse, but 22 (10110) is not. For a given input N, find the smallest sparse number greater than or equal to N.

Do this in faster than `O(N log N)` time.

### Problem #1578 <a id="p1578"></a>

**Hard** · Facebook · 2024-04-20

#### Problem

Given an array of numbers of length N, find both the minimum and maximum using less than 2 \* (N - 2) comparisons.

### Problem #1579 <a id="p1579"></a>

**Medium** · Facebook · 2024-04-21

#### Problem

Given a circular array, compute its maximum subarray sum in `O(n)` time. A subarray can be empty, and in this case the sum is 0.

For example, given `[8, -1, 3, 4]`, return 15 as we choose the numbers 3, 4, and 8 where the 8 is obtained from wrapping around.

Given `[-4, 5, 1, 0]`, return 6 as we choose the numbers 5 and 1.

### Problem #1580 <a id="p1580"></a>

**Medium** · Google · 2024-04-22

#### Problem

Given a set of distinct positive integers, find the largest subset such that every pair of elements in the subset (i, j) satisfies either i % j = 0 or j % i = 0.

For example, given the set `[3, 5, 10, 20, 21]`, you should return `[5, 10, 20]`. Given `[1, 3, 6, 24]`, return `[1, 3, 6, 24]`.

### Problem #1581 <a id="p1581"></a>

**Easy** · Google · 2024-04-23

#### Problem

Given two rectangles on a 2D graph, return the area of their intersection. If the rectangles don't intersect, return 0.

For example, given the following rectangles:

```
{
    "top_left": (1, 4),
    "dimensions": (3, 3) # width, height
}
```

and

```
{
    "top_left": (0, 5),
    "dimensions": (4, 3) # width, height
}
```

return 6.

### Problem #1582 <a id="p1582"></a>

**Hard** · Salesforce · 2024-04-24

#### Problem

Connect 4 is a game where opponents take turns dropping red or black discs into a 7 x 6 vertically suspended grid. The game ends either when one player creates a line of four consecutive discs of their color (horizontally, vertically, or diagonally), or when there are no more spots left in the grid.

Design and implement Connect 4.

### Problem #1583 <a id="p1583"></a>

**Medium** · Yelp · 2024-04-25

#### Problem

The horizontal distance of a binary tree node describes how far left or right the node will be when the tree is printed out.

More rigorously, we can define it as follows:

- The horizontal distance of the root is 0.
- The horizontal distance of a left child is `hd(parent)` - 1.
- The horizontal distance of a right child is `hd(parent)` + 1.

For example, for the following tree, `hd(1)` = -2, and `hd(6)` = 0.

```
           5
        /     \
      3         7
    /  \      /   \
  1     4    6     9
 /                /
0                8
```

The bottom view of a tree, then, consists of the lowest node at each horizontal distance. If there are two nodes at the same depth and horizontal distance, either is acceptable.

For this tree, for example, the bottom view could be `[0, 1, 3, 6, 8, 9]`.

Given the root to a binary tree, return its bottom view.

### Problem #1584 <a id="p1584"></a>

**Medium** · Uber · 2024-04-26

#### Problem

Implement a 2D iterator class. It will be initialized with an array of arrays, and should implement the following methods:

- `next()`: returns the next element in the array of arrays. If there are no more elements, raise an exception.
- `has_next()`: returns whether or not the iterator still has elements left.

For example, given the input `[[1, 2], [3], [], [4, 5, 6]]`, calling `next()` repeatedly should output 1, 2, 3, 4, 5, 6.

Do not use flatten or otherwise clone the arrays. Some of the arrays can be empty.

### Problem #1585 <a id="p1585"></a>

**Easy** · Square · 2024-04-27

#### Problem

The Sieve of Eratosthenes is an algorithm used to generate all prime numbers smaller than N. The method is to take increasingly larger prime numbers, and mark their multiples as composite.

For example, to find all primes less than 100, we would first mark `[4, 6, 8, ...]` (multiples of two), then `[6, 9, 12, ...]` (multiples of three), and so on. Once we have done this for all primes less than N, the unmarked numbers that remain will be prime.

Implement this algorithm.

Bonus: Create a generator that produces primes indefinitely (that is, without taking N as an input).

### Problem #1586 <a id="p1586"></a>

**Hard** · Airbnb · 2024-04-28

#### Problem

An 8-puzzle is a game played on a 3 x 3 board of tiles, with the ninth tile missing. The remaining tiles are labeled 1 through 8 but shuffled randomly. Tiles may slide horizontally or vertically into an empty space, but may not be removed from the board.

Design a class to represent the board, and find a series of steps to bring the board to the state `[[1, 2, 3], [4, 5, 6], [7, 8, None]]`.

### Problem #1587 <a id="p1587"></a>

**Medium** · Yelp · 2024-04-29

#### Problem

You are given an array of integers, where each element represents the maximum number of steps that can be jumped going forward from that element. Write a function to return the minimum number of jumps you must take in order to get from the start to the end of the array.

For example, given `[6, 2, 4, 0, 5, 1, 1, 4, 2, 9]`, you should return 2, as the optimal solution involves jumping from 6 to 5, and then from 5 to 9.

### Problem #1588 <a id="p1588"></a>

**Medium** · Slack · 2024-04-30

#### Problem

You are given an N by M matrix of 0s and 1s. Starting from the top left corner, how many ways are there to reach the bottom right corner?

You can only move right and down. 0 represents an empty space while 1 represents a wall you cannot walk through.

For example, given the following matrix:

```
[[0, 0, 1],
 [0, 0, 1],
 [1, 0, 0]]
```

Return two, as there are only two ways to get to the bottom right:

- Right, down, down, right
- Down, right, down, right

The top left corner and bottom right corner will always be 0.

### Problem #1589 <a id="p1589"></a>

**Easy** · Facebook · 2024-05-01

#### Problem

On a mysterious island there are creatures known as Quxes which come in three colors: red, green, and blue. One power of the Qux is that if two of them are standing next to each other, they can transform into a single creature of the third color.

Given N Quxes standing in a line, determine the smallest number of them remaining after any possible sequence of such transformations.

For example, given the input `['R', 'G', 'B', 'G', 'B']`, it is possible to end up with a single Qux through the following steps:

```
        Arrangement       |   Change
----------------------------------------
['R', 'G', 'B', 'G', 'B'] | (R, G) -> B
['B', 'B', 'G', 'B']      | (B, G) -> R
['B', 'R', 'B']           | (R, B) -> G
['B', 'G']                | (B, G) -> R
['R']                     |
```

### Problem #1590 <a id="p1590"></a>

**Medium** · Amazon · 2024-05-02

#### Problem

Given a string, find the length of the smallest window that contains every distinct character. Characters may appear more than once in the window.

For example, given `"jiujitsu"`, you should return 5, corresponding to the final five letters.

### Problem #1591 <a id="p1591"></a>

**Hard** · Google · 2024-05-03

#### Problem

Given a list of integers S and a target number k, write a function that returns a subset of S that adds up to k. If such a subset cannot be made, then return null.

Integers can appear more than once in the list. You may assume all numbers in the list are positive.

For example, given S = `[12, 1, 61, 5, 9, 2]` and k = 24, return `[12, 9, 2, 1]` since it sums up to 24.

### Problem #1592 <a id="p1592"></a>

**Easy** · Microsoft · 2024-05-04

#### Problem

Using a `read7()` method that returns 7 characters from a file, implement `readN(n)` which reads n characters.

For example, given a file with the content “Hello world”, three `read7()` returns “Hello w”, “orld” and then “”.

### Problem #1593 <a id="p1593"></a>

**Easy** · Two Sigma · 2024-05-05

#### Problem

Using a function `rand7()` that returns an integer from 1 to 7 (inclusive) with uniform probability, implement a function `rand5()` that returns an integer from 1 to 5 (inclusive).

### Problem #1594 <a id="p1594"></a>

**Easy** · Google · 2024-05-06

#### Problem

You are given given a list of rectangles represented by min and max x- and y-coordinates. Compute whether or not a pair of rectangles overlap each other. If one rectangle completely covers another, it is considered overlapping.

For example, given the following rectangles:

```
{
    "top_left": (1, 4),
    "dimensions": (3, 3) # width, height
},
{
    "top_left": (-1, 3),
    "dimensions": (2, 1)
},
{
    "top_left": (0, 5),
    "dimensions": (4, 3)
}
```

return true as the first and third rectangle overlap each other.

### Problem #1595 <a id="p1595"></a>

**Hard** · Google · 2024-05-07

#### Problem

Given a string, split it into as few strings as possible such that each string is a palindrome.

For example, given the input string racecarannakayak, return `["racecar", "anna", "kayak"]`.

Given the input string abc, return `["a", "b", "c"]`.

### Problem #1596 <a id="p1596"></a>

**Easy** · Amazon · 2024-05-08

#### Problem

A tree is symmetric if its data and shape remain unchanged when it is reflected about the root node. The following tree is an example:

```
        4
      / | \
    3   5   3
  /           \
9              9
```

Given a k-ary tree, determine whether it is symmetric.

### Problem #1597 <a id="p1597"></a>

**Medium** · Google · 2024-05-09

#### Problem

Given pre-order and in-order traversals of a binary tree, write a function to reconstruct the tree.

For example, given the following preorder traversal:

`[a, b, d, e, c, f, g]`

And the following inorder traversal:

`[d, b, e, a, f, c, g]`

You should return the following tree:

```
    a
   / \
  b   c
 / \ / \
d  e f  g
```

### Problem #1598 <a id="p1598"></a>

**Hard** · Airbnb · 2024-05-10

#### Problem

You are given an array X of floating-point numbers x1, x2, ... xn. These can be rounded up or down to create a corresponding array Y of integers y1, y2, ... yn.

Write an algorithm that finds an appropriate Y array with the following properties:

- The rounded sums of both arrays should be equal.
- The absolute pairwise difference between elements is minimized. In other words, |x1- y1| + |x2- y2| + ... + |xn- yn| should be as small as possible.

For example, suppose your input is `[1.3, 2.3, 4.4]`. In this case you cannot do better than `[1, 2, 5]`, which has an absolute difference of |1.3 - 1| + |2.3 - 2| + |4.4 - 5| = 1.

### Problem #1599 <a id="p1599"></a>

**Medium** · Snapchat · 2024-05-11

#### Problem

Given a string of digits, generate all possible valid IP address combinations.

IP addresses must follow the format A.B.C.D, where A, B, C, and D are numbers between 0 and 255. Zero-prefixed numbers, such as 01 and 065, are not allowed, except for 0 itself.

For example, given `"2542540123"`, you should return `['254.25.40.123', '254.254.0.123']`.

### Problem #1600 <a id="p1600"></a>

**Easy** · Amazon · 2024-05-12

#### Problem

Implement a stack API using only a heap. A stack implements the following methods:

- `push(item)`, which adds an element to the stack
- `pop()`, which removes and returns the most recently added element (or throws an error if there is nothing on the stack)

Recall that a heap has the following operations:

- `push(item)`, which adds a new key to the heap
- `pop()`, which removes and returns the max value of the heap

### Problem #1601 <a id="p1601"></a>

**Easy** · Apple · 2024-05-13

#### Problem

Given a binary tree, find a minimum path sum from root to a leaf.

For example, the minimum path in this tree is `[10, 5, 1, -1]`, which has sum 15.

```
  10
 /  \
5    5
 \     \
   2    1
       /
     -1
```

### Problem #1602 <a id="p1602"></a>

**Medium** · LinkedIn · 2024-05-14

#### Problem

You are given a string consisting of the letters x and y, such as xyxxxyxyy. In addition, you have an operation called flip, which changes a single x to y or vice versa.

Determine how many times you would need to apply this operation to ensure that all x's come before all y's. In the preceding example, it suffices to flip the second and sixth characters, so you should return 2.

### Problem #1603 <a id="p1603"></a>

**Easy** · Pivotal · 2024-05-15

#### Problem

Write an algorithm that finds the total number of set bits in all integers between 1 and N.

### Problem #1604 <a id="p1604"></a>

**Easy** · Oracle · 2024-05-16

#### Problem

Given a binary search tree, find the floor and ceiling of a given integer. The floor is the highest element in the tree less than or equal to an integer, while the ceiling is the lowest element in the tree greater than or equal to an integer.

If either value does not exist, return None.

### Problem #1605 <a id="p1605"></a>

**Easy** · Wayfair · 2024-05-17

#### Problem

You are given a 2 x N board, and instructed to completely cover the board with the following shapes:

- Dominoes, or 2 x 1 rectangles.
- Trominoes, or L-shapes.

For example, if N = 4, here is one possible configuration, where A is a domino, and B and C are trominoes.

```
A B B C
A B C C
```

Given an integer N, determine in how many ways this task is possible.

### Problem #1606 <a id="p1606"></a>

**Easy** · PayPal · 2024-05-18

#### Problem

Given a binary tree, determine whether or not it is height-balanced. A height-balanced binary tree can be defined as one in which the heights of the two subtrees of any node never differ by more than one.

### Problem #1607 <a id="p1607"></a>

**Easy** · Bloomberg · 2024-05-19

#### Problem

Determine whether there exists a one-to-one character mapping from one string s1 to another s2.

For example, given s1 = abc and s2 = bcd, return true since we can map a to b, b to c, and c to d.

Given s1 = foo and s2 = bar, return false since the o cannot map to two characters.

### Problem #1608 <a id="p1608"></a>

**Medium** · Microsoft · 2024-05-20

#### Problem

Write a program to determine how many distinct ways there are to create a max heap from a list of N given integers.

For example, if N = 3, and our integers are `[1, 2, 3]`, there are two ways, shown below.

```
  3      3
 / \    / \
1   2  2   1
```

### Problem #1609 <a id="p1609"></a>

**Hard** · Twitter · 2024-05-21

#### Problem

You are given an array of length 24, where each element represents the number of new subscribers during the corresponding hour. Implement a data structure that efficiently supports the following:

- `update(hour: int, value: int)`: Increment the element at index hour by value.
- `query(start: int, end: int)`: Retrieve the number of subscribers that have signed up between start and end (inclusive).

You can assume that all values get cleared at the end of the day, and that you will not be asked for start and end values that wrap around midnight.

### Problem #1610 <a id="p1610"></a>

**Medium** · Facebook · 2024-05-22

#### Problem

Given a stream of elements too large to store in memory, pick a random element from the stream with uniform probability.

### Problem #1611 <a id="p1611"></a>

**Easy** · Yahoo · 2024-05-23

#### Problem

You are given a string of length N and a parameter k. The string can be manipulated by taking one of the first k letters and moving it to the end.

Write a program to determine the lexicographically smallest string that can be created after an unlimited number of moves.

For example, suppose we are given the string daily and k = 1. The best we can create in this case is ailyd.

### Problem #1612 <a id="p1612"></a>

**Hard** · Etsy · 2024-05-24

#### Problem

Given a sorted array, convert it into a height-balanced binary search tree.

### Problem #1613 <a id="p1613"></a>

**Hard** · VMware · 2024-05-25

#### Problem

The skyline of a city is composed of several buildings of various widths and heights, possibly overlapping one another when viewed from a distance. We can represent the buildings using an array of (left, right, height) tuples, which tell us where on an imaginary x-axis a building begins and ends, and how tall it is. The skyline itself can be described by a list of (x, height) tuples, giving the locations at which the height visible to a distant observer changes, and each new height.

Given an array of buildings as described above, create a function that returns the skyline.

For example, suppose the input consists of the buildings `[(0, 15, 3), (4, 11, 5), (19, 23, 4)]`. In aggregate, these buildings would create a skyline that looks like the one below.

### Problem #1614 <a id="p1614"></a>

**Medium** · Facebook · 2024-05-26

#### Problem

Given the mapping a = 1, b = 2, ... z = 26, and an encoded message, count the number of ways it can be decoded.

For example, the message `'111'` would give 3, since it could be decoded as `'aaa'`, `'ka'`, and `'ak'`.

You can assume that the messages are decodable. For example, `'001'` is not allowed.

### Problem #1615 <a id="p1615"></a>

**Easy** · Amazon · 2024-05-27

#### Problem

Given an array and a number k that's smaller than the length of the array, rotate the array to the right k elements in-place.

### Problem #1616 <a id="p1616"></a>

**Easy** · Alibaba · 2024-05-28

#### Problem

Given an even number (greater than 2), return two prime numbers whose sum will be equal to the given number.

A solution will always exist. See Goldbach’s conjecture .

Example:

```
Input: 4
Output: 2 + 2 = 4
```

If there are more than one solution possible, return the lexicographically smaller solution.

If `[a, b]` is one solution with a \<= b, and `[c, d]` is another solution with c \<= d, then

`[a, b]` \< `[c, d]`

If a \< c OR a==c AND b \< d.

### Problem #1617 <a id="p1617"></a>

**Easy** · Microsoft · 2024-05-29

#### Problem

You are given an string representing the initial conditions of some dominoes. Each element can take one of three values:

- L, meaning the domino has just been pushed to the left,
- R, meaning the domino has just been pushed to the right, or
- ., meaning the domino is standing still.

Determine the orientation of each tile when the dominoes stop falling. Note that if a domino receives a force from the left and right side simultaneously, it will remain upright.

For example, given the string .L.R....L, you should return LL.RRRLLL.

Given the string ..R...L.L, you should return ..RR.LLLL.

### Problem #1618 <a id="p1618"></a>

**Easy** · Zillow · 2024-05-30

#### Problem

Let's define a `"sevenish"` number to be one which is either a power of 7, or the sum of unique powers of 7. The first few sevenish numbers are 1, 7, 8, 49, and so on. Create an algorithm to find the nth sevenish number.

### Problem #1619 <a id="p1619"></a>

**Hard** · Google · 2024-05-31

#### Problem

Given two non-empty binary trees s and t, check whether tree t has exactly the same structure and node values with a subtree of s. A subtree of s is a tree consists of a node in s and all of this node's descendants. The tree s could also be considered as a subtree of itself.

### Problem #1620 <a id="p1620"></a>

**Hard** · Dropbox · 2024-06-01

#### Problem

Sudoku is a puzzle where you're given a partially-filled 9 by 9 grid with digits. The objective is to fill the grid with the constraint that every row, column, and box (3 by 3 subgrid) must contain all of the digits from 1 to 9.

Implement an efficient sudoku solver.

### Problem #1621 <a id="p1621"></a>

**Easy** · Google · 2024-06-02

#### Problem

A regular number in mathematics is defined as one which evenly divides some power of 60. Equivalently, we can say that a regular number is one whose only prime divisors are 2, 3, and 5.

These numbers have had many applications, from helping ancient Babylonians keep time to tuning instruments according to the diatonic scale.

Given an integer N, write a program that returns, in order, the first N regular numbers.

### Problem #1622 <a id="p1622"></a>

**Hard** · Amazon · 2024-06-03

#### Problem

Given an integer k and a string s, find the length of the longest substring that contains at most k distinct characters.

For example, given s = `"abcba"` and k = 2, the longest substring with k distinct characters is `"bcb"`.

### Problem #1623 <a id="p1623"></a>

**Medium** · Google · 2024-06-04

#### Problem

You are given a set of synonyms, such as (big, large) and (eat, consume). Using this set, determine if two sentences with the same number of words are equivalent.

For example, the following two sentences are equivalent:

- `"He wants to eat food."`
- `"He wants to consume food."`

Note that the synonyms (a, b) and (a, c) do not necessarily imply (b, c): consider the case of (coach, bus) and (coach, teacher).

Follow-up: what if we can assume that (a, b) and (a, c) do in fact imply (b, c)?

### Problem #1624 <a id="p1624"></a>

**Medium** · Salesforce · 2024-06-05

#### Problem

The number 6174 is known as Kaprekar's contant, after the mathematician who discovered an associated property: for all four-digit numbers with at least two distinct digits, repeatedly applying a simple procedure eventually results in this value. The procedure is as follows:

- For a given input x, create two new numbers that consist of the digits in x in ascending and descending order.
- Subtract the smaller number from the larger number.

For example, this algorithm terminates in three steps when starting from 1234:

- 4321 - 1234 = 3087
- 8730 - 0378 = 8352
- 8532 - 2358 = 6174

Write a function that returns how many steps this will take for a given input N.

### Problem #1625 <a id="p1625"></a>

**Medium** · Amazon · 2024-06-06

#### Problem

Given a node in a binary search tree, return the next bigger element, also known as the inorder successor.

For example, the inorder successor of 22 is 30.

```
  10
 /  \
5    30
    /  \
  22    35
```

You can assume each node has a parent pointer.

### Problem #1626 <a id="p1626"></a>

**Easy** · Microsoft · 2024-06-07

#### Problem

Let's represent an integer in a linked list format by having each node represent a digit in the number. The nodes make up the number in reversed order.

For example, the following linked list:

1 -> 2 -> 3 -> 4 -> 5

is the number 54321.

Given two linked lists in this format, return their sum in the same linked list format.

For example, given

9 -> 9

5 -> 2

return 124 (99 + 25) as:

4 -> 2 -> 1

### Problem #1627 <a id="p1627"></a>

**Easy** · Facebook · 2024-06-08

#### Problem

Given a function f, and N return a debounced f of N milliseconds.

That is, as long as the debounced f continues to be invoked, f itself will not be called for N milliseconds.

### Problem #1628 <a id="p1628"></a>

**Medium** · Facebook · 2024-06-09

#### Problem

Given an N by N matrix, rotate it by 90 degrees clockwise.

For example, given the following matrix:

```
[[1, 2, 3],
 [4, 5, 6],
 [7, 8, 9]]
```

you should return:

```
[[7, 4, 1],
 [8, 5, 2],
 [9, 6, 3]]
```

Follow-up: What if you couldn't use any extra space?

### Problem #1629 <a id="p1629"></a>

**Medium** · Amazon · 2024-06-10

#### Problem

Given a pivot x, and a list lst, partition the list into three parts.

- The first part contains all elements in lst that are less than x
- The second part contains all elements in lst that are equal to x
- The third part contains all elements in lst that are larger than x

Ordering within a part can be arbitrary.

For example, given x = 10 and lst = `[9, 12, 3, 5, 14, 10, 10]`, one partition may be `[9, 3, 5, 10, 10, 12, 14]`.

### Problem #1630 <a id="p1630"></a>

**Medium** · Facebook · 2024-06-11

#### Problem

Given a binary tree, return all paths from the root to leaves.

For example, given the tree:

```
  1
 / \
2   3
   / \
  4   5
```

Return `[[1, 2], [1, 3, 4], [1, 3, 5]]`.

### Problem #1631 <a id="p1631"></a>

**Medium** · LinkedIn · 2024-06-12

#### Problem

Determine whether a tree is a valid binary search tree.

A binary search tree is a tree with two children, left and right, and satisfies the constraint that the key in the left child must be less than or equal to the root and the key in the right child must be greater than or equal to the root.

### Problem #1632 <a id="p1632"></a>

**Hard** · Microsoft · 2024-06-13

#### Problem

Given an array of positive integers, divide the array into two subsets such that the difference between the sum of the subsets is as small as possible.

For example, given `[5, 10, 15, 20, 25]`, return the sets `{10, 25}` and `{5, 15, 20}`, which has a difference of 5, which is the smallest possible difference.

### Problem #1633 <a id="p1633"></a>

**Medium** · Facebook · 2024-06-14

#### Problem

Given three 32-bit integers x, y, and b, return x if b is 1 and y if b is 0, using only mathematical or bit operations. You can assume b can only be 1 or 0.

### Problem #1634 <a id="p1634"></a>

**Medium** · Facebook · 2024-06-15

#### Problem

Given a start word, an end word, and a dictionary of valid words, find the shortest transformation sequence from start to end such that only one letter is changed at each step of the sequence, and each transformed word exists in the dictionary. If there is no possible transformation, return null. Each word in the dictionary have the same length as start and end and is lowercase.

For example, given start = `"dog"`, end = `"cat"`, and dictionary = `{"dot", "dop", "dat", "cat"}`, return `["dog", "dot", "dat", "cat"]`.

Given start = `"dog"`, end = `"cat"`, and dictionary = `{"dot", "tod", "dat", "dar"}`, return null as there is no possible transformation from dog to cat.

### Problem #1635 <a id="p1635"></a>

**Hard** · Microsoft · 2024-06-16

#### Problem

Let X be a set of n intervals on the real line. We say that a set of points P `"stabs"` X if every interval in X contains at least one point in P. Compute the smallest set of points that stabs X.

For example, given the intervals `[(1, 4), (4, 5), (7, 9), (9, 12)]`, you should return `[4, 9]`.

### Problem #1636 <a id="p1636"></a>

**Hard** · Google · 2024-06-17

#### Problem

Given a string which we can delete at most k, return whether you can make a palindrome.

For example, given `'waterrfetawx'` and a k of 2, you could delete f and x to get `'waterretaw'`.

### Problem #1637 <a id="p1637"></a>

**Hard** · Netflix · 2024-06-18

#### Problem

Given a sorted list of integers of length N, determine if an element x is in the list without performing any multiplication, division, or bit-shift operations.

Do this in `O(log N)` time.

### Problem #1638 <a id="p1638"></a>

**Easy** · Google · 2024-06-19

#### Problem

Given the head of a singly linked list, swap every two nodes and return its head.

For example, given 1 -> 2 -> 3 -> 4, return 2 -> 1 -> 4 -> 3.

### Problem #1639 <a id="p1639"></a>

**Medium** · Google · 2024-06-20

#### Problem

We can determine how `"out of order"` an array A is by counting the number of inversions it has. Two elements `A[i]` and `A[j]` form an inversion if `A[i]` > `A[j]` but i \< j. That is, a smaller element appears after a larger element.

Given an array, count the number of inversions it has. Do this faster than `O(N^2)` time.

You may assume each element in the array is distinct.

For example, a sorted list has zero inversions. The array `[2, 4, 1, 3, 5]` has three inversions: (2, 1), (4, 1), and (4, 3). The array `[5, 4, 3, 2, 1]` has ten inversions: every distinct pair forms an inversion.

### Problem #1640 <a id="p1640"></a>

**Hard** · Google · 2024-06-21

#### Problem

You're given a string consisting solely of (, ), and \*. \* can represent either a (, ), or an empty string. Determine whether the parentheses are balanced.

For example, (()\* and (\*) are balanced. )\*( is not balanced.

### Problem #1641 <a id="p1641"></a>

**Easy** · Amazon · 2024-06-22

#### Problem

Run-length encoding is a fast and simple method of encoding strings. The basic idea is to represent repeated successive characters as a single count and character. For example, the string `"AAAABBBCCDAA"` would be encoded as `"4A3B2C1D2A"`.

Implement run-length encoding and decoding. You can assume the string to be encoded have no digits and consists solely of alphabetic characters. You can assume the string to be decoded is valid.

### Problem #1642 <a id="p1642"></a>

**Hard** · 2024-06-23

#### Problem

Find an efficient algorithm to find the smallest distance (measured in number of words) between any two given words in a string.

For example, given words `"hello"`, and `"world"` and a text content of `"dog cat hello cat dog dog hello cat world"`, return 1 because there's only one word `"cat"` in between the two words.

### Problem #1643 <a id="p1643"></a>

**Easy** · Facebook · 2024-06-24

#### Problem

Given a 32-bit integer, return the number with its bits reversed.

For example, given the binary number 1111 0000 1111 0000 1111 0000 1111 0000, return 0000 1111 0000 1111 0000 1111 0000 1111.

### Problem #1644 <a id="p1644"></a>

**Easy** · IBM · 2024-06-25

#### Problem

Given an integer, find the next permutation of it in absolute order. For example, given 48975, the next permutation would be 49578.

### Problem #1645 <a id="p1645"></a>

**Hard** · Facebook · 2024-06-26

#### Problem

Implement regular expression matching with the following special characters:

- . (period) which matches any single character
- \* (asterisk) which matches zero or more of the preceding element

That is, implement a function that takes in a string and a valid regular expression and returns whether or not the string matches the regular expression.

For example, given the regular expression `"ra."` and the string `"ray"`, your function should return true. The same regular expression on the string `"raymond"` should return false.

Given the regular expression `".*at"` and the string `"chat"`, your function should return true. The same regular expression on the string `"chats"` should return false.

### Problem #1646 <a id="p1646"></a>

**Medium** · Facebook · 2024-06-27

#### Problem

Write a function that rotates a list by k elements. For example, `[1, 2, 3, 4, 5, 6]` rotated by two becomes `[3, 4, 5, 6, 1, 2]`. Try solving this without creating a copy of the list. How many swap or move operations do you need?

### Problem #1647 <a id="p1647"></a>

**Medium** · Square · 2024-06-28

#### Problem

In front of you is a row of N coins, with values v1, v1, ..., vn.

You are asked to play the following game. You and an opponent take turns choosing either the first or last coin from the row, removing it from the row, and receiving the value of the coin.

Write a program that returns the maximum amount of money you can win with certainty, if you move first, assuming your opponent plays optimally.

### Problem #1648 <a id="p1648"></a>

**Medium** · Quora · 2024-06-29

#### Problem

Given an absolute pathname that may have . or .. as part of it, return the shortest standardized path.

For example, given `"/usr/bin/../bin/./scripts/../"`, return `"/usr/bin/"`.

### Problem #1649 <a id="p1649"></a>

**Easy** · Dropbox · 2024-06-30

#### Problem

Spreadsheets often use this alphabetical encoding for its columns: `"A"`, `"B"`, `"C"`, ..., `"AA"`, `"AB"`, ..., `"ZZ"`, `"AAA"`, `"AAB"`, ....

Given a column number, return its alphabetical column id. For example, given 1, return `"A"`. Given 27, return `"AA"`.

### Problem #1650 <a id="p1650"></a>

**Hard** · Microsoft · 2024-07-01

#### Problem

Recall that the minimum spanning tree is the subset of edges of a tree that connect all its vertices with the smallest possible total edge weight. Given an undirected graph with weighted edges, compute the maximum weight spanning tree.

### Problem #1651 <a id="p1651"></a>

**Hard** · Apple · 2024-07-02

#### Problem

You are going on a road trip, and would like to create a suitable music playlist. The trip will require N songs, though you only have M songs downloaded, where M \< N. A valid playlist should select each song at least once, and guarantee a buffer of B songs between repeats.

Given N, M, and B, determine the number of valid playlists.

### Problem #1652 <a id="p1652"></a>

**Medium** · Quora · 2024-07-03

#### Problem

Given a string, find the palindrome that can be made by inserting the fewest number of characters as possible anywhere in the word. If there is more than one palindrome of minimum length that can be made, return the lexicographically earliest one (the first one alphabetically).

For example, given the string `"race"`, you should return `"ecarace"`, since we can add three letters to it (which is the smallest amount to make a palindrome). There are seven other palindromes that can be made from `"race"` by adding three letters, but `"ecarace"` comes first alphabetically.

As another example, given the string `"google"`, you should return `"elgoogle"`.

### Problem #1653 <a id="p1653"></a>

**Easy** · Microsoft · 2024-07-04

#### Problem

Suppose an arithmetic expression is given as a binary tree. Each leaf is an integer and each internal node is one of `'+'`, `'−'`, `'∗'`, or `'/'`.

Given the root to such a tree, write a function to evaluate it.

For example, given the following tree:

```
    *
   / \
  +    +
 / \  / \
3  2  4  5
```

You should return 45, as it is (3 + 2) \* (4 + 5).

### Problem #1654 <a id="p1654"></a>

**Hard** · Quora · 2024-07-05

#### Problem

Word sense disambiguation is the problem of determining which sense a word takes on in a particular setting, if that word has multiple meanings. For example, in the sentence `"I went to get money from the bank"`, bank probably means the place where people deposit money, not the land beside a river or lake.

Suppose you are given a list of meanings for several words, formatted like so:

```
{
    "word_1": ["meaning one", "meaning two", ...],
    ...
    "word_n": ["meaning one", "meaning two", ...]
}
```

Given a sentence, most of whose words are contained in the meaning list above, create an algorithm that determines the likely sense of each possibly ambiguous word.

### Problem #1655 <a id="p1655"></a>

**Medium** · Google · 2024-07-06

#### Problem

What will this code print out?

```
def make_functions():
    flist = []
```

```
for i in [1, 2, 3]:
    def print_i():
        print(i)
    flist.append(print_i)
```

```
return flist
```

```
functions = make_functions()
for f in functions:
    f()
```

How can we make it print out what we apparently want?

### Problem #1656 <a id="p1656"></a>

**Medium** · Twitter · 2024-07-07

#### Problem

Implement an autocomplete system. That is, given a query string s and a set of all possible query strings, return all strings in the set that have s as a prefix.

For example, given the query string de and the set of strings `[dog, deer, deal]`, return `[deer, deal]`.

Hint: Try preprocessing the dictionary into a more efficient data structure to speed up queries.

### Problem #1657 <a id="p1657"></a>

**Easy** · Airbnb · 2024-07-08

#### Problem

Given a linked list and a positive integer k, rotate the list to the right by k places.

For example, given the linked list 7 -> 7 -> 3 -> 5 and k = 2, it should become 3 -> 5 -> 7 -> 7.

Given the linked list 1 -> 2 -> 3 -> 4 -> 5 and k = 3, it should become 3 -> 4 -> 5 -> 1 -> 2.

### Problem #1658 <a id="p1658"></a>

**Easy** · Google · 2024-07-09

#### Problem

You are given an array of arrays of integers, where each array corresponds to a row in a triangle of numbers. For example, `[[1], [2, 3], [1, 5, 1]]` represents the triangle:

```
  1
 2 3
1 5 1
```

We define a path in the triangle to start at the top and go down one row at a time to an adjacent value, eventually ending with an entry on the bottom row. For example, 1 -> 3 -> 5. The weight of the path is the sum of the entries.

Write a program that returns the weight of the maximum weight path.

### Problem #1659 <a id="p1659"></a>

**Easy** · WhatsApp · 2024-07-10

#### Problem

Given an array of integers out of order, determine the bounds of the smallest window that must be sorted in order for the entire array to be sorted. For example, given `[3, 7, 5, 6, 9]`, you should return (1, 3).

### Problem #1660 <a id="p1660"></a>

**Hard** · Adobe · 2024-07-11

#### Problem

You are given a tree with an even number of nodes. Consider each connection between a parent and child node to be an `"edge"`. You would like to remove some of these edges, such that the disconnected subtrees that remain each have an even number of nodes.

For example, suppose your input was the following tree:

```
   1
  / \ 
 2   3
    / \ 
   4   5
 / | \
6  7  8
```

In this case, removing the edge (3, 4) satisfies our requirement.

Write a function that returns the maximum number of edges you can remove while still satisfying this requirement.

### Problem #1661 <a id="p1661"></a>

**Medium** · Triplebyte · 2024-07-12

#### Problem

Implement a data structure which carries out the following operations without resizing the underlying array:

- `add(value)`: Add a value to the set of values.
- `check(value)`: Check whether a value is in the set.

The check method may return occasional false positives (in other words, incorrectly identifying an element as part of the set), but should always correctly identify a true element.

### Problem #1662 <a id="p1662"></a>

**Hard** · Google · 2024-07-13

#### Problem

Implement an LFU (Least Frequently Used) cache. It should be able to be initialized with a cache size n, and contain the following methods:

- `set(key, value)`: sets key to value. If there are already n items in the cache and we are adding a new item, then it should also remove the least frequently used item. If there is a tie, then the least recently used key should be removed.
- `get(key)`: gets the value at key. If no such key exists, return null.

Each operation should run in `O(1)` time.

### Problem #1663 <a id="p1663"></a>

**Hard** · Apple · 2024-07-14

#### Problem

Given a linked list, uniformly shuffle the nodes. What if we want to prioritize space over time?

### Problem #1664 <a id="p1664"></a>

**Easy** · Twitter · 2024-07-15

#### Problem

A permutation can be specified by an array P, where `P[i]` represents the location of the element at i in the permutation. For example, `[2, 1, 0]` represents the permutation where elements at the index 0 and 2 are swapped.

Given an array and a permutation, apply the permutation to the array. For example, given the array `["a", "b", "c"]` and the permutation `[2, 1, 0]`, return `["c", "b", "a"]`.

### Problem #1665 <a id="p1665"></a>

**Medium** · LinkedIn · 2024-07-16

#### Problem

A wall consists of several rows of bricks of various integer lengths and uniform height. Your goal is to find a vertical line going from the top to the bottom of the wall that cuts through the fewest number of bricks. If the line goes through the edge between two bricks, this does not count as a cut.

For example, suppose the input is as follows, where values in each row represent the lengths of bricks in that row:

```
[[3, 5, 1, 1],
 [2, 3, 3, 2],
 [5, 5],
 [4, 4, 2],
 [1, 3, 3, 3],
 [1, 1, 6, 1, 1]]
```

The best we can we do here is to draw a line after the eighth brick, which will only require cutting through the bricks in the third and fifth row.

Given an input consisting of brick lengths for each row such as the one above, return the fewest number of bricks that must be cut to create a vertical line.

### Problem #1666 <a id="p1666"></a>

**Easy** · Amazon · 2024-07-17

#### Problem

Given n numbers, find the greatest common denominator between them.

For example, given the numbers `[42, 56, 14]`, return 14.

### Problem #1667 <a id="p1667"></a>

**Hard** · Airbnb · 2024-07-18

#### Problem

We're given a hashmap associating each courseId key with a list of courseIds values, which represents that the prerequisites of courseId are courseIds. Return a sorted ordering of courses such that we can finish all courses.

Return null if there is no such ordering.

For example, given `{'CSC300': ['CSC100', 'CSC200'], 'CSC200': ['CSC100'], 'CSC100': []}`, should return `['CSC100', 'CSC200', 'CSCS300']`.

### Problem #1668 <a id="p1668"></a>

**Easy** · Microsoft · 2024-07-19

#### Problem

A number is considered perfect if its digits sum up to exactly 10.

Given a positive integer n, return the n-th perfect number.

For example, given 1, you should return 19. Given 2, you should return 28.

### Problem #1669 <a id="p1669"></a>

**Easy** · Google · 2024-07-20

#### Problem

Given a sorted list of integers, square the elements and give the output in sorted order.

For example, given `[-9, -2, 0, 2, 3]`, return `[0, 4, 4, 9, 81]`.

### Problem #1670 <a id="p1670"></a>

**Medium** · Lyft · 2024-07-21

#### Problem

Given a list of integers and a number K, return which contiguous elements of the list sum to K.

For example, if the list is `[1, 2, 3, 4, 5]` and K is 9, then it should return `[2, 3, 4]`, since 2 + 3 + 4 = 9.

### Problem #1671 <a id="p1671"></a>

**Medium** · Google · 2024-07-22

#### Problem

You are given an N by M 2D matrix of lowercase letters. Determine the minimum number of columns that can be removed to ensure that each row is ordered from top to bottom lexicographically. That is, the letter at each column is lexicographically later as you go down each row. It does not matter whether each row itself is ordered lexicographically.

For example, given the following table:

```
cba
daf
ghi
```

This is not ordered because of the a in the center. We can remove the second column to make it ordered:

```
ca
df
gi
```

So your function should return 1, since we only needed to remove 1 column.

As another example, given the following table:

abcdef

Your function should return 0, since the rows are already ordered (there's only one row).

As another example, given the following table:

```
zyx
wvu
tsr
```

Your function should return 3, since we would need to remove all the columns to order it.

### Problem #1672 <a id="p1672"></a>

**Hard** · Oracle · 2024-07-23

#### Problem

You are presented with an 8 by 8 matrix representing the positions of pieces on a chess board. The only pieces on the board are the black king and various white pieces. Given this matrix, determine whether the king is in check.

For details on how each piece moves, see here.

For example, given the following matrix:

```
...K....
........
.B......
......P.
.......R
..N.....
........
.....Q..
```

You should return True, since the bishop is attacking the king diagonally.

### Problem #1673 <a id="p1673"></a>

**Hard** · Citrix · 2024-07-24

#### Problem

You are given a circular lock with three wheels, each of which display the numbers 0 through 9 in order. Each of these wheels rotate clockwise and counterclockwise.

In addition, the lock has a certain number of `"dead ends"`, meaning that if you turn the wheels to one of these combinations, the lock becomes stuck in that state and cannot be opened.

Let us consider a `"move"` to be a rotation of a single wheel by one digit, in either direction. Given a lock initially set to 000, a target combination, and a list of dead ends, write a function that returns the minimum number of moves required to reach the target state, or None if this is impossible.

### Problem #1674 <a id="p1674"></a>

**Hard** · Google · 2024-07-25

#### Problem

Implement a file syncing algorithm for two computers over a low-bandwidth network. What if we know the files in the two computers are mostly the same?

### Problem #1675 <a id="p1675"></a>

**Easy** · Pandora · 2024-07-26

#### Problem

Given an undirected graph, determine if it contains a cycle.

### Problem #1676 <a id="p1676"></a>

**Easy** · Google · 2024-07-27

#### Problem

Implement a PrefixMapSum class with the following methods:

- `insert(key: str, value: int)`: Set a given key's value in the map. If the key already exists, overwrite the value.
- `sum(prefix: str)`: Return the sum of all values of keys that begin with a given prefix.

For example, you should be able to run the following code:

```
mapsum.insert("columnar", 3)
assert mapsum.sum("col") == 3
```

```
mapsum.insert("column", 2)
assert mapsum.sum("col") == 5
```

### Problem #1677 <a id="p1677"></a>

**Easy** · Google · 2024-07-28

#### Problem

Determine whether a doubly linked list is a palindrome. What if it’s singly linked?

For example, 1 -> 4 -> 3 -> 4 -> 1 returns True while 1 -> 4 returns False.

### Problem #1678 <a id="p1678"></a>

**Medium** · ContextLogic · 2024-07-29

#### Problem

Implement division of two positive integers without using the division, multiplication, or modulus operators. Return the quotient as an integer, ignoring the remainder.

### Problem #1679 <a id="p1679"></a>

**Hard** · Apple · 2024-07-30

#### Problem

Given a tree, find the largest tree/subtree that is a BST.

Given a tree, return the size of the largest tree/subtree that is a BST.

### Problem #1680 <a id="p1680"></a>

**Medium** · Nvidia · 2024-07-31

#### Problem

You are given a list of N points (x1, y1), (x2, y2), ..., (xN, yN) representing a polygon. You can assume these points are given in order; that is, you can construct the polygon by connecting point 1 to point 2, point 2 to point 3, and so on, finally looping around to connect point N to point 1.

Determine if a new point p lies inside this polygon. (If p is on the boundary of the polygon, you should return False).

### Problem #1681 <a id="p1681"></a>

**Medium** · Google · 2024-08-01

#### Problem

Given a string of parentheses, write a function to compute the minimum number of parentheses to be removed to make the string valid (i.e. each open parenthesis is eventually closed).

For example, given the string `"()())()"`, you should return 1. Given the string `")("`, you should return 2, since we must remove all of them.

### Problem #1682 <a id="p1682"></a>

**Medium** · Google · 2024-08-02

#### Problem

Given an undirected graph represented as an adjacency matrix and an integer k, write a function to determine whether each vertex in the graph can be colored such that no two adjacent vertices share the same color using at most k colors.

### Problem #1683 <a id="p1683"></a>

**Easy** · Microsoft · 2024-08-03

#### Problem

Given a 2D matrix of characters and a target word, write a function that returns whether the word can be found in the matrix by going left-to-right, or up-to-down.

For example, given the following matrix:

```
[['F', 'A', 'C', 'I'],
 ['O', 'B', 'Q', 'P'],
 ['A', 'N', 'O', 'B'],
 ['M', 'A', 'S', 'S']]
```

and the target word `'FOAM'`, you should return true, since it's the leftmost column. Similarly, given the target word `'MASS'`, you should return true, since it's the last row.

### Problem #1684 <a id="p1684"></a>

**Easy** · Google · 2024-08-04

#### Problem

The edit distance between two strings refers to the minimum number of character insertions, deletions, and substitutions required to change one string to the other. For example, the edit distance between “kitten” and “sitting” is three: substitute the “k” for “s”, substitute the “e” for “i”, and append a “g”.

Given two strings, compute the edit distance between them.

### Problem #1685 <a id="p1685"></a>

**Medium** · Spotify · 2024-08-05

#### Problem

You have access to ranked lists of songs for various users. Each song is represented as an integer, and more preferred songs appear earlier in each list. For example, the list `[4, 1, 7]` indicates that a user likes song 4 the best, followed by songs 1 and 7.

Given a set of these ranked lists, interleave them to create a playlist that satisfies everyone's priorities.

For example, suppose your input is `{[1, 7, 3], [2, 1, 6, 7, 9], [3, 9, 5]}`. In this case a satisfactory playlist could be `[2, 1, 6, 7, 3, 9, 5]`.

### Problem #1686 <a id="p1686"></a>

**Hard** · 2024-08-06

#### Problem

Given a list, sort it using this method: `reverse(lst, i, j)`, which reverses lst from i to j.

### Problem #1687 <a id="p1687"></a>

**Easy** · Google · 2024-08-07

#### Problem

Given the root of a binary tree, return a deepest node. For example, in the following tree, return d.

```
    a
   / \
  b   c
 /
d
```

### Problem #1688 <a id="p1688"></a>

**Medium** · Jane Street · 2024-08-08

#### Problem

`cons(a, b)` constructs a pair, and `car(pair)` and `cdr(pair)` returns the first and last element of that pair. For example, `car(cons(3, 4))` returns 3, and `cdr(cons(3, 4))` returns 4.

Given this implementation of cons:

```
def cons(a, b):
    def pair(f):
        return f(a, b)
    return pair
```

Implement car and cdr.

### Problem #1689 <a id="p1689"></a>

**Medium** · Google · 2024-08-09

#### Problem

Given a linked list, sort it in `O(n log n)` time and constant space.

For example, the linked list 4 -> 1 -> -3 -> 99 should become -3 -> 1 -> 4 -> 99.

### Problem #1690 <a id="p1690"></a>

**Easy** · Facebook · 2024-08-10

#### Problem

Boggle is a game played on a 4 x 4 grid of letters. The goal is to find as many words as possible that can be formed by a sequence of adjacent letters in the grid, using each cell at most once. Given a game board and a dictionary of valid words, implement a Boggle solver.

### Problem #1691 <a id="p1691"></a>

**Medium** · Facebook · 2024-08-11

#### Problem

Given an integer n, find the next biggest integer with the same number of 1-bits on. For example, given the number 6 (0110 in binary), return 9 (1001).

### Problem #1692 <a id="p1692"></a>

**Easy** · Microsoft · 2024-08-12

#### Problem

Given a clock time in hh:mm format, determine, to the nearest degree, the angle between the hour and the minute hands.

Bonus: When, during the course of a day, will the angle be zero?

### Problem #1693 <a id="p1693"></a>

**Easy** · Zillow · 2024-08-13

#### Problem

A ternary search tree is a trie-like data structure where each node may have up to three children. Here is an example which represents the words code, cob, be, ax, war, and we.

```
       c
    /  |  \
   b   o   w
 / |   |   |
a  e   d   a
|    / |   | \ 
x   b  e   r  e
```

The tree is structured according to the following rules:

- left child nodes link to words lexicographically earlier than the parent prefix
- right child nodes link to words lexicographically later than the parent prefix
- middle child nodes continue the current word

For instance, since code is the first word inserted in the tree, and cob lexicographically precedes cod, cob is represented as a left child extending from cod.

Implement insertion and search functions for a ternary search tree.

### Problem #1694 <a id="p1694"></a>

**Medium** · Fitbit · 2024-08-14

#### Problem

Given a linked list, rearrange the node values such that they appear in alternating low -> high -> low -> high ... form. For example, given 1 -> 2 -> 3 -> 4 -> 5, you should return 1 -> 3 -> 2 -> 5 -> 4.

### Problem #1695 <a id="p1695"></a>

**Easy** · Google · 2024-08-15

#### Problem

Given two singly linked lists that intersect at some point, find the intersecting node. The lists are non-cyclical.

For example, given A = 3 -> 7 -> 8 -> 10 and B = 99 -> 1 -> 8 -> 10, return the node with value 8.

In this example, assume nodes with the same value are the exact same node objects.

Do this in `O(M + N)` time (where M and N are the lengths of the lists) and constant space.

### Problem #1696 <a id="p1696"></a>

**Hard** · Microsoft · 2024-08-16

#### Problem

You have an N by N board. Write a function that, given N, returns the number of possible arrangements of the board where N queens can be placed on the board without threatening each other, i.e. no two queens share the same row, column, or diagonal.

### Problem #1697 <a id="p1697"></a>

**Easy** · Palantir · 2024-08-17

#### Problem

A typical American-style crossword puzzle grid is an N x N matrix with black and white squares, which obeys the following rules:

- Every white square must be part of an `"across"` word and a `"down"` word.
- No word can be fewer than three letters long.
- Every white square must be reachable from every other white square.
- The grid is rotationally symmetric (for example, the colors of the top left and bottom right squares must match).

Write a program to determine whether a given matrix qualifies as a crossword grid.

### Problem #1698 <a id="p1698"></a>

**Medium** · Snapchat · 2024-08-18

#### Problem

You are given an array of length N, where each element i represents the number of ways we can produce i units of change. For example, `[1, 0, 1, 1, 2]` would indicate that there is only one way to make 0, 2, or 3 units, and two ways of making 4 units.

Given such an array, determine the denominations that must be in use. In the case above, for example, there must be coins with value 2, 3, and 4.

### Problem #1699 <a id="p1699"></a>

**Medium** · Spotify · 2024-08-19

#### Problem

Write a function, `throw_dice(N, faces, total)`, that determines how many ways it is possible to throw N dice with some number of faces each to get a specific total.

For example, `throw_dice(3, 6, 7)` should equal 15.

### Problem #1700 <a id="p1700"></a>

**Hard** · Google · 2024-08-20

#### Problem

A quack is a data structure combining properties of both stacks and queues. It can be viewed as a list of elements written left to right such that three operations are possible:

- `push(x)`: add a new item x to the left end of the list
- `pop()`: remove and return the item on the left end of the list
- `pull()`: remove the item on the right end of the list.

Implement a quack using three stacks and `O(1)` additional memory, so that the amortized time for any push, pop, or pull operation is `O(1)`.

### Problem #1701 <a id="p1701"></a>

**Easy** · Uber · 2024-08-21

#### Problem

On election day, a voting machine writes data in the form (`voter_id`, `candidate_id`) to a text file. Write a program that reads this file as a stream and returns the top 3 candidates at any given time. If you find a voter voting more than once, report this as fraud.

### Problem #1702 <a id="p1702"></a>

**Medium** · Google · 2024-08-22

#### Problem

Given an N by M matrix consisting only of 1's and 0's, find the largest rectangle containing only 1's and return its area.

For example, given the following matrix:

```
[[1, 0, 0, 0],
 [1, 0, 1, 1],
 [1, 0, 1, 1],
 [0, 1, 0, 0]]
```

Return 4.

### Problem #1703 <a id="p1703"></a>

**Hard** · Goldman Sachs · 2024-08-23

#### Problem

Given a list of numbers L, implement a method `sum(i, j)` which returns the sum from the sublist `L[i:j]` (including i, excluding j).

For example, given L = `[1, 2, 3, 4, 5]`, `sum(1, 3)` should return `sum([2, 3])`, which is 5.

You can assume that you can do some pre-processing. `sum()` should be optimized over the pre-processing step.

### Problem #1704 <a id="p1704"></a>

**Medium** · Amazon · 2024-08-24

#### Problem

At a popular bar, each customer has a set of favorite drinks, and will happily accept any drink among this set. For example, in the following situation, customer 0 will be satisfied with drinks 0, 1, 3, or 6.

```
preferences = {
    0: [0, 1, 3, 6],
    1: [1, 4, 7],
    2: [2, 4, 7, 5],
    3: [3, 2, 5],
    4: [5, 8]
}
```

A lazy bartender working at this bar is trying to reduce his effort by limiting the drink recipes he must memorize. Given a dictionary input such as the one above, return the fewest number of drinks he must learn in order to satisfy all customers.

For the input above, the answer would be 2, as drinks 1 and 5 will satisfy everyone.

### Problem #1705 <a id="p1705"></a>

**Medium** · Yext · 2024-08-25

#### Problem

Two nodes in a binary tree can be called cousins if they are on the same level of the tree but have different parents. For example, in the following diagram 4 and 6 are cousins.

```
    1
   / \
  2   3
 / \   \
4   5   6
```

Given a binary tree and a particular node, find all cousins of that node.

### Problem #1706 <a id="p1706"></a>

**Medium** · Microsoft · 2024-08-26

#### Problem

Given an unsorted array of integers, find the length of the longest consecutive elements sequence.

For example, given `[100, 4, 200, 1, 3, 2]`, the longest consecutive element sequence is `[1, 2, 3, 4]`. Return its length: 4.

Your algorithm should run in `O(n)` complexity.

### Problem #1707 <a id="p1707"></a>

**Medium** · Facebook · 2024-08-27

#### Problem

In chess, the Elo rating system is used to calculate player strengths based on game results.

A simplified description of the Elo system is as follows. Every player begins at the same score. For each subsequent game, the loser transfers some points to the winner, where the amount of points transferred depends on how unlikely the win is. For example, a 1200-ranked player should gain much more points for beating a 2000-ranked player than for beating a 1300-ranked player.

Implement this system.

### Problem #1708 <a id="p1708"></a>

**Medium** · Indeed · 2024-08-28

#### Problem

Given a 32-bit positive integer N, determine whether it is a power of four in faster than `O(log N)` time.

### Problem #1709 <a id="p1709"></a>

**Medium** · Facebook · 2024-08-29

#### Problem

Given an array of integers, write a function to determine whether the array could become non-decreasing by modifying at most 1 element.

For example, given the array `[10, 5, 7]`, you should return true, since we can modify the 10 into a 1 to make the array non-decreasing.

Given the array `[10, 5, 1]`, you should return false, since we can't modify any one element to get a non-decreasing array.

### Problem #1710 <a id="p1710"></a>

**Medium** · Snapchat · 2024-08-30

#### Problem

Given the head to a singly linked list, where each node also has a “random” pointer that points to anywhere in the linked list, deep clone the list.

### Problem #1711 <a id="p1711"></a>

**Easy** · Microsoft · 2024-08-31

#### Problem

The transitive closure of a graph is a measure of which vertices are reachable from other vertices. It can be represented as a matrix M, where `M[i][j]` == 1 if there is a path between vertices i and j, and otherwise 0.

For example, suppose we are given the following graph in adjacency list form:

```
graph = [
    [0, 1, 3],
    [1, 2],
    [2],
    [3]
]
```

The transitive closure of this graph would be:

```
[1, 1, 1, 1]
[0, 1, 1, 0]
[0, 0, 1, 0]
[0, 0, 0, 1]
```

Given a graph, find its transitive closure.

### Problem #1712 <a id="p1712"></a>

**Easy** · Google · 2024-09-01

#### Problem

Given the root of a binary search tree, and a target K, return two nodes in the tree whose sum equals K.

For example, given the following tree and K of 20

```
   10
  /   \
5      15
      /  \
    11    15
```

Return the nodes 5 and 15.

### Problem #1713 <a id="p1713"></a>

**Hard** · Netflix · 2024-09-02

#### Problem

Implement a queue using a set of fixed-length arrays.

The queue should support enqueue, dequeue, and `get_size` operations.

### Problem #1714 <a id="p1714"></a>

**Easy** · Google · 2024-09-03

#### Problem

You are given an N by N matrix of random letters and a dictionary of words. Find the maximum number of words that can be packed on the board from the given dictionary.

A word is considered to be able to be packed on the board if:

- It can be found in the dictionary
- It can be constructed from untaken letters by other words found so far on the board
- The letters are adjacent to each other (vertically and horizontally, not diagonally).

Each tile can be visited only once by any word.

For example, given the following dictionary:

`{ 'eat', 'rain', 'in', 'rat' }`

and matrix:

```
[['e', 'a', 'n'],
 ['t', 't', 'i'],
 ['a', 'r', 'a']]
```

Your function should return 3, since we can make the words `'eat'`, `'in'`, and `'rat'` without them touching each other. We could have alternatively made `'eat'` and `'rain'`, but that would be incorrect since that's only 2 words.

### Problem #1715 <a id="p1715"></a>

**Hard** · Affirm · 2024-09-04

#### Problem

Given a array of numbers representing the stock prices of a company in chronological order, write a function that calculates the maximum profit you could have made from buying and selling that stock. You're also given a number fee that represents a transaction fee for each buy and sell transaction.

You must buy before you can sell the stock, but you can make as many transactions as you like.

For example, given `[1, 3, 2, 8, 4, 10]` and fee = 2, you should return 9, since you could buy the stock at 1 dollar, and sell at 8 dollars, and then buy it at 4 dollars and sell it at 10 dollars. Since we did two transactions, there is a 4 dollar fee, so we have 7 + 6 = 13 profit minus 4 dollars of fees.

### Problem #1716 <a id="p1716"></a>

**Easy** · Amazon · 2024-09-05

#### Problem

Given a N by M matrix of numbers, print out the matrix in a clockwise spiral.

For example, given the following matrix:

```
[[1,  2,  3,  4,  5],
 [6,  7,  8,  9,  10],
 [11, 12, 13, 14, 15],
 [16, 17, 18, 19, 20]]
```

You should print out the following:

```
1
2
3
4
5
10
15
20
19
18
17
16
11
6
7
8
9
14
13
12
```

### Problem #1717 <a id="p1717"></a>

**Medium** · Palantir · 2024-09-06

#### Problem

Write an algorithm to justify text. Given a sequence of words and an integer line length k, return a list of strings which represents each line, fully justified.

More specifically, you should have as many words as possible in each line. There should be at least one space between each word. Pad extra spaces when necessary so that each line has exactly length k. Spaces should be distributed as equally as possible, with the extra spaces, if any, distributed starting from the left.

If you can only fit one word on a line, then you should pad the right-hand side with spaces.

Each word is guaranteed not to be longer than k.

For example, given the list of words `["the", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]` and k = 16, you should return the following:

```
["the  quick brown", # 1 extra space on the left
"fox  jumps  over", # 2 extra spaces distributed evenly
"the   lazy   dog"] # 4 extra spaces distributed evenly
```

### Problem #1718 <a id="p1718"></a>

**Easy** · Facebook · 2024-09-07

#### Problem

Suppose you are given two lists of n points, one list p1, p2, ..., pn on the line y = 0 and the other list q1, q2, ..., qn on the line y = 1. Imagine a set of n line segments connecting each point pi to qi. Write an algorithm to determine how many pairs of the line segments intersect.

### Problem #1719 <a id="p1719"></a>

**Easy** · Palantir · 2024-09-08

#### Problem

In academia, the h-index is a metric used to calculate the impact of a researcher's papers. It is calculated as follows:

A researcher has index h if at least h of her N papers have h citations each. If there are multiple h satisfying this formula, the maximum is chosen.

For example, suppose N = 5, and the respective citations of each paper are `[4, 3, 0, 1, 5]`. Then the h-index would be 3, since the researcher has 3 papers with at least 3 citations.

Given a list of paper citations of a researcher, calculate their h-index.

### Problem #1720 <a id="p1720"></a>

**Medium** · Uber · 2024-09-09

#### Problem

Write a program that determines the smallest number of perfect squares that sum up to N.

Here are a few examples:

- Given N = 4, return 1 (4)
- Given N = 17, return 2 (16 + 1)
- Given N = 18, return 2 (9 + 9)

### Problem #1721 <a id="p1721"></a>

**Hard** · Facebook · 2024-09-10

#### Problem

Given a string and a set of delimiters, reverse the words in the string while maintaining the relative order of the delimiters. For example, given `"hello/world:here"`, return `"here/world:hello"`

Follow-up: Does your solution work for the following cases: `"hello/world:here/"`, `"hello//world:here"`

### Problem #1722 <a id="p1722"></a>

**Medium** · Dropbox · 2024-09-11

#### Problem

Create an algorithm to efficiently compute the approximate median of a list of numbers.

More precisely, given an unordered list of N numbers, find an element whose rank is between N / 4 and 3 \* N / 4, with a high level of certainty, in less than `O(N)` time.

### Problem #1723 <a id="p1723"></a>

**Medium** · Amazon · 2024-09-12

#### Problem

An sorted array of integers was rotated an unknown number of times.

Given such an array, find the index of the element in the array in faster than linear time. If the element doesn't exist in the array, return null.

For example, given the array `[13, 18, 25, 2, 8, 10]` and the element 8, return 4 (the index of 8 in the array).

You can assume all the integers in the array are unique.

### Problem #1724 <a id="p1724"></a>

**Medium** · Airbnb · 2024-09-13

#### Problem

You are given a huge list of airline ticket prices between different cities around the world on a given day. These are all direct flights. Each element in the list has the format (`source_city`, destination, price).

Consider a user who is willing to take up to k connections from their origin city A to their destination B. Find the cheapest fare possible for this journey and print the itinerary for that journey.

For example, our traveler wants to go from JFK to LAX with up to 3 connections, and our input flights are as follows:

```
[
    ('JFK', 'ATL', 150),
    ('ATL', 'SFO', 400),
    ('ORD', 'LAX', 200),
    ('LAX', 'DFW', 80),
    ('JFK', 'HKG', 800),
    ('ATL', 'ORD', 90),
    ('JFK', 'LAX', 500),
]
```

Due to some improbably low flight prices, the cheapest itinerary would be JFK -> ATL -> ORD -> LAX, costing \$440.

### Problem #1725 <a id="p1725"></a>

**Easy** · Stripe · 2024-09-14

#### Problem

Given a collection of intervals, find the minimum number of intervals you need to remove to make the rest of the intervals non-overlapping.

Intervals can `"touch"`, such as `[0, 1]` and `[1, 2]`, but they won't be considered overlapping.

For example, given the intervals (7, 9), (2, 4), (5, 8), return 1 as the last interval can be removed and the first two won't overlap.

The intervals are not necessarily sorted in any order.

### Problem #1726 <a id="p1726"></a>

**Medium** · Palantir · 2024-09-15

#### Problem

You are given a list of N numbers, in which each number is located at most k places away from its sorted position. For example, if k = 1, a given element at index 4 might end up at indices 3, 4, or 5.

Come up with an algorithm that sorts this list in `O(N log k)` time.

### Problem #1727 <a id="p1727"></a>

**Easy** · Facebook · 2024-09-16

#### Problem

Given a binary tree, return the level of the tree with minimum sum.

### Problem #1728 <a id="p1728"></a>

**Medium** · Square · 2024-09-17

#### Problem

Assume you have access to a function `toss_biased()` which returns 0 or 1 with a probability that's not 50-50 (but also not 0-100 or 100-0). You do not know the bias of the coin.

Write a function to simulate an unbiased coin toss.

### Problem #1729 <a id="p1729"></a>

**Medium** · Mailchimp · 2024-09-18

#### Problem

You are given an array representing the heights of neighboring buildings on a city street, from east to west. The city assessor would like you to write an algorithm that returns how many of these buildings have a view of the setting sun, in order to properly value the street.

For example, given the array `[3, 7, 8, 3, 6, 1]`, you should return 3, since the top floors of the buildings with heights 8, 6, and 1 all have an unobstructed view to the west.

Can you do this using just one forward pass through the array?

### Problem #1730 <a id="p1730"></a>

**Medium** · Google · 2024-09-19

#### Problem

Implement integer exponentiation. That is, implement the `pow(x, y)` function, where x and y are integers and returns x^y.

Do this faster than the naive method of repeated multiplication.

For example, `pow(2, 10)` should return 1024.

### Problem #1731 <a id="p1731"></a>

**Medium** · Stitch Fix · 2024-09-20

#### Problem

Pascal's triangle is a triangular array of integers constructed with the following formula:

- The first row consists of the number 1.
- For each subsequent row, each element is the sum of the numbers directly above it, on either side.

For example, here are the first few rows:

```
    1
   1 1
  1 2 1
 1 3 3 1
1 4 6 4 1
```

Given an input k, return the kth row of Pascal's triangle.

Bonus: Can you do this using only `O(k)` space?

### Problem #1732 <a id="p1732"></a>

**Medium** · Yahoo · 2024-09-21

#### Problem

Write a function that returns the bitwise AND of all integers between M and N, inclusive.

### Problem #1733 <a id="p1733"></a>

**Hard** · Google · 2024-09-22

#### Problem

Implement an LRU (Least Recently Used) cache. It should be able to be initialized with a cache size n, and contain the following methods:

- `set(key, value)`: sets key to value. If there are already n items in the cache and we are adding a new item, then it should also remove the least recently used item.
- `get(key)`: gets the value at key. If no such key exists, return null.

Each operation should run in `O(1)` time.

### Problem #1734 <a id="p1734"></a>

**Easy** · Apple · 2024-09-23

#### Problem

A fixed point in an array is an element whose value is equal to its index. Given a sorted array of distinct elements, return a fixed point, if one exists. Otherwise, return False.

For example, given `[-6, 0, 2, 40]`, you should return 2. Given `[1, 5, 7, 8]`, you should return False.

### Problem #1735 <a id="p1735"></a>

**Easy** · Google · 2024-09-24

#### Problem

Given the head of a singly linked list, reverse it in-place.

### Problem #1736 <a id="p1736"></a>

**Medium** · Apple · 2024-09-25

#### Problem

Implement a queue using two stacks. Recall that a queue is a FIFO (first-in, first-out) data structure with the following methods: enqueue, which inserts an element into the queue, and dequeue, which removes it.

### Problem #1737 <a id="p1737"></a>

**Medium** · Facebook · 2024-09-26

#### Problem

Given an array of integers in which two elements appear exactly once and all other elements appear exactly twice, find the two elements that appear only once.

For example, given the array `[2, 4, 6, 8, 10, 2, 6, 10]`, return 4 and 8. The order does not matter.

Follow-up: Can you do this in linear time and constant space?

### Problem #1738 <a id="p1738"></a>

**Hard** · Google · 2024-09-27

#### Problem

In a directed graph, each node is assigned an uppercase letter. We define a path's value as the number of most frequently-occurring letter along that path. For example, if a path in the graph goes through `"ABACA"`, the value of the path is 3, since there are 3 occurrences of `'A'` on the path.

Given a graph with n nodes and m directed edges, return the largest value path of the graph. If the largest value is infinite, then return null.

The graph is represented with a string and an edge list. The i-th character represents the uppercase letter of the i-th node. Each tuple in the edge list (i, j) means there is a directed edge from the i-th node to the j-th node. Self-edges are possible, as well as multi-edges.

For example, the following input graph:

ABACA

```
[(0, 1),
 (0, 2),
 (2, 3),
 (3, 4)]
```

Would have maximum value 3 using the path of vertices `[0, 2, 3, 4]`, (A, A, C, A).

The following input graph:

A

`[(0, 0)]`

Should return null, since we have an infinite loop.

### Problem #1739 <a id="p1739"></a>

**Medium** · Google · 2024-09-28

#### Problem

Implement locking in a binary tree. A binary tree node can be locked or unlocked only if all of its descendants or ancestors are not locked.

Design a binary tree node class with the following methods:

- `is_locked`, which returns whether the node is locked
- lock, which attempts to lock the node. If it cannot be locked, then it should return false. Otherwise, it should lock it and return true.
- unlock, which unlocks the node. If it cannot be unlocked, then it should return false. Otherwise, it should unlock it and return true.

You may augment the node to add parent pointers or any other property you would like. You may assume the class is used in a single-threaded program, so there is no need for actual locks or mutexes. Each method should run in `O(h)`, where h is the height of the tree.

### Problem #1740 <a id="p1740"></a>

**Hard** · LinkedIn · 2024-09-29

#### Problem

Given a string, return whether it represents a number. Here are the different kinds of numbers:

- `"10"`, a positive integer
- `"-10"`, a negative integer
- `"10.1"`, a positive real number
- `"-10.1"`, a negative real number
- `"1e5"`, a number in scientific notation

And here are examples of non-numbers:

- `"a"`
- `"x 1"`
- `"a -2"`
- `"-"`

### Problem #1741 <a id="p1741"></a>

**Hard** · Google · 2024-09-30

#### Problem

Design a system to crawl and copy all of Wikipedia using a distributed network of machines.

More specifically, suppose your server has access to a set of client machines. Your client machines can execute code you have written to access Wikipedia pages, download and parse their data, and write the results to a database.

Some questions you may want to consider as part of your solution are:

- How will you reach as many pages as possible?
- How can you keep track of pages that have already been visited?
- How will you deal with your client machines being blacklisted?
- How can you update your database when Wikipedia pages are added or updated?

### Problem #1742 <a id="p1742"></a>

**Easy** · Amazon · 2024-10-01

#### Problem

Given an integer N, construct all possible binary search trees with N nodes.

### Problem #1743 <a id="p1743"></a>

**Medium** · Stripe · 2024-10-02

#### Problem

reduce (also known as fold) is a function that takes in an array, a combining function, and an initial value and builds up a result by calling the combining function on each element of the array, left to right. For example, we can write `sum()` in terms of reduce:

```
def add(a, b):
    return a + b
```

```
def sum(lst):
    return reduce(lst, add, 0)
```

This should call add on the initial value with the first element of the array, and then the result of that with the second element of the array, and so on until we reach the end, when we return the sum of the array.

Implement your own version of reduce.

### Problem #1744 <a id="p1744"></a>

**Medium** · Google · 2024-10-03

#### Problem

Given k sorted singly linked lists, write a function to merge all the lists into one sorted singly linked list.

### Problem #1745 <a id="p1745"></a>

**Medium** · Pinterest · 2024-10-04

#### Problem

At a party, there is a single person who everyone knows, but who does not know anyone in return (the `"celebrity"`). To help figure out who this is, you have access to an `O(1)` method called `knows(a, b)`, which returns True if person a knows person b, else False.

Given a list of N people and the above operation, find a way to identify the celebrity in `O(N)` time.

### Problem #1746 <a id="p1746"></a>

**Medium** · Triplebyte · 2024-10-05

#### Problem

You are given n numbers as well as n probabilities that sum up to 1. Write a function to generate one of the numbers with its corresponding probability.

For example, given the numbers `[1, 2, 3, 4]` and probabilities `[0.1, 0.5, 0.2, 0.2]`, your function should return 1 10% of the time, 2 50% of the time, and 3 and 4 20% of the time.

You can generate random numbers between 0 and 1 uniformly.

### Problem #1747 <a id="p1747"></a>

**Easy** · Microsoft · 2024-10-06

#### Problem

Given a number in the form of a list of digits, return all possible permutations.

For example, given `[1,2,3]`, return `[[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]`.

### Problem #1748 <a id="p1748"></a>

**Easy** · Slack · 2024-10-07

#### Problem

You are given a string formed by concatenating several words corresponding to the integers zero through nine and then anagramming.

For example, the input could be `'niesevehrtfeev'`, which is an anagram of `'threefiveseven'`. Note that there can be multiple instances of each integer.

Given this string, return the original integers in sorted order. In the example above, this would be 357.

### Problem #1749 <a id="p1749"></a>

**Medium** · Facebook · 2024-10-08

#### Problem

Given an array of numbers representing the stock prices of a company in chronological order and an integer k, return the maximum profit you can make from k buys and sells. You must buy the stock before you can sell it, and you must sell the stock before you can buy it again.

For example, given k = 2 and the array `[5, 2, 4, 0, 1]`, you should return 3.

### Problem #1750 <a id="p1750"></a>

**Medium** · Google · 2024-10-09

#### Problem

You are given an array of nonnegative integers. Let's say you start at the beginning of the array and are trying to advance to the end. You can advance at most, the number of steps that you're currently on. Determine whether you can get to the end of the array.

For example, given the array `[1, 3, 1, 2, 0, 1]`, we can go from indices 0 -> 1 -> 3 -> 5, so return true.

Given the array `[1, 2, 1, 0, 0]`, we can't reach the end, so return false.

### Problem #1751 <a id="p1751"></a>

**Easy** · Google · 2024-10-10

#### Problem

You are in an infinite 2D grid where you can move in any of the 8 directions:

```
(x,y) to
   (x+1, y),
   (x - 1, y),
   (x, y+1),
   (x, y-1),
   (x-1, y-1),
   (x+1,y+1),
   (x-1,y+1),
   (x+1,y-1)
```

You are given a sequence of points and the order in which you need to cover the points. Give the minimum number of steps in which you can achieve it. You start from the first point.

Example:

```
Input: [(0, 0), (1, 1), (1, 2)]
Output: 2
```

It takes 1 step to move from (0, 0) to (1, 1). It takes one more step to move from (1, 1) to (1, 2).

### Problem #1752 <a id="p1752"></a>

**Medium** · Google · 2024-10-11

#### Problem

Given a singly linked list and an integer k, remove the kth last element from the list. k is guaranteed to be smaller than the length of the list.

The list is very long, so making more than one pass is prohibitively expensive.

Do this in constant space and in one pass.

### Problem #1753 <a id="p1753"></a>

**Easy** · Palantir · 2024-10-12

#### Problem

The ancient Egyptians used to express fractions as a sum of several terms where each numerator is one. For example, 4 / 13 can be represented as 1 / 4 + 1 / 18 + 1 / 468.

Create an algorithm to turn an ordinary fraction a / b, where a \< b, into an Egyptian fraction.

### Problem #1754 <a id="p1754"></a>

**Easy** · Twitter · 2024-10-13

#### Problem

A strobogrammatic number is a positive number that appears the same after being rotated 180 degrees. For example, 16891 is strobogrammatic.

Create a program that finds all strobogrammatic numbers with N digits.

### Problem #1755 <a id="p1755"></a>

**Easy** · Amazon · 2024-10-14

#### Problem

Given a complete binary tree, count the number of nodes in faster than `O(n)` time. Recall that a complete binary tree has every level filled except the last, and the nodes in the last level are filled starting from the left.

### Problem #1756 <a id="p1756"></a>

**Easy** · Riot Games · 2024-10-15

#### Problem

Design and implement a HitCounter class that keeps track of requests (or hits). It should support the following operations:

- `record(timestamp)`: records a hit that happened at timestamp
- `total()`: returns the total number of hits recorded
- `range(lower, upper)`: returns the number of hits that occurred between timestamps lower and upper (inclusive)

Follow-up: What if our system has limited memory?

### Problem #1757 <a id="p1757"></a>

**Hard** · Amazon · 2024-10-16

#### Problem

There exists a staircase with N steps, and you can climb up either 1 or 2 steps at a time. Given N, write a function that returns the number of unique ways you can climb the staircase. The order of the steps matters.

For example, if N is 4, then there are 5 unique ways:

- 1, 1, 1, 1
- 2, 1, 1
- 1, 2, 1
- 1, 1, 2
- 2, 2

What if, instead of being able to climb 1 or 2 steps at a time, you could climb any number from a set of positive integers X? For example, if X = `{1, 3, 5}`, you could climb 1, 3, or 5 steps at a time.

### Problem #1758 <a id="p1758"></a>

**Hard** · Google · 2024-10-17

#### Problem

Given an array of strictly the characters `'R'`, `'G'`, and `'B'`, segregate the values of the array so that all the Rs come first, the Gs come second, and the Bs come last. You can only swap elements of the array.

Do this in linear time and in-place.

For example, given the array `['G', 'B', 'R', 'R', 'B', 'R', 'G']`, it should become `['R', 'R', 'R', 'G', 'G', 'B', 'B']`.

### Problem #1759 <a id="p1759"></a>

**Hard** · Twitch · 2024-10-18

#### Problem

Describe what happens when you type a URL into your browser and press Enter.

### Problem #1760 <a id="p1760"></a>

**Hard** · Google · 2024-10-19

#### Problem

Given an array of integers and a number k, where 1 \<= k \<= length of the array, compute the maximum values of each subarray of length k.

For example, given array = `[10, 5, 2, 7, 8, 7]` and k = 3, we should get: `[10, 7, 8, 8]`, since:

- 10 = `max(10, 5, 2)`
- 7 = `max(5, 2, 7)`
- 8 = `max(2, 7, 8)`
- 8 = `max(7, 8, 7)`

Do this in `O(n)` time and `O(k)` space. You can modify the input array in-place and you do not need to store the results. You can simply print them out as you compute them.

### Problem #1761 <a id="p1761"></a>

**Medium** · Google · 2024-10-20

#### Problem

Given a stack of N elements, interleave the first half of the stack with the second half reversed using only one other queue. This should be done in-place.

Recall that you can only push or pop from a stack, and enqueue or dequeue from a queue.

For example, if the stack is `[1, 2, 3, 4, 5]`, it should become `[1, 5, 2, 4, 3]`. If the stack is `[1, 2, 3, 4]`, it should become `[1, 4, 2, 3]`.

Hint: Try working backwards from the end state.

### Problem #1762 <a id="p1762"></a>

**Hard** · Quantcast · 2024-10-21

#### Problem

You are presented with an array representing a Boolean expression. The elements are of two kinds:

- T and F, representing the values True and False.
- &, |, and ^, representing the bitwise operators for AND, OR, and XOR.

Determine the number of ways to group the array elements using parentheses so that the entire expression evaluates to True.

For example, suppose the input is `['F', '|', 'T', '&', 'T']`. In this case, there are two acceptable groupings: (F | T) & T and F | (T & T).

### Problem #1763 <a id="p1763"></a>

**Medium** · Amazon · 2024-10-22

#### Problem

Given an array of a million integers between zero and a billion, out of order, how can you efficiently sort it? Assume that you cannot store an array of a billion elements in memory.

### Problem #1764 <a id="p1764"></a>

**Hard** · Grammarly · 2024-10-23

#### Problem

Soundex is an algorithm used to categorize phonetically, such that two names that sound alike but are spelled differently have the same representation.

Soundex maps every name to a string consisting of one letter and three numbers, like M460.

One version of the algorithm is as follows:

- Remove consecutive consonants with the same sound (for example, change ck -> c).
- Keep the first letter. The remaining steps only apply to the rest of the string.
- Remove all vowels, including y, w, and h.
- Replace all consonants with the following digits:

- b, f, p, v → 1
- c, g, j, k, q, s, x, z → 2
- d, t → 3
- l → 4
- m, n → 5
- r → 6

- If you don't have three numbers yet, append zeros until you do. Keep the first three numbers.

Using this scheme, Jackson and Jaxen both map to J250.

Implement Soundex.

### Problem #1765 <a id="p1765"></a>

**Medium** · Facebook · 2024-10-24

#### Problem

Given a number in Roman numeral format, convert it to decimal.

The values of Roman numerals are as follows:

```
{
    'M': 1000,
    'D': 500,
    'C': 100,
    'L': 50,
    'X': 10,
    'V': 5,
    'I': 1
}
```

In addition, note that the Roman numeral system uses subtractive notation for numbers such as IV and XL.

For the input XIV, for instance, you should return 14.

### Problem #1766 <a id="p1766"></a>

**Easy** · Snapchat · 2024-10-25

#### Problem

Given a list of possibly overlapping intervals, return a new list of intervals where all overlapping intervals have been merged.

The input list is not necessarily ordered in any way.

For example, given `[(1, 3), (5, 8), (4, 10), (20, 25)]`, you should return `[(1, 3), (4, 10), (20, 25)]`.

### Problem #1767 <a id="p1767"></a>

**Medium** · Facebook · 2024-10-26

#### Problem

You are given an array of non-negative integers that represents a two-dimensional elevation map where each element is unit-width wall and the integer is the height. Suppose it will rain and all spots between two walls get filled up.

Compute how many units of water remain trapped on the map in `O(N)` time and `O(1)` space.

For example, given the input `[2, 1, 2]`, we can hold 1 unit of water in the middle.

Given the input `[3, 0, 1, 3, 0, 5]`, we can hold 3 units in the first index, 2 in the second, and 3 in the fourth index (we cannot hold 5 since it would run off to the left), so we can trap 8 units of water.

### Problem #1768 <a id="p1768"></a>

**Hard** · Uber · 2024-10-27

#### Problem

A rule looks like this:

A NE B

This means this means point A is located northeast of point B.

A SW C

means that point A is southwest of C.

Given a list of rules, check if the sum of the rules validate. For example:

```
A N B
B NE C
C N A
```

does not validate, since A cannot be both north and south of C.

```
A NW B
A N B
```

is considered valid.

### Problem #1769 <a id="p1769"></a>

**Medium** · Square · 2024-10-28

#### Problem

A competitive runner would like to create a route that starts and ends at his house, with the condition that the route goes entirely uphill at first, and then entirely downhill.

Given a dictionary of places of the form `{location: elevation}`, and a dictionary mapping paths between some of these locations to their corresponding distances, find the length of the shortest route satisfying the condition above. Assume the runner's home is location 0.

For example, suppose you are given the following input:

```
elevations = {0: 5, 1: 25, 2: 15, 3: 20, 4: 10}
paths = {
    (0, 1): 10,
    (0, 2): 8,
    (0, 3): 15,
    (1, 3): 12,
    (2, 4): 10,
    (3, 4): 5,
    (3, 0): 17,
    (4, 0): 10
}
```

In this case, the shortest valid path would be 0 -> 2 -> 4 -> 0, with a distance of 28.

### Problem #1770 <a id="p1770"></a>

**Medium** · 2024-10-29

#### Problem

Given a 2-D matrix representing an image, a location of a pixel in the screen and a color C, replace the color of the given pixel and all adjacent same colored pixels with C.

For example, given the following matrix, and location pixel of (2, 2), and `'G'` for green:

```
B B W
W W W
W W W
B B B
```

Becomes

```
B B G
G G G
G G G
B B B
```

### Problem #1771 <a id="p1771"></a>

**Hard** · Facebook · 2024-10-30

#### Problem

Given a string consisting of parentheses, single digits, and positive and negative signs, convert the string into a mathematical expression to obtain the answer.

Don't use eval or a similar built-in parser.

For example, given `'-1 + (2 + 3)'`, you should return 4.

### Problem #1772 <a id="p1772"></a>

**Medium** · Flipkart · 2024-10-31

#### Problem

Starting from 0 on a number line, you would like to make a series of jumps that lead to the integer N.

On the ith jump, you may move exactly i places to the left or right.

Find a path with the fewest number of jumps required to get from 0 to N.

### Problem #1773 <a id="p1773"></a>

**Easy** · Jane Street · 2024-11-01

#### Problem

Given integers M and N, write a program that counts how many positive integer pairs (a, b) satisfy the following conditions:

- a + b = M
- a XOR b = N

### Problem #1774 <a id="p1774"></a>

**Easy** · PagerDuty · 2024-11-02

#### Problem

Given a positive integer N, find the smallest number of steps it will take to reach 1.

There are two kinds of permitted steps:

- You may decrement N to N - 1.
- If a \* b = N, you may decrement N to the larger of a and b.

For example, given 100, you can reach 1 in five steps with the following route: 100 -> 10 -> 9 -> 3 -> 2 -> 1.

### Problem #1775 <a id="p1775"></a>

**Medium** · Facebook · 2024-11-03

#### Problem

Mastermind is a two-player game in which the first player attempts to guess the secret code of the second. In this version, the code may be any six-digit number with all distinct digits.

Each turn the first player guesses some number, and the second player responds by saying how many digits in this number correctly matched their location in the secret code. For example, if the secret code were 123456, then a guess of 175286 would score two, since 1 and 6 were correctly placed.

Write an algorithm which, given a sequence of guesses and their scores, determines whether there exists some secret code that could have produced them.

For example, for the following scores you should return True, since they correspond to the secret code 123456:

`{175286: 2, 293416: 3, 654321: 0}`

However, it is impossible for any key to result in the following scores, so in this case you should return False:

`{123456: 4, 345678: 4, 567890: 4}`

### Problem #1776 <a id="p1776"></a>

**Medium** · Uber · 2024-11-05

#### Problem

One way to unlock an Android phone is through a pattern of swipes across a 1-9 keypad.

For a pattern to be valid, it must satisfy the following:

- All of its keys must be distinct.
- It must not connect two keys by jumping over a third key, unless that key has already been used.

For example, 4 - 2 - 1 - 7 is a valid pattern, whereas 2 - 1 - 7 is not.

Find the total number of valid unlock patterns of length N, where 1 \<= N \<= 9.

### Problem #1777 <a id="p1777"></a>

**Medium** · Google · 2024-11-06

#### Problem

On our special chessboard, two bishops attack each other if they share the same diagonal. This includes bishops that have another bishop located between them, i.e. bishops can attack through pieces.

You are given N bishops, represented as (row, column) tuples on a M by M chessboard. Write a function to count the number of pairs of bishops that attack each other. The ordering of the pair doesn't matter: (1, 2) is considered the same as (2, 1).

For example, given M = 5 and the list of bishops:

- (0, 0)
- (1, 2)
- (2, 2)
- (4, 0)

The board would look like this:

```
[b 0 0 0 0]
[0 0 b 0 0]
[0 0 b 0 0]
[0 0 0 0 0]
[b 0 0 0 0]
```

You should return 2, since bishops 1 and 3 attack each other, as well as bishops 3 and 4.

### Problem #1778 <a id="p1778"></a>

**Hard** · Airbnb · 2024-11-07

#### Problem

Given a list of integers, write a function that returns the largest sum of non-adjacent numbers. Numbers can be 0 or negative.

For example, `[2, 4, 6, 2, 5]` should return 13, since we pick 2, 6, and 5. `[5, 1, 1, 5]` should return 10, since we pick 5 and 5.

Follow-up: Can you do this in `O(N)` time and constant space?

### Problem #1779 <a id="p1779"></a>

**Hard** · Google · 2024-11-08

#### Problem

Find the minimum number of coins required to make n cents.

You can use standard American denominations, that is, 1¢, 5¢, 10¢, and 25¢.

For example, given n = 16, return 3 since we can make it with a 10¢, a 5¢, and a 1¢.

### Problem #1780 <a id="p1780"></a>

**Medium** · Cisco · 2024-11-09

#### Problem

Given an unsigned 8-bit integer, swap its even and odd bits. The 1st and 2nd bit should be swapped, the 3rd and 4th bit should be swapped, and so on.

For example, 10101010 should be 01010101. 11100010 should be 11010001.

Bonus: Can you do this in one line?

### Problem #1781 <a id="p1781"></a>

**Easy** · Jane Street · 2024-11-10

#### Problem

The United States uses the imperial system of weights and measures, which means that there are many different, seemingly arbitrary units to measure distance. There are 12 inches in a foot, 3 feet in a yard, 22 yards in a chain, and so on.

Create a data structure that can efficiently convert a certain quantity of one unit to the correct amount of any other unit. You should also allow for additional units to be added to the system.

### Problem #1782 <a id="p1782"></a>

**Medium** · Facebook · 2024-11-11

#### Problem

Given a positive integer n, find the smallest number of squared integers which sum to n.

For example, given n = 13, return 2 since 13 = 32 + 22 = 9 + 4.

Given n = 27, return 3 since 27 = 32 + 32 + 32 = 9 + 9 + 9.

### Problem #1783 <a id="p1783"></a>

**Medium** · Uber · 2024-11-12

#### Problem

You are given a 2-d matrix where each cell consists of either /, \\, or an empty space. Write an algorithm that determines into how many regions the slashes divide the space.

For example, suppose the input for a three-by-six grid is the following:

```
\    /
 \  /
  \/
```

Considering the edges of the matrix as boundaries, this divides the grid into three triangles, so you should return 3.

### Problem #1784 <a id="p1784"></a>

**Hard** · Palantir · 2024-11-13

#### Problem

Typically, an implementation of in-order traversal of a binary tree has `O(h)` space complexity, where h is the height of the tree. Write a program to compute the in-order traversal of a binary tree using `O(1)` space.

### Problem #1785 <a id="p1785"></a>

**Medium** · Facebook · 2024-11-14

#### Problem

Given an unordered list of flights taken by someone, each represented as (origin, destination) pairs, and a starting airport, compute the person's itinerary. If no such itinerary exists, return null. If there are multiple possible itineraries, return the lexicographically smallest one. All flights must be used in the itinerary.

For example, given the list of flights `[('SFO', 'HKO'), ('YYZ', 'SFO'), ('YUL', 'YYZ'), ('HKO', 'ORD')]` and starting airport `'YUL'`, you should return the list `['YUL', 'YYZ', 'SFO', 'HKO', 'ORD']`.

Given the list of flights `[('SFO', 'COM'), ('COM', 'YYZ')]` and starting airport `'COM'`, you should return null.

Given the list of flights `[('A', 'B'), ('A', 'C'), ('B', 'C'), ('C', 'A')]` and starting airport `'A'`, you should return the list `['A', 'B', 'C', 'A', 'C']` even though `['A', 'C', 'A', 'B', 'C']` is also a valid itinerary. However, the first one is lexicographically smaller.

### Problem #1786 <a id="p1786"></a>

**Easy** · Twitter · 2024-11-15

#### Problem

The 24 game is played as follows. You are given a list of four integers, each between 1 and 9, in a fixed order. By placing the operators +, -, \*, and / between the numbers, and grouping them with parentheses, determine whether it is possible to reach the value 24.

For example, given the input `[5, 2, 7, 8]`, you should return True, since (5 \* 2 - 7) \* 8 = 24.

Write a function that plays the 24 game.

### Problem #1787 <a id="p1787"></a>

**Easy** · Bloomberg · 2024-11-16

#### Problem

There are N prisoners standing in a circle, waiting to be executed. The executions are carried out starting with the kth person, and removing every successive kth person going clockwise until there is no one left.

Given N and k, write an algorithm to determine where a prisoner should stand in order to be the last survivor.

For example, if N = 5 and k = 2, the order of executions would be `[2, 4, 1, 5, 3]`, so you should return 3.

Bonus: Find an `O(log N)` solution if k = 2.

### Problem #1788 <a id="p1788"></a>

**Hard** · LinkedIn · 2024-11-17

#### Problem

Given a list of points, a central point, and an integer k, find the nearest k points from the central point.

For example, given the list of points `[(0, 0), (5, 4), (3, 1)]`, the central point (1, 2), and k = 2, return `[(0, 0), (3, 1)]`.

### Problem #1789 <a id="p1789"></a>

**Medium** · Spotify · 2024-11-18

#### Problem

You are the technical director of WSPT radio, serving listeners nationwide. For simplicity's sake we can consider each listener to live along a horizontal line stretching from 0 (west) to 1000 (east).

Given a list of N listeners, and a list of M radio towers, each placed at various locations along this line, determine what the minimum broadcast range would have to be in order for each listener's home to be covered.

For example, suppose listeners = `[1, 5, 11, 20]`, and towers = `[4, 8, 15]`. In this case the minimum range would be 5, since that would be required for the tower at position 15 to reach the listener at position 20.

### Problem #1790 <a id="p1790"></a>

**Easy** · Apple · 2024-11-19

#### Problem

Implement the function `fib(n)`, which returns the nth number in the Fibonacci sequence, using only `O(1)` space.

### Problem #1791 <a id="p1791"></a>

**Hard** · Dropbox · 2024-11-20

#### Problem

Create a data structure that performs all the following operations in `O(1)` time:

- plus: Add a key with value 1. If the key already exists, increment its value by one.
- minus: Decrement the value of a key. If the key's value is currently 1, remove it.
- `get_max`: Return a key with the highest value.
- `get_min`: Return a key with the lowest value.

### Problem #1792 <a id="p1792"></a>

**Easy** · Google · 2024-11-21

#### Problem

Given a binary tree of integers, find the maximum path sum between two nodes. The path must go through at least one node, and does not need to go through the root.

### Problem #1793 <a id="p1793"></a>

**Medium** · Walmart Labs · 2024-11-22

#### Problem

There are M people sitting in a row of N seats, where M \< N. Your task is to redistribute people such that there are no gaps between any of them, while keeping overall movement to a minimum.

For example, suppose you are faced with an input of `[0, 1, 1, 0, 1, 0, 0, 0, 1]`, where 0 represents an empty seat and 1 represents a person. In this case, one solution would be to place the person on the right in the fourth seat. We can consider the cost of a solution to be the sum of the absolute distance each person must move, so that the cost here would be five.

Given an input such as the one above, return the lowest possible cost of moving people to remove all gaps.

### Problem #1794 <a id="p1794"></a>

**Hard** · Google · 2024-11-23

#### Problem

The game of Nim is played as follows. Starting with three heaps, each containing a variable number of items, two players take turns removing one or more items from a single pile. The player who eventually is forced to take the last stone loses. For example, if the initial heap sizes are 3, 4, and 5, a game could be played as shown below:

```
  A  |  B  |  C
-----------------
  3  |  4  |  5
  3  |  1  |  3
  3  |  1  |  3
  0  |  1  |  3
  0  |  1  |  0
  0  |  0  |  0
```

In other words, to start, the first player takes three items from pile B. The second player responds by removing two stones from pile C. The game continues in this way until player one takes last stone and loses.

Given a list of non-zero starting values `[a, b, c]`, and assuming optimal play, determine whether the first player has a forced win.

### Problem #1795 <a id="p1795"></a>

**Medium** · Google · 2024-11-24

#### Problem

Given an iterator with methods `next()` and `hasNext()`, create a wrapper iterator, PeekableInterface, which also implements `peek()`. peek shows the next element that would be returned on `next()`.

Here is the interface:

```
class PeekableInterface(object):
    def __init__(self, iterator):
        pass
```

```
def peek(self):
    pass
```

```
def next(self):
    pass
```

```
def hasNext(self):
    pass
```

### Problem #1796 <a id="p1796"></a>

**Medium** · Google · 2024-11-25

#### Problem

Given an integer n and a list of integers l, write a function that randomly generates a number from 0 to n-1 that isn't in l (uniform).

### Problem #1797 <a id="p1797"></a>

**Medium** · Amazon · 2024-11-26

#### Problem

Given a string s and an integer k, break up the string into multiple lines such that each line has a length of k or less. You must break it up so that words don't break across lines. Each line has to have the maximum possible amount of words. If there's no way to break the text up, then return null.

You can assume that there are no spaces at the ends of the string and that there is exactly one space between each word.

For example, given the string `"the quick brown fox jumps over the lazy dog"` and k = 10, you should return: `["the quick", "brown fox", "jumps over", "the lazy", "dog"]`. No string in the list has a length of more than 10.

### Problem #1798 <a id="p1798"></a>

**Easy** · Google · 2024-11-27

#### Problem

You are given an M by N matrix consisting of booleans that represents a board. Each True boolean represents a wall. Each False boolean represents a tile you can walk on.

Given this matrix, a start coordinate, and an end coordinate, return the minimum number of steps required to reach the end coordinate from the start. If there is no possible path, then return null. You can move up, left, down, and right. You cannot move through walls. You cannot wrap around the edges of the board.

For example, given the following board:

```
[[f, f, f, f],
[t, t, f, t],
[f, f, f, f],
[f, f, f, f]]
```

and start = (3, 0) (bottom left) and end = (0, 0) (top left), the minimum number of steps required to reach the end is 7, since we would need to go through (1, 2) because there is a wall everywhere else on the second row.

### Problem #1799 <a id="p1799"></a>

**Easy** · Google · 2024-11-28

#### Problem

A girl is walking along an apple orchard with a bag in each hand. She likes to pick apples from each tree as she goes along, but is meticulous about not putting different kinds of apples in the same bag.

Given an input describing the types of apples she will pass on her path, in order, determine the length of the longest portion of her path that consists of just two types of apple trees.

For example, given the input `[2, 1, 2, 3, 3, 1, 3, 5]`, the longest portion will involve types 1 and 3, with a length of four.

### Problem #1800 <a id="p1800"></a>

**Easy** · Palantir · 2024-11-29

#### Problem

Write a program that checks whether an integer is a palindrome. For example, 121 is a palindrome, as well as 888. 678 is not a palindrome. Do not convert the integer into a string.

### Problem #1801 <a id="p1801"></a>

**Hard** · Facebook · 2024-11-30

#### Problem

Given a string of parentheses, find the balanced string that can be produced from it using the minimum number of insertions and deletions. If there are multiple solutions, return any of them.

For example, given `"(()"`, you could return `"(())"`. Given `"))()("`, you could return `"()()()()"`.

### Problem #1802 <a id="p1802"></a>

**Hard** · Airbnb · 2024-12-01

#### Problem

Given a list of words, find all pairs of unique indices such that the concatenation of the two words is a palindrome.

For example, given the list `["code", "edoc", "da", "d"]`, return `[(0, 1), (1, 0), (2, 3)]`.

### Problem #1803 <a id="p1803"></a>

**Medium** · Microsoft · 2024-12-02

#### Problem

Implement the singleton pattern with a twist. First, instead of storing one instance, store two instances. And in every even call of `getInstance()`, return the first instance and in every odd call of `getInstance()`, return the second instance.

### Problem #1804 <a id="p1804"></a>

**Medium** · Google · 2024-12-03

#### Problem

You are given an array of length n + 1 whose elements belong to the set `{1, 2, ..., n}`. By the pigeonhole principle, there must be a duplicate. Find it in linear time and space.

### Problem #1805 <a id="p1805"></a>

**Medium** · Mozilla · 2024-12-04

#### Problem

A bridge in a connected (undirected) graph is an edge that, if removed, causes the graph to become disconnected. Find all the bridges in a graph.

### Problem #1806 <a id="p1806"></a>

**Medium** · 2024-12-05

#### Problem

The Tower of Hanoi is a puzzle game with three rods and n disks, each a different size.

All the disks start off on the first rod in a stack. They are ordered by size, with the largest disk on the bottom and the smallest one at the top.

The goal of this puzzle is to move all the disks from the first rod to the last rod while following these rules:

- You can only move one disk at a time.
- A move consists of taking the uppermost disk from one of the stacks and placing it on top of another stack.
- You cannot place a larger disk on top of a smaller disk.

Write a function that prints out all the steps necessary to complete the Tower of Hanoi. You should assume that the rods are numbered, with the first rod being 1, the second (auxiliary) rod being 2, and the last (goal) rod being 3.

For example, with n = 3, we can do this in 7 moves:

```
Move 1 to 3
Move 1 to 2
Move 3 to 2
Move 1 to 3
Move 2 to 1
Move 2 to 3
Move 1 to 3
```

### Problem #1807 <a id="p1807"></a>

**Medium** · Etsy · 2024-12-06

#### Problem

Given an array of numbers N and an integer k, your task is to split N into k partitions such that the maximum sum of any partition is minimized. Return this sum.

For example, given N = `[5, 1, 2, 7, 3, 4]` and k = 3, you should return 8, since the optimal partition is `[5, 1, 2]`, `[7]`, `[3, 4]`.

### Problem #1808 <a id="p1808"></a>

**Medium** · Squarespace · 2024-12-07

#### Problem

Write a function, `add_subtract`, which alternately adds and subtracts curried arguments. Here are some sample operations:

`add_subtract(7)` -> 7

`add_subtract(1)(2)(3)` -> 1 + 2 - 3 -> 0

`add_subtract(-5)(10)(3)(9)` -> -5 + 10 - 3 + 9 -> 11

### Problem #1809 <a id="p1809"></a>

**Hard** · Two Sigma · 2024-12-08

#### Problem

A knight is placed on a given square on an 8 x 8 chessboard. It is then moved randomly several times, where each move is a standard knight move. If the knight jumps off the board at any point, however, it is not allowed to jump back on.

After k moves, what is the probability that the knight remains on the board?

### Problem #1810 <a id="p1810"></a>

**Easy** · Microsoft · 2024-12-09

#### Problem

Given an array of numbers and a number k, determine if there are three entries in the array which add up to the specified number k. For example, given `[20, 303, 3, 4, 25]` and k = 49, return true as 20 + 4 + 25 = 49.

### Problem #1811 <a id="p1811"></a>

**Hard** · Microsoft · 2024-12-10

#### Problem

Given an array of numbers, find the length of the longest increasing subsequence in the array. The subsequence does not necessarily have to be contiguous.

For example, given the array `[0, 8, 4, 12, 2, 10, 6, 14, 1, 9, 5, 13, 3, 11, 7, 15]`, the longest increasing subsequence has length 6: it is 0, 2, 6, 9, 11, 15.

### Problem #1812 <a id="p1812"></a>

**Hard** · Twitter · 2024-12-11

#### Problem

A teacher must divide a class of students into two teams to play dodgeball. Unfortunately, not all the kids get along, and several refuse to be put on the same team as that of their enemies.

Given an adjacency list of students and their enemies, write an algorithm that finds a satisfactory pair of teams, or returns False if none exists.

For example, given the following enemy graph you should return the teams `{0, 1, 4, 5}` and `{2, 3}`.

```
students = {
    0: [3],
    1: [2],
    2: [1, 4],
    3: [0, 4, 5],
    4: [2, 3],
    5: [3]
}
```

On the other hand, given the input below, you should return False.

```
students = {
    0: [3],
    1: [2],
    2: [1, 3, 4],
    3: [0, 2, 4, 5],
    4: [2, 3],
    5: [3]
}
```

### Problem #1813 <a id="p1813"></a>

**Easy** · Twitter · 2024-12-12

#### Problem

You run an e-commerce website and want to record the last N order ids in a log. Implement a data structure to accomplish this, with the following API:

- `record(order_id)`: adds the `order_id` to the log
- `get_last(i)`: gets the ith last element from the log. i is guaranteed to be smaller than or equal to N.

You should be as efficient with time and space as possible.

### Problem #1814 <a id="p1814"></a>

**Medium** · Flipkart · 2024-12-13

#### Problem

Snakes and Ladders is a game played on a 10 x 10 board, the goal of which is get from square 1 to square 100. On each turn players will roll a six-sided die and move forward a number of spaces equal to the result. If they land on a square that represents a snake or ladder, they will be transported ahead or behind, respectively, to a new square.

Find the smallest number of turns it takes to play snakes and ladders.

For convenience, here are the squares representing snakes and ladders, and their outcomes:

```
snakes = {16: 6, 48: 26, 49: 11, 56: 53, 62: 19, 64: 60, 87: 24, 93: 73, 95: 75, 98: 78}
ladders = {1: 38, 4: 14, 9: 31, 21: 42, 28: 84, 36: 44, 51: 67, 71: 91, 80: 100}
```

### Problem #1815 <a id="p1815"></a>

**Medium** · Apple · 2024-12-14

#### Problem

Gray code is a binary code where each successive value differ in only one bit, as well as when wrapping around. Gray code is common in hardware so that we don't see temporary spurious values during transitions.

Given a number of bits n, generate a possible gray code for it.

For example, for n = 2, one gray code would be `[00, 01, 11, 10]`.

### Problem #1816 <a id="p1816"></a>

**Medium** · Stripe · 2024-12-15

#### Problem

Write a map implementation with a get function that lets you retrieve the value of a key at a particular time.

It should contain the following methods:

- `set(key, value, time)`: sets key to value for t = time.
- `get(key, time)`: gets the key at t = time.

The map should work like this. If we set a key at a particular time, it will maintain that value forever or until it gets set at a later time. In other words, when we get a key at a time, it should return the value that was set for that key set at the most recent time.

Consider the following examples:

```
d.set(1, 1, 0) # set key 1 to value 1 at time 0
d.set(1, 2, 2) # set key 1 to value 2 at time 2
d.get(1, 1) # get key 1 at time 1 should be 1
d.get(1, 3) # get key 1 at time 3 should be 2
```

```
d.set(1, 1, 5) # set key 1 to value 1 at time 5
d.get(1, 0) # get key 1 at time 0 should be null
d.get(1, 10) # get key 1 at time 10 should be 1
```

```
d.set(1, 1, 0) # set key 1 to value 1 at time 0
d.set(1, 2, 0) # set key 1 to value 2 at time 0
d.get(1, 0) # get key 1 at time 0 should be 2
```

### Problem #1817 <a id="p1817"></a>

**Hard** · LinkedIn · 2024-12-16

#### Problem

Given a set of characters C and an integer k, a De Bruijn sequence is a cyclic sequence in which every possible k-length string of characters in C occurs exactly once.

For example, suppose C = `{0, 1}` and k = 3. Then our sequence should contain the substrings `{'000', '001', '010', '011', '100', '101', '110', '111'}`, and one possible solution would be 00010111.

Create an algorithm that finds a De Bruijn sequence.

### Problem #1818 <a id="p1818"></a>

**Easy** · Google · 2024-12-17

#### Problem

Given a set of points (x, y) on a 2D cartesian plane, find the two closest points. For example, given the points `[(1, 1), (-1, -1), (3, 4), (6, 1), (-1, -6), (-4, -3)]`, return `[(-1, -1), (1, 1)]`.

### Problem #1819 <a id="p1819"></a>

**Hard** · Uber · 2024-12-18

#### Problem

Given a tree where each edge has a weight, compute the length of the longest path in the tree.

For example, given the following tree:

```
  a
 /|\
b c d
   / \
  e   f
 / \
g   h
```

and the weights: a-b: 3, a-c: 5, a-d: 8, d-e: 2, d-f: 4, e-g: 1, e-h: 1, the longest path would be c -> a -> d -> f, with a length of 17.

The path does not have to pass through the root, and each node can have any amount of children.

### Problem #1820 <a id="p1820"></a>

**Easy** · Google · 2024-12-19

#### Problem

Given two strings A and B, return whether or not A can be shifted some number of times to get B.

For example, if A is abcde and B is cdeab, return true. If A is abc and B is acb, return false.

### Problem #1821 <a id="p1821"></a>

**Medium** · 2024-12-20

#### Problem

Given a real number n, find the square root of n. For example, given n = 9, return 3.

### Problem #1822 <a id="p1822"></a>

**Medium** · Dropbox · 2024-12-21

#### Problem

Given an undirected graph G, check whether it is bipartite. Recall that a graph is bipartite if its vertices can be divided into two independent sets, U and V, such that no edge connects vertices of the same set.

### Problem #1823 <a id="p1823"></a>

**Medium** · Yahoo · 2024-12-22

#### Problem

Recall that a full binary tree is one in which each node is either a leaf node, or has two children. Given a binary tree, convert it to a full one by removing nodes with only one child.

For example, given the following tree:

```
         0
      /     \
    1         2
  /            \
3                 4
  \             /   \
    5          6     7
```

You should convert it to:

```
     0
  /     \
5         4
        /   \
       6     7
```

### Problem #1824 <a id="p1824"></a>

**Hard** · Google · 2024-12-23

#### Problem

A knight's tour is a sequence of moves by a knight on a chessboard such that all squares are visited once.

Given N, write a function to return the number of knight's tours on an N by N chessboard.

### Problem #1825 <a id="p1825"></a>

**Medium** · Dropbox · 2024-12-24

#### Problem

Given a list of words, determine whether the words can be chained to form a circle. A word X can be placed in front of another word Y in a circle if the last character of X is same as the first character of Y.

For example, the words `['chair', 'height', 'racket', touch', 'tunic']` can form the following circle: chair --> racket --> touch --> height --> tunic --> chair.

### Problem #1826 <a id="p1826"></a>

**Medium** · Twitter · 2024-12-25

#### Problem

A network consists of nodes labeled 0 to N. You are given a list of edges (a, b, t), describing the time t it takes for a message to be sent from node a to node b. Whenever a node receives a message, it immediately passes the message on to a neighboring node, if possible.

Assuming all nodes are connected, determine how long it will take for every node to receive a message that begins at node 0.

For example, given N = 5, and the following edges:

```
edges = [
    (0, 1, 5),
    (0, 2, 3),
    (0, 5, 4),
    (1, 3, 8),
    (2, 3, 1),
    (3, 5, 10),
    (3, 4, 5)
]
```

You should return 9, because propagating the message from 0 -> 2 -> 3 -> 4 will take that much time.

### Problem #1827 <a id="p1827"></a>

**Medium** · Goldman Sachs · 2024-12-26

#### Problem

You are given N identical eggs and access to a building with k floors. Your task is to find the lowest floor that will cause an egg to break, if dropped from that floor. Once an egg breaks, it cannot be dropped again. If an egg breaks when dropped from the xth floor, you can assume it will also break when dropped from any floor greater than x.

Write an algorithm that finds the minimum number of trial drops it will take, in the worst case, to identify this floor.

For example, if N = 1 and k = 5, we will need to try dropping the egg at every floor, beginning with the first, until we reach the fifth floor, so our solution will be 5.

### Problem #1828 <a id="p1828"></a>

**Hard** · Jane Street · 2024-12-27

#### Problem

Suppose you are given a table of currency exchange rates, represented as a 2D array. Determine whether there is a possible arbitrage: that is, whether there is some sequence of trades you can make, starting with some amount A of any currency, so that you can end up with some amount greater than A of that currency.

There are no transaction costs and you can trade fractional quantities.

### Problem #1829 <a id="p1829"></a>

**Hard** · Two Sigma · 2024-12-28

#### Problem

Ghost is a two-person word game where players alternate appending letters to a word. The first person who spells out a word, or creates a prefix for which there is no possible continuation, loses. Here is a sample game:

- Player 1: g
- Player 2: h
- Player 1: o
- Player 2: s
- Player 1: t \[loses\]

Given a dictionary of words, determine the letters the first player should start with, such that with optimal play they cannot lose.

For example, if the dictionary is `["cat", "calf", "dog", "bear"]`, the only winning start letter would be b.

### Problem #1830 <a id="p1830"></a>

**Easy** · Coursera · 2024-12-29

#### Problem

Given a 2D board of characters and a word, find if the word exists in the grid.

The word can be constructed from letters of sequentially adjacent cell, where `"adjacent"` cells are those horizontally or vertically neighboring. The same letter cell may not be used more than once.

For example, given the following board:

```
[
  ['A','B','C','E'],
  ['S','F','C','S'],
  ['A','D','E','E']
]
```

`exists(board, "ABCCED")` returns true, `exists(board, "SEE")` returns true, `exists(board, "ABCB")` returns false.

### Problem #1831 <a id="p1831"></a>

**Easy** · Amazon · 2024-12-30

#### Problem

Given a sorted array, find the smallest positive integer that is not the sum of a subset of the array.

For example, for the input `[1, 2, 3, 10]`, you should return 7.

Do this in `O(N)` time.

### Problem #1832 <a id="p1832"></a>

**Easy** · Microsoft · 2024-12-31

#### Problem

You have n fair coins and you flip them all at the same time. Any that come up tails you set aside. The ones that come up heads you flip again. How many rounds do you expect to play before only one coin remains?

Write a function that, given n, returns the number of rounds you'd expect to play until one coin remains.

### Problem #1833 <a id="p1833"></a>

**Medium** · Pinterest · 2025-01-01

#### Problem

The sequence `[0, 1, ..., N]` has been jumbled, and the only clue you have for its order is an array representing whether each number is larger or smaller than the last. Given this information, reconstruct an array that is consistent with it. For example, given `[None, +, +, -, +]`, you could return `[1, 2, 3, 0, 4]`.

### Problem #1834 <a id="p1834"></a>

**Hard** · Spotify · 2025-01-02

#### Problem

There are N couples sitting in a row of length 2 \* N. They are currently ordered randomly, but would like to rearrange themselves so that each couple's partners can sit side by side.

What is the minimum number of swaps necessary for this to happen?

### Problem #1835 <a id="p1835"></a>

**Easy** · Microsoft · 2025-01-03

#### Problem

Compute the running median of a sequence of numbers. That is, given a stream of numbers, print out the median of the list so far on each new element.

Recall that the median of an even-numbered list is the average of the two middle numbers.

For example, given the sequence `[2, 1, 5, 7, 2, 0, 5]`, your algorithm should print out:

```
2
1.5
2
3.5
2
2
2
```

### Problem #1836 <a id="p1836"></a>

**Medium** · Microsoft · 2025-01-04

#### Problem

Describe and give an example of each of the following types of polymorphism:

- Ad-hoc polymorphism
- Parametric polymorphism
- Subtype polymorphism

### Problem #1837 <a id="p1837"></a>

**Hard** · Amazon · 2025-01-05

#### Problem

The stable marriage problem is defined as follows:

Suppose you have N men and N women, and each person has ranked their prospective opposite-sex partners in order of preference.

For example, if N = 3, the input could be something like this:

```
guy_preferences = {
    'andrew': ['caroline', 'abigail', 'betty'],
    'bill': ['caroline', 'betty', 'abigail'],
    'chester': ['betty', 'caroline', 'abigail'],
}
```

```
gal_preferences = {
    'abigail': ['andrew', 'bill', 'chester'],
    'betty': ['bill', 'andrew', 'chester'],
    'caroline': ['bill', 'chester', 'andrew']
}
```

Write an algorithm that pairs the men and women together in such a way that no two people of opposite sex would both rather be with each other than with their current partners.

### Problem #1838 <a id="p1838"></a>

**Medium** · Google · 2025-01-06

#### Problem

Given an array of integers, return a new array where each element in the new array is the number of smaller elements to the right of that element in the original input array.

For example, given the array `[3, 4, 9, 6, 1]`, return `[1, 1, 2, 1, 0]`, since:

- There is 1 smaller element to the right of 3
- There is 1 smaller element to the right of 4
- There are 2 smaller elements to the right of 9
- There is 1 smaller element to the right of 6
- There are no smaller elements to the right of 1

### Problem #1839 <a id="p1839"></a>

**Medium** · Uber · 2025-01-07

#### Problem

Suppose an array sorted in ascending order is rotated at some pivot unknown to you beforehand. Find the minimum element in `O(log N)` time. You may assume the array does not contain duplicates.

For example, given `[5, 7, 10, 3, 4]`, return 3.

### Problem #1840 <a id="p1840"></a>

**Easy** · Stripe · 2025-01-08

#### Problem

Write a function to flatten a nested dictionary. Namespace the keys with a period.

For example, given the following dictionary:

```
{
    "key": 3,
    "foo": {
        "a": 5,
        "bar": {
            "baz": 8
        }
    }
}
```

it should become:

```
{
    "key": 3,
    "foo.a": 5,
    "foo.bar.baz": 8
}
```

You can assume keys do not contain dots in them, i.e. no clobbering will occur.

### Problem #1841 <a id="p1841"></a>

**Medium** · Quora · 2025-01-09

#### Problem

You are given a list of (website, user) pairs that represent users visiting websites. Come up with a program that identifies the top k pairs of websites with the greatest similarity.

For example, suppose k = 1, and the list of tuples is:

```
[('a', 1), ('a', 3), ('a', 5),
 ('b', 2), ('b', 6),
 ('c', 1), ('c', 2), ('c', 3), ('c', 4), ('c', 5)
 ('d', 4), ('d', 5), ('d', 6), ('d', 7),
 ('e', 1), ('e', 3), ('e': 5), ('e', 6)]
```

Then a reasonable similarity metric would most likely conclude that a and e are the most similar, so your program should return `[('a', 'e')]`.

### Problem #1842 <a id="p1842"></a>

**Medium** · MongoDB · 2025-01-10

#### Problem

Given a list of elements, find the majority element, which appears more than half the time (> `floor(len(lst) / 2.0)`).

You can assume that such element exists.

For example, given `[1, 2, 1, 1, 3, 4, 0]`, return 1.

### Problem #1843 <a id="p1843"></a>

**Hard** · Nvidia · 2025-01-11

#### Problem

Find the maximum of two numbers without using any if-else statements, branching, or direct comparisons.

### Problem #1844 <a id="p1844"></a>

**Medium** · Square · 2025-01-12

#### Problem

You are given a histogram consisting of rectangles of different heights. These heights are represented in an input list, such that `[1, 3, 2, 5]` corresponds to the following diagram:

```
      x
      x  
  x   x
  x x x
x x x x
```

Determine the area of the largest rectangle that can be formed only from the bars of the histogram. For the diagram above, for example, this would be six, representing the 2 x 3 area at the bottom right.

### Problem #1845 <a id="p1845"></a>

**Medium** · Google · 2025-01-13

#### Problem

Given an array of numbers and an index i, return the index of the nearest larger number of the number at index i, where distance is measured in array indices.

For example, given `[4, 1, 3, 5, 6]` and index 0, you should return 3.

If two distances to larger numbers are the equal, then return any one of them. If the array at i doesn't have a nearest larger integer, then return null.

Follow-up: If you can preprocess the array, can you do this in constant time?

### Problem #1846 <a id="p1846"></a>

**Medium** · Google · 2025-01-14

#### Problem

Given the root to a binary tree, implement `serialize(root)`, which serializes the tree into a string, and deserialize(s), which deserializes the string back into the tree.

For example, given the following Node class

```
class Node:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
```

The following test should pass:

```
node = Node('root', Node('left', Node('left.left')), Node('right'))
assert deserialize(serialize(node)).left.left.val == 'left.left'
```

### Problem #1847 <a id="p1847"></a>

**Hard** · Dropbox · 2025-01-15

#### Problem

A Boolean formula can be said to be satisfiable if there is a way to assign truth values to each variable such that the entire formula evaluates to true.

For example, suppose we have the following formula, where the symbol ¬ is used to denote negation:

(¬c OR b) AND (b OR c) AND (¬b OR c) AND (¬c OR ¬a)

One way to satisfy this formula would be to let a = False, b = True, and c = True.

This type of formula, with AND statements joining tuples containing exactly one OR, is known as 2-CNF.

Given a 2-CNF formula, find a way to assign truth values to satisfy it, or return False if this is impossible.

### Problem #1848 <a id="p1848"></a>

**Hard** · YouTube · 2025-01-16

#### Problem

Write a program that computes the length of the longest common subsequence of three given strings. For example, given `"epidemiologist"`, `"refrigeration"`, and `"supercalifragilisticexpialodocious"`, it should return 5, since the longest common subsequence is `"eieio"`.

### Problem #1849 <a id="p1849"></a>

**Easy** · Facebook · 2025-01-17

#### Problem

You have a large array with most of the elements as zero.

Use a more space-efficient data structure, SparseArray, that implements the same interface:

- `init(arr, size)`: initialize with the original large array and size.
- `set(i, val)`: updates index at i with val.
- `get(i)`: gets the value at index i.

### Problem #1850 <a id="p1850"></a>

**Easy** · Two Sigma · 2025-01-18

#### Problem

Using a function `rand5()` that returns an integer from 1 to 5 (inclusive) with uniform probability, implement a function `rand7()` that returns an integer from 1 to 7 (inclusive).

### Problem #1851 <a id="p1851"></a>

**Medium** · PayPal · 2025-01-19

#### Problem

Given a string and a number of lines k, print the string in zigzag form. In zigzag, characters are printed out diagonally from top left to bottom right until reaching the kth line, then back up to top right, and so on.

For example, given the sentence `"thisisazigzag"` and k = 4, you should print:

```
t     a     g
 h   s z   a
  i i   i z
   s     g
```

### Problem #1852 <a id="p1852"></a>

**Medium** · Facebook · 2025-01-20

#### Problem

Describe an algorithm to compute the longest increasing subsequence of an array of numbers in `O(n log n)` time.

### Problem #1853 <a id="p1853"></a>

**Easy** · Amazon · 2025-01-21

#### Problem

Implement a stack that has the following methods:

- `push(val)`, which pushes an element onto the stack
- `pop()`, which pops off and returns the topmost element of the stack. If there are no elements in the stack, then it should throw an error or return null.
- `max()`, which returns the maximum value in the stack currently. If there are no elements in the stack, then it should throw an error or return null.

Each method should run in constant time.

### Problem #1854 <a id="p1854"></a>

**Medium** · Square · 2025-01-22

#### Problem

Given a string and a set of characters, return the shortest substring containing all the characters in the set.

For example, given the string `"figehaeci"` and the set of characters `{a, e, i}`, you should return `"aeci"`.

If there is no substring containing all the characters in the set, return null.

### Problem #1855 <a id="p1855"></a>

**Hard** · Twitter · 2025-01-23

#### Problem

Given a binary tree, find the lowest common ancestor (LCA) of two given nodes in the tree. Assume that each node in the tree also has a pointer to its parent.

According to the definition of LCA on Wikipedia: “The lowest common ancestor is defined between two nodes v and w as the lowest node in T that has both v and w as descendants (where we allow a node to be a descendant of itself).”

### Problem #1856 <a id="p1856"></a>

**Easy** · Apple · 2025-01-24

#### Problem

A Collatz sequence in mathematics can be defined as follows. Starting with any positive integer:

- if n is even, the next number in the sequence is n / 2
- if n is odd, the next number in the sequence is 3n + 1

It is conjectured that every such sequence eventually reaches the number 1. Test this conjecture.

Bonus: What input n \<= 1000000 gives the longest sequence?

### Problem #1857 <a id="p1857"></a>

**Hard** · Jane Street · 2025-01-25

#### Problem

Given an arithmetic expression in Reverse Polish Notation, write a program to evaluate it.

The expression is given as a list of numbers and operands. For example: `[5, 3, '+']` should return 5 + 3 = 8.

For example, `[15, 7, 1, 1, '+', '-', '/', 3, '*', 2, 1, 1, '+', '+', '-']` should return 5, since it is equivalent to ((15 / (7 - (1 + 1))) \* 3) - (2 + (1 + 1)) = 5.

You can assume the given expression is always valid.

In [ ]:
# Progress. Run this cell once per session; after that the summary refreshes itself
# every time you run a cell (for example a new solution). It reads the saved notebook,
# so a solution counts once it is saved. A problem is solved when its section has code.
import json
import re

from IPython.core.getipython import get_ipython
from IPython.display import Markdown, display, update_display

NOTEBOOK = globals().get("__vsc_ipynb_file__", "dcp-solutions.ipynb")


def progress_markdown(path=NOTEBOOK):
    cells = json.load(open(path, encoding="utf-8"))["cells"]
    problems = []  # [number, difficulty, solved]
    for cell in cells:
        if cell.get("id") == "dcp-progress":
            continue  # this cell is not a solution
        text = "".join(cell["source"])
        heading = re.match(r"#{2,3} Problem #(\d+)", text) if cell["cell_type"] == "markdown" else None
        if heading:
            level = re.search(r"\*\*(Easy|Medium|Hard)\*\*", text)
            problems.append([int(heading[1]), level[1] if level else "", False])
        elif problems and cell["cell_type"] == "code" and text.strip():
            problems[-1][2] = True

    def link(p):
        return f"[{p[0]}](#p{p[0]})"

    solved = [p for p in problems if p[2]]
    next_up = [p for p in problems if not p[2]][:3]
    levels = " · ".join(
        f"{d} {sum(p[2] for p in problems if p[1] == d)}/{sum(p[1] == d for p in problems)}"
        for d in ("Easy", "Medium", "Hard")
    )
    return (
        "### Progress\n\n"
        f"**{len(solved)} / {len(problems)} solved** · {levels}\n\n"
        f"**Solved:** {' · '.join(link(p) for p in solved) or 'none yet'}  \n"
        f"**Next up:** {' · '.join(f'{link(p)} {p[1]}' for p in next_up) or 'all done'}"
    )


def _refresh_progress(*_):
    try:
        update_display(Markdown(progress_markdown()), display_id="dcp-progress")
    except Exception:
        pass  # never let the summary break the cell that just ran


display(Markdown(progress_markdown()), display_id="dcp-progress")

# Refresh after every cell run; re-running this cell replaces the old hook.
_ip = get_ipython()
if _ip is not None:
    _old_hook = globals().get("_progress_hook")
    if _old_hook is not None:
        try:
            _ip.events.unregister("post_run_cell", _old_hook)
        except ValueError:
            pass
    _progress_hook = _refresh_progress
    _ip.events.register("post_run_cell", _progress_hook)

### Progress

**4 / 1853 solved** · Easy 3/604 · Medium 0/778 · Hard 1/471

**Solved:** [1](#p1) · [2](#p2) · [1296](#p1296) · [1297](#p1297)  
**Next up:** [3](#p3) Medium · [4](#p4) Hard · [5](#p5) Medium